In [1]:
import matplotlib.pyplot as plt
tex_fonts = {
    # Use LaTeX to write all text
    "text.usetex": True,
    "font.family": "times",
    # Use 10pt font in plots, to match 10pt font in document
    "axes.labelsize": 10,
    "font.size": 10,
    # Make the legend/bel fonts a little smaller
    "legend.fontsize": 8,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
}

plt.rcParams.update(tex_fonts)
# 以下函数与 1.py 中的函数保持一致
# 以下函数与 1.py 中的函数保持一致

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
# [MODIFIED] Keep 2D plots as tracking-error L2 plots; add/standardize equilibrium lines.
"""
Unified controller comparison for Saint-Venant canal regulation.

Purpose:
    Compare seven controllers under one shared Saint-Venant test condition:
    TDPSAC, TDPSAC-FPC, DPSAC, DO-SAC, SAC-FPC, SAC, and analytical
    Backstepping. Plotting, data saving, 3D surfaces, terminal summaries, and
    the evidence table retain the previous unified comparison style.

Method mapping:
    1) TDPSAC:
       the latest partially frozen Hadamard DeepONet-SAC controller.
    2) TDPSAC-FPC:
       DeepONet actor/critic features plus an independent frozen prior controller.
    3) DPSAC:
       Hadamard DeepONet-SAC with all DeepONet parameters fine-tuned.
    4) DO-SAC:
       Hadamard DeepONet is used only as the SAC feature extractor.
    5) SAC-FPC:
       ordinary SAC actor/critic plus an independent frozen DeepONet prior.
    6) SAC:
       newly trained ordinary SAC without DeepONet, using the raw physical H/Q state.
    7) BS:
       analytical Backstepping controller.

All learned controllers receive the raw physical state
    [H_1,...,H_N,Q_1,...,Q_N],
with the same ODE equilibrium and the same physical initial condition.

Reference and test initial condition:
    - Qbar(x) is fixed at 11.93 m^3/s.
    - Hbar(L) is obtained from the downstream equilibrium gate relation.
    - Hbar(x) is obtained by solving the steady Saint-Venant ODE backward from
      x=L to x=0 with scipy.integrate.solve_ivp.
    - Every controller uses the same deterministic linear initial condition:

          H(x,0) = 3.75 + (4.65 - 3.75) x/L,
          Q(x,0) = 10.0.

      The initial condition is independent of the random seed. The seed is kept
      only for deterministic environment/model evaluation.

Extension:
    Additional learned controllers can be added later through
    LEARNED_CONTROLLER_SPECS without changing the common plotting and table
    functions.
"""

from __future__ import annotations

import argparse
import contextlib
import io
import os
import sys
import warnings
import zipfile
from pathlib import Path
from typing import Tuple, Dict, Optional

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from gymnasium import spaces
from scipy.integrate import solve_ivp
from stable_baselines3 import SAC
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from stable_baselines3.sac.policies import Actor, SACPolicy


# ============================================================
# Figure-only output mode
# ============================================================
# Suppress all script-side terminal print output. Exceptions/tracebacks are
# intentionally not suppressed.
warnings.filterwarnings("ignore")

def print(*args, **kwargs):
    return None


from cac_saint_venant_env_no_bs import (
    CACParams,
    CACSaintVenantPDE1D,
    source,
)

try:
    from pde_control_gym.src.rewards.base_reward import BaseReward
except Exception:
    class BaseReward:
        pass


# ============================================================
# Common physical / reward settings
# ============================================================

N = 102
dt = 5.0
control_sample_rate = 400.0

max_state_value = 1e6

# The learned controller emits a normalized action in [-1,1]. The environment
# maps it to this physical downstream-gate interval. These limits are identical
# to the corresponding raw-HQ training program.
min_control_value = 0.15
max_control_value = 0.40

# H and Q remain raw physical states in the policy observation. These scales are
# used only inside the reward calculation.
H_scale = 0.30
Q_scale = 3.00
U_scale = 0.10
obs_clip = 50.0

cfl = 0.45
check_cfl = True
well_balanced = True

# Collection-matched equilibrium.
Q_BAR_FIXED = 11.93
REFERENCE_MODE = "ode"
U_ref = 0.30

# First-order low-pass filter applied to the CONTROL COMMAND of every RL controller.
# The filtered control is passed to env.step(), so it is the actual closed-loop input.
# Backstepping is intentionally left unfiltered.
#
#   U_k^f = (1-alpha_f) U_{k-1}^f + alpha_f U_k^RL
#
# alpha_f = 1.0 -> no filtering; smaller values -> stronger smoothing.
RL_CONTROL_FILTER_ALPHA = 0.70

# Shared deterministic linear test initial condition.
TEST_H_LEFT = 3.75
TEST_H_RIGHT = 4.65
TEST_Q_VALUE = 10.8

# Reward used by the new training program:
# r_k = -alpha*tracking - beta*state_rate
#       -gamma*control - delta*delta_control.
reward_alpha = 1.0
reward_beta = 0.10
reward_gamma = 0.05
reward_delta = 0.10
reward_w_H = 1.0
reward_w_Q = 1.0
terminal_tracking_weight = 0
truncate_penalty = -200.0

# Keep the test horizon and plotting/table workflow consistent with the previous
# unified comparison program.
SAC_T = 80000.0
BS_T = SAC_T

BACKSTEPPING_GAIN = 1.0

learning_rate = 1e-4
deeponet_learning_rate = 1e-6

LATENT_DIM = 102
RAW_STATE_DIM = 2 * N
SAC_FEATURE_DIM = LATENT_DIM

# This file is only needed while the policy object is being constructed. The
# trained SB3 checkpoint subsequently restores all actor/critic/DeepONet weights.
DEEPNET_PRETRAINED_PATH = "modedlNet10000QqQqQ14444_hadamard.pt"

# ============================================================
# New Hadamard DeepONet controller and Hybrid SAC modules
# ============================================================

def preprocess_obs(obs: torch.Tensor) -> torch.Tensor:
    if not isinstance(obs, torch.Tensor):
        obs = torch.as_tensor(obs)
    return obs.float()


class BranchNet(nn.Module):
    def __init__(self, shape=N, output_dim=LATENT_DIM):
        super().__init__()
        self.shape = int(shape)
        self.output_dim = int(output_dim)

        self.conv1 = nn.Conv1d(
            in_channels=2,
            out_channels=16,
            kernel_size=5,
            stride=2,
        )
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv1d(
            in_channels=16,
            out_channels=32,
            kernel_size=5,
            stride=2,
        )

        with torch.no_grad():
            tmp = torch.zeros(1, 2, self.shape)
            tmp = self.relu(self.conv1(tmp))
            tmp = self.relu(self.conv2(tmp))
            flat_dim = tmp.reshape(1, -1).shape[1]

        self.fc1 = nn.Linear(flat_dim, self.output_dim)

    def forward(self, x):
        x = preprocess_obs(x)
        if x.dim() == 1:
            x = x.unsqueeze(0)

        if x.shape[1] != 2 * self.shape:
            raise ValueError(
                f"Branch input dimension is {x.shape[1]}, expected {2 * self.shape}."
            )

        x = x.reshape(x.shape[0], 2, self.shape)
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = torch.flatten(x, start_dim=1)
        x = self.relu(self.fc1(x))
        return x


class TrunkNet(nn.Module):
    def __init__(self, dim_x=1, width=LATENT_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim_x, 16),
            nn.ReLU(),
            nn.Linear(16, 32),
            nn.ReLU(),
            nn.Linear(32, width),
            nn.ReLU(),
        )

    def forward(self, x):
        return self.net(x.float())


class DeepONetModified(nn.Module):
    """
    Hadamard-fusion DeepONet controller.

        b(s) in R^p,
        t(x) in R^p,
        z(s,x) = b(s) * t(x) in R^p,
        U_prior = f_ctrl(z) in R.
    """

    def __init__(self, shape=N, latent_dim=LATENT_DIM):
        super().__init__()
        self.latent_dim = int(latent_dim)

        self.branch = BranchNet(shape, output_dim=self.latent_dim)
        self.trunk = TrunkNet(dim_x=1, width=self.latent_dim)

        # Hadamard fusion preserves latent_dim features.
        self.fc3 = nn.Linear(self.latent_dim, 32)
        self.relu = nn.ReLU()
        self.fc4 = nn.Linear(32, 16)
        self.fc5 = nn.Linear(16, 1)

    def extract_latent(self, obs, grid):
        b = self.branch(obs)       # [batch, latent_dim]
        t = self.trunk(grid)       # [n_query, latent_dim]

        if t.shape[0] == 1:
            t = t.expand(b.shape[0], -1)
        elif t.shape[0] != b.shape[0]:
            raise ValueError(
                f"Branch batch {b.shape[0]} and trunk batch {t.shape[0]} do not match."
            )

        z = b * t                  # Hadamard product
        return z

    def control_head(self, z):
        y = self.relu(self.fc3(z))
        y = self.relu(self.fc4(y))
        y = self.fc5(y)
        return y

    def forward(self, obs, grid, return_latent=False):
        z = self.extract_latent(obs, grid)
        u_prior_physical = self.control_head(z)

        if return_latent:
            return u_prior_physical, z

        return u_prior_physical


class HybridCACDeepONetFeatureExtractor(BaseFeaturesExtractor):
    """
    Return the Hadamard-fused DeepONet latent feature.

    Input observation and DeepONet input:
        [H_1,...,H_N,Q_1,...,Q_N]

    SAC feature:
        [z_1,...,z_p]
    """

    def __init__(
        self,
        observation_space: spaces.Box,
        features_dim: int = SAC_FEATURE_DIM,
        pretrained_path: str = DEEPNET_PRETRAINED_PATH,
        query_point: float = 1.0,
        freeze_deeponet: bool = False,
    ):
        if features_dim != SAC_FEATURE_DIM:
            raise ValueError(
                f"features_dim must be {SAC_FEATURE_DIM}, got {features_dim}."
            )

        super().__init__(observation_space, features_dim)

        self.query_point = float(query_point)
        self.net1 = DeepONetModified(N, latent_dim=LATENT_DIM)

        # Normalized action used by the hybrid SAC actor.
        self.u_deep_norm: Optional[torch.Tensor] = None

        # Physical prior gate retained for diagnostics.
        self.u_prior_physical: Optional[torch.Tensor] = None

        self.register_buffer(
            "grid",
            torch.tensor([[self.query_point]], dtype=torch.float32),
        )

        self._load_pretrained(pretrained_path)
        self._apply_freeze(freeze_all=freeze_deeponet)

    def _apply_freeze(self, freeze_all=False):
        if freeze_all:
            for param in self.net1.parameters():
                param.requires_grad = False
            print("All DeepONet parameters are frozen.")
            return

        # Keep the same transfer-learning structure used during training.
        for param in self.net1.branch.conv1.parameters():
            param.requires_grad = False
        for param in self.net1.branch.conv2.parameters():
            param.requires_grad = False

        for param in self.net1.branch.fc1.parameters():
            param.requires_grad = True

        for param in self.net1.trunk.parameters():
            param.requires_grad = False

        last_linear = None
        for module in reversed(list(self.net1.trunk.net)):
            if isinstance(module, nn.Linear):
                last_linear = module
                break

        if last_linear is not None:
            for param in last_linear.parameters():
                param.requires_grad = True

        for layer in [self.net1.fc3, self.net1.fc4, self.net1.fc5]:
            for param in layer.parameters():
                param.requires_grad = True

    @staticmethod
    def _clean_state_dict_keys(state_dict):
        cleaned = {}
        prefixes = ["module.", "model.", "net1.", "deeponet."]

        for key, value in state_dict.items():
            new_key = key
            changed = True
            while changed:
                changed = False
                for prefix in prefixes:
                    if new_key.startswith(prefix):
                        new_key = new_key[len(prefix):]
                        changed = True
            cleaned[new_key] = value

        return cleaned

    def _load_pretrained(self, pretrained_path: str):
        path = Path(pretrained_path)

        if not path.exists():
            raise FileNotFoundError(
                f"Pretrained Hadamard DeepONet checkpoint not found: {path}"
            )

        state_dict = torch.load(str(path), map_location="cpu")

        if isinstance(state_dict, dict) and "state_dict" in state_dict:
            state_dict = state_dict["state_dict"]

        if not isinstance(state_dict, dict):
            raise TypeError(
                f"Unsupported checkpoint type: {type(state_dict).__name__}"
            )

        cleaned = self._clean_state_dict_keys(state_dict)

        try:
            self.net1.load_state_dict(cleaned, strict=True)
        except RuntimeError as exc:
            raise RuntimeError(
                "The checkpoint is incompatible with the Hadamard-fusion "
                "DeepONet. Re-pretrain DeepONet with fc3=Linear(102,32) and "
                "z=b*t before running this test script.\n"
                f"Original error:\n{exc}"
            ) from exc

        print(f"Loaded pretrained Hadamard DeepONet: {path}")

    @staticmethod
    def _physical_gate_to_normalized_action(u_physical):
        scale = max_control_value - min_control_value
        if scale <= 0.0:
            raise ValueError(
                "max_control_value must be greater than min_control_value."
            )
        return 2.0 * (u_physical - min_control_value) / scale - 1.0

    def forward(self, observations: torch.Tensor) -> torch.Tensor:
        obs = preprocess_obs(observations)
        if obs.dim() == 1:
            obs = obs.unsqueeze(0)

        if obs.shape[1] != RAW_STATE_DIM:
            raise ValueError(
                f"Raw observation dimension is {obs.shape[1]}, "
                f"expected {RAW_STATE_DIM}."
            )

        u_prior_physical, z = self.net1(
            obs,
            self.grid.to(obs.device),
            return_latent=True,
        )

        # Clip the physical DeepONet prior before mapping it to [-1,1].
        u_prior_physical = torch.clamp(
            u_prior_physical,
            min_control_value,
            max_control_value,
        )
        self.u_prior_physical = u_prior_physical

        u_deep_norm = self._physical_gate_to_normalized_action(u_prior_physical)
        self.u_deep_norm = torch.clamp(u_deep_norm, -1.0, 1.0)

        # The SAC representation contains only the fused DeepONet feature z.
        return z.view(z.size(0), -1)


class HybridCACActor(Actor):
    """SAC actor whose normalized output is blended with DeepONet prior action."""

    def __init__(self, *args, alpha: float = 0.4, **kwargs):
        super().__init__(*args, **kwargs)
        init_alpha = float(np.clip(alpha, 0.0, 1.0))

        self.register_buffer(
            "alpha_tensor",
            torch.tensor(init_alpha, dtype=torch.float32),
        )

        self.last_deep_action = None
        self.last_rl_action = None
        self.last_hybrid_action = None

    @property
    def alpha(self) -> float:
        return float(self.alpha_tensor.detach().cpu().item())

    def set_alpha(self, alpha: float):
        alpha_value = float(np.clip(alpha, 0.0, 1.0))
        with torch.no_grad():
            self.alpha_tensor.fill_(alpha_value)

    def _deep_action(self, obs: torch.Tensor) -> torch.Tensor:
        u_deep = getattr(self.features_extractor, "u_deep_norm", None)

        if u_deep is None or u_deep.shape[0] != obs.shape[0]:
            _ = self.features_extractor(obs)
            u_deep = self.features_extractor.u_deep_norm

        return torch.clamp(u_deep, -1.0, 1.0)

    def _hybridize(self, rl_action: torch.Tensor, obs: torch.Tensor) -> torch.Tensor:
        u_deep = self._deep_action(obs)

        alpha = self.alpha_tensor.to(
            device=rl_action.device,
            dtype=rl_action.dtype,
        )

        hybrid = (1.0 - alpha) * u_deep + alpha * rl_action
        hybrid = torch.clamp(hybrid, -1.0, 1.0)

        self.last_deep_action = u_deep.detach()
        self.last_rl_action = rl_action.detach()
        self.last_hybrid_action = hybrid.detach()

        return hybrid

    def forward(self, obs: torch.Tensor, deterministic: bool = False):
        rl_action = super().forward(obs, deterministic=deterministic)
        return self._hybridize(rl_action, obs)

    def action_log_prob(self, obs: torch.Tensor):
        # The log probability is that of the SAC stochastic component.
        # The deterministic affine hybrid mapping is not included in log_prob.
        rl_action, log_prob = super().action_log_prob(obs)
        hybrid_action = self._hybridize(rl_action, obs)
        return hybrid_action, log_prob


class HybridCACCritic(nn.Module):
    """Twin-Q critic using the Hadamard-fused DeepONet feature."""

    def __init__(
        self,
        observation_space: spaces.Box,
        action_space: spaces.Box,
        hidden_dim: int = 256,
        features_dim: int = SAC_FEATURE_DIM,
        pretrained_path: str = DEEPNET_PRETRAINED_PATH,
    ):
        super().__init__()

        self.features_extractor = HybridCACDeepONetFeatureExtractor(
            observation_space,
            features_dim=features_dim,
            pretrained_path=pretrained_path,
            query_point=1.0,
            freeze_deeponet=False,
        )

        action_dim = int(np.prod(action_space.shape))
        input_dim = features_dim + action_dim

        self.q1 = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )
        self.q2 = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )

    def set_training_mode(self, mode: bool):
        self.train(mode)

    def forward(self, obs: torch.Tensor, actions: torch.Tensor):
        features = self.features_extractor(obs)
        q_input = torch.cat([features, actions], dim=1)
        return self.q1(q_input), self.q2(q_input)


class HybridCACSACPolicy(SACPolicy):
    """SAC policy with Hadamard DeepONet representation/control transfer."""

    def __init__(
        self,
        *args,
        hybrid_alpha_initial: float = 0.4,
        deeponet_pretrained_path: str = DEEPNET_PRETRAINED_PATH,
        **kwargs,
    ):
        self.hybrid_alpha_initial = float(hybrid_alpha_initial)
        self.deeponet_pretrained_path = deeponet_pretrained_path

        kwargs.setdefault(
            "features_extractor_class",
            HybridCACDeepONetFeatureExtractor,
        )
        kwargs.setdefault(
            "features_extractor_kwargs",
            dict(
                features_dim=SAC_FEATURE_DIM,
                pretrained_path=deeponet_pretrained_path,
                query_point=1.0,
                freeze_deeponet=False,
            ),
        )

        super().__init__(*args, **kwargs)

    def make_actor(self, features_extractor=None):
        kwargs = self._update_features_extractor(
            self.actor_kwargs,
            features_extractor,
        )
        return HybridCACActor(
            **kwargs,
            alpha=self.hybrid_alpha_initial,
        ).to(self.device)

    def make_critic(self, features_extractor=None):
        return HybridCACCritic(
            self.observation_space,
            self.action_space,
            hidden_dim=256,
            features_dim=SAC_FEATURE_DIM,
            pretrained_path=self.deeponet_pretrained_path,
        ).to(self.device)

    def set_alpha(self, alpha: float):
        self.actor.set_alpha(alpha)


# ============================================================
# 9. Optimizer helper: separate DeepONet learning rate
# ============================================================

def _make_param_groups_with_deeponet_lr(module: nn.Module):
    deeponet_params = []
    other_params = []

    for name, param in module.named_parameters():
        if not param.requires_grad:
            continue

        if "features_extractor.net1" in name or name.startswith("net1."):
            deeponet_params.append(param)
        else:
            other_params.append(param)

    groups = []

    if other_params:
        groups.append(
            {
                "params": other_params,
                "lr": learning_rate,
                "is_deeponet_group": False,
            }
        )

    if deeponet_params:
        groups.append(
            {
                "params": deeponet_params,
                "lr": deeponet_learning_rate,
                "is_deeponet_group": True,
            }
        )

    return groups


def set_deeponet_small_lr_optimizers(model):
    actor_groups = _make_param_groups_with_deeponet_lr(model.actor)
    critic_groups = _make_param_groups_with_deeponet_lr(model.critic)

    model.actor.optimizer = torch.optim.Adam(actor_groups, eps=1e-5)
    model.critic.optimizer = torch.optim.Adam(critic_groups, eps=1e-5)

    n_actor_deep = sum(
        p.numel()
        for group in actor_groups
        if group.get("is_deeponet_group", False)
        for p in group["params"]
    )
    n_actor_other = sum(
        p.numel()
        for group in actor_groups
        if not group.get("is_deeponet_group", False)
        for p in group["params"]
    )
    n_critic_deep = sum(
        p.numel()
        for group in critic_groups
        if group.get("is_deeponet_group", False)
        for p in group["params"]
    )
    n_critic_other = sum(
        p.numel()
        for group in critic_groups
        if not group.get("is_deeponet_group", False)
        for p in group["params"]
    )

    print("Custom optimizers with separate DeepONet learning rate are set.")
    print(f"Actor params: DeepONet={n_actor_deep}, other={n_actor_other}")
    print(f"Critic params: DeepONet={n_critic_deep}, other={n_critic_other}")
    print(
        f"Base SAC learning_rate={learning_rate}, "
        f"DeepONet learning_rate={deeponet_learning_rate}"
    )


class HybridSAC(SAC):
    """Preserve the smaller learning rate of DeepONet parameter groups."""

    def _update_learning_rate(self, optimizers):
        super()._update_learning_rate(optimizers)

        if not isinstance(optimizers, (list, tuple)):
            optimizers = [optimizers]

        current_lr = self.lr_schedule(self._current_progress_remaining)

        for optimizer in optimizers:
            for group in optimizer.param_groups:
                if group.get("is_deeponet_group", False):
                    group["lr"] = deeponet_learning_rate
                elif "is_deeponet_group" in group:
                    group["lr"] = current_lr




class EvalHybridSAC(HybridSAC):
    """Evaluation loader compatible with DeepONet-specific optimizer groups."""

    def _setup_model(self):
        super()._setup_model()
        set_deeponet_small_lr_optimizers(self)

    def set_parameters(self, load_path_or_dict, exact_match: bool = True, device="auto"):
        try:
            return super().set_parameters(
                load_path_or_dict,
                exact_match=exact_match,
                device=device,
            )
        except ValueError as exc:
            message = str(exc)
            optimizer_error = (
                "parameter group" in message
                or "Names of parameters" in message
                or "optimizer" in message.lower()
            )

            if isinstance(load_path_or_dict, dict) and optimizer_error:
                print(
                    "Evaluation load: incompatible optimizer states are skipped; "
                    "policy, actor, critic, DeepONet, and alpha weights are retained."
                )
                filtered = {
                    name: value
                    for name, value in load_path_or_dict.items()
                    if "optimizer" not in name.lower()
                }
                return super().set_parameters(
                    filtered,
                    exact_match=False,
                    device=device,
                )
            raise


# ============================================================
# Physical setup: ODE equilibrium + shared linear test initial condition
# ============================================================

def make_cac_table_params(
    q_bar: float = Q_BAR_FIXED,
    h_bar_0: float = 3.80,
    h_bar_L: float = 4.70,
) -> CACParams:
    """Create CAC parameters consistent with the new training program."""
    return CACParams(
        L=11239.0,
        B=40.0,
        g=9.81,
        I=7.92e-5,
        K=33.0,
        mu0=0.4,
        muL=0.4,
        z_up=5.00,
        U0_bar=0.1548,
        U_L_bar=U_ref,
        h_s=4.094,
        Q_bar=float(q_bar),
        H_bar_0=float(h_bar_0),
        H_bar_L=float(h_bar_L),
        H_min=0.20,
        max_state_value=1e10,
    )


def make_grid(p: CACParams, n_grid: int) -> Tuple[np.ndarray, float]:
    X = np.linspace(0.0, p.L, n_grid)
    dx = p.L / (n_grid - 1)
    return X, dx


def downstream_equilibrium_depth_from_q(Q: float, p: CACParams) -> float:
    """Compute Hbar(L) from the downstream equilibrium gate relation."""
    Q = float(Q)
    if not np.isfinite(Q) or Q <= 0.0:
        raise ValueError("The equilibrium discharge must be positive and finite.")

    kinetic_head = (
        Q**2 / (2.0 * p.g * p.B**2 * p.muL**2)
    ) ** (1.0 / 3.0)
    return float(p.h_s + p.U_L_bar + kinetic_head)


def integrate_steady_fixed_q_ode(
    X: np.ndarray,
    p: CACParams,
    Q_candidate: float,
) -> Tuple[np.ndarray, np.ndarray]:
    """Integrate the steady Saint-Venant ODE backward from x=L to x=0."""
    X = np.asarray(X, dtype=float)
    if X.ndim != 1 or X.size < 2:
        raise ValueError("X must be a one-dimensional grid with at least two points.")
    if not np.all(np.diff(X) > 0.0):
        raise ValueError("X must be strictly increasing.")

    Q_candidate = float(Q_candidate)
    if not np.isfinite(Q_candidate) or Q_candidate <= 0.0:
        raise ValueError("Q_candidate must be positive and finite.")

    Hbar_L = downstream_equilibrium_depth_from_q(Q_candidate, p)

    def rhs(_x: float, state: np.ndarray) -> np.ndarray:
        H = float(state[0])
        Q = float(state[1])

        if not np.isfinite(H) or not np.isfinite(Q):
            raise FloatingPointError("Non-finite state in steady reference ODE.")
        if H <= float(p.H_min):
            raise FloatingPointError("Steady reference ODE violates minimum depth.")

        area = p.B * H
        hydraulic_radius = p.B * H / (p.B + 2.0 * H)
        friction_slope = Q * abs(Q) / (
            p.K**2 * area**2 * hydraulic_radius ** (4.0 / 3.0)
        )

        denominator = p.g * p.B * H - Q**2 / (p.B * H**2)
        if abs(denominator) <= 1e-10:
            raise FloatingPointError(
                "Steady ODE approaches the critical-flow singularity."
            )

        dHdx = p.g * p.B * H * (p.I - friction_slope) / denominator
        if not np.isfinite(dHdx):
            raise FloatingPointError("Non-finite dHbar/dx in steady ODE.")

        return np.asarray([dHdx, 0.0], dtype=float)

    sol = solve_ivp(
        fun=rhs,
        t_span=(float(X[-1]), float(X[0])),
        y0=np.asarray([Hbar_L, Q_candidate], dtype=float),
        t_eval=X[::-1],
        rtol=1e-9,
        atol=1e-11,
        max_step=p.L / 1000.0,
    )

    if not sol.success:
        raise RuntimeError(f"Failed to solve steady ODE: {sol.message}")
    if sol.y.shape[1] != X.size:
        raise RuntimeError("Steady ODE did not return all requested grid points.")

    Hbar = np.asarray(sol.y[0][::-1], dtype=float)
    Qbar = np.asarray(sol.y[1][::-1], dtype=float)

    if not np.all(np.isfinite(Hbar)) or not np.all(np.isfinite(Qbar)):
        raise FloatingPointError("Steady reference contains NaN or Inf.")
    if float(np.min(Hbar)) <= float(p.H_min):
        raise RuntimeError("Steady Hbar profile violates the minimum depth.")

    return Hbar, Qbar


def solve_steady_reference_ode(
    X: np.ndarray,
    p: CACParams,
    q_bar_fixed: float = Q_BAR_FIXED,
) -> Tuple[np.ndarray, np.ndarray, float]:
    """Solve Hbar(x) for the prescribed constant Qbar=11.93 m^3/s."""
    q_bar_fixed = float(q_bar_fixed)
    Hbar, _ = integrate_steady_fixed_q_ode(X, p, q_bar_fixed)
    Qbar = np.full_like(X, q_bar_fixed, dtype=float)
    return Hbar, Qbar, q_bar_fixed


def make_reference_profiles(
    X: np.ndarray,
    p: CACParams,
    reference: str = REFERENCE_MODE,
    q_bar_fixed: float = Q_BAR_FIXED,
) -> Tuple[np.ndarray, np.ndarray, float]:
    if reference in {"ode", "strict"}:
        return solve_steady_reference_ode(X, p, q_bar_fixed=q_bar_fixed)
    if reference == "linear":
        Hbar = p.H_bar_0 + (p.H_bar_L - p.H_bar_0) * (X / p.L)
        Qbar = np.full_like(X, q_bar_fixed, dtype=float)
        return Hbar, Qbar, float(q_bar_fixed)
    raise ValueError("reference must be 'ode', 'strict', or 'linear'.")


def build_collection_matched_reference():
    """Return one p, X, dx, Hbar, Qbar tuple shared by every controller."""
    nominal_p = make_cac_table_params(q_bar=Q_BAR_FIXED)
    X, dx = make_grid(nominal_p, N)
    Hbar, Qbar, q_equilibrium = make_reference_profiles(
        X,
        nominal_p,
        reference=REFERENCE_MODE,
        q_bar_fixed=Q_BAR_FIXED,
    )

    p = make_cac_table_params(
        q_bar=q_equilibrium,
        h_bar_0=float(Hbar[0]),
        h_bar_L=float(Hbar[-1]),
    )
    return p, X, dx, Hbar, Qbar


def generate_test_initial_condition(
    X: np.ndarray,
    Hbar: np.ndarray,
    Qbar: np.ndarray,
    H_left: float = TEST_H_LEFT,
    H_right: float = TEST_H_RIGHT,
    Q_value: float = TEST_Q_VALUE,
) -> Tuple[np.ndarray, np.ndarray, Dict[str, float]]:
    """Generate the shared deterministic linear test initial condition.

    H(x,0) = H_left + (H_right-H_left) x/L,
    Q(x,0) = Q_value.

    Hbar and Qbar are accepted so the function can also report the exact
    initial deviations from the ODE equilibrium used in the test.
    """
    X = np.asarray(X, dtype=float)
    Hbar = np.asarray(Hbar, dtype=float)
    Qbar = np.asarray(Qbar, dtype=float)

    if X.ndim != 1 or Hbar.shape != X.shape or Qbar.shape != X.shape:
        raise ValueError(
            "X, Hbar, and Qbar must be one-dimensional arrays of equal length."
        )
    L = float(X[-1])
    if not np.isfinite(L) or L <= 0.0:
        raise ValueError("The spatial-domain length must be positive and finite.")

    H_left = float(H_left)
    H_right = float(H_right)
    Q_value = float(Q_value)
    if not all(np.isfinite(v) for v in (H_left, H_right, Q_value)):
        raise ValueError("H_left, H_right, and Q_value must be finite.")

    H0 = H_left + (H_right - H_left) * (X / L)
    Q0 = np.full_like(X, Q_value, dtype=float)

    if float(np.min(H0)) <= 0.20:
        raise ValueError("The linear initial H profile violates H_min=0.20 m.")
    if float(np.min(Q0)) <= 0.0:
        raise ValueError("The initial discharge must be positive.")

    details = {
        "H_left": H_left,
        "H_right": H_right,
        "Q_value": Q_value,
        "H_realized_max_deviation": float(np.max(np.abs(H0 - Hbar))),
        "Q_realized_max_deviation": float(np.max(np.abs(Q0 - Qbar))),
    }
    return H0, Q0, details


def make_fixed_initial_condition(
    X: np.ndarray,
    Hbar: np.ndarray,
    Qbar: np.ndarray,
    H_left: float = TEST_H_LEFT,
    H_right: float = TEST_H_RIGHT,
    Q_value: float = TEST_Q_VALUE,
) -> Tuple[np.ndarray, np.ndarray]:
    """Return the shared deterministic linear test initial state."""
    H0, Q0, _ = generate_test_initial_condition(
        X,
        Hbar,
        Qbar,
        H_left=H_left,
        H_right=H_right,
        Q_value=Q_value,
    )
    return H0, Q0


def make_test_initial_condition_func(
    H_left: float = TEST_H_LEFT,
    H_right: float = TEST_H_RIGHT,
    Q_value: float = TEST_Q_VALUE,
):
    """Return the same deterministic linear reset callback for all controllers."""
    def reset_func(X, Hbar, Qbar, rng):
        del rng
        H0, Q0, _ = generate_test_initial_condition(
            X,
            Hbar,
            Qbar,
            H_left=H_left,
            H_right=H_right,
            Q_value=Q_value,
        )
        return H0, Q0

    return reset_func

# ============================================================
# Hybrid action diagnostics
# ============================================================

def normalized_action_to_physical_ul(a):
    """Map normalized action a in [-1, 1] to physical downstream gate U_L."""
    a = float(np.clip(float(a), -1.0, 1.0))
    return float(min_control_value + 0.5 * (a + 1.0) * (max_control_value - min_control_value))


def physical_ul_to_normalized_action(u_physical):
    """Map physical downstream-gate command U_L back to normalized action [-1,1]."""
    scale = float(max_control_value - min_control_value)
    if scale <= 0.0:
        raise ValueError("max_control_value must be greater than min_control_value.")
    u_physical = float(
        np.clip(float(u_physical), min_control_value, max_control_value)
    )
    action = 2.0 * (u_physical - min_control_value) / scale - 1.0
    return float(np.clip(action, -1.0, 1.0))


def lowpass_filter_rl_control(
    action_raw,
    previous_u_filtered,
    alpha=RL_CONTROL_FILTER_ALPHA,
):
    """Apply the same first-order low-pass filter to one RL gate command.

    The RL action is converted to physical U_L, filtered in physical units,
    converted back to normalized action, and the filtered action is then sent
    to env.step().
    """
    alpha = float(alpha)
    if not (0.0 < alpha <= 1.0):
        raise ValueError("RL_CONTROL_FILTER_ALPHA must satisfy 0 < alpha <= 1.")

    a_raw = float(np.asarray(action_raw, dtype=float).reshape(-1)[0])
    a_raw = float(np.clip(a_raw, -1.0, 1.0))
    u_command_raw = normalized_action_to_physical_ul(a_raw)

    previous_u_filtered = float(
        np.clip(float(previous_u_filtered), min_control_value, max_control_value)
    )
    u_filtered = (
        (1.0 - alpha) * previous_u_filtered
        + alpha * u_command_raw
    )
    u_filtered = float(
        np.clip(u_filtered, min_control_value, max_control_value)
    )

    a_filtered = physical_ul_to_normalized_action(u_filtered)
    action_filtered = np.asarray([a_filtered], dtype=np.float32)
    return action_filtered, u_command_raw, u_filtered


def get_hybrid_action_components(model, obs, action_hybrid):
    """
    Extract DeepONet, RL, and hybrid normalized actions for the loaded hybrid SAC policy.

    The actor actually outputs
        a_hybrid = (1-alpha) * a_deep + alpha * a_RL.
    During testing, model.predict(obs) returns a_hybrid. We compute a_deep by calling
    the actor feature extractor and infer a_RL from the affine mixing formula.
    """
    actor = model.policy.actor
    device = model.device

    obs_tensor = torch.as_tensor(obs, dtype=torch.float32, device=device)
    if obs_tensor.dim() == 1:
        obs_tensor = obs_tensor.unsqueeze(0)

    with torch.no_grad():
        # This call updates actor.features_extractor.u_deep_norm.
        _ = actor.features_extractor(obs_tensor)
        u_deep = getattr(actor.features_extractor, "u_deep_norm", None)

    if u_deep is None:
        a_deep = np.nan
    else:
        a_deep = float(u_deep.detach().cpu().numpy().reshape(-1)[0])
        a_deep = float(np.clip(a_deep, -1.0, 1.0))

    a_hybrid = float(np.asarray(action_hybrid).reshape(-1)[0])
    a_hybrid = float(np.clip(a_hybrid, -1.0, 1.0))

    alpha = float(getattr(actor, "alpha", np.nan))

    if np.isfinite(alpha) and alpha > 1e-8 and np.isfinite(a_deep):
        a_rl = (a_hybrid - (1.0 - alpha) * a_deep) / alpha
        a_rl = float(np.clip(a_rl, -1.0, 1.0))
    else:
        a_rl = np.nan

    return {
        "alpha": alpha,
        "a_deep": a_deep,
        "a_rl": a_rl,
        "a_hybrid": a_hybrid,
        "U_deep": normalized_action_to_physical_ul(a_deep) if np.isfinite(a_deep) else np.nan,
        "U_rl": normalized_action_to_physical_ul(a_rl) if np.isfinite(a_rl) else np.nan,
        "U_hybrid": normalized_action_to_physical_ul(a_hybrid),
    }


# ============================================================
# Reward class
# ============================================================

class CACTrackingSmoothReward(BaseReward):
    """
    Discrete reward at the control instant t_k:

        r_k = -alpha ||y_tilde_k-y_tilde_star||^2
              -beta  ||(y_tilde_k-y_tilde_{k-1})/Delta_t||^2
              -gamma ||U_tilde_k||^2
              -delta ||U_tilde_k-U_tilde_{k-1}||^2.

    Here:
        y_tilde_star = 0,
        H_tilde_error = (H-H_eq)/H_scale,
        Q_tilde_error = (Q-Q_eq)/Q_scale,
        U_tilde = (U-U_eq)/U_scale.

    Spatial means are used instead of unnormalized sums so that the reward scale
    does not grow with the number of grid points N.
    """

    def __init__(
        self,
        H_eq,
        Q_eq,
        delta_t=control_sample_rate,
        H_scale=0.10,
        Q_scale=3.00,
        U_scale=0.10,
        alpha=1.0,
        beta=0.10,
        gamma=0.05,
        delta=0.10,
        w_H=1.0,
        w_Q=1.0,
        U_ref=0.3,
        terminal_tracking_weight=10.0,
        truncate_penalty=-200.0,
    ):
        self.H_eq = np.asarray(H_eq, dtype=float)
        self.Q_eq = np.asarray(Q_eq, dtype=float)
        self.N = len(self.H_eq)

        self.delta_t = float(delta_t)
        self.H_scale = float(H_scale)
        self.Q_scale = float(Q_scale)
        self.U_scale = float(U_scale)

        self.alpha = float(alpha)
        self.beta = float(beta)
        self.gamma = float(gamma)
        self.delta = float(delta)
        self.w_H = float(w_H)
        self.w_Q = float(w_Q)

        self.U_ref = float(U_ref)
        self.terminal_tracking_weight = float(terminal_tracking_weight)
        self.truncate_penalty = float(truncate_penalty)

        # Previous values at the preceding reward/control call.
        self.prev_H: Optional[np.ndarray] = None
        self.prev_Q: Optional[np.ndarray] = None
        self.prev_action: Optional[float] = None

        # Diagnostics.
        self.last_terms = {
            "tracking_cost": 0.0,
            "state_rate_cost": 0.0,
            "control_cost": 0.0,
            "delta_control_cost": 0.0,
        }

    def _reset_memory(self):
        self.prev_H = None
        self.prev_Q = None
        self.prev_action = None

    def _split_state(self, state_vec):
        state_vec = np.asarray(state_vec, dtype=float).reshape(-1)
        if state_vec.size < 2 * self.N:
            raise ValueError(
                f"PDE state length is {state_vec.size}, expected at least {2 * self.N}."
            )
        H = state_vec[:self.N]
        Q = state_vec[self.N:2 * self.N]
        return H, Q

    def _action_value(self, action):
        # The original environment passes the physical downstream gate to reward.
        if action is None:
            return self.U_ref
        value = float(np.asarray(action).reshape(-1)[0])
        return float(
            np.nan_to_num(
                value,
                nan=self.U_ref,
                posinf=self.U_ref,
                neginf=self.U_ref,
            )
        )

    def reward(
        self,
        uVec=None,
        time_index=None,
        terminate=None,
        truncate=None,
        action=None,
        control_sample_rate=None,
    ):
        if uVec is None or time_index is None:
            return 0.0

        # A new episode normally starts with a small time index.
        if int(time_index) <= 1:
            self._reset_memory()

        if truncate:
            self._reset_memory()
            return float(self.truncate_penalty)

        Hk, Qk = self._split_state(uVec[time_index])

        if (
            not np.all(np.isfinite(Hk))
            or not np.all(np.isfinite(Qk))
            or np.any(Hk <= 0.0)
        ):
            self._reset_memory()
            return float(self.truncate_penalty)

        # ----------------------------------------------------
        # 1) Tracking term
        # ----------------------------------------------------
        H_error_tilde = (Hk - self.H_eq) / self.H_scale
        Q_error_tilde = (Qk - self.Q_eq) / self.Q_scale

        tracking_cost = (
            self.w_H * np.mean(H_error_tilde ** 2)
            + self.w_Q * np.mean(Q_error_tilde ** 2)
        )

        # ----------------------------------------------------
        # 2) State-rate term
        # ----------------------------------------------------
        if self.prev_H is None or self.prev_Q is None:
            state_rate_cost = 0.0
        else:
            delta_t_k = (
                float(control_sample_rate)
                if control_sample_rate is not None
                else self.delta_t
            )
            delta_t_k = max(delta_t_k, 1e-12)

            dH_tilde = (Hk - self.prev_H) / (delta_t_k * self.H_scale)
            dQ_tilde = (Qk - self.prev_Q) / (delta_t_k * self.Q_scale)

            state_rate_cost = (
                self.w_H * np.mean(dH_tilde ** 2)
                + self.w_Q * np.mean(dQ_tilde ** 2)
            )

        # ----------------------------------------------------
        # 3) Control-magnitude term
        # ----------------------------------------------------
        Uk = self._action_value(action)
        U_tilde = (Uk - self.U_ref) / self.U_scale
        control_cost = U_tilde ** 2

        # ----------------------------------------------------
        # 4) Control-increment term
        # ----------------------------------------------------
        if self.prev_action is None:
            delta_control_cost = 0.0
        else:
            U_prev_tilde = (self.prev_action - self.U_ref) / self.U_scale
            delta_control_cost = (U_tilde - U_prev_tilde) ** 2

        reward_value = (
            -self.alpha * tracking_cost
            -self.beta * state_rate_cost
            -self.gamma * control_cost
            -self.delta * delta_control_cost
        )

        if terminate:
            reward_value -= self.terminal_tracking_weight * tracking_cost

        self.prev_H = Hk.copy()
        self.prev_Q = Qk.copy()
        self.prev_action = Uk

        self.last_terms = {
            "tracking_cost": float(tracking_cost),
            "state_rate_cost": float(state_rate_cost),
            "control_cost": float(control_cost),
            "delta_control_cost": float(delta_control_cost),
        }

        return float(reward_value)




# ============================================================
# Environments
# ============================================================

def _make_reward(Hbar, Qbar):
    return CACTrackingSmoothReward(
        H_eq=Hbar,
        Q_eq=Qbar,
        delta_t=control_sample_rate,
        H_scale=H_scale,
        Q_scale=Q_scale,
        U_scale=U_scale,
        alpha=reward_alpha,
        beta=reward_beta,
        gamma=reward_gamma,
        delta=reward_delta,
        w_H=reward_w_H,
        w_Q=reward_w_Q,
        U_ref=U_ref,
        terminal_tracking_weight=terminal_tracking_weight,
        truncate_penalty=truncate_penalty,
    )


def make_env_sac(
    H_left=TEST_H_LEFT,
    H_right=TEST_H_RIGHT,
    Q_value=TEST_Q_VALUE,
    seed=123,
    dH_source=None,
):
    """Create the exact evaluation environment for the raw-HQ learned model.

    The policy receives raw physical [H,Q]. Every controller is reset to the
    same deterministic linear H profile and constant Q profile.
    """
    p, X, dx, Hbar, Qbar = build_collection_matched_reference()
    H_init, Q_init = make_fixed_initial_condition(
        X,
        Hbar,
        Qbar,
        H_left=H_left,
        H_right=H_right,
        Q_value=Q_value,
    )

    env_kwargs = dict(
        params=p,
        X_grid=X,
        Hbar=Hbar,
        Qbar=Qbar,
        H_init=H_init,
        Q_init=Q_init,
        sensing_noise_func=lambda observation: observation,
        reset_init_condition_func=make_test_initial_condition_func(
            H_left=H_left,
            H_right=H_right,
            Q_value=Q_value,
        ),
        sensing_loc="full",
        control_type="Gate",
        limit_pde_state_size=True,
        max_state_value=max_state_value,
        min_control_value=min_control_value,
        max_control_value=max_control_value,
        control_sample_rate=control_sample_rate,
        reward_class=_make_reward(Hbar, Qbar),
        normalize=True,
        observation_mode="raw",
        H_scale=H_scale,
        Q_scale=Q_scale,
        obs_clip=obs_clip,
        T=SAC_T,
        dt=dt,
        cfl=cfl,
        check_cfl=check_cfl,
        well_balanced=well_balanced,
    )

    if dH_source is not None:
        env_kwargs["dH_source"] = float(dH_source)

    env = CACSaintVenantPDE1D(**env_kwargs)

    env.reference_X = X
    env.reference_H = Hbar
    env.reference_Q = Qbar
    env.reference_dx = dx

    return env, X, Hbar, Qbar, p


def make_env_sac_error_state(
    H_left=TEST_H_LEFT,
    H_right=TEST_H_RIGHT,
    Q_value=TEST_Q_VALUE,
    seed=123,
    dH_source=None,
):
    """Create the evaluation environment for the old error-state SAC model.

    The physical test problem is unchanged: the ODE equilibrium, linear H0,
    constant Q0, PDE parameters, action range, time step, and test horizon are
    identical to the raw-HQ controllers. Only the policy observation changes to

        [(H-Hbar)/H_scale, (Q-Qbar)/Q_scale].

    This matches the old SAC checkpoint trained with
    observation_mode="normalized_error".
    """
    p, X, dx, Hbar, Qbar = build_collection_matched_reference()
    H_init, Q_init = make_fixed_initial_condition(
        X,
        Hbar,
        Qbar,
        H_left=H_left,
        H_right=H_right,
        Q_value=Q_value,
    )

    env_kwargs = dict(
        params=p,
        X_grid=X,
        Hbar=Hbar,
        Qbar=Qbar,
        H_init=H_init,
        Q_init=Q_init,
        sensing_noise_func=lambda observation: observation,
        reset_init_condition_func=make_test_initial_condition_func(
            H_left=H_left,
            H_right=H_right,
            Q_value=Q_value,
        ),
        sensing_loc="full",
        control_type="Gate",
        limit_pde_state_size=True,
        max_state_value=max_state_value,
        min_control_value=min_control_value,
        max_control_value=max_control_value,
        control_sample_rate=control_sample_rate,
        reward_class=_make_reward(Hbar, Qbar),
        normalize=True,
        observation_mode="normalized_error",
        H_scale=H_scale,
        Q_scale=Q_scale,
        obs_clip=obs_clip,
        T=SAC_T,
        dt=dt,
        cfl=cfl,
        check_cfl=check_cfl,
        well_balanced=well_balanced,
    )

    if dH_source is not None:
        env_kwargs["dH_source"] = float(dH_source)

    env = CACSaintVenantPDE1D(**env_kwargs)
    env.reference_X = X
    env.reference_H = Hbar
    env.reference_Q = Qbar
    env.reference_dx = dx
    env.policy_observation_mode = "normalized_error"

    return env, X, Hbar, Qbar, p


def make_env_backstepping(
    H_left=TEST_H_LEFT,
    H_right=TEST_H_RIGHT,
    Q_value=TEST_Q_VALUE,
    seed=123,
    dH_source=None,
):
    """Create Backstepping with the same ODE reference and linear initial state."""
    p, X, dx, Hbar, Qbar = build_collection_matched_reference()
    H_init, Q_init = make_fixed_initial_condition(
        X,
        Hbar,
        Qbar,
        H_left=H_left,
        H_right=H_right,
        Q_value=Q_value,
    )

    env_kwargs = dict(
        params=p,
        X_grid=X,
        Hbar=Hbar,
        Qbar=Qbar,
        H_init=H_init,
        Q_init=Q_init,
        sensing_noise_func=lambda state: state,
        reset_init_condition_func=make_test_initial_condition_func(
            H_left=H_left,
            H_right=H_right,
            Q_value=Q_value,
        ),
        sensing_loc="full",
        control_type="Gate",
        limit_pde_state_size=True,
        max_state_value=max_state_value,
        min_control_value=min_control_value,
        max_control_value=max_control_value,
        control_sample_rate=control_sample_rate,
        reward_class=_make_reward(Hbar, Qbar),
        normalize=False,
        observation_mode="raw",
        H_scale=H_scale,
        Q_scale=Q_scale,
        obs_clip=obs_clip,
        T=BS_T,
        dt=dt,
        cfl=cfl,
        check_cfl=check_cfl,
        well_balanced=well_balanced,
    )

    if dH_source is not None:
        env_kwargs["dH_source"] = float(dH_source)

    env = CACSaintVenantPDE1D(**env_kwargs)

    env.reference_X = X
    env.reference_H = Hbar
    env.reference_Q = Qbar
    env.reference_dx = dx

    return env, X, Hbar, Qbar, p

# ============================================================
# Backstepping functions copied from the standalone Backstepping script
# ============================================================

def left_eigenvector_matrix(H: float, Q: float, p: CACParams) -> Tuple[np.ndarray, float, float]:
    H = max(float(H), p.H_min)
    V = Q / (p.B * H)
    c = np.sqrt(p.g * H)
    lp = V + c
    lm = V - c

    R = np.array([[1.0, 1.0], [p.B * lp, p.B * lm]], dtype=float)
    Tmat = np.linalg.inv(R)
    return Tmat, lp, lm


def characteristic_variables(H: np.ndarray, Q: np.ndarray, Hbar: np.ndarray, p: CACParams) -> Tuple[np.ndarray, np.ndarray]:
    u = np.zeros_like(H)
    v = np.zeros_like(H)

    for i in range(len(H)):
        Tmat, _, _ = left_eigenvector_matrix(Hbar[i], p.Q_bar, p)
        z = Tmat @ np.array([H[i] - Hbar[i], Q[i] - p.Q_bar])
        u[i] = z[0]
        v[i] = z[1]

    return u, v


def linearized_matrices_profiles(X: np.ndarray, Hbar: np.ndarray, p: CACParams) -> Dict[str, np.ndarray]:
    Np = len(X)
    Qbar = p.Q_bar
    Hx = np.gradient(Hbar, X, edge_order=2)

    T_arr = np.zeros((Np, 2, 2))
    R_arr = np.zeros((Np, 2, 2))
    A_arr = np.zeros((Np, 2, 2))
    lam_plus = np.zeros(Np)
    lam_minus = np.zeros(Np)

    for i in range(Np):
        H = float(Hbar[i])
        Q = Qbar

        Tmat, lp, lm = left_eigenvector_matrix(H, Q, p)
        T_arr[i] = Tmat
        R_arr[i] = np.linalg.inv(Tmat)

        A_arr[i] = np.array(
            [[0.0, 1.0 / p.B],
             [p.g * p.B * H - Q**2 / (p.B * H**2), 2.0 * Q / (p.B * H)]],
            dtype=float,
        )

        lam_plus[i] = lp
        lam_minus[i] = lm

    R_x = np.gradient(R_arr, X, axis=0, edge_order=2)

    def G_of_state(Hval: float, Qval: float, Hx_val: float) -> np.ndarray:
        A = np.array(
            [[0.0, 1.0 / p.B],
             [p.g * p.B * Hval - Qval**2 / (p.B * Hval**2), 2.0 * Qval / (p.B * Hval)]],
            dtype=float,
        )
        Xbar_x = np.array([Hx_val, 0.0])
        F = source(np.array([Hval]), np.array([Qval]), p)[0]
        return A @ Xbar_x - F

    Bstar = np.zeros((Np, 2, 2))

    for i in range(Np):
        H = float(Hbar[i])
        Q = Qbar
        hx = float(Hx[i])

        dh = max(1e-6, abs(H) * 1e-6)
        dq = max(1e-6, abs(Q) * 1e-6)

        dG_dH = (G_of_state(H + dh, Q, hx) - G_of_state(H - dh, Q, hx)) / (2.0 * dh)
        dG_dQ = (G_of_state(H, Q + dq, hx) - G_of_state(H, Q - dq, hx)) / (2.0 * dq)

        Bstar[i, :, 0] = dG_dH
        Bstar[i, :, 1] = dG_dQ

    M = np.zeros((Np, 2, 2))
    for i in range(Np):
        M[i] = T_arr[i] @ (A_arr[i] @ R_x[i] + Bstar[i] @ R_arr[i])

    eps1 = lam_plus.copy()
    eps2 = -lam_minus.copy()

    int1 = np.zeros(Np)
    int2 = np.zeros(Np)

    f1 = M[:, 0, 0] / eps1
    f2 = -M[:, 1, 1] / eps2

    for i in range(1, Np):
        h = X[i] - X[i - 1]
        int1[i] = int1[i - 1] + 0.5 * h * (f1[i] + f1[i - 1])
        int2[i] = int2[i - 1] + 0.5 * h * (f2[i] + f2[i - 1])

    phi1 = np.exp(int1)
    phi2 = np.exp(int2)

    c1 = -(phi1 / phi2) * M[:, 0, 1]
    c2 = -(phi2 / phi1) * M[:, 1, 0]

    Hbar0 = float(Hbar[0])
    kappa0 = -p.U0_bar * p.B * p.mu0 * p.g / np.sqrt(2.0 * p.g * (p.z_up - Hbar0))

    T0 = T_arr[0]
    q0 = (T0[0, 0] + T0[0, 1] * kappa0) / (T0[1, 0] + T0[1, 1] * kappa0)

    return {
        "eps1": eps1,
        "eps2": eps2,
        "eps1_x": np.gradient(eps1, X, edge_order=2),
        "eps2_x": np.gradient(eps2, X, edge_order=2),
        "c1": c1,
        "c2": c2,
        "q0": float(q0),
    }


def compute_variable_kernels(X: np.ndarray, coeff: Dict[str, np.ndarray]) -> Tuple[np.ndarray, np.ndarray]:
    Np = len(X)
    dx = X[1] - X[0]

    e1 = coeff["eps1"]
    e2 = coeff["eps2"]
    e1x = coeff["eps1_x"]
    e2x = coeff["eps2_x"]
    c1 = coeff["c1"]
    c2 = coeff["c2"]
    q0 = coeff["q0"]

    ratio = q0 * e1[0] / e2[0]

    Kvu = np.zeros((Np, Np))
    Kvv = np.zeros((Np, Np))

    for i in range(Np):
        Kvu[i, i] = -c2[i] / (e1[i] + e2[i])

    for i in range(1, Np):
        A = (e2[i] + e1[0]) / dx - e1x[0]
        rhs = e2[i] * Kvu[i - 1, 0] / dx + e1[0] * Kvu[i, 1] / dx
        denom = A - c2[0] * ratio

        Kvu[i, 0] = rhs / denom
        Kvv[i, 0] = ratio * Kvu[i, 0]

        for j in range(1, i):
            A = (e2[i] + e1[j]) / dx - e1x[j]
            B = (e2[i] + e2[j]) / dx + e2x[j]

            rhsK = e2[i] * Kvu[i - 1, j] / dx + e1[j] * Kvu[i, j + 1] / dx
            rhsL = e2[i] * Kvv[i - 1, j] / dx + e2[j] * Kvv[i, j - 1] / dx

            mat = np.array([[A, -c2[j]], [c1[j], B]], dtype=float)
            sol = np.linalg.solve(mat, np.array([rhsK, rhsL], dtype=float))

            Kvu[i, j] = sol[0]
            Kvv[i, j] = sol[1]

        Bdiag = e2[i] / dx + e2x[i]
        rhsdiag = e2[i] * Kvv[i - 1, i - 1] / dx - c1[i] * Kvu[i, i]
        Kvv[i, i] = rhsdiag / Bdiag

    return Kvu, Kvv


def backstepping_action(env: CACSaintVenantPDE1D, Kvu_L: np.ndarray, Kvv_L: np.ndarray, gain: float = 1.0):
    p = env.params
    H, Q = env.get_current_HQ()
    Hbar = env.Hbar
    X = env.X_grid

    u, v = characteristic_variables(H, Q, Hbar, p)
    Ubs = gain * float(np.trapezoid(Kvu_L * u + Kvv_L * v, X))

    T_L, _, _ = left_eigenvector_matrix(float(Hbar[-1]), p.Q_bar, p)
    t21, t22 = T_L[1, 0], T_L[1, 1]

    QL = float(Q[-1])
    Hbs_L = float(Hbar[-1]) + (Ubs - t22 * (QL - p.Q_bar)) / t21

    UL = Hbs_L - p.h_s - (QL**2 / (2.0 * p.g * p.B**2 * p.muL**2)) ** (1.0 / 3.0)

    return float(UL), float(Ubs)




# ============================================================
# SB3 evaluation-only loading helper
# ============================================================

def strip_optimizer_states_from_sb3_model(model_path: str) -> str:
    """
    Create an evaluation-only SB3 zip by removing optimizer states.

    This fixes errors such as:
        ValueError: loaded state dict has a different number of parameter groups

    It is safe for evaluation because optimizer states are only needed when
    continuing training. The policy/actor/critic weights are kept unchanged.

    Return value is a path without the .zip suffix, matching SB3's usual API.
    """
    model_path = Path(model_path)

    if model_path.suffix == ".zip":
        src_zip = model_path
    else:
        src_zip = model_path.with_suffix(".zip")

    if not src_zip.exists():
        raise FileNotFoundError(f"Cannot find SB3 model zip file: {src_zip}")

    dst_zip = src_zip.with_name(src_zip.stem + "_eval_only.zip")

    if dst_zip.exists():
        print(f"Using existing eval-only model: {dst_zip}")
        return str(dst_zip.with_suffix(""))

    with zipfile.ZipFile(src_zip, "r") as zin:
        with zipfile.ZipFile(dst_zip, "w", compression=zipfile.ZIP_DEFLATED) as zout:
            for item in zin.infolist():
                name = item.filename
                # Remove all optimizer states. These can have different parameter-group
                # structures after using DeepONet-specific learning rates.
                if "optimizer" in name.lower():
                    print(f"Skip optimizer state: {name}")
                    continue
                zout.writestr(item, zin.read(name))

    print(f"Created eval-only model: {dst_zip}")
    return str(dst_zip.with_suffix(""))


# ============================================================
# Rollouts
# ============================================================

def run_sac_exact(
    model_path,
    seed=123,
    H_left=TEST_H_LEFT,
    H_right=TEST_H_RIGHT,
    Q_value=TEST_Q_VALUE,
    dH_source=None,
):
    env, X, Hbar, Qbar, p = make_env_sac(
        H_left=H_left,
        H_right=H_right,
        Q_value=Q_value,
        seed=seed,
        dH_source=dH_source,
    )

    # Directly load the trained HybridSAC model.
    # EvalHybridSAC only rebuilds the optimizer structure required by SB3 loading;
    # the actual closed-loop control still comes directly from model.predict().
    # Because this training version saves alpha as actor.alpha_tensor, no manual
    # alpha restoration is needed here.
    model = EvalHybridSAC.load(model_path, env=env)
    obs, _ = env.reset(seed=seed)

    H_hist, Q_hist, U_hist, t_hist, reward_hist = [], [], [], [], []

    # Record the common prescribed initial state at t=0.
    H0, Q0 = env.get_current_HQ()
    H_hist.append(H0.copy())
    Q_hist.append(Q0.copy())
    U_hist.append(np.nan)
    t_hist.append(0.0)
    reward_hist.append(0.0)

    # Initialize the RL control filter from the equilibrium gate opening.
    previous_u_filtered = float(U_ref)

    done = False
    while not done:
        # Raw trained-controller output.
        action, _ = model.predict(obs, deterministic=True)

        # Same online low-pass filter as the previous comparison program.
        action_filtered, u_command_raw, previous_u_filtered = lowpass_filter_rl_control(
            action,
            previous_u_filtered,
            alpha=RL_CONTROL_FILTER_ALPHA,
        )

        # The FILTERED command is the actual closed-loop input.
        obs, reward, terminated, truncated, info = env.step(action_filtered)

        H, Q = env.get_current_HQ()

        H_hist.append(H.copy())
        Q_hist.append(Q.copy())
        U_hist.append(float(env.last_control))
        t_hist.append(float(info["time"]))
        reward_hist.append(float(reward))

        done = terminated or truncated

    env.close()

    H_hist = np.asarray(H_hist)
    Q_hist = np.asarray(Q_hist)
    U_hist = np.asarray(U_hist)
    t_hist = np.asarray(t_hist)
    reward_hist = np.asarray(reward_hist)

    H_err = H_hist - Hbar[None, :]
    Q_err = Q_hist - Qbar[None, :]

    return {
        "X": X,
        "Hbar": Hbar,
        "Qbar": Qbar,
        "t": t_hist,
        "H": H_hist,
        "Q": Q_hist,
        "U": U_hist,
        "reward": reward_hist,
        "H_l2": np.sqrt(np.mean(H_err ** 2, axis=1)),
        "Q_l2": np.sqrt(np.mean(Q_err ** 2, axis=1)),
        "final_H_minus_Hbar": H_err[-1],
        "final_Q_minus_Qbar": Q_err[-1],
        "total_reward": float(np.sum(reward_hist)),
    }


def run_backstepping_exact(
    seed=123,
    H_left=TEST_H_LEFT,
    H_right=TEST_H_RIGHT,
    Q_value=TEST_Q_VALUE,
    dH_source=None,
):
    """
    Backstepping rollout unified to the SAC standalone recording style:
        action -> env.step(action) -> record H,Q,U,t,reward.

    Therefore this is no longer the original Backstepping standalone recording mode;
    it is intentionally aligned with the SAC plot.
    """
    env, X, Hbar, Qbar, p = make_env_backstepping(
        H_left=H_left,
        H_right=H_right,
        Q_value=Q_value,
        seed=seed,
        dH_source=dH_source,
    )

    coeff = linearized_matrices_profiles(X, Hbar, p)
    Kvu, Kvv = compute_variable_kernels(X, coeff)
    Kvu_L = Kvu[-1, :]
    Kvv_L = Kvv[-1, :]

    obs, _ = env.reset(seed=seed)

    H_hist, Q_hist, U_hist, t_hist, reward_hist = [], [], [], [], []

    # Record the common prescribed initial state at t=0, exactly as for the
    # learned controller. The initial control is undefined and stored as NaN.
    H0, Q0 = env.get_current_HQ()
    H_hist.append(H0.copy())
    Q_hist.append(Q0.copy())
    U_hist.append(np.nan)
    t_hist.append(0.0)
    reward_hist.append(0.0)

    done = False
    while not done:
        UL_raw, Ubs = backstepping_action(env, Kvu_L, Kvv_L, gain=BACKSTEPPING_GAIN)
        action_env = float(np.clip(UL_raw, min_control_value, max_control_value))

        obs, reward, terminated, truncated, info = env.step(np.array([action_env], dtype=np.float32))

        # SAC plotting style: record AFTER env.step().
        H, Q = env.get_current_HQ()
        H_hist.append(H.copy())
        Q_hist.append(Q.copy())
        U_hist.append(action_env)
        t_hist.append(float(info["time"]))
        reward_hist.append(float(reward))

        done = terminated or truncated

    env.close()

    H_hist = np.asarray(H_hist)
    Q_hist = np.asarray(Q_hist)
    U_hist = np.asarray(U_hist)
    t_hist = np.asarray(t_hist)
    reward_hist = np.asarray(reward_hist)

    H_err = H_hist - Hbar[None, :]
    Q_err = Q_hist - Qbar[None, :]

    return {
        "X": X,
        "Hbar": Hbar,
        "Qbar": Qbar,
        "t": t_hist,
        "H": H_hist,
        "Q": Q_hist,
        "U": U_hist,
        "reward": reward_hist,
        "H_l2": np.sqrt(np.mean(H_err ** 2, axis=1)),
        "Q_l2": np.sqrt(np.mean(Q_err ** 2, axis=1)),
        "final_H_minus_Hbar": H_err[-1],
        "final_Q_minus_Qbar": Q_err[-1],
        "total_reward": float(np.sum(reward_hist)),
    }


# ============================================================
# Plot / save
# ============================================================

def plot_boundary_comparison(res_sac, res_bs, save_dir, tag):
    t_sac = res_sac["t"] / 60.0
    t_bs = res_bs["t"] / 60.0

    Hbar = res_sac["Hbar"]
    Qbar = res_sac["Qbar"]

    fig, ax = plt.subplots(4, 1, figsize=(8, 10), dpi=300, sharex=True)

    ax[0].plot(t_sac, res_sac["H"][:, 0], label="SAC H(0,t)")
    ax[0].plot(t_bs, res_bs["H"][:, 0], "--", label="BS H(0,t)")
    ax[0].axhline(Hbar[0], linestyle=":", linewidth=1, label="Hbar(0)")
    ax[0].set_ylabel("H(0,t) [m]", fontsize=18, fontweight="bold", labelpad=8)
    ax[0].legend(**_legend_kwargs_two_columns(loc="best")); ax[0].grid(True, alpha=0.3)

    ax[1].plot(t_sac, res_sac["H"][:, -1], label="SAC H(L,t)")
    ax[1].plot(t_bs, res_bs["H"][:, -1], "--", label="BS H(L,t)")
    ax[1].axhline(Hbar[-1], linestyle=":", linewidth=1, label="Hbar(L)")
    ax[1].set_ylabel("H(L,t) [m]", fontsize=18, fontweight="bold", labelpad=8)
    ax[1].legend(**_legend_kwargs_two_columns(loc="best")); ax[1].grid(True, alpha=0.3)

    ax[2].plot(t_sac, res_sac["Q"][:, 0], label="SAC Q(0,t)")
    ax[2].plot(t_bs, res_bs["Q"][:, 0], "--", label="BS Q(0,t)")
    ax[2].axhline(Qbar[0], linestyle=":", linewidth=1, label="Qbar")
    ax[2].set_ylabel("Q(0,t) [m^3/s]", fontsize=18, fontweight="bold", labelpad=8)
    ax[2].legend(**_legend_kwargs_two_columns(loc="best")); ax[2].grid(True, alpha=0.3)

    ax[3].plot(t_sac, res_sac["Q"][:, -1], label="SAC Q(L,t)")
    ax[3].plot(t_bs, res_bs["Q"][:, -1], "--", label="BS Q(L,t)")
    ax[3].axhline(Qbar[-1], linestyle=":", linewidth=1, label="Qbar")
    ax[3].set_ylabel("Q(L,t) [m^3/s]", fontsize=18, fontweight="bold", labelpad=8)
    ax[3].set_xlabel("Time [min]", fontsize=18, fontweight="bold", labelpad=10)
    ax[3].legend(**_legend_kwargs_two_columns(loc="best")); ax[3].grid(True, alpha=0.3)

    fig.suptitle("SAC-style boundary comparison: SAC vs Backstepping")
    fig.tight_layout()

    path = os.path.join(save_dir, f"{tag}_SAC_vs_Backstepping_boundary_only.png")
    fig.savefig(path, dpi=500, bbox_inches="tight")
    print(f"Saved comparison figure to: {path}")
    plt.show()


def plot_hybrid_action_diagnostics(res_sac, save_dir, tag):
    """Plot normalized actions and physical gate commands of DeepONet/RL/hybrid."""
    required = ["A_deep", "A_hybrid", "alpha", "U_deep", "U_hybrid"]
    if any(k not in res_sac for k in required):
        print("Hybrid diagnostics are not available in res_sac.")
        return

    t = np.asarray(res_sac["t"], dtype=float) / 60.0
    a_deep = np.asarray(res_sac["A_deep"], dtype=float)
    a_rl = np.asarray(res_sac["A_rl"], dtype=float)
    a_hybrid = np.asarray(res_sac["A_hybrid"], dtype=float)
    alpha = np.asarray(res_sac["alpha"], dtype=float)
    U_deep = np.asarray(res_sac["U_deep"], dtype=float)
    U_rl = np.asarray(res_sac["U_rl"], dtype=float)
    U_hybrid = np.asarray(res_sac["U_hybrid"], dtype=float)

    fig, ax = plt.subplots(3, 1, figsize=(8, 8), dpi=300, sharex=True)

    ax[0].plot(t, a_deep, label=r"$a_{DeepONet}$")
    if np.any(np.isfinite(a_rl)):
        ax[0].plot(t, a_rl, label=r"$a_{RL}$")
    ax[0].plot(t, a_hybrid, label=r"$a_{hybrid}$")
    ax[0].axhline(1.0, linestyle=":", linewidth=1)
    ax[0].axhline(-1.0, linestyle=":", linewidth=1)
    ax[0].set_ylabel("normalized action", fontsize=18, fontweight="bold", labelpad=8)
    ax[0].legend(**_legend_kwargs_two_columns(loc="best"))
    ax[0].grid(True, alpha=0.3)

    ax[1].plot(t, U_deep, label=r"$U_{DeepONet}$")
    if np.any(np.isfinite(U_rl)):
        ax[1].plot(t, U_rl, label=r"$U_{RL}$")
    ax[1].plot(t, U_hybrid, label=r"$U_{hybrid}$")
    ax[1].axhline(U_ref, linestyle=":", linewidth=1, label=r"$\bar U_L$")
    ax[1].axhline(max_control_value, linestyle=":", linewidth=1, label=r"$U_{max}$")
    ax[1].set_ylabel(r"$U_L$ [m]", fontsize=18, fontweight="bold", labelpad=8)
    ax[1].legend(**_legend_kwargs_two_columns(loc="best"))
    ax[1].grid(True, alpha=0.3)

    ax[2].plot(t, alpha, label=r"$\alpha$")
    ax[2].set_ylabel(r"RL weight $\alpha$", fontsize=18, fontweight="bold", labelpad=8)
    ax[2].set_xlabel("Time [min]", fontsize=18, fontweight="bold", labelpad=10)
    ax[2].set_ylim([-0.05, 1.05])
    ax[2].legend(**_legend_kwargs_two_columns(loc="best"))
    ax[2].grid(True, alpha=0.3)

    fig.suptitle("Hybrid DeepONet-SAC action diagnostics")
    fig.tight_layout()

    path = os.path.join(save_dir, f"{tag}_hybrid_action_diagnostics.png")
    fig.savefig(path, dpi=500, bbox_inches="tight")
    print(f"Saved hybrid action diagnostic figure to: {path}")
    plt.show()


def plot_control_input_comparison(res_sac, res_bs, save_dir, tag):
    """
    Plot downstream gate input U_L(t) for SAC and Backstepping.

    Upper panel:
        U_L^{SAC}(t) and U_L^{Backstepping}(t)

    Lower panel:
        Delta U_L(t) = U_L^{SAC}(t) - U_L^{Backstepping}(t)

    This figure is used to show how the learned SAC controller changes the
    physical downstream gate command compared with the nominal backstepping
    controller.
    """
    t_sac = np.asarray(res_sac["t"], dtype=float) / 60.0
    t_bs = np.asarray(res_bs["t"], dtype=float) / 60.0

    U_sac = np.asarray(res_sac["U"], dtype=float)
    U_bs = np.asarray(res_bs["U"], dtype=float)

    # Robust length alignment in case one rollout records one extra sample.
    n_sac = min(len(t_sac), len(U_sac))
    n_bs = min(len(t_bs), len(U_bs))
    t_sac = t_sac[:n_sac]
    U_sac = U_sac[:n_sac]
    t_bs = t_bs[:n_bs]
    U_bs = U_bs[:n_bs]

    # Interpolate Backstepping U onto the SAC time grid for Delta U.
    if len(t_bs) >= 2 and len(t_sac) >= 1:
        U_bs_on_sac = np.interp(t_sac, t_bs, U_bs)
        dU = U_sac - U_bs_on_sac
        t_delta = t_sac
    else:
        dU = np.array([])
        t_delta = np.array([])

    fig, ax = plt.subplots(2, 1, figsize=(8, 6), dpi=300, sharex=True)

    ax[0].plot(t_sac, U_sac, label=r"SAC $U_L(t)$")
    ax[0].plot(t_bs, U_bs, "--", label=r"BS $U_L(t)$")
    ax[0].axhline(U_ref, linestyle=":", linewidth=1, label=r"$\bar U_L$")
    ax[0].set_ylabel(r"$U_L(t)$ [m]", fontsize=18, fontweight="bold", labelpad=8)
    ax[0].legend(**_legend_kwargs_two_columns(loc="best"))
    ax[0].grid(True, alpha=0.3)

    if len(dU) > 0:
        ax[1].plot(t_delta, dU, label=r"$U_L^{SAC}(t)-U_L^{Backstepping}(t)$")
        ax[1].axhline(0.0, linestyle=":", linewidth=1)
    ax[1].set_ylabel(r"$\Delta U_L(t)$ [m]", fontsize=18, fontweight="bold", labelpad=8)
    ax[1].set_xlabel("Time [min]", fontsize=18, fontweight="bold", labelpad=10)
    ax[1].legend(**_legend_kwargs_two_columns(loc="best"))
    ax[1].grid(True, alpha=0.3)

    fig.suptitle("Control input comparison: SAC vs Backstepping")
    fig.tight_layout()

    path = os.path.join(save_dir, f"{tag}_SAC_vs_Backstepping_control_input.png")
    fig.savefig(path, dpi=500, bbox_inches="tight")
    print(f"Saved control input figure to: {path}")
    plt.show()


def _resolve_output_dir():
    """Return the active output directory used by this script."""
    if "SAVE_DIR" in globals():
        out = globals()["SAVE_DIR"]
    elif "DEFAULT_SAVE_DIR" in globals():
        out = globals()["DEFAULT_SAVE_DIR"]
    elif "save_dir" in globals():
        out = globals()["save_dir"]
    elif "out_dir" in globals():
        out = globals()["out_dir"]
    else:
        out = "./test_results_sac_style_backstepping_vs_sac_with_l2_3d"
    out = Path(out)
    out.mkdir(parents=True, exist_ok=True)
    return str(out)


def _compute_spatial_l2_over_time(res, state_key):
    """
    Compute spatial L2 norm over time:
        ||H-Hbar||_L2 = sqrt( integral_0^1 (H(x,t)-Hbar(x))^2 ds )
        ||Q-Qbar||_L2 = sqrt( integral_0^1 (Q(x,t)-Qbar(x))^2 ds )
    where s=x/L is the normalized spatial coordinate.
    """
    X = res["X"]
    s = X / X[-1]

    if state_key == "H":
        err = res["H"] - res["Hbar"][None, :]
    elif state_key == "Q":
        err = res["Q"] - res["Qbar"][None, :]
    else:
        raise ValueError("state_key must be 'H' or 'Q'.")

    return np.sqrt(np.trapezoid(err ** 2, s, axis=1))


def plot_l2_norm_comparison(res_sac, res_bs, tag="case_Q9p5_H3p5_4p4"):
    """
    Plot time evolution of spatial L2 norms:
        ||H-Hbar||_L2
        ||Q-Qbar||_L2
    for SAC and Backstepping.
    """
    t_sac = res_sac["t"] / 60.0
    t_bs = res_bs["t"] / 60.0

    H_l2_sac = _compute_spatial_l2_over_time(res_sac, "H")
    Q_l2_sac = _compute_spatial_l2_over_time(res_sac, "Q")
    H_l2_bs = _compute_spatial_l2_over_time(res_bs, "H")
    Q_l2_bs = _compute_spatial_l2_over_time(res_bs, "Q")

    fig, axes = plt.subplots(2, 1, figsize=(9, 7), sharex=True)

    axes[0].plot(t_sac, H_l2_sac, label=r"SAC $\Vert H-\bar H\Vert_{L^2}$ [m]")
    axes[0].plot(t_bs, H_l2_bs, "--", label=r"BS $\Vert H-\bar H\Vert_{L^2}$ [m]")
    axes[0].set_ylabel(r"$\Vert H-\bar H\Vert_{L^2}$ [m]", fontsize=18, fontweight="bold", labelpad=8)
    axes[0].legend(**_legend_kwargs_two_columns(loc="best"))
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(t_sac, Q_l2_sac, label=r"SAC $\Vert Q-\bar Q\Vert_{L^2}$ [m$^3$/s]")
    axes[1].plot(t_bs, Q_l2_bs, "--", label=r"BS $\Vert Q-\bar Q\Vert_{L^2}$ [m$^3$/s]")
    axes[1].set_ylabel(r"$\Vert Q-\bar Q\Vert_{L^2}$ [m$^3$/s]", fontsize=18, fontweight="bold", labelpad=8)
    axes[1].set_xlabel("Time [min]", fontsize=18, fontweight="bold", labelpad=10)
    axes[1].legend(**_legend_kwargs_two_columns(loc="best"))
    axes[1].grid(True, alpha=0.3)

    fig.suptitle("Spatial L2 norm tracking errors")
    fig.tight_layout()

    fig_path = os.path.join(_resolve_output_dir(), f"{tag}_H_Q_L2_norm_errors.png")
    fig.savefig(fig_path, dpi=300, bbox_inches="tight")
    print(f"L2-norm error figure saved to: {fig_path}")
    plt.show()

    # Save L2 norm time series as .dat files for later plotting.
    np.savetxt(
        os.path.join(_resolve_output_dir(), f"{tag}_SAC_H_L2.dat"),
        np.column_stack([t_sac, H_l2_sac]),
        header="time_min H_L2",
    )
    np.savetxt(
        os.path.join(_resolve_output_dir(), f"{tag}_SAC_Q_L2.dat"),
        np.column_stack([t_sac, Q_l2_sac]),
        header="time_min Q_L2",
    )
    np.savetxt(
        os.path.join(_resolve_output_dir(), f"{tag}_Backstepping_H_L2.dat"),
        np.column_stack([t_bs, H_l2_bs]),
        header="time_min H_L2",
    )
    np.savetxt(
        os.path.join(_resolve_output_dir(), f"{tag}_Backstepping_Q_L2.dat"),
        np.column_stack([t_bs, Q_l2_bs]),
        header="time_min Q_L2",
    )
    print("L2-norm time series saved.")


def plot_surface_single(X, t, Z, zlabel, title, save_path):
    """
    Draw one 3D surface plot for H(x,t) or Q(x,t).
    Plotting style is matched to the first program; all data and calls are unchanged.
    """
    X_km = X / 1000.0
    t_min = np.asarray(t, dtype=float) / 60.0
    Z = np.asarray(Z, dtype=float)

    # Uniform time-row subsampling is used only for 3D rendering.
    # Keep rstride=1 below so adjacent rendered rows are always connected,
    # avoiding the coarse-stride polygon artifact that can look like spikes.
    t_min, Z = _subsample_3d_time_rows(t_min, Z)

    meshx, mesht = np.meshgrid(X_km, t_min)

    fig = plt.figure(figsize=(8.8, 6.6), dpi=600)
    subfigs = fig.subfigures(nrows=1, ncols=1, hspace=0)
    subfig = subfigs
    subfig.subplots_adjust(left=0.18, bottom=0.08, right=0.98, top=0.96)

    axes = subfig.subplots(
        nrows=1,
        ncols=1,
        subplot_kw={"projection": "3d", "computed_zorder": False},
    )

    for axis in [axes.xaxis, axes.yaxis, axes.zaxis]:
        axis._axinfo["axisline"]["linewidth"] = 10
        axis._axinfo["axisline"]["color"] = "b"
        axis._axinfo["grid"]["linewidth"] = 0.8
        axis._axinfo["grid"]["linestyle"] = "--"
        axis._axinfo["grid"]["color"] = "#d1d1d1"
        axis.set_pane_color((1, 1, 1))

    axes.view_init(10, 15)
    axes.set_xlabel(r"$x$ [km]", fontsize=16, labelpad=5, fontweight="bold")
    axes.set_ylabel(r"$Time$ [min]", fontsize=16, labelpad=5, fontweight="bold")
    axes.set_zlabel(zlabel, rotation=90, fontsize=16, labelpad=4, fontweight="bold")
    axes.zaxis.set_rotate_label(False)
    axes.zaxis.label.set_clip_on(False)

    x_ticks = np.array([X_km[0], 0.5 * (X_km[0] + X_km[-1]), X_km[-1]], dtype=float)
    y_ticks = np.array([t_min[0], 0.5 * (t_min[0] + t_min[-1]), t_min[-1]], dtype=float)
    z_min, z_max = float(np.nanmin(Z)), float(np.nanmax(Z))
    z_ticks = np.linspace(z_min, z_max, 5)

    axes.set_xticks(x_ticks)
    axes.set_yticks(y_ticks)
    axes.set_zticks(z_ticks)
    axes.set_xticklabels(_format_three_ticklabels(x_ticks), fontweight="bold", fontsize=16)
    axes.set_yticklabels(_format_three_ticklabels(y_ticks), fontweight="bold", fontsize=16)
    axes.set_zticklabels(_format_z_ticklabels(z_ticks), fontweight="bold", fontsize=16)

    rstride = 1
    surface_cmap = "viridis" if "H(" in str(zlabel) else "plasma"
    axes.plot_surface(
        meshx,
        mesht,
        Z,
        edgecolor="black",
        lw=0.8,
        rstride=rstride,
        cstride=5,
        alpha=1,
        cmap=surface_cmap,
        shade=False,
        rasterized=True,
        antialiased=True,
    )

    # Downstream boundary highlight curve remains removed as in the target program.

    axes.set_title(title, fontsize=14)
    fig.savefig(save_path, dpi=600, bbox_inches="tight", pad_inches=0.35)
    print(f"3D surface saved to: {save_path}")
    plt.show()
    plt.close(fig)

def plot_HQ_3d_surfaces(res_sac, res_bs, tag="case_Q9p5_H3p5_4p4"):
    """
    Plot H(x,t) and Q(x,t) 3D surfaces for SAC and Backstepping.
    Four figures are generated:
        SAC H, SAC Q, Backstepping H, Backstepping Q.
    """
    plot_surface_single(
        res_sac["X"],
        res_sac["t"],
        res_sac["H"],
        r"$H(x,t)$ [m]",
        "SAC: $H(x,t)$",
        os.path.join(_resolve_output_dir(), f"{tag}_SAC_H_3d.png"),
    )

    plot_surface_single(
        res_sac["X"],
        res_sac["t"],
        res_sac["Q"],
        r"$Q(x,t)$ [m$^3$/s]",
        "SAC: $Q(x,t)$",
        os.path.join(_resolve_output_dir(), f"{tag}_SAC_Q_3d.png"),
    )

    plot_surface_single(
        res_bs["X"],
        res_bs["t"],
        res_bs["H"],
        r"$H(x,t)$ [m]",
        "Backstepping: $H(x,t)$",
        os.path.join(_resolve_output_dir(), f"{tag}_Backstepping_H_3d.png"),
    )

    plot_surface_single(
        res_bs["X"],
        res_bs["t"],
        res_bs["Q"],
        r"$Q(x,t)$ [m$^3$/s]",
        "Backstepping: $Q(x,t)$",
        os.path.join(_resolve_output_dir(), f"{tag}_Backstepping_Q_3d.png"),
    )


def save_final_error_vectors(res_sac, res_bs, tag="case_Q9p5_H3p5_4p4"):
    """
    Save final H-Hbar and Q-Qbar vectors for both controllers.
    """
    np.savetxt(
        os.path.join(_resolve_output_dir(), f"{tag}_SAC_final_H_minus_Hbar.dat"),
        res_sac["H"][-1] - res_sac["Hbar"],
    )
    np.savetxt(
        os.path.join(_resolve_output_dir(), f"{tag}_SAC_final_Q_minus_Qbar.dat"),
        res_sac["Q"][-1] - res_sac["Qbar"],
    )
    np.savetxt(
        os.path.join(_resolve_output_dir(), f"{tag}_Backstepping_final_H_minus_Hbar.dat"),
        res_bs["H"][-1] - res_bs["Hbar"],
    )
    np.savetxt(
        os.path.join(_resolve_output_dir(), f"{tag}_Backstepping_final_Q_minus_Qbar.dat"),
        res_bs["Q"][-1] - res_bs["Qbar"],
    )
    print("Final H-Hbar and Q-Qbar vectors saved.")

def save_npz(res_sac, res_bs, save_dir, tag):
    path = os.path.join(save_dir, f"{tag}_SAC_vs_Backstepping_exact.npz")
    np.savez(
        path,
        X=res_sac["X"],
        Hbar=res_sac["Hbar"],
        Qbar=res_sac["Qbar"],
        t_sac=res_sac["t"],
        H_sac=res_sac["H"],
        Q_sac=res_sac["Q"],
        U_sac=res_sac["U"],
        reward_sac=res_sac["reward"],
        t_backstepping=res_bs["t"],
        H_backstepping=res_bs["H"],
        Q_backstepping=res_bs["Q"],
        U_backstepping=res_bs["U"],
        reward_backstepping=res_bs["reward"],
        total_reward_sac=res_sac["total_reward"],
        total_reward_backstepping=res_bs["total_reward"],
    )
    print(f"Saved data to: {path}")


def print_final(name, res):
    H = res["H"]
    Q = res["Q"]
    Hbar = res["Hbar"]
    Qbar = res["Qbar"]
    print("")
    print("=" * 90)
    print(f"{name} final")
    print("=" * 90)
    print(f"final H(0) = {H[-1,0]:.6f}, target = {Hbar[0]:.6f}, error = {H[-1,0]-Hbar[0]:.6f}")
    print(f"final H(L) = {H[-1,-1]:.6f}, target = {Hbar[-1]:.6f}, error = {H[-1,-1]-Hbar[-1]:.6f}")
    print(f"final Q(0) = {Q[-1,0]:.6f}, target = {Qbar[0]:.6f}, error = {Q[-1,0]-Qbar[0]:.6f}")
    print(f"final Q(L) = {Q[-1,-1]:.6f}, target = {Qbar[-1]:.6f}, error = {Q[-1,-1]-Qbar[-1]:.6f}")
    print(f"total_reward = {res['total_reward']:.6f}")


def print_final_error_values(name, res):
    """
    Print the final-time values of H-Hbar and Q-Qbar at every spatial grid point.
    """
    X = np.asarray(res["X"], dtype=float)
    H_err_final = np.asarray(res["H"][-1] - res["Hbar"], dtype=float)
    Q_err_final = np.asarray(res["Q"][-1] - res["Qbar"], dtype=float)

    print("")
    print("=" * 90)
    print(f"{name} final H-Hbar and Q-Qbar values")
    print("=" * 90)
    print("index, x[m], H-Hbar[m], Q-Qbar[m^3/s]")
    for i, (x_i, h_i, q_i) in enumerate(zip(X, H_err_final, Q_err_final)):
        print(f"{i:4d}, {x_i:12.6f}, {h_i: .10e}, {q_i: .10e}")


def save_final_error_value_table(res_sac, res_bs, save_dir, tag):
    """
    Save the final-time H-Hbar and Q-Qbar values of SAC and Backstepping
    into one .dat file for direct inspection.
    """
    X = np.asarray(res_sac["X"], dtype=float)

    sac_H_err = np.asarray(res_sac["H"][-1] - res_sac["Hbar"], dtype=float)
    sac_Q_err = np.asarray(res_sac["Q"][-1] - res_sac["Qbar"], dtype=float)

    bs_H_err = np.asarray(res_bs["H"][-1] - res_bs["Hbar"], dtype=float)
    bs_Q_err = np.asarray(res_bs["Q"][-1] - res_bs["Qbar"], dtype=float)

    data = np.column_stack([
        X,
        sac_H_err,
        sac_Q_err,
        bs_H_err,
        bs_Q_err,
    ])

    path = os.path.join(save_dir, f"{tag}_final_H_Q_error_values.dat")
    np.savetxt(
        path,
        data,
        header=(
            "x_m "
            "SAC_H_minus_Hbar_m "
            "SAC_Q_minus_Qbar_m3s "
            "Backstepping_H_minus_Hbar_m "
            "Backstepping_Q_minus_Qbar_m3s"
        ),
    )
    print(f"Final H-Hbar and Q-Qbar value table saved to: {path}")



# ============================================================
# Controller-comparison comparison utilities
# ============================================================

LABEL_BACKSTEPPING = "BS"
LABEL_NEW_MODEL = "TDPSAC"

# Retain the previous label constants so the common style helper can be reused
# when additional checkpoints are added later.
LABEL_SAC_CURVE = "SAC"
LABEL_FP_SAC = "SAC-FPC"
LABEL_SAC = "DO-SAC"
LABEL_HYBRID = "DPSAC"
LABEL_DFP_DON_SAC = "TDPSAC-FPC"
LABEL_HYBRID_FREEZE = LABEL_NEW_MODEL

# Fixed order for 2D legends, terminal summaries, evidence tables, and saved
# controller arrays.  This is independent of the order in which models are run.
CONTROLLER_DISPLAY_ORDER = [
    LABEL_BACKSTEPPING,
    LABEL_SAC_CURVE,
    LABEL_FP_SAC,
    LABEL_SAC,
    LABEL_HYBRID,
    LABEL_DFP_DON_SAC,
    LABEL_NEW_MODEL,
]


def _ordered_results(results: Dict[str, dict]) -> Dict[str, dict]:
    """Return a dictionary in the requested display/table order."""
    ordered = {
        label: results[label]
        for label in CONTROLLER_DISPLAY_ORDER
        if label in results
    }
    # Preserve any future additional controllers after the seven fixed entries.
    for label, res in results.items():
        if label not in ordered:
            ordered[label] = res
    return ordered


def _safe_load_hybrid_model(model_path: str, env):
    """
    Load a HybridSAC checkpoint for evaluation.

    The HybridSAC training script may save optimizers with special parameter
    groups for DeepONet. EvalHybridSAC rebuilds the optimizer structure. If the
    optimizer groups still do not match, an eval-only copy without optimizer
    states is created and loaded. This does not change the policy weights.
    """
    try:
        return EvalHybridSAC.load(model_path, env=env)
    except Exception as e:
        msg = str(e)
        if ("parameter group" in msg) or ("Names of parameters" in msg) or ("optimizer" in msg):
            print("Hybrid load with optimizer states failed; trying eval-only checkpoint.")
            print("Original error:", repr(e))
            eval_path = strip_optimizer_states_from_sb3_model(model_path)
            return EvalHybridSAC.load(eval_path, env=env)
        raise



def _checkpoint_exists(model_path: str) -> bool:
    """
    Return True only when model_path resolves to a valid SB3 ZIP checkpoint.

    This deliberately rejects arbitrary existing files such as Jupyter's
    kernel-*.json connection file. Stable-Baselines3 checkpoints are ZIP files,
    although callers may pass the path either with or without the .zip suffix.
    """
    if model_path is None or str(model_path).strip() == "":
        return False

    path = Path(str(model_path)).expanduser()

    candidates = [path]
    if path.suffix.lower() != ".zip":
        candidates.append(path.with_suffix(".zip"))

    for candidate in candidates:
        if candidate.is_file() and zipfile.is_zipfile(candidate):
            return True

    return False


def _safe_load_generic_sac_model(model_path: str, env):
    """
    Load a custom SAC-family checkpoint without assuming one exact policy layout.

    This is used for TDPSAC-FPC and SAC-FPC because their policy
    architectures may differ. The loader first asks SB3 to reconstruct the
    policy class stored in the checkpoint. If optimizer parameter groups are
    incompatible during evaluation, it creates an evaluation-only copy with
    optimizer states removed and retries.

    Policy, actor, critic, normalization, and saved alpha weights are preserved.
    """
    errors = []

    try:
        return SAC.load(model_path, env=env)
    except Exception as exc:
        errors.append(("SAC.load", exc))

    try:
        eval_path = strip_optimizer_states_from_sb3_model(model_path)
        return SAC.load(eval_path, env=env)
    except Exception as exc:
        errors.append(("SAC.load(eval-only)", exc))

    # Final fallback for checkpoints trained with the HybridSAC subclass used
    # in this file.
    try:
        return _safe_load_hybrid_model(model_path, env=env)
    except Exception as exc:
        errors.append(("_safe_load_hybrid_model", exc))

    details = "\n".join(f"  {name}: {error!r}" for name, error in errors)
    raise RuntimeError(
        f"Unable to load SAC-family checkpoint: {model_path}\n"
        f"Tried the generic and HybridSAC evaluation loaders:\n{details}"
    )

def run_policy_exact(
    model_path: str,
    model_kind: str,
    label: str,
    seed: int = 123,
    H_left: float = TEST_H_LEFT,
    H_right: float = TEST_H_RIGHT,
    Q_value: float = TEST_Q_VALUE,
    dH_source=None,
):
    """
    Run one learned controller in the CAC environment.

    model_kind:
        "sac"       -> ordinary raw-HQ SAC.load(...)
        "sac_error" -> old SAC checkpoint using normalized H/Q errors
        "hybrid"    -> EvalHybridSAC.load(...), for the existing hybrid checkpoints
        "auto"      -> reconstruct the saved SAC policy class automatically; used
                       for TDPSAC-FPC and SAC-FPC

    Every learned controller sees the same physical PDE reference, initial
    state, and raw physical H/Q observation interface in the current comparison.
    The reset state is recorded first at t=0, then later states are recorded
    after env.step().
    """
    model_kind_lower = model_kind.lower()
    if model_kind_lower == "sac_error":
        env, X, Hbar, Qbar, p = make_env_sac_error_state(
            H_left=H_left,
            H_right=H_right,
            Q_value=Q_value,
            seed=seed,
            dH_source=dH_source,
        )
    else:
        env, X, Hbar, Qbar, p = make_env_sac(
            H_left=H_left,
            H_right=H_right,
            Q_value=Q_value,
            seed=seed,
            dH_source=dH_source,
        )

    if model_kind_lower in {"sac", "sac_error"}:
        model = SAC.load(model_path, env=env)
    elif model_kind_lower == "hybrid":
        model = _safe_load_hybrid_model(model_path, env=env)
    elif model_kind_lower == "auto":
        model = _safe_load_generic_sac_model(model_path, env=env)
    else:
        raise ValueError(
            f"Unknown model_kind: {model_kind}. Use 'sac', 'sac_error', "
            "'hybrid', or 'auto'."
        )

    obs, _ = env.reset(seed=seed)
    obs = np.asarray(obs, dtype=np.float32).reshape(-1)
    if obs.size != RAW_STATE_DIM:
        raise RuntimeError(
            f"The reset observation has dimension {obs.size}; expected "
            f"{RAW_STATE_DIM} for the H/Q or H/Q-error interface."
        )

    observation_mode_used = (
        "normalized_error" if model_kind_lower == "sac_error" else "raw"
    )
    print(
        f"{label} observation mode: {observation_mode_used}, "
        f"dimension={obs.size}"
    )

    H_hist, Q_hist, U_hist, t_hist, reward_hist = [], [], [], [], []

    # Record the reset state before any controller acts.
    # This makes the learned controllers start from
    # exactly the same L2 error at t=0.
    H0, Q0 = env.get_current_HQ()
    H_hist.append(H0.copy())
    Q_hist.append(Q0.copy())
    U_hist.append(np.nan)
    t_hist.append(0.0)
    reward_hist.append(0.0)

    # Initialize the RL control filter from the equilibrium gate opening.
    previous_u_filtered = float(U_ref)

    done = False
    while not done:
        action, _ = model.predict(obs, deterministic=True)

        # Same online low-pass filter for every learned/RL controller.
        action_filtered, u_command_raw, previous_u_filtered = lowpass_filter_rl_control(
            action,
            previous_u_filtered,
            alpha=RL_CONTROL_FILTER_ALPHA,
        )

        # Use the filtered command in the actual Saint-Venant closed loop.
        obs, reward, terminated, truncated, info = env.step(action_filtered)

        H, Q = env.get_current_HQ()

        H_hist.append(H.copy())
        Q_hist.append(Q.copy())
        U_hist.append(float(env.last_control))
        t_hist.append(float(info["time"]))
        reward_hist.append(float(reward))

        done = terminated or truncated

    env.close()

    H_hist = np.asarray(H_hist)
    Q_hist = np.asarray(Q_hist)
    U_hist = np.asarray(U_hist)
    t_hist = np.asarray(t_hist)
    reward_hist = np.asarray(reward_hist)

    H_err = H_hist - Hbar[None, :]
    Q_err = Q_hist - Qbar[None, :]

    return {
        "label": label,
        "model_kind": model_kind,
        "observation_mode": observation_mode_used,
        "X": X,
        "Hbar": Hbar,
        "Qbar": Qbar,
        "t": t_hist,
        "H": H_hist,
        "Q": Q_hist,
        "U": U_hist,
        "reward": reward_hist,
        "H_l2": np.sqrt(np.mean(H_err ** 2, axis=1)),
        "Q_l2": np.sqrt(np.mean(Q_err ** 2, axis=1)),
        "final_H_minus_Hbar": H_err[-1],
        "final_Q_minus_Qbar": Q_err[-1],
        "total_reward": float(np.sum(reward_hist)),
    }


def run_backstepping_for_four(
    seed: int = 123,
    H_left: float = TEST_H_LEFT,
    H_right: float = TEST_H_RIGHT,
    Q_value: float = TEST_Q_VALUE,
    dH_source=None,
):
    res = run_backstepping_exact(
        seed=seed,
        H_left=H_left,
        H_right=H_right,
        Q_value=Q_value,
        dH_source=dH_source,
    )
    res["label"] = LABEL_BACKSTEPPING
    res["model_kind"] = "backstepping"
    return res


def _time_min(res):
    return np.asarray(res["t"], dtype=float) / 60.0


def _get_l2(res, key):
    return _compute_spatial_l2_over_time(res, key)


def _get_linf(res, key):
    """Return the spatial L-infinity tracking-error norm at each time instant."""
    state = np.asarray(res[key], dtype=float)
    reference_key = "Hbar" if key == "H" else "Qbar"
    reference = np.asarray(res[reference_key], dtype=float).reshape(1, -1)
    if state.ndim != 2 or reference.shape[1] != state.shape[1]:
        return np.asarray([], dtype=float)
    return np.max(np.abs(state - reference), axis=1)


def set_size(width, fraction=1, ncols=1, nrows=1, height_add=0):
    """Figure size helper copied from the third comparison-program style."""
    inches_per_pt = 1 / 72.27
    fig_width = width * inches_per_pt * fraction
    fig_height = fig_width * 0.618 * (nrows / ncols) + height_add
    return fig_width, fig_height



def _line_style(label):
    """Return the fixed colors and line styles used by the old comparison program."""
    if label == LABEL_BACKSTEPPING:  # BS
        return dict(color="orange", linestyle=":", linewidth=2.0, alpha=0.8)
    if label == LABEL_SAC_CURVE:  # SAC
        return dict(color="purple", linestyle="-", linewidth=1.5, alpha=0.8)
    if label == LABEL_FP_SAC:  # SAC-FPC
        return dict(
            color="saddlebrown",
            linestyle=(0, (6, 2, 1.5, 2)),
            linewidth=2.2,
            alpha=0.95,
        )
    if label == LABEL_SAC:  # DO-SAC
        return dict(color="blue", linestyle="-.", linewidth=2.0, alpha=0.8)
    if label == LABEL_HYBRID:  # DPSAC
        return dict(color="green", linestyle="--", linewidth=2.0, alpha=0.8)
    if label == LABEL_DFP_DON_SAC:  # TDPSAC-FPC
        return dict(color="black", linestyle="--", linewidth=2.2, alpha=0.95)
    if label == LABEL_HYBRID_FREEZE:  # TDPSAC
        return dict(color="red", linestyle="-", linewidth=3, alpha=0.8)
    return dict(linewidth=2.0, alpha=0.8)


def _line_style_2d(label):
    """2D-only linewidth settings; colors and linestyles remain unchanged."""
    style = dict(_line_style(label))

    if label == LABEL_HYBRID_FREEZE:      # TDPSAC, red
        style["linewidth"] = 2.3
    elif label == LABEL_DFP_DON_SAC:      # TDPSAC-FPC, black
        style["linewidth"] = 2.3
    elif label == LABEL_SAC_CURVE:        # SAC, purple
        style["linewidth"] = 1.5
    else:
        # BS, SAC-FPC, DO-SAC, DPSAC, and any other displayed curve
        style["linewidth"] = 2.0

    return style


def _ordered_results_for_2d_drawing(results: Dict[str, dict]) -> Dict[str, dict]:
    """2D drawing order only: red second-to-last, black last."""
    ordered = {}

    # Preserve the current order for all remaining curves.
    for label, res in results.items():
        if label not in (LABEL_HYBRID_FREEZE, LABEL_DFP_DON_SAC):
            ordered[label] = res

    # Red TDPSAC: second-to-last.
    if LABEL_HYBRID_FREEZE in results:
        ordered[LABEL_HYBRID_FREEZE] = results[LABEL_HYBRID_FREEZE]

    # Black TDPSAC-FPC: last.
    if LABEL_DFP_DON_SAC in results:
        ordered[LABEL_DFP_DON_SAC] = results[LABEL_DFP_DON_SAC]

    return ordered




def _legend_kwargs_two_columns(loc="upper center", bbox_to_anchor=None):
    """
    Shared legend style for the displayed methods.
    """
    kwargs = dict(
        fontsize=11,
        frameon=True,
        shadow=True,
        ncol=2,
        columnspacing=1.2,
        handlelength=2.8,
        borderaxespad=0.6,
    )
    kwargs["loc"] = loc
    if bbox_to_anchor is not None:
        kwargs["bbox_to_anchor"] = bbox_to_anchor
    return kwargs


def _apply_ordered_two_column_legend(
    ax,
    *,
    loc="best",
    fontsize=11,
    frameon=True,
    shadow=True,
    columnspacing=1.2,
    handlelength=2.8,
    borderaxespad=0.6,
):
    """Draw a two-column legend in top-to-bottom column order.

    The legend names follow CONTROLLER_DISPLAY_ORDER from top to bottom.
    With seven methods and two columns, the displayed order is:

        left column:  BS, SAC, SAC-FPC, DO-SAC
        right column: DPSAC, TDPSAC-FPC, TDPSAC
    """
    handles, labels = ax.get_legend_handles_labels()
    if not handles:
        return None

    pair_by_label = {
        label: (handle, label)
        for handle, label in zip(handles, labels)
    }

    ordered_pairs = [
        pair_by_label[label]
        for label in CONTROLLER_DISPLAY_ORDER
        if label in pair_by_label
    ]
    ordered_pairs.extend(
        (handle, label)
        for handle, label in zip(handles, labels)
        if label not in CONTROLLER_DISPLAY_ORDER
    )

    # Matplotlib fills a multi-column legend from top to bottom within each
    # column, so pass the requested order directly without row-wise repacking.
    ordered_handles = [pair[0] for pair in ordered_pairs]
    ordered_labels = [pair[1] for pair in ordered_pairs]

    return ax.legend(
        ordered_handles,
        ordered_labels,
        fontsize=fontsize,
        frameon=frameon,
        shadow=shadow,
        loc=loc,
        ncol=2,
        columnspacing=columnspacing,
        handlelength=handlelength,
        borderaxespad=borderaxespad,
    )


def _style_axes_second(ax, xlabel=None, ylabel=None, legend=True, legend_loc="best"):
    """2D axis style matched to the first program."""
    if xlabel is not None:
        ax.set_xlabel(xlabel, fontsize=18, fontweight="bold", labelpad=10)
    if ylabel is not None:
        ax.set_ylabel(ylabel, fontsize=18, fontweight="bold", labelpad=8)
    ax.tick_params(axis="both", labelsize=18)
    for tick in ax.get_xticklabels() + ax.get_yticklabels():
        tick.set_fontweight("bold")
    ax.grid(True, linestyle="--", alpha=0.8)
    if legend:
        _apply_ordered_two_column_legend(ax, loc=legend_loc, fontsize=11)


# Keep old function name for compatibility with the rest of the script.
def _style_axes(ax, xlabel=None, ylabel=None, legend=True, legend_loc="best"):
    _style_axes_second(ax, xlabel=xlabel, ylabel=ylabel, legend=legend, legend_loc=legend_loc)


def _safe_label_for_plot(label):
    """Keep algorithm names exactly the same as the second program."""
    return label


def _common_time_end_min(results: Dict[str, dict]) -> float:
    """Return the common end time in minutes across all results."""
    ends = []
    for res in results.values():
        t = _time_min(res)
        if len(t) > 0 and np.any(np.isfinite(t)):
            ends.append(float(np.nanmax(t)))
    return min(ends) if ends else 0.0


def _style_inset_axes(ax_inset):
    """Inset style copied from the second program."""
    ax_inset.grid(True, linestyle="--", alpha=0.8)
    ax_inset.tick_params(axis="both", labelsize=14)


def _add_second_style_inset(
    ax,
    results: Dict[str, dict],
    y_getter,
    ref_value=None,
    zoom_ratio: float = 0.75,
    bounds=(0.55, 0.15, 0.40, 0.40),
):
    """
    Add a local zoom inset with the same style as the second program:
        ax.inset_axes([...]) + ax.indicate_inset_zoom(...).

    This function only changes plotting. It does not change model loading,
    controller output, reward, rollout data, or saved numerical results.
    """
    min_t_end = _common_time_end_min(results)
    if min_t_end <= 0:
        return None

    zoom_start = float(zoom_ratio) * min_t_end
    ax_inset = ax.inset_axes(list(bounds))

    y_all = []
    results_draw = _ordered_results_for_2d_drawing(results)
    for label, res in results_draw.items():
        style = _line_style_2d(label)
        t = _time_min(res)
        y = np.asarray(y_getter(res), dtype=float).reshape(-1)
        n = min(len(t), len(y))
        tt = t[:n]
        yy = y[:n]
        mask = (tt >= zoom_start) & (tt <= min_t_end)
        if np.any(mask):
            ax_inset.plot(tt[mask], yy[mask], **style)
            y_all.extend(yy[mask][np.isfinite(yy[mask])].tolist())

    if ref_value is not None:
        ax_inset.axhline(ref_value, linestyle="--", linewidth=1.4, color="dimgray", alpha=0.95)
        if np.isfinite(ref_value):
            y_all.append(float(ref_value))

    if y_all:
        y_min, y_max = float(np.min(y_all)), float(np.max(y_all))
        margin = (y_max - y_min) * 0.1 if y_max != y_min else 0.1
        ax_inset.set_ylim(y_min - margin, y_max + margin)
    ax_inset.set_xlim(zoom_start, min_t_end)
    _style_inset_axes(ax_inset)

    # Same guide-line style as the second program.
    try:
        ax.indicate_inset_zoom(ax_inset, edgecolor="black", linewidth=1.5)
    except Exception:
        pass
    return ax_inset


# Backward-compatible alias for old calls.
def _add_end_time_inset(
    ax,
    results: Dict[str, dict],
    y_getter,
    ref_value=None,
    zoom_ratio: float = 0.75,
    bounds=(0.55, 0.15, 0.40, 0.40),
):
    return _add_second_style_inset(
        ax,
        results,
        y_getter,
        ref_value=ref_value,
        zoom_ratio=zoom_ratio,
        bounds=bounds,
    )


def plot_l2_norm_four(results: Dict[str, dict], save_dir: str, tag: str):
    """Controller-comparison L2 plots with the same 2D style as the second program.

    H-Hbar and Q-Qbar are drawn in two separate figures. Other logic remains unchanged.
    """
    results = _ordered_results(results)
    results_draw = _ordered_results_for_2d_drawing(results)
    # --------------------------------------------------------
    # Figure 1: ||H-Hbar||_L2
    # --------------------------------------------------------
    fig_h, ax_h = plt.subplots(
        figsize=set_size(433, 0.99, ncols=1, nrows=1, height_add=1.0),
        dpi=600,
    )

    for label, res in results_draw.items():
        style = _line_style_2d(label)
        plot_label = _safe_label_for_plot(label)
        ax_h.plot(_time_min(res), _get_l2(res, "H"), label=plot_label, **style)

    ax_h.axhline(y=0.0, color="dimgray", linestyle="--", alpha=0.95, linewidth=1.8, label="_nolegend_")
    _style_axes_second(
        ax_h,
        xlabel=r"$Time$ [min]",
        ylabel=r"$\Vert H-\bar H\Vert_{L^2}$ [m]",
        legend=True,
    )

    fig_h.tight_layout()
    path_h = os.path.join(save_dir, f"{tag}_seven_algorithms_H_L2_norm_error_second_style.png")
    fig_h.savefig(path_h, dpi=600, bbox_inches="tight")
    print(f"Saved seven-controller H L2 figure to: {path_h}")
    plt.show()

    # --------------------------------------------------------
    # Figure 2: ||Q-Qbar||_L2
    # --------------------------------------------------------
    fig_q, ax_q = plt.subplots(
        figsize=set_size(433, 0.99, ncols=1, nrows=1, height_add=1.0),
        dpi=600,
    )

    for label, res in results_draw.items():
        style = _line_style_2d(label)
        plot_label = _safe_label_for_plot(label)
        ax_q.plot(_time_min(res), _get_l2(res, "Q"), label=plot_label, **style)

    ax_q.axhline(y=0.0, color="dimgray", linestyle="--", alpha=0.95, linewidth=1.8, label="_nolegend_")
    _style_axes_second(
        ax_q,
        xlabel=r"$Time$ [min]",
        ylabel=r"$\Vert Q-\bar Q\Vert_{L^2}$ [m$^3$/s]",
        legend=True,
    )

    # Only Q-Qbar keeps local zoom, as previously requested.
    _add_second_style_inset(
        ax_q,
        results,
        lambda res: _get_l2(res, "Q"),
        ref_value=0.0,
        zoom_ratio=0.75,
        bounds=(0.55, 0.15, 0.40, 0.40),
    )

    fig_q.tight_layout()
    path_q = os.path.join(save_dir, f"{tag}_seven_algorithms_Q_L2_norm_error_second_style.png")
    fig_q.savefig(path_q, dpi=600, bbox_inches="tight")
    print(f"Saved seven-controller Q L2 figure to: {path_q}")
    plt.show()

def _pad_to_length_1d(data, target_len):
    data = np.asarray(data, dtype=float).reshape(-1)
    if len(data) < target_len:
        return np.pad(data, (0, target_len - len(data)), mode="edge")
    return data[:target_len]


def plot_control_input_four(results: Dict[str, dict], save_dir: str, tag: str):
    """U_L(t) plot with the same 2D style as the second program; no inset."""
    results = _ordered_results(results)
    results_draw = _ordered_results_for_2d_drawing(results)
    fig, ax = plt.subplots(
        figsize=set_size(433, 0.99, ncols=1, nrows=1, height_add=1.0),
        dpi=600,
    )

    min_t_end = min(float(np.nanmax(_time_min(res))) for res in results.values())
    for label, res in results_draw.items():
        style = _line_style_2d(label)
        t = _time_min(res)
        U = np.asarray(res["U"], dtype=float)
        n = min(len(t), len(U))
        ax.plot(t[:n], U[:n], label=_safe_label_for_plot(label), **style)

    ax.axhline(y=U_ref, color="dimgray", linestyle="--", alpha=0.95, linewidth=1.8, label="_nolegend_")
    ax.set_xlim(0, min_t_end)
    _style_axes_second(ax, xlabel=r"$Time$ [min]", ylabel=r"$U_L(t)$ [m]", legend=True)

    fig.tight_layout()
    path = os.path.join(save_dir, f"{tag}_seven_algorithms_control_input_second_style.png")
    fig.savefig(path, dpi=600, bbox_inches="tight")
    print(f"Saved seven-controller control input figure to: {path}")
    plt.show()


def plot_boundary_comparison_four(results: Dict[str, dict], save_dir: str, tag: str):
    """Boundary H/Q comparison with the same 2D style as the second program."""
    results = _ordered_results(results)
    results_draw = _ordered_results_for_2d_drawing(results)
    first = next(iter(results.values()))
    Hbar = first["Hbar"]
    Qbar = first["Qbar"]

    fig, axes = plt.subplots(
        2, 2,
        figsize=set_size(780, 0.99, ncols=2, nrows=2, height_add=1.1),
        dpi=600,
        sharex=True,
    )
    axes = axes.ravel()

    specs = [
        ("H", 0, r"$H(0,t)$ [m]", Hbar[0]),
        ("H", -1, r"$H(L,t)$ [m]", Hbar[-1]),
        ("Q", 0, r"$Q(0,t)$ [m$^3$/s]", Qbar[0]),
        ("Q", -1, r"$Q(L,t)$ [m$^3$/s]", Qbar[-1]),
    ]

    for ax, (var, idx, ylabel, ref) in zip(axes, specs):
        for label, res in results_draw.items():
            style = _line_style_2d(label)
            y = np.asarray(res[var])[:, idx]
            t = _time_min(res)
            n = min(len(t), len(y))
            ax.plot(t[:n], y[:n], label=_safe_label_for_plot(label), **style)

        ax.axhline(y=ref, color="dimgray", linestyle="--", alpha=0.95, linewidth=1.8, label="_nolegend_")
        _style_axes_second(ax, ylabel=ylabel, legend=False)

        # Only Q(L,t) keeps local zoom, as previously requested.
        if var == "Q" and idx == -1:
            _add_second_style_inset(
                ax,
                results,
                lambda res, var=var, idx=idx: np.asarray(res[var])[:, idx],
                ref_value=float(ref),
                zoom_ratio=0.75,
                bounds=(0.55, 0.52, 0.40, 0.40),
            )

    _style_axes_second(axes[2], xlabel=r"$Time$ [min]", ylabel=specs[2][2], legend=False)
    _style_axes_second(axes[3], xlabel=r"$Time$ [min]", ylabel=specs[3][2], legend=False)
    _apply_ordered_two_column_legend(
        axes[0],
        fontsize=14,
        frameon=True,
        shadow=True,
        loc="best",
        columnspacing=1.2,
        handlelength=2.8,
        borderaxespad=0.6,
    )

    fig.tight_layout()
    path = os.path.join(save_dir, f"{tag}_seven_algorithms_boundary_comparison_second_style.png")
    fig.savefig(path, dpi=600, bbox_inches="tight")
    print(f"Saved seven-controller boundary figure to: {path}")
    plt.show()


def _format_three_ticklabels(values):
    """Format three x/y tick labels in the same style as the first program's 3D figures."""
    values = np.asarray(values, dtype=float)
    labels = []
    for value in values:
        if abs(value) >= 10:
            labels.append(f"{value:.0f}")
        elif abs(value) >= 1:
            labels.append(f"{value:.1f}")
        else:
            labels.append(f"{value:.2f}")
    return labels

# Maximum number of recorded time slices used only for 3D rendering.
# The simulation data themselves are not filtered or modified.
MAX_3D_TIME_POINTS = 51


def _subsample_3d_time_rows(t_values, Z_values, max_points=MAX_3D_TIME_POINTS):
    """Uniformly subsample recorded time rows for 3D rendering only.

    No H/Q filtering or interpolation is applied. The first and last recorded
    times are always retained. plot_surface still uses rstride=1.
    """
    t_values = np.asarray(t_values, dtype=float).reshape(-1)
    Z_values = np.asarray(Z_values, dtype=float)

    if Z_values.ndim != 2:
        raise ValueError("3D surface data must be a two-dimensional array.")

    n = min(t_values.size, Z_values.shape[0])
    if n <= 0:
        raise ValueError("3D surface data contain no time rows.")

    t_values = t_values[:n]
    Z_values = Z_values[:n, :]

    max_points = max(2, int(max_points))
    if n <= max_points:
        return t_values, Z_values

    indices = np.linspace(0, n - 1, max_points, dtype=int)
    indices = np.unique(indices)
    return t_values[indices], Z_values[indices, :]


def _format_z_ticklabels(values):
    """Format z-axis tick labels in the same style as the first program's 3D figures."""
    values = np.asarray(values, dtype=float)
    span = float(np.nanmax(values) - np.nanmin(values)) if len(values) > 0 else 0.0
    if span >= 10:
        return [f"{v:.0f}" for v in values]
    if span >= 1:
        return [f"{v:.1f}" for v in values]
    return [f"{v:.2f}" for v in values]

def _surface_plot_second_style(res, var: str, title: str, save_path: str):
    """Draw one independent 3D surface using the same style as the first program."""
    X = np.asarray(res["X"], dtype=float)
    t_min = _time_min(res)
    Z = np.asarray(res[var], dtype=float)

    # Use a uniform subset of recorded time rows for 3D rendering only.
    # No state smoothing/filtering is performed. Together with rstride=1,
    # this avoids artificial long-polygon spikes.
    t_min, Z = _subsample_3d_time_rows(t_min, Z)

    spatial = X / 1000.0
    meshx, mesht = np.meshgrid(spatial, t_min)

    fig = plt.figure(figsize=(8.8, 6.6), dpi=600)
    subfigs = fig.subfigures(nrows=1, ncols=1, hspace=0)
    subfig = subfigs
    subfig.subplots_adjust(left=0.18, bottom=0.08, right=0.98, top=0.96)

    axes = subfig.subplots(
        nrows=1,
        ncols=1,
        subplot_kw={"projection": "3d", "computed_zorder": False},
    )

    for axis in [axes.xaxis, axes.yaxis, axes.zaxis]:
        axis._axinfo["axisline"]["linewidth"] = 10
        axis._axinfo["axisline"]["color"] = "b"
        axis._axinfo["grid"]["linewidth"] = 0.8
        axis._axinfo["grid"]["linestyle"] = "--"
        axis._axinfo["grid"]["color"] = "#d1d1d1"
        axis.set_pane_color((1, 1, 1))

    axes.view_init(10, 15)
    axes.set_xlabel(r"$x$ [km]", fontsize=16, labelpad=5, fontweight="bold")
    axes.set_ylabel(r"$Time$ [min]", fontsize=16, labelpad=5, fontweight="bold")
    if var == "H":
        zlabel = r"$H(t,x)$ [m]"
    else:
        zlabel = r"$Q(t,x)$ [m$^3$/s]"
    axes.set_zlabel(zlabel, rotation=90, fontsize=16, labelpad=4, fontweight="bold")
    axes.zaxis.set_rotate_label(False)
    axes.zaxis.label.set_clip_on(False)

    x_ticks = np.array([spatial[0], 0.5 * (spatial[0] + spatial[-1]), spatial[-1]], dtype=float)
    y_ticks = np.array([t_min[0], 0.5 * (t_min[0] + t_min[-1]), t_min[-1]], dtype=float)
    z_min, z_max = float(np.nanmin(Z)), float(np.nanmax(Z))
    z_ticks = np.linspace(z_min, z_max, 5)

    axes.set_xticks(x_ticks)
    axes.set_yticks(y_ticks)
    axes.set_zticks(z_ticks)
    axes.set_xticklabels(_format_three_ticklabels(x_ticks), fontweight="bold", fontsize=16)
    axes.set_yticklabels(_format_three_ticklabels(y_ticks), fontweight="bold", fontsize=16)
    axes.set_zticklabels(_format_z_ticklabels(z_ticks), fontweight="bold", fontsize=16)

    rstride = 1
    surface_cmap = "viridis" if var == "H" else "plasma"
    axes.plot_surface(
        meshx,
        mesht,
        Z,
        edgecolor="black",
        lw=0.8,
        rstride=rstride,
        cstride=5,
        alpha=1,
        cmap=surface_cmap,
        shade=False,
        rasterized=True,
        antialiased=True,
    )

    # Red downstream boundary curve remains removed as in the target program.

    axes.set_title(_safe_label_for_plot(title), fontsize=14)
    if save_path is not None:
        fig.savefig(save_path, dpi=600, bbox_inches="tight", pad_inches=0.35)
        print(f"3D surface saved to: {save_path}")
    plt.show()
    plt.close(fig)

def plot_3d_surfaces_four(results: Dict[str, dict], save_dir: str, tag: str):
    """
    Draw independent 3D figures for each algorithm and each state variable,
    using the same style as the second program. No combined 3D figure is drawn.
    """
    results = _ordered_results(results)
    for var in ["H", "Q"]:
        for label, res in results.items():
            safe_label = label.replace("+", "_plus_").replace(" ", "_").replace("/", "_")
            path = os.path.join(save_dir, f"{tag}_{safe_label}_{var}_3D_surface_second_style.png")
            _surface_plot_second_style(res, var, label, path)


def save_npz_four(results: Dict[str, dict], save_dir: str, tag: str):
    results = _ordered_results(results)
    arrays = {}
    for label, res in results.items():
        safe = label.replace("+", "_plus_").replace(" ", "_").replace("/", "_")
        arrays[f"t_{safe}"] = res["t"]
        arrays[f"H_{safe}"] = res["H"]
        arrays[f"Q_{safe}"] = res["Q"]
        arrays[f"U_{safe}"] = res["U"]
        arrays[f"reward_{safe}"] = res["reward"]
        arrays[f"total_reward_{safe}"] = np.asarray(res["total_reward"])

    first = next(iter(results.values()))
    arrays["X"] = first["X"]
    arrays["Hbar"] = first["Hbar"]
    arrays["Qbar"] = first["Qbar"]

    e_H0_rel, e_Q0_rel = _initial_relative_deviation_percentages_from_result(first)
    arrays["e_H0_rel_pct"] = np.asarray(e_H0_rel)
    arrays["e_Q0_rel_pct"] = np.asarray(e_Q0_rel)

    path = os.path.join(save_dir, f"{tag}_controller_comparison.npz")
    np.savez(path, **arrays)
    print(f"Saved seven-controller data to: {path}")



def print_final_summary_four(results: Dict[str, dict]):
    results = _ordered_results(results)
    print("\n" + "=" * 90)
    print("Final summary of controller comparison")
    print("=" * 90)
    for label, res in results.items():
        H = np.asarray(res["H"])
        Q = np.asarray(res["Q"])
        Hbar = np.asarray(res["Hbar"])
        Qbar = np.asarray(res["Qbar"])
        H_l2_final = _get_l2(res, "H")[-1]
        Q_l2_final = _get_l2(res, "Q")[-1]

        print(f"\n[{label}]")
        print(f"final H(0) = {H[-1,0]:.6f}, target = {Hbar[0]:.6f}, error = {H[-1,0]-Hbar[0]:.6f}")
        print(f"final H(L) = {H[-1,-1]:.6f}, target = {Hbar[-1]:.6f}, error = {H[-1,-1]-Hbar[-1]:.6f}")
        print(f"final Q(0) = {Q[-1,0]:.6f}, target = {Qbar[0]:.6f}, error = {Q[-1,0]-Qbar[0]:.6f}")
        print(f"final Q(L) = {Q[-1,-1]:.6f}, target = {Qbar[-1]:.6f}, error = {Q[-1,-1]-Qbar[-1]:.6f}")
        print(f"final ||H-Hbar||_L2 = {H_l2_final:.6f} m")
        print(f"final ||Q-Qbar||_L2 = {Q_l2_final:.6f} m^3/s")
        print(f"total_reward = {res['total_reward']:.6f}")



# ============================================================
# Evidence-summary table for the selected single scenario
# ============================================================

# Keep the steady-state evaluation window identical to the local enlarged plot.
# The inset starts at 0.75 of the common end time, so the table uses the final 25%.
LOCAL_ZOOM_START_RATIO = 0.75

# Settling bands are constructed from the steady-state mean error:
# threshold = steady_state_mean + 5% * |initial_error - steady_state_mean|.
SETTLING_BAND_FRACTION = 0.05


def _safe_metric_value(value):
    """Return a finite float when possible; otherwise return np.nan."""
    try:
        value = float(value)
    except Exception:
        return np.nan
    if not np.isfinite(value):
        return np.nan
    return value


def _format_metric(value, precision=6):
    """Format one metric for terminal table output."""
    value = _safe_metric_value(value)
    if not np.isfinite(value):
        return "—"
    return f"{value:.{precision}f}"


def _time_average_on_interval(
    t_min: np.ndarray,
    values: np.ndarray,
    start_min: float,
    end_min: float,
) -> float:
    """
    Calculate the time average on the same interval displayed by the local inset.

    The mask is intentionally identical to the inset logic:
        start_min <= t <= end_min.
    """
    t_min = np.asarray(t_min, dtype=float).reshape(-1)
    values = np.asarray(values, dtype=float).reshape(-1)

    n = min(t_min.size, values.size)
    if n == 0:
        return np.nan

    t_min = t_min[:n]
    values = values[:n]

    mask = (
        np.isfinite(t_min)
        & np.isfinite(values)
        & (t_min >= float(start_min))
        & (t_min <= float(end_min))
    )

    t_window = t_min[mask]
    value_window = values[mask]

    if value_window.size == 0:
        return np.nan

    if value_window.size == 1:
        return float(value_window[0])

    duration = float(t_window[-1] - t_window[0])
    if duration <= 0.0:
        return float(np.mean(value_window))

    return float(
        np.trapezoid(value_window, t_window)
        / duration
    )



def _normalized_spatial_l2_state_over_time(res: dict, key: str) -> np.ndarray:
    """
    Return the normalized spatial L2 norm of the state itself:

        ||v(.,t)||_{L2,n}
            = sqrt((1/L) * integral_0^L v(x,t)^2 dx)
            = sqrt(integral_0^1 v(s,t)^2 ds),  s=x/L.

    This is different from _get_l2(), which computes the tracking-error norm
    ||v-vbar||_{L2,n}.
    """
    X = np.asarray(res.get("X", []), dtype=float).reshape(-1)
    state = np.asarray(res.get(key, []), dtype=float)

    if X.size < 2 or state.ndim != 2 or state.shape[1] != X.size:
        return np.asarray([], dtype=float)

    s = X / float(X[-1])
    return np.sqrt(np.trapezoid(state ** 2, s, axis=1))


def _initial_relative_deviation_percentages_from_arrays(
    X: np.ndarray,
    H0: np.ndarray,
    Q0: np.ndarray,
    Hbar: np.ndarray,
    Qbar: np.ndarray,
) -> Tuple[float, float]:
    """
    Compute the shared initial relative deviations:

        e_H0_rel = ||H0-Hbar||_L2 / ||Hbar||_L2 * 100%
        e_Q0_rel = ||Q0-Qbar||_L2 / ||Qbar||_L2 * 100%.

    The same normalized spatial coordinate s=x/L is used in every norm.
    Because numerator and denominator use the same convention, the ratio is
    independent of whether one writes the normalized or unnormalized L2 norm.
    """
    X = np.asarray(X, dtype=float).reshape(-1)
    H0 = np.asarray(H0, dtype=float).reshape(-1)
    Q0 = np.asarray(Q0, dtype=float).reshape(-1)
    Hbar = np.asarray(Hbar, dtype=float).reshape(-1)
    Qbar = np.asarray(Qbar, dtype=float).reshape(-1)

    if (
        X.size < 2
        or H0.size != X.size
        or Q0.size != X.size
        or Hbar.size != X.size
        or Qbar.size != X.size
    ):
        return np.nan, np.nan

    s = X / float(X[-1])

    H_err_norm = float(np.sqrt(np.trapezoid((H0 - Hbar) ** 2, s)))
    Q_err_norm = float(np.sqrt(np.trapezoid((Q0 - Qbar) ** 2, s)))
    H_ref_norm = float(np.sqrt(np.trapezoid(Hbar ** 2, s)))
    Q_ref_norm = float(np.sqrt(np.trapezoid(Qbar ** 2, s)))

    e_H0_rel = (
        100.0 * H_err_norm / H_ref_norm
        if np.isfinite(H_ref_norm) and H_ref_norm > 1e-12
        else np.nan
    )
    e_Q0_rel = (
        100.0 * Q_err_norm / Q_ref_norm
        if np.isfinite(Q_ref_norm) and Q_ref_norm > 1e-12
        else np.nan
    )
    return float(e_H0_rel), float(e_Q0_rel)


def _initial_relative_deviation_percentages_from_result(
    res: dict,
) -> Tuple[float, float]:
    """Read the common t=0 state from one rollout result and compute both percentages."""
    H = np.asarray(res.get("H", []), dtype=float)
    Q = np.asarray(res.get("Q", []), dtype=float)
    if H.ndim != 2 or Q.ndim != 2 or H.shape[0] == 0 or Q.shape[0] == 0:
        return np.nan, np.nan
    return _initial_relative_deviation_percentages_from_arrays(
        X=np.asarray(res.get("X", []), dtype=float),
        H0=H[0],
        Q0=Q[0],
        Hbar=np.asarray(res.get("Hbar", []), dtype=float),
        Qbar=np.asarray(res.get("Qbar", []), dtype=float),
    )



def validate_shared_reference_and_initial_state(
    results: Dict[str, dict],
    X_expected: np.ndarray,
    Hbar_expected: np.ndarray,
    Qbar_expected: np.ndarray,
    H0_expected: np.ndarray,
    Q0_expected: np.ndarray,
    save_dir: str,
    tag: str,
    atol: float = 1e-10,
) -> Dict[str, Dict[str, float]]:
    """Verify that every controller uses exactly the same test problem.

    This check is intentionally strict.  It compares, controller by controller,
    the spatial grid, ODE equilibrium, reset state, and the initial relative-L2
    percentages against the single profiles generated in ``main``.

    A mismatch raises ``RuntimeError`` before figures are produced, preventing a
    visually plausible but unfair comparison from being saved.
    """
    X_expected = np.asarray(X_expected, dtype=float).reshape(-1)
    Hbar_expected = np.asarray(Hbar_expected, dtype=float).reshape(-1)
    Qbar_expected = np.asarray(Qbar_expected, dtype=float).reshape(-1)
    H0_expected = np.asarray(H0_expected, dtype=float).reshape(-1)
    Q0_expected = np.asarray(Q0_expected, dtype=float).reshape(-1)

    expected_eH, expected_eQ = _initial_relative_deviation_percentages_from_arrays(
        X=X_expected,
        H0=H0_expected,
        Q0=Q0_expected,
        Hbar=Hbar_expected,
        Qbar=Qbar_expected,
    )

    report: Dict[str, Dict[str, float]] = {}
    violations = []

    for label, res in results.items():
        X = np.asarray(res.get("X", []), dtype=float).reshape(-1)
        Hbar = np.asarray(res.get("Hbar", []), dtype=float).reshape(-1)
        Qbar = np.asarray(res.get("Qbar", []), dtype=float).reshape(-1)
        H = np.asarray(res.get("H", []), dtype=float)
        Q = np.asarray(res.get("Q", []), dtype=float)
        t = np.asarray(res.get("t", []), dtype=float).reshape(-1)

        if (
            X.shape != X_expected.shape
            or Hbar.shape != Hbar_expected.shape
            or Qbar.shape != Qbar_expected.shape
            or H.ndim != 2
            or Q.ndim != 2
            or H.shape[0] == 0
            or Q.shape[0] == 0
            or H.shape[1] != X_expected.size
            or Q.shape[1] != X_expected.size
            or t.size == 0
        ):
            violations.append(f"{label}: incompatible/missing result-array shapes")
            continue

        metrics = {
            "max_abs_X_difference": float(np.max(np.abs(X - X_expected))),
            "max_abs_Hbar_difference": float(np.max(np.abs(Hbar - Hbar_expected))),
            "max_abs_Qbar_difference": float(np.max(np.abs(Qbar - Qbar_expected))),
            "max_abs_H0_difference": float(np.max(np.abs(H[0] - H0_expected))),
            "max_abs_Q0_difference": float(np.max(np.abs(Q[0] - Q0_expected))),
            "initial_time_s": float(t[0]),
        }
        eH, eQ = _initial_relative_deviation_percentages_from_arrays(
            X=X,
            H0=H[0],
            Q0=Q[0],
            Hbar=Hbar,
            Qbar=Qbar,
        )
        metrics["e_H0_rel_pct"] = float(eH)
        metrics["e_Q0_rel_pct"] = float(eQ)
        metrics["e_H0_rel_difference_pct"] = float(abs(eH - expected_eH))
        metrics["e_Q0_rel_difference_pct"] = float(abs(eQ - expected_eQ))
        report[label] = metrics

        for key in (
            "max_abs_X_difference",
            "max_abs_Hbar_difference",
            "max_abs_Qbar_difference",
            "max_abs_H0_difference",
            "max_abs_Q0_difference",
        ):
            if not np.isfinite(metrics[key]) or metrics[key] > atol:
                violations.append(f"{label}: {key}={metrics[key]:.3e} > {atol:.3e}")

        if abs(metrics["initial_time_s"]) > atol:
            violations.append(
                f"{label}: first recorded time is {metrics['initial_time_s']:.12g} s, not 0 s"
            )
        if metrics["e_H0_rel_difference_pct"] > 1e-9:
            violations.append(
                f"{label}: e_H0_rel differs by {metrics['e_H0_rel_difference_pct']:.3e}%"
            )
        if metrics["e_Q0_rel_difference_pct"] > 1e-9:
            violations.append(
                f"{label}: e_Q0_rel differs by {metrics['e_Q0_rel_difference_pct']:.3e}%"
            )

    # Figure-only mode: keep the strict validation itself, but do not
    # write a validation report or print a validation summary.

    if violations:
        raise RuntimeError(
            "The controller comparison does not use an identical reference/initial state:\n"
            + "\n".join(f"- {item}" for item in violations)
        )

    return report


def save_backstepping_diagnostic_data(
    res_bs: dict,
    save_dir: str,
    tag: str,
) -> None:
    """Save the BS trajectory separately, independent of multi-curve autoscaling.

    The NPZ makes it straightforward to compare this run numerically with an
    earlier two-controller run.  If the same environment file, seed, reference,
    and initial-condition parameters are used, the arrays should match up to
    floating-point roundoff.
    """
    path = Path(save_dir) / f"{tag}_BS_only_diagnostic.npz"
    np.savez(
        path,
        X=np.asarray(res_bs["X"]),
        Hbar=np.asarray(res_bs["Hbar"]),
        Qbar=np.asarray(res_bs["Qbar"]),
        t=np.asarray(res_bs["t"]),
        H=np.asarray(res_bs["H"]),
        Q=np.asarray(res_bs["Q"]),
        U=np.asarray(res_bs["U"]),
        H_l2=_get_l2(res_bs, "H"),
        Q_l2=_get_l2(res_bs, "Q"),
    )
    print(f"Backstepping-only diagnostic data saved to: {path}")


def _normalized_l2_norm_overshoot_percentage(
    res: dict,
    key: str,
) -> float:
    r"""
    Compute the normalized-L2-norm overshoot of a distributed state field:

        M_v = 100 * max_{t>0}
              [ (||v(.,t)||_{L2,n} - ||vbar||_{L2,n})
                / ||vbar||_{L2,n} ]_+,

    where v is H or Q and

        ||v(.,t)||_{L2,n}
            = sqrt((1/L) * integral_0^L v(x,t)^2 dx).

    The common initial sample t=0 is excluded. For the spatially constant
    Qbar used here, ||Qbar||_{L2,n}=Qbar, so this reduces exactly to the
    previously requested Q formula.
    """
    if key not in {"H", "Q"}:
        raise ValueError("key must be 'H' or 'Q'.")

    t_min = _time_min(res)
    state_norm = _normalized_spatial_l2_state_over_time(res, key)
    reference_key = "Hbar" if key == "H" else "Qbar"
    reference = np.asarray(res.get(reference_key, []), dtype=float).reshape(-1)
    X = np.asarray(res.get("X", []), dtype=float).reshape(-1)

    if (
        t_min.size == 0
        or state_norm.size == 0
        or reference.size == 0
        or X.size < 2
        or reference.size != X.size
    ):
        return np.nan

    n = min(t_min.size, state_norm.size)
    t_min = t_min[:n]
    state_norm = state_norm[:n]

    finite_time = t_min[np.isfinite(t_min)]
    if finite_time.size == 0:
        return np.nan

    initial_time = float(np.min(finite_time))
    tolerance = max(1e-12, 1e-10 * max(abs(initial_time), 1.0))
    time_mask = np.isfinite(t_min) & (t_min > initial_time + tolerance)
    if not np.any(time_mask):
        return np.nan

    if not np.all(np.isfinite(reference)):
        return np.nan

    domain_length = float(X[-1] - X[0])
    if not np.isfinite(domain_length) or domain_length <= 0.0:
        return np.nan

    # Use s=(x-x0)/L, so this is the normalized spatial L2 norm and has the
    # same physical unit as the state itself.
    s = (X - X[0]) / domain_length
    reference_norm = float(np.sqrt(np.trapezoid(reference ** 2, s)))
    if not np.isfinite(reference_norm) or reference_norm <= 1e-12:
        return np.nan

    relative_excursion = (
        state_norm[time_mask] - reference_norm
    ) / reference_norm
    relative_excursion = relative_excursion[np.isfinite(relative_excursion)]
    if relative_excursion.size == 0:
        return np.nan

    return float(100.0 * max(0.0, np.max(relative_excursion)))


def _h_normalized_l2_norm_overshoot_percentage(res: dict) -> float:
    """Return M_H using the same normalized-L2-norm definition as M_Q."""
    return _normalized_l2_norm_overshoot_percentage(res, "H")


def _q_normalized_l2_norm_overshoot_percentage(res: dict) -> float:
    """Return M_Q; for constant Qbar this equals the requested Q formula."""
    return _normalized_l2_norm_overshoot_percentage(res, "Q")


def _state_overshoot_percentage(
    res: dict,
    key: str,
) -> float:
    """
    Return the maximum positive state overshoot percentage over the full response.

    The pointwise definition is
        max((state-reference)/abs(reference), 0) * 100%.
    The common initial instant is excluded, so the value describes the
    controller-induced response rather than the prescribed initial condition.
    All subsequent samples in the controller's complete simulation horizon are
    included. This is a state overshoot measure, not an absolute tracking-error
    measure.
    """
    t_min = _time_min(res)
    state = np.asarray(res.get(key, []), dtype=float)
    reference_key = "Hbar" if key == "H" else "Qbar"
    reference = np.asarray(res.get(reference_key, []), dtype=float).reshape(-1)

    if state.ndim != 2 or reference.size != state.shape[1] or t_min.size == 0:
        return np.nan

    n = min(t_min.size, state.shape[0])
    t_min = t_min[:n]
    state = state[:n]

    finite_time = t_min[np.isfinite(t_min)]
    if finite_time.size == 0:
        return np.nan

    initial_time = float(np.min(finite_time))
    tolerance = max(1e-12, 1e-10 * max(abs(initial_time), 1.0))
    time_mask = np.isfinite(t_min) & (t_min > initial_time + tolerance)
    if not np.any(time_mask):
        return np.nan

    denominator = np.abs(reference)
    valid_reference = np.isfinite(reference) & (denominator > 1e-12)
    if not np.any(valid_reference):
        return np.nan

    state_window = state[time_mask][:, valid_reference]
    ref_window = reference[valid_reference].reshape(1, -1)
    denominator = denominator[valid_reference].reshape(1, -1)

    relative_positive_excursion = (state_window - ref_window) / denominator
    relative_positive_excursion = relative_positive_excursion[
        np.isfinite(relative_positive_excursion)
    ]
    if relative_positive_excursion.size == 0:
        return np.nan

    return float(100.0 * max(0.0, np.max(relative_positive_excursion)))


def _peak_spatial_linf(res: dict, key: str) -> float:
    """
    Return the peak spatial L-infinity tracking error over the full simulation.

    At each time instant, the spatial L-infinity norm is
        max_x |state(t, x) - reference(x)|.
    The reported peak then takes the maximum of this norm over all simulated
    time instants, including the initial state:
        max_t max_x |state(t, x) - reference(x)|.
    """
    linf_over_time = np.asarray(_get_linf(res, key), dtype=float).reshape(-1)
    finite_values = linf_over_time[np.isfinite(linf_over_time)]
    if finite_values.size == 0:
        return np.nan
    return float(np.max(finite_values))


def _joint_settling_time(
    t_min: np.ndarray,
    H_l2: np.ndarray,
    Q_l2: np.ndarray,
    steady_H_l2: float,
    steady_Q_l2: float,
    common_end_min: float,
    band_fraction: float = SETTLING_BAND_FRACTION,
) -> float:
    """
    Find the earliest time after which both H and Q L2 errors remain in their
    respective steady-state bands through the common comparison end time.

    H threshold:
        steady_H_l2 + band_fraction * |H_l2(0) - steady_H_l2|

    Q threshold:
        steady_Q_l2 + band_fraction * |Q_l2(0) - steady_Q_l2|
    """
    t_min = np.asarray(t_min, dtype=float).reshape(-1)
    H_l2 = np.asarray(H_l2, dtype=float).reshape(-1)
    Q_l2 = np.asarray(Q_l2, dtype=float).reshape(-1)

    n = min(t_min.size, H_l2.size, Q_l2.size)
    if n == 0:
        return np.nan

    t_min = t_min[:n]
    H_l2 = H_l2[:n]
    Q_l2 = Q_l2[:n]

    valid = (
        np.isfinite(t_min)
        & np.isfinite(H_l2)
        & np.isfinite(Q_l2)
        & (t_min <= float(common_end_min))
    )

    t_eval = t_min[valid]
    H_eval = H_l2[valid]
    Q_eval = Q_l2[valid]

    if (
        t_eval.size == 0
        or not np.isfinite(steady_H_l2)
        or not np.isfinite(steady_Q_l2)
    ):
        return np.nan

    H_threshold = float(
        steady_H_l2
        + float(band_fraction)
        * abs(float(H_eval[0]) - steady_H_l2)
    )
    Q_threshold = float(
        steady_Q_l2
        + float(band_fraction)
        * abs(float(Q_eval[0]) - steady_Q_l2)
    )

    inside_both_bands = (
        (H_eval <= H_threshold)
        & (Q_eval <= Q_threshold)
    )

    # suffix_all[i] is True only when every sample from i to the end remains
    # inside both steady-state bands.
    suffix_all = np.logical_and.accumulate(
        inside_both_bands[::-1]
    )[::-1]

    settled_indices = np.flatnonzero(suffix_all)
    if settled_indices.size == 0:
        return np.nan

    return float(t_eval[settled_indices[0]])



def build_selected_case_evidence_rows(
    results: Dict[str, dict],
    expected_T: Optional[float] = None,
):
    """
    Build the compact quantitative table for the selected scenario.

    Measures:
        1) steady-state mean spatial L2 tracking errors for H and Q;
        2) peak spatial L-infinity tracking errors for H and Q;
        3) H normalized-L2-norm overshoot M_H;
        4) Q normalized-L2-norm overshoot M_Q;
        5) joint H/Q settling time and convergence status.
    """
    results = _ordered_results(results)
    rows = []

    common_end_min = float(_common_time_end_min(results))
    steady_start_min = (
        LOCAL_ZOOM_START_RATIO * common_end_min
        if common_end_min > 0.0
        else np.nan
    )

    for label, res in results.items():
        t = np.asarray(res.get("t", []), dtype=float)
        t_min = t / 60.0 if t.size > 0 else np.asarray([], dtype=float)

        H_l2 = np.asarray(_get_l2(res, "H"), dtype=float)
        Q_l2 = np.asarray(_get_l2(res, "Q"), dtype=float)
        H_linf = np.asarray(_get_linf(res, "H"), dtype=float)
        Q_linf = np.asarray(_get_linf(res, "Q"), dtype=float)

        has_nonfinite = (
            t.size == 0
            or H_l2.size == 0
            or Q_l2.size == 0
            or H_linf.size == 0
            or Q_linf.size == 0
            or not np.all(np.isfinite(t))
            or not np.all(np.isfinite(H_l2))
            or not np.all(np.isfinite(Q_l2))
            or not np.all(np.isfinite(H_linf))
            or not np.all(np.isfinite(Q_linf))
        )

        early_stop = False
        if expected_T is not None and t.size > 0:
            early_stop = bool(t[-1] < float(expected_T) - 1e-9)

        diverged = bool(has_nonfinite or early_stop)

        if diverged:
            steady_H_l2 = np.nan
            steady_Q_l2 = np.nan
            peak_H_linf = np.nan
            peak_Q_linf = np.nan
            H_norm_overshoot_pct = np.nan
            Q_norm_overshoot_pct = np.nan
            settling_time_min = np.nan
            convergence_status = "Diverged"
        else:
            steady_H_l2 = _time_average_on_interval(
                t_min, H_l2, steady_start_min, common_end_min
            )
            steady_Q_l2 = _time_average_on_interval(
                t_min, Q_l2, steady_start_min, common_end_min
            )
            peak_H_linf = _peak_spatial_linf(res, "H")
            peak_Q_linf = _peak_spatial_linf(res, "Q")

            # H and Q now use the same normalized-L2-norm overshoot rule.
            H_norm_overshoot_pct = _h_normalized_l2_norm_overshoot_percentage(res)
            Q_norm_overshoot_pct = _q_normalized_l2_norm_overshoot_percentage(res)

            settling_time_min = _joint_settling_time(
                t_min=t_min,
                H_l2=H_l2,
                Q_l2=Q_l2,
                steady_H_l2=steady_H_l2,
                steady_Q_l2=steady_Q_l2,
                common_end_min=common_end_min,
                band_fraction=SETTLING_BAND_FRACTION,
            )

            convergence_status = (
                "Converged" if np.isfinite(settling_time_min) else "Not settled"
            )

        rows.append({
            "Controller": label,
            "SS mean L2(H) [m]": steady_H_l2,
            "SS mean L2(Q) [m^3/s]": steady_Q_l2,
            "Peak Linf(H) [m]": peak_H_linf,
            "Peak Linf(Q) [m^3/s]": peak_Q_linf,
            "M_H norm overshoot [%]": H_norm_overshoot_pct,
            "M_Q norm overshoot [%]": Q_norm_overshoot_pct,
            "Settling time [min]": settling_time_min,
            "Convergence status": convergence_status,
        })

    return rows




def print_selected_case_evidence_summary(
    results: Dict[str, dict],
    H_left: float,
    H_right: float,
    Q_value: float,
    seed: int,
    dH_source=None,
):
    """Print the evidence-summary table for the shared seven-controller scenario."""
    rows = build_selected_case_evidence_rows(
        results,
        expected_T=SAC_T,
    )

    headers = [
        "Controller",
        "SS mean L2(H) [m]",
        "SS mean L2(Q) [m^3/s]",
        "Peak Linf(H) [m]",
        "Peak Linf(Q) [m^3/s]",
        "M_H norm overshoot [%]",
        "M_Q norm overshoot [%]",
        "Settling time [min]",
        "Convergence status",
    ]

    common_end_min = float(_common_time_end_min(results))
    steady_start_min = (
        LOCAL_ZOOM_START_RATIO * common_end_min
        if common_end_min > 0.0
        else np.nan
    )
    steady_percentage = 100.0 * (1.0 - LOCAL_ZOOM_START_RATIO)

    first_result = next(iter(results.values())) if results else None
    if first_result is not None:
        H0 = np.asarray(first_result["H"], dtype=float)[0]
        Q0 = np.asarray(first_result["Q"], dtype=float)[0]
        Hbar0 = np.asarray(first_result["Hbar"], dtype=float)
        Qbar0 = np.asarray(first_result["Qbar"], dtype=float)
        actual_H = float(np.max(np.abs(H0 - Hbar0)))
        actual_Q = float(np.max(np.abs(Q0 - Qbar0)))
        e_H0_rel, e_Q0_rel = _initial_relative_deviation_percentages_from_result(
            first_result
        )
    else:
        actual_H = np.nan
        actual_Q = np.nan
        e_H0_rel = np.nan
        e_Q0_rel = np.nan

    scenario = (
        "Selected linear initial condition: "
        f"H(0)={H_left:g} m, H(L)={H_right:g} m, "
        f"Q(x,0)={Q_value:g} m^3/s, seed={seed}, "
        f"max deviations from ODE equilibrium="
        f"({actual_H:.6f} m, {actual_Q:.6f} m^3/s)"
    )
    if dH_source is not None:
        scenario += f", dH_source={dH_source:g}"

    printable_rows = []
    for row in rows:
        printable_rows.append([
            str(row["Controller"]),
            _format_metric(row["SS mean L2(H) [m]"]),
            _format_metric(row["SS mean L2(Q) [m^3/s]"]),
            _format_metric(row["Peak Linf(H) [m]"]),
            _format_metric(row["Peak Linf(Q) [m^3/s]"]),
            _format_metric(row["M_H norm overshoot [%]"], precision=3),
            _format_metric(row["M_Q norm overshoot [%]"], precision=3),
            _format_metric(row["Settling time [min]"]),
            str(row["Convergence status"]),
        ])

    widths = []
    for i, header in enumerate(headers):
        widths.append(
            max(
                len(header),
                max(len(r[i]) for r in printable_rows),
            )
        )

    header_line = " | ".join(
        header.ljust(widths[i])
        for i, header in enumerate(headers)
    )
    sep_line = "-+-".join(
        "-" * widths[i]
        for i in range(len(headers))
    )
    table_width = max(120, len(header_line))

    print("\n" + "=" * table_width)
    print("Closed-loop performance table for the selected single scenario")
    print(scenario)
    print(
        "Shared initial relative deviations: "
        f"e_H,0^rel={e_H0_rel:.6f}%, "
        f"e_Q,0^rel={e_Q0_rel:.6f}%."
    )
    print(
        "Steady-state averaging window "
        f"(same as local zoom): "
        f"[{steady_start_min:.6f}, {common_end_min:.6f}] min, "
        f"final {steady_percentage:.1f}% of the common horizon."
    )
    print(
        "Peak Linf is max_t max_x |state-reference| over the complete "
        "simulation horizon. H and Q use the same normalized-L2-norm "
        "overshoot definition: "
        "M_H=100*max_{t>0}[(||H(.,t)||_{L2,n}-||Hbar||_{L2,n})/"
        "||Hbar||_{L2,n}]_+ and "
        "M_Q=100*max_{t>0}[(||Q(.,t)||_{L2,n}-Qbar)/Qbar]_+. "
        "Settling requires both H and Q L2 tracking errors to remain inside "
        f"their {100.0 * SETTLING_BAND_FRACTION:.1f}% steady-state bands."
    )
    print("=" * table_width)

    print(header_line)
    print(sep_line)
    for row in printable_rows:
        print(
            " | ".join(
                row[i].ljust(widths[i])
                for i in range(len(headers))
            )
        )
    print("=" * table_width)



# ============================================================
# Main: seven-controller comparison
# ============================================================


DEFAULT_NEW_MODEL_PATH = (
     "./sactx/seed_42/"
    "sac_cac_hybrid_yul_ode_cheb_raw_HQ_100000_steps"
) 
DEFAULT_TDPSAC_FPC_MODEL_PATH = (
    "./sacddx/seed_42/"
    "sac_cac_hybrid_fixed_control_yul_ode_cheb_raw_HQ_100000_steps" 
)
DEFAULT_DPSAC_MODEL_PATH = (
    "./saccontrolx/seed_42/"
    "sac_cac_new_hadamard_hybrid_all_trainable_raw_HQ_100000_steps"
)
DEFAULT_DO_SAC_MODEL_PATH = (
    "./sactrainx/seed_42/"
    "sac_cac_deeponet_feature_raw_hq_ode_cheb_100000_steps"
)
DEFAULT_SAC_FPC_MODEL_PATH = (
    "./sacnoxv/seed_42/"
    "sac_fnn_frozen_new_deeponet_prior_raw_HQ_100000_steps"
)
# Newly trained ordinary SAC checkpoint using raw physical H/Q observations.
DEFAULT_SAC_MODEL_PATH = (
    "./saczx/seed_42/"
    "sac_cac_raw_hq_ode_cheb_100000_steps")
DEFAULT_SAVE_DIR_COMPARISON = "./test_results_seven_algorithms_new_hadamard_linear_IC"

# Display/run order follows the old multi-controller comparison hierarchy.
LEARNED_CONTROLLER_SPECS = [
    {
        "label": LABEL_SAC_CURVE,          # SAC: newly trained raw-HQ model
        "model_kind": "sac",
        "required": True,
    },
    {
        "label": LABEL_FP_SAC,             # SAC-FPC
        "model_kind": "auto",
        "required": True,
    },
    {
        "label": LABEL_SAC,                # DO-SAC
        "model_kind": "auto",
        "required": True,
    },
    {
        "label": LABEL_HYBRID,             # DPSAC
        "model_kind": "auto",
        "required": True,
    },
    {
        "label": LABEL_DFP_DON_SAC,        # TDPSAC-FPC
        "model_kind": "auto",
        "required": True,
    },
    {
        "label": LABEL_NEW_MODEL,           # TDPSAC
        "model_kind": "hybrid",
        "required": True,
    },
]


def _run_optional_controller(
    results: Dict[str, dict],
    *,
    label: str,
    model_path: str,
    model_kind: str,
    seed: int,
    H_left: float,
    H_right: float,
    Q_value: float,
    dH_source,
    required: bool = False,
):
    """Run one learned controller and keep the common result-data interface."""
    if not _checkpoint_exists(model_path):
        message = (
            f"No valid SB3 ZIP checkpoint was found for {label}.\n"
            f"Current path: {model_path!r}\n"
            "Expected either '/path/model' with '/path/model.zip' present, "
            "or a direct '/path/model.zip' path."
        )
        if required:
            raise FileNotFoundError(message)
        print(f"\n[SKIP] {message}")
        return

    print(f"\nRunning {label}...")
    results[label] = run_policy_exact(
        model_path=model_path,
        model_kind=model_kind,
        label=label,
        seed=seed,
        H_left=H_left,
        H_right=H_right,
        Q_value=Q_value,
        dH_source=dH_source,
    )


def _clean_runtime_argv(argv=None):
    """Remove Jupyter/IPython kernel connection arguments before argparse."""
    raw = list(sys.argv[1:] if argv is None else argv)
    cleaned = []
    i = 0

    while i < len(raw):
        arg = str(raw[i])

        if arg in {"-f", "--f"} and i + 1 < len(raw):
            candidate = str(raw[i + 1])
            if candidate.endswith(".json") and (
                "jupyter" in candidate.lower()
                or "kernel-" in Path(candidate).name.lower()
            ):
                i += 2
                continue

        if arg.startswith(("--f=", "-f=")):
            candidate = arg.split("=", 1)[1]
            if candidate.endswith(".json") and (
                "jupyter" in candidate.lower()
                or "kernel-" in Path(candidate).name.lower()
            ):
                i += 1
                continue

        if arg.endswith(".json") and (
            "jupyter" in arg.lower()
            or "kernel-" in Path(arg).name.lower()
        ):
            i += 1
            continue

        cleaned.append(arg)
        i += 1

    return cleaned



def main():
    parser = argparse.ArgumentParser(
        allow_abbrev=False,
        description=(
            "Compare BS, new raw-HQ SAC, SAC-FPC, DO-SAC, DPSAC, "
            "TDPSAC-FPC, and TDPSAC under one shared ODE-equilibrium/linear-IC test."
        ),
    )
    parser.add_argument(
        "--new-model",
        "--tdpsac-model",
        dest="new_model",
        type=str,
        default=DEFAULT_NEW_MODEL_PATH,
        help="Checkpoint of the latest partially frozen TDPSAC model.",
    )
    parser.add_argument(
        "--tdpsac-fpc-model",
        type=str,
        default=DEFAULT_TDPSAC_FPC_MODEL_PATH,
        help="Checkpoint of TDPSAC-FPC with an independent frozen control prior.",
    )
    parser.add_argument(
        "--dpsac-model",
        type=str,
        default=DEFAULT_DPSAC_MODEL_PATH,
        help="Checkpoint of the all-trainable Hadamard DeepONet-SAC model.",
    )
    parser.add_argument(
        "--do-sac-model",
        type=str,
        default=DEFAULT_DO_SAC_MODEL_PATH,
        help="Checkpoint of the DeepONet-feature-only SAC model.",
    )
    parser.add_argument(
        "--sac-fpc-model",
        type=str,
        default=DEFAULT_SAC_FPC_MODEL_PATH,
        help="Checkpoint of ordinary SAC with a frozen DeepONet action prior.",
    )
    parser.add_argument(
        "--sac-model",
        type=str,
        default=DEFAULT_SAC_MODEL_PATH,
        help=(
            "Checkpoint of the newly trained ordinary SAC model using "
            "raw physical H/Q observations."
        ),
    )
    parser.add_argument(
        "--out-dir",
        type=str,
        default=DEFAULT_SAVE_DIR_COMPARISON,
    )
    parser.add_argument(
        "--seed",
        type=int,
        default=123,
        help="Shared deterministic environment/model evaluation seed.",
    )
    parser.add_argument(
        "--H-left",
        type=float,
        default=TEST_H_LEFT,
        help="Initial upstream water depth H(0,0) [m].",
    )
    parser.add_argument(
        "--H-right",
        type=float,
        default=TEST_H_RIGHT,
        help="Initial downstream water depth H(L,0) [m].",
    )
    parser.add_argument(
        "--Q-value",
        type=float,
        default=TEST_Q_VALUE,
        help="Spatially constant initial discharge Q(x,0) [m^3/s].",
    )
    parser.add_argument("--dH-source", type=float, default=None)
    parser.add_argument(
        "--skip-3d",
        action="store_true",
        help="Skip H/Q 3D surface plots.",
    )

    args, unknown_args = parser.parse_known_args(_clean_runtime_argv())
    if unknown_args:
        print(f"Ignoring unrecognized runtime arguments: {unknown_args}")

    save_dir = Path(args.out_dir)
    save_dir.mkdir(parents=True, exist_ok=True)
    globals()["SAVE_DIR"] = str(save_dir)

    dH_tag = "default" if args.dH_source is None else f"{args.dH_source:.0e}"
    tag = (
        f"case_linear_H{args.H_left:g}_{args.H_right:g}_"
        f"Q{args.Q_value:g}_seed{args.seed}_src{dH_tag}_"
        "seven_algorithm_comparison"
    )
    tag = tag.replace(".", "p").replace("-", "m").replace("+", "")

    # Build and report the exact ODE reference and the one common initial state.
    p_ref, X_ref, dx_ref, Hbar_ref, Qbar_ref = build_collection_matched_reference()
    H0_ref, Q0_ref, ic_details = generate_test_initial_condition(
        X_ref,
        Hbar_ref,
        Qbar_ref,
        H_left=args.H_left,
        H_right=args.H_right,
        Q_value=args.Q_value,
    )
    e_H0_rel, e_Q0_rel = _initial_relative_deviation_percentages_from_arrays(
        X=X_ref,
        H0=H0_ref,
        Q0=Q0_ref,
        Hbar=Hbar_ref,
        Qbar=Qbar_ref,
    )

    print("=" * 110)
    print("Seven-controller Saint-Venant comparison")
    print("=" * 110)
    print(f"1. {LABEL_BACKSTEPPING}: analytical controller")
    print(f"2. {LABEL_SAC_CURVE}: {args.sac_model}")
    print(f"3. {LABEL_FP_SAC}: {args.sac_fpc_model}")
    print(f"4. {LABEL_SAC}: {args.do_sac_model}")
    print(f"5. {LABEL_HYBRID}: {args.dpsac_model}")
    print(f"6. {LABEL_DFP_DON_SAC}: {args.tdpsac_fpc_model}")
    print(f"7. {LABEL_NEW_MODEL}: {args.new_model}")
    print(f"Test horizon: {SAC_T} s")
    print(
        "Policy observations: all learned controllers use raw physical "
        f"[H,Q]. Dimension={RAW_STATE_DIM}."
    )
    print(
        "ODE equilibrium: "
        f"Hbar(0)={Hbar_ref[0]:.9f}, "
        f"Hbar(L)={Hbar_ref[-1]:.9f}, "
        f"Qbar={Qbar_ref[0]:.9f}, dx={dx_ref:.6f}"
    )
    print(
        "Linear test initial condition: "
        f"H(x,0)={args.H_left:g}+({args.H_right:g}-{args.H_left:g})x/L m, "
        f"Q(x,0)={args.Q_value:g} m^3/s"
    )
    print(
        "Maximum initial deviations from the ODE equilibrium: "
        f"max|H0-Hbar|={ic_details['H_realized_max_deviation']:.9f} m, "
        f"max|Q0-Qbar|={ic_details['Q_realized_max_deviation']:.9f} m^3/s"
    )
    print(
        "Initial-state ranges: "
        f"H0 in [{np.min(H0_ref):.9f}, {np.max(H0_ref):.9f}] m, "
        f"Q0 in [{np.min(Q0_ref):.9f}, {np.max(Q0_ref):.9f}] m^3/s"
    )
    print(
        "Initial relative deviations: "
        f"e_H,0^rel={e_H0_rel:.9f}%, "
        f"e_Q,0^rel={e_Q0_rel:.9f}%"
    )
    print(
        "H/Q overshoot used in the table: "
        "M_H=100*max_{t>0}[(||H(.,t)||_{L2,n}-||Hbar||_{L2,n})/"
        "||Hbar||_{L2,n}]_+; "
        "M_Q=100*max_{t>0}[(||Q(.,t)||_{L2,n}-Qbar)/Qbar]_+."
    )
    print(f"Source disturbance: {args.dH_source}")
    print(f"Output directory: {save_dir}")
    print(
        "RL control low-pass filter: "
        f"alpha={RL_CONTROL_FILTER_ALPHA:.3f} "
        "(all learned controllers; Backstepping unfiltered)"
    )
    print(f"3D time rendering: at most {MAX_3D_TIME_POINTS} uniformly sampled recorded time points, rstride=1.")
    print("=" * 110)

    results: Dict[str, dict] = {}

    controller_paths = {
        LABEL_NEW_MODEL: args.new_model,
        LABEL_DFP_DON_SAC: args.tdpsac_fpc_model,
        LABEL_HYBRID: args.dpsac_model,
        LABEL_FP_SAC: args.sac_fpc_model,
        LABEL_SAC: args.do_sac_model,
        LABEL_SAC_CURVE: args.sac_model,
    }

    # Run and insert controllers in the exact order requested for 2D legends and
    # all quantitative tables.
    print(f"\nRunning {LABEL_BACKSTEPPING}...")
    results[LABEL_BACKSTEPPING] = run_backstepping_for_four(
        seed=args.seed,
        H_left=args.H_left,
        H_right=args.H_right,
        Q_value=args.Q_value,
        dH_source=args.dH_source,
    )

    for spec in LEARNED_CONTROLLER_SPECS:
        _run_optional_controller(
            results,
            label=spec["label"],
            model_path=controller_paths[spec["label"]],
            model_kind=spec["model_kind"],
            required=bool(spec.get("required", False)),
            seed=args.seed,
            H_left=args.H_left,
            H_right=args.H_right,
            Q_value=args.Q_value,
            dH_source=args.dH_source,
        )

    results = _ordered_results(results)

    # Strictly verify that adding more algorithms did not change the physical
    # test problem seen by Backstepping or any learned controller.
    validate_shared_reference_and_initial_state(
        results=results,
        X_expected=X_ref,
        Hbar_expected=Hbar_ref,
        Qbar_expected=Qbar_ref,
        H0_expected=H0_ref,
        Q0_expected=Q0_ref,
        save_dir=str(save_dir),
        tag=tag,
    )
    # Figure-only mode: do not save NPZ/DAT/TXT summaries or print tables.
    plot_boundary_comparison_four(results, str(save_dir), tag)
    plot_control_input_four(results, str(save_dir), tag)
    plot_l2_norm_four(results, str(save_dir), tag)

    if not args.skip_3d:
        plot_3d_surfaces_four(results, str(save_dir), tag)



if __name__ == "__main__":
    # Suppress stdout emitted by Stable-Baselines3/model-loading helpers.
    # Figures are still saved and displayed normally.
    with contextlib.redirect_stdout(io.StringIO()):
        main()


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Unified controller comparison for Saint-Venant canal regulation.

Purpose:
    Compare six learned controllers under one shared Saint-Venant test condition:
    TDPSAC, TDPSAC-FPC, DPSAC, DO-SAC, SAC-FPC, and SAC. The retained 2D
    comparison figures contain only these six learned controllers. In addition,
    independent colored 3D H(x,t)/Q(x,t) surfaces are generated for the six
    learned controllers and analytical Backstepping.

Method mapping:
    1) TDPSAC:
       the latest partially frozen Hadamard DeepONet-SAC controller.
    2) TDPSAC-FPC:
       DeepONet actor/critic features plus an independent frozen prior controller.
    3) DPSAC:
       Hadamard DeepONet-SAC with all DeepONet parameters fine-tuned.
    4) DO-SAC:
       Hadamard DeepONet is used only as the SAC feature extractor.
    5) SAC-FPC:
       ordinary SAC actor/critic plus an independent frozen DeepONet prior.
    6) SAC:
       newly trained ordinary SAC without DeepONet, using the raw physical H/Q state.

All learned controllers receive the raw physical state
    [H_1,...,H_N,Q_1,...,Q_N],
with the same ODE equilibrium and the same physical initial condition.

Reference and test initial condition:
    - Qbar(x) is fixed at 11.93 m^3/s.
    - Hbar(L) is obtained from the downstream equilibrium gate relation.
    - Hbar(x) is obtained by solving the steady Saint-Venant ODE backward from
      x=L to x=0 with scipy.integrate.solve_ivp.
    - Every controller uses the same deterministic linear initial condition:

          H(x,0) = 3.75 + (4.65 - 3.75) x/L,
          Q(x,0) = 10.0.

      The initial condition is independent of the random seed. The seed is kept
      only for deterministic environment/model evaluation.

Extension:
    Additional learned controllers can be added later through
    LEARNED_CONTROLLER_SPECS without changing the retained 2D plotting functions.
"""

from __future__ import annotations

import argparse
import contextlib
import io
import os
import sys
import warnings
import zipfile
from pathlib import Path
from typing import Tuple, Dict, Optional

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from gymnasium import spaces
from scipy.integrate import solve_ivp
from stable_baselines3 import SAC
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from stable_baselines3.sac.policies import Actor, SACPolicy


# ============================================================
# Figure-only silent-output mode
# ============================================================
# Hide normal terminal messages and third-party warnings.
# Exceptions/tracebacks are not suppressed.
warnings.filterwarnings("ignore")


def print(*args, **kwargs):
    return None


from cac_saint_venant_env_no_bs import (
    CACParams,
    CACSaintVenantPDE1D,
    source,
)

try:
    from pde_control_gym.src.rewards.base_reward import BaseReward
except Exception:
    class BaseReward:
        pass


# ============================================================
# Common physical / reward settings
# ============================================================

N = 102
dt = 5.0
control_sample_rate = 400.0

max_state_value = 1e6

# The learned controller emits a normalized action in [-1,1]. The environment
# maps it to this physical downstream-gate interval. These limits are identical
# to the corresponding raw-HQ training program.
min_control_value = 0.15
max_control_value = 0.40

# H and Q remain raw physical states in the policy observation. These scales are
# used only inside the reward calculation.
H_scale = 0.30
Q_scale = 3.00
U_scale = 0.10
obs_clip = 50.0

cfl = 0.45
check_cfl = True
well_balanced = True

# Collection-matched equilibrium.
Q_BAR_FIXED = 11.93
REFERENCE_MODE = "ode"
U_ref = 0.30

# First-order low-pass filter applied to the CONTROL COMMAND of every RL controller.
# The filter is applied before env.step(), i.e. to the actual closed-loop input,
# not to H/Q plotting data. Backstepping is intentionally left unfiltered.
#
#   U_k^f = (1-alpha_f) U_{k-1}^f + alpha_f U_k^RL
#
# alpha_f = 1.0 -> no filtering; smaller values -> stronger smoothing.
RL_CONTROL_FILTER_ALPHA = 0.70


# Shared deterministic linear test initial condition.
TEST_H_LEFT = 3.54
TEST_H_RIGHT = 4.44
TEST_Q_VALUE =9.54
# Reward used by the new training program:
# r_k = -alpha*tracking - beta*state_rate
#       -gamma*control - delta*delta_control.
reward_alpha = 1.0
reward_beta = 0.10
reward_gamma = 0.05
reward_delta = 0.10
reward_w_H = 1.0
reward_w_Q = 1.0
terminal_tracking_weight = 0
truncate_penalty = -200.0

# Keep the test horizon consistent with the previous unified comparison program.
SAC_T = 80000.0
BS_T = SAC_T

BACKSTEPPING_GAIN = 1.0

learning_rate = 1e-4
deeponet_learning_rate = 1e-6

LATENT_DIM = 102
RAW_STATE_DIM = 2 * N
SAC_FEATURE_DIM = LATENT_DIM

# This file is only needed while the policy object is being constructed. The
# trained SB3 checkpoint subsequently restores all actor/critic/DeepONet weights.
DEEPNET_PRETRAINED_PATH = "modedlNet10000QqQqQ14444_hadamard.pt"

# ============================================================
# New Hadamard DeepONet controller and Hybrid SAC modules
# ============================================================

def preprocess_obs(obs: torch.Tensor) -> torch.Tensor:
    if not isinstance(obs, torch.Tensor):
        obs = torch.as_tensor(obs)
    return obs.float()


class BranchNet(nn.Module):
    def __init__(self, shape=N, output_dim=LATENT_DIM):
        super().__init__()
        self.shape = int(shape)
        self.output_dim = int(output_dim)

        self.conv1 = nn.Conv1d(
            in_channels=2,
            out_channels=16,
            kernel_size=5,
            stride=2,
        )
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv1d(
            in_channels=16,
            out_channels=32,
            kernel_size=5,
            stride=2,
        )

        with torch.no_grad():
            tmp = torch.zeros(1, 2, self.shape)
            tmp = self.relu(self.conv1(tmp))
            tmp = self.relu(self.conv2(tmp))
            flat_dim = tmp.reshape(1, -1).shape[1]

        self.fc1 = nn.Linear(flat_dim, self.output_dim)

    def forward(self, x):
        x = preprocess_obs(x)
        if x.dim() == 1:
            x = x.unsqueeze(0)

        if x.shape[1] != 2 * self.shape:
            raise ValueError(
                f"Branch input dimension is {x.shape[1]}, expected {2 * self.shape}."
            )

        x = x.reshape(x.shape[0], 2, self.shape)
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = torch.flatten(x, start_dim=1)
        x = self.relu(self.fc1(x))
        return x


class TrunkNet(nn.Module):
    def __init__(self, dim_x=1, width=LATENT_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim_x, 16),
            nn.ReLU(),
            nn.Linear(16, 32),
            nn.ReLU(),
            nn.Linear(32, width),
            nn.ReLU(),
        )

    def forward(self, x):
        return self.net(x.float())


class DeepONetModified(nn.Module):
    """
    Hadamard-fusion DeepONet controller.

        b(s) in R^p,
        t(x) in R^p,
        z(s,x) = b(s) * t(x) in R^p,
        U_prior = f_ctrl(z) in R.
    """

    def __init__(self, shape=N, latent_dim=LATENT_DIM):
        super().__init__()
        self.latent_dim = int(latent_dim)

        self.branch = BranchNet(shape, output_dim=self.latent_dim)
        self.trunk = TrunkNet(dim_x=1, width=self.latent_dim)

        # Hadamard fusion preserves latent_dim features.
        self.fc3 = nn.Linear(self.latent_dim, 32)
        self.relu = nn.ReLU()
        self.fc4 = nn.Linear(32, 16)
        self.fc5 = nn.Linear(16, 1)

    def extract_latent(self, obs, grid):
        b = self.branch(obs)       # [batch, latent_dim]
        t = self.trunk(grid)       # [n_query, latent_dim]

        if t.shape[0] == 1:
            t = t.expand(b.shape[0], -1)
        elif t.shape[0] != b.shape[0]:
            raise ValueError(
                f"Branch batch {b.shape[0]} and trunk batch {t.shape[0]} do not match."
            )

        z = b * t                  # Hadamard product
        return z

    def control_head(self, z):
        y = self.relu(self.fc3(z))
        y = self.relu(self.fc4(y))
        y = self.fc5(y)
        return y

    def forward(self, obs, grid, return_latent=False):
        z = self.extract_latent(obs, grid)
        u_prior_physical = self.control_head(z)

        if return_latent:
            return u_prior_physical, z

        return u_prior_physical


class HybridCACDeepONetFeatureExtractor(BaseFeaturesExtractor):
    """
    Return the Hadamard-fused DeepONet latent feature.

    Input observation and DeepONet input:
        [H_1,...,H_N,Q_1,...,Q_N]

    SAC feature:
        [z_1,...,z_p]
    """

    def __init__(
        self,
        observation_space: spaces.Box,
        features_dim: int = SAC_FEATURE_DIM,
        pretrained_path: str = DEEPNET_PRETRAINED_PATH,
        query_point: float = 1.0,
        freeze_deeponet: bool = False,
    ):
        if features_dim != SAC_FEATURE_DIM:
            raise ValueError(
                f"features_dim must be {SAC_FEATURE_DIM}, got {features_dim}."
            )

        super().__init__(observation_space, features_dim)

        self.query_point = float(query_point)
        self.net1 = DeepONetModified(N, latent_dim=LATENT_DIM)

        # Normalized action used by the hybrid SAC actor.
        self.u_deep_norm: Optional[torch.Tensor] = None

        # Physical prior gate retained for diagnostics.
        self.u_prior_physical: Optional[torch.Tensor] = None

        self.register_buffer(
            "grid",
            torch.tensor([[self.query_point]], dtype=torch.float32),
        )

        self._load_pretrained(pretrained_path)
        self._apply_freeze(freeze_all=freeze_deeponet)

    def _apply_freeze(self, freeze_all=False):
        if freeze_all:
            for param in self.net1.parameters():
                param.requires_grad = False
            print("All DeepONet parameters are frozen.")
            return

        # Keep the same transfer-learning structure used during training.
        for param in self.net1.branch.conv1.parameters():
            param.requires_grad = False
        for param in self.net1.branch.conv2.parameters():
            param.requires_grad = False

        for param in self.net1.branch.fc1.parameters():
            param.requires_grad = True

        for param in self.net1.trunk.parameters():
            param.requires_grad = False

        last_linear = None
        for module in reversed(list(self.net1.trunk.net)):
            if isinstance(module, nn.Linear):
                last_linear = module
                break

        if last_linear is not None:
            for param in last_linear.parameters():
                param.requires_grad = True

        for layer in [self.net1.fc3, self.net1.fc4, self.net1.fc5]:
            for param in layer.parameters():
                param.requires_grad = True

    @staticmethod
    def _clean_state_dict_keys(state_dict):
        cleaned = {}
        prefixes = ["module.", "model.", "net1.", "deeponet."]

        for key, value in state_dict.items():
            new_key = key
            changed = True
            while changed:
                changed = False
                for prefix in prefixes:
                    if new_key.startswith(prefix):
                        new_key = new_key[len(prefix):]
                        changed = True
            cleaned[new_key] = value

        return cleaned

    def _load_pretrained(self, pretrained_path: str):
        path = Path(pretrained_path)

        if not path.exists():
            raise FileNotFoundError(
                f"Pretrained Hadamard DeepONet checkpoint not found: {path}"
            )

        state_dict = torch.load(str(path), map_location="cpu")

        if isinstance(state_dict, dict) and "state_dict" in state_dict:
            state_dict = state_dict["state_dict"]

        if not isinstance(state_dict, dict):
            raise TypeError(
                f"Unsupported checkpoint type: {type(state_dict).__name__}"
            )

        cleaned = self._clean_state_dict_keys(state_dict)

        try:
            self.net1.load_state_dict(cleaned, strict=True)
        except RuntimeError as exc:
            raise RuntimeError(
                "The checkpoint is incompatible with the Hadamard-fusion "
                "DeepONet. Re-pretrain DeepONet with fc3=Linear(102,32) and "
                "z=b*t before running this test script.\n"
                f"Original error:\n{exc}"
            ) from exc

        print(f"Loaded pretrained Hadamard DeepONet: {path}")

    @staticmethod
    def _physical_gate_to_normalized_action(u_physical):
        scale = max_control_value - min_control_value
        if scale <= 0.0:
            raise ValueError(
                "max_control_value must be greater than min_control_value."
            )
        return 2.0 * (u_physical - min_control_value) / scale - 1.0

    def forward(self, observations: torch.Tensor) -> torch.Tensor:
        obs = preprocess_obs(observations)
        if obs.dim() == 1:
            obs = obs.unsqueeze(0)

        if obs.shape[1] != RAW_STATE_DIM:
            raise ValueError(
                f"Raw observation dimension is {obs.shape[1]}, "
                f"expected {RAW_STATE_DIM}."
            )

        u_prior_physical, z = self.net1(
            obs,
            self.grid.to(obs.device),
            return_latent=True,
        )

        # Clip the physical DeepONet prior before mapping it to [-1,1].
        u_prior_physical = torch.clamp(
            u_prior_physical,
            min_control_value,
            max_control_value,
        )
        self.u_prior_physical = u_prior_physical

        u_deep_norm = self._physical_gate_to_normalized_action(u_prior_physical)
        self.u_deep_norm = torch.clamp(u_deep_norm, -1.0, 1.0)

        # The SAC representation contains only the fused DeepONet feature z.
        return z.view(z.size(0), -1)


class HybridCACActor(Actor):
    """SAC actor whose normalized output is blended with DeepONet prior action."""

    def __init__(self, *args, alpha: float = 0.4, **kwargs):
        super().__init__(*args, **kwargs)
        init_alpha = float(np.clip(alpha, 0.0, 1.0))

        self.register_buffer(
            "alpha_tensor",
            torch.tensor(init_alpha, dtype=torch.float32),
        )

        self.last_deep_action = None
        self.last_rl_action = None
        self.last_hybrid_action = None

    @property
    def alpha(self) -> float:
        return float(self.alpha_tensor.detach().cpu().item())

    def set_alpha(self, alpha: float):
        alpha_value = float(np.clip(alpha, 0.0, 1.0))
        with torch.no_grad():
            self.alpha_tensor.fill_(alpha_value)

    def _deep_action(self, obs: torch.Tensor) -> torch.Tensor:
        u_deep = getattr(self.features_extractor, "u_deep_norm", None)

        if u_deep is None or u_deep.shape[0] != obs.shape[0]:
            _ = self.features_extractor(obs)
            u_deep = self.features_extractor.u_deep_norm

        return torch.clamp(u_deep, -1.0, 1.0)

    def _hybridize(self, rl_action: torch.Tensor, obs: torch.Tensor) -> torch.Tensor:
        u_deep = self._deep_action(obs)

        alpha = self.alpha_tensor.to(
            device=rl_action.device,
            dtype=rl_action.dtype,
        )

        hybrid = (1.0 - alpha) * u_deep + alpha * rl_action
        hybrid = torch.clamp(hybrid, -1.0, 1.0)

        self.last_deep_action = u_deep.detach()
        self.last_rl_action = rl_action.detach()
        self.last_hybrid_action = hybrid.detach()

        return hybrid

    def forward(self, obs: torch.Tensor, deterministic: bool = False):
        rl_action = super().forward(obs, deterministic=deterministic)
        return self._hybridize(rl_action, obs)

    def action_log_prob(self, obs: torch.Tensor):
        # The log probability is that of the SAC stochastic component.
        # The deterministic affine hybrid mapping is not included in log_prob.
        rl_action, log_prob = super().action_log_prob(obs)
        hybrid_action = self._hybridize(rl_action, obs)
        return hybrid_action, log_prob


class HybridCACCritic(nn.Module):
    """Twin-Q critic using the Hadamard-fused DeepONet feature."""

    def __init__(
        self,
        observation_space: spaces.Box,
        action_space: spaces.Box,
        hidden_dim: int = 256,
        features_dim: int = SAC_FEATURE_DIM,
        pretrained_path: str = DEEPNET_PRETRAINED_PATH,
    ):
        super().__init__()

        self.features_extractor = HybridCACDeepONetFeatureExtractor(
            observation_space,
            features_dim=features_dim,
            pretrained_path=pretrained_path,
            query_point=1.0,
            freeze_deeponet=False,
        )

        action_dim = int(np.prod(action_space.shape))
        input_dim = features_dim + action_dim

        self.q1 = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )
        self.q2 = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )

    def set_training_mode(self, mode: bool):
        self.train(mode)

    def forward(self, obs: torch.Tensor, actions: torch.Tensor):
        features = self.features_extractor(obs)
        q_input = torch.cat([features, actions], dim=1)
        return self.q1(q_input), self.q2(q_input)


class HybridCACSACPolicy(SACPolicy):
    """SAC policy with Hadamard DeepONet representation/control transfer."""

    def __init__(
        self,
        *args,
        hybrid_alpha_initial: float = 0.4,
        deeponet_pretrained_path: str = DEEPNET_PRETRAINED_PATH,
        **kwargs,
    ):
        self.hybrid_alpha_initial = float(hybrid_alpha_initial)
        self.deeponet_pretrained_path = deeponet_pretrained_path

        kwargs.setdefault(
            "features_extractor_class",
            HybridCACDeepONetFeatureExtractor,
        )
        kwargs.setdefault(
            "features_extractor_kwargs",
            dict(
                features_dim=SAC_FEATURE_DIM,
                pretrained_path=deeponet_pretrained_path,
                query_point=1.0,
                freeze_deeponet=False,
            ),
        )

        super().__init__(*args, **kwargs)

    def make_actor(self, features_extractor=None):
        kwargs = self._update_features_extractor(
            self.actor_kwargs,
            features_extractor,
        )
        return HybridCACActor(
            **kwargs,
            alpha=self.hybrid_alpha_initial,
        ).to(self.device)

    def make_critic(self, features_extractor=None):
        return HybridCACCritic(
            self.observation_space,
            self.action_space,
            hidden_dim=256,
            features_dim=SAC_FEATURE_DIM,
            pretrained_path=self.deeponet_pretrained_path,
        ).to(self.device)

    def set_alpha(self, alpha: float):
        self.actor.set_alpha(alpha)


# ============================================================
# 9. Optimizer helper: separate DeepONet learning rate
# ============================================================

def _make_param_groups_with_deeponet_lr(module: nn.Module):
    deeponet_params = []
    other_params = []

    for name, param in module.named_parameters():
        if not param.requires_grad:
            continue

        if "features_extractor.net1" in name or name.startswith("net1."):
            deeponet_params.append(param)
        else:
            other_params.append(param)

    groups = []

    if other_params:
        groups.append(
            {
                "params": other_params,
                "lr": learning_rate,
                "is_deeponet_group": False,
            }
        )

    if deeponet_params:
        groups.append(
            {
                "params": deeponet_params,
                "lr": deeponet_learning_rate,
                "is_deeponet_group": True,
            }
        )

    return groups


def set_deeponet_small_lr_optimizers(model):
    actor_groups = _make_param_groups_with_deeponet_lr(model.actor)
    critic_groups = _make_param_groups_with_deeponet_lr(model.critic)

    model.actor.optimizer = torch.optim.Adam(actor_groups, eps=1e-5)
    model.critic.optimizer = torch.optim.Adam(critic_groups, eps=1e-5)

    n_actor_deep = sum(
        p.numel()
        for group in actor_groups
        if group.get("is_deeponet_group", False)
        for p in group["params"]
    )
    n_actor_other = sum(
        p.numel()
        for group in actor_groups
        if not group.get("is_deeponet_group", False)
        for p in group["params"]
    )
    n_critic_deep = sum(
        p.numel()
        for group in critic_groups
        if group.get("is_deeponet_group", False)
        for p in group["params"]
    )
    n_critic_other = sum(
        p.numel()
        for group in critic_groups
        if not group.get("is_deeponet_group", False)
        for p in group["params"]
    )

    print("Custom optimizers with separate DeepONet learning rate are set.")
    print(f"Actor params: DeepONet={n_actor_deep}, other={n_actor_other}")
    print(f"Critic params: DeepONet={n_critic_deep}, other={n_critic_other}")
    print(
        f"Base SAC learning_rate={learning_rate}, "
        f"DeepONet learning_rate={deeponet_learning_rate}"
    )


class HybridSAC(SAC):
    """Preserve the smaller learning rate of DeepONet parameter groups."""

    def _update_learning_rate(self, optimizers):
        super()._update_learning_rate(optimizers)

        if not isinstance(optimizers, (list, tuple)):
            optimizers = [optimizers]

        current_lr = self.lr_schedule(self._current_progress_remaining)

        for optimizer in optimizers:
            for group in optimizer.param_groups:
                if group.get("is_deeponet_group", False):
                    group["lr"] = deeponet_learning_rate
                elif "is_deeponet_group" in group:
                    group["lr"] = current_lr




class EvalHybridSAC(HybridSAC):
    """Evaluation loader compatible with DeepONet-specific optimizer groups."""

    def _setup_model(self):
        super()._setup_model()
        set_deeponet_small_lr_optimizers(self)

    def set_parameters(self, load_path_or_dict, exact_match: bool = True, device="auto"):
        try:
            return super().set_parameters(
                load_path_or_dict,
                exact_match=exact_match,
                device=device,
            )
        except ValueError as exc:
            message = str(exc)
            optimizer_error = (
                "parameter group" in message
                or "Names of parameters" in message
                or "optimizer" in message.lower()
            )

            if isinstance(load_path_or_dict, dict) and optimizer_error:
                print(
                    "Evaluation load: incompatible optimizer states are skipped; "
                    "policy, actor, critic, DeepONet, and alpha weights are retained."
                )
                filtered = {
                    name: value
                    for name, value in load_path_or_dict.items()
                    if "optimizer" not in name.lower()
                }
                return super().set_parameters(
                    filtered,
                    exact_match=False,
                    device=device,
                )
            raise


# ============================================================
# Physical setup: ODE equilibrium + shared linear test initial condition
# ============================================================

def make_cac_table_params(
    q_bar: float = Q_BAR_FIXED,
    h_bar_0: float = 3.80,
    h_bar_L: float = 4.70,
) -> CACParams:
    """Create CAC parameters consistent with the new training program."""
    return CACParams(
        L=11239.0,
        B=40.0,
        g=9.81,
        I=7.92e-5,
        K=33.0,
        mu0=0.4,
        muL=0.4,
        z_up=5.00,
        U0_bar=0.1548,
        U_L_bar=U_ref,
        h_s=4.094,
        Q_bar=float(q_bar),
        H_bar_0=float(h_bar_0),
        H_bar_L=float(h_bar_L),
        H_min=0.20,
        max_state_value=1e10,
    )


def make_grid(p: CACParams, n_grid: int) -> Tuple[np.ndarray, float]:
    X = np.linspace(0.0, p.L, n_grid)
    dx = p.L / (n_grid - 1)
    return X, dx


def downstream_equilibrium_depth_from_q(Q: float, p: CACParams) -> float:
    """Compute Hbar(L) from the downstream equilibrium gate relation."""
    Q = float(Q)
    if not np.isfinite(Q) or Q <= 0.0:
        raise ValueError("The equilibrium discharge must be positive and finite.")

    kinetic_head = (
        Q**2 / (2.0 * p.g * p.B**2 * p.muL**2)
    ) ** (1.0 / 3.0)
    return float(p.h_s + p.U_L_bar + kinetic_head)


def integrate_steady_fixed_q_ode(
    X: np.ndarray,
    p: CACParams,
    Q_candidate: float,
) -> Tuple[np.ndarray, np.ndarray]:
    """Integrate the steady Saint-Venant ODE backward from x=L to x=0."""
    X = np.asarray(X, dtype=float)
    if X.ndim != 1 or X.size < 2:
        raise ValueError("X must be a one-dimensional grid with at least two points.")
    if not np.all(np.diff(X) > 0.0):
        raise ValueError("X must be strictly increasing.")

    Q_candidate = float(Q_candidate)
    if not np.isfinite(Q_candidate) or Q_candidate <= 0.0:
        raise ValueError("Q_candidate must be positive and finite.")

    Hbar_L = downstream_equilibrium_depth_from_q(Q_candidate, p)

    def rhs(_x: float, state: np.ndarray) -> np.ndarray:
        H = float(state[0])
        Q = float(state[1])

        if not np.isfinite(H) or not np.isfinite(Q):
            raise FloatingPointError("Non-finite state in steady reference ODE.")
        if H <= float(p.H_min):
            raise FloatingPointError("Steady reference ODE violates minimum depth.")

        area = p.B * H
        hydraulic_radius = p.B * H / (p.B + 2.0 * H)
        friction_slope = Q * abs(Q) / (
            p.K**2 * area**2 * hydraulic_radius ** (4.0 / 3.0)
        )

        denominator = p.g * p.B * H - Q**2 / (p.B * H**2)
        if abs(denominator) <= 1e-10:
            raise FloatingPointError(
                "Steady ODE approaches the critical-flow singularity."
            )

        dHdx = p.g * p.B * H * (p.I - friction_slope) / denominator
        if not np.isfinite(dHdx):
            raise FloatingPointError("Non-finite dHbar/dx in steady ODE.")

        return np.asarray([dHdx, 0.0], dtype=float)

    sol = solve_ivp(
        fun=rhs,
        t_span=(float(X[-1]), float(X[0])),
        y0=np.asarray([Hbar_L, Q_candidate], dtype=float),
        t_eval=X[::-1],
        rtol=1e-9,
        atol=1e-11,
        max_step=p.L / 1000.0,
    )

    if not sol.success:
        raise RuntimeError(f"Failed to solve steady ODE: {sol.message}")
    if sol.y.shape[1] != X.size:
        raise RuntimeError("Steady ODE did not return all requested grid points.")

    Hbar = np.asarray(sol.y[0][::-1], dtype=float)
    Qbar = np.asarray(sol.y[1][::-1], dtype=float)

    if not np.all(np.isfinite(Hbar)) or not np.all(np.isfinite(Qbar)):
        raise FloatingPointError("Steady reference contains NaN or Inf.")
    if float(np.min(Hbar)) <= float(p.H_min):
        raise RuntimeError("Steady Hbar profile violates the minimum depth.")

    return Hbar, Qbar


def solve_steady_reference_ode(
    X: np.ndarray,
    p: CACParams,
    q_bar_fixed: float = Q_BAR_FIXED,
) -> Tuple[np.ndarray, np.ndarray, float]:
    """Solve Hbar(x) for the prescribed constant Qbar=11.93 m^3/s."""
    q_bar_fixed = float(q_bar_fixed)
    Hbar, _ = integrate_steady_fixed_q_ode(X, p, q_bar_fixed)
    Qbar = np.full_like(X, q_bar_fixed, dtype=float)
    return Hbar, Qbar, q_bar_fixed


def make_reference_profiles(
    X: np.ndarray,
    p: CACParams,
    reference: str = REFERENCE_MODE,
    q_bar_fixed: float = Q_BAR_FIXED,
) -> Tuple[np.ndarray, np.ndarray, float]:
    if reference in {"ode", "strict"}:
        return solve_steady_reference_ode(X, p, q_bar_fixed=q_bar_fixed)
    if reference == "linear":
        Hbar = p.H_bar_0 + (p.H_bar_L - p.H_bar_0) * (X / p.L)
        Qbar = np.full_like(X, q_bar_fixed, dtype=float)
        return Hbar, Qbar, float(q_bar_fixed)
    raise ValueError("reference must be 'ode', 'strict', or 'linear'.")


def build_collection_matched_reference():
    """Return one p, X, dx, Hbar, Qbar tuple shared by every controller."""
    nominal_p = make_cac_table_params(q_bar=Q_BAR_FIXED)
    X, dx = make_grid(nominal_p, N)
    Hbar, Qbar, q_equilibrium = make_reference_profiles(
        X,
        nominal_p,
        reference=REFERENCE_MODE,
        q_bar_fixed=Q_BAR_FIXED,
    )

    p = make_cac_table_params(
        q_bar=q_equilibrium,
        h_bar_0=float(Hbar[0]),
        h_bar_L=float(Hbar[-1]),
    )
    return p, X, dx, Hbar, Qbar


def generate_test_initial_condition(
    X: np.ndarray,
    Hbar: np.ndarray,
    Qbar: np.ndarray,
    H_left: float = TEST_H_LEFT,
    H_right: float = TEST_H_RIGHT,
    Q_value: float = TEST_Q_VALUE,
) -> Tuple[np.ndarray, np.ndarray, Dict[str, float]]:
    """Generate the shared deterministic linear test initial condition.

    H(x,0) = H_left + (H_right-H_left) x/L,
    Q(x,0) = Q_value.

    Hbar and Qbar are accepted so the function can also report the exact
    initial deviations from the ODE equilibrium used in the test.
    """
    X = np.asarray(X, dtype=float)
    Hbar = np.asarray(Hbar, dtype=float)
    Qbar = np.asarray(Qbar, dtype=float)

    if X.ndim != 1 or Hbar.shape != X.shape or Qbar.shape != X.shape:
        raise ValueError(
            "X, Hbar, and Qbar must be one-dimensional arrays of equal length."
        )
    L = float(X[-1])
    if not np.isfinite(L) or L <= 0.0:
        raise ValueError("The spatial-domain length must be positive and finite.")

    H_left = float(H_left)
    H_right = float(H_right)
    Q_value = float(Q_value)
    if not all(np.isfinite(v) for v in (H_left, H_right, Q_value)):
        raise ValueError("H_left, H_right, and Q_value must be finite.")

    H0 = H_left + (H_right - H_left) * (X / L)
    Q0 = np.full_like(X, Q_value, dtype=float)

    if float(np.min(H0)) <= 0.20:
        raise ValueError("The linear initial H profile violates H_min=0.20 m.")
    if float(np.min(Q0)) <= 0.0:
        raise ValueError("The initial discharge must be positive.")

    details = {
        "H_left": H_left,
        "H_right": H_right,
        "Q_value": Q_value,
        "H_realized_max_deviation": float(np.max(np.abs(H0 - Hbar))),
        "Q_realized_max_deviation": float(np.max(np.abs(Q0 - Qbar))),
    }
    return H0, Q0, details


def make_fixed_initial_condition(
    X: np.ndarray,
    Hbar: np.ndarray,
    Qbar: np.ndarray,
    H_left: float = TEST_H_LEFT,
    H_right: float = TEST_H_RIGHT,
    Q_value: float = TEST_Q_VALUE,
) -> Tuple[np.ndarray, np.ndarray]:
    """Return the shared deterministic linear test initial state."""
    H0, Q0, _ = generate_test_initial_condition(
        X,
        Hbar,
        Qbar,
        H_left=H_left,
        H_right=H_right,
        Q_value=Q_value,
    )
    return H0, Q0


def make_test_initial_condition_func(
    H_left: float = TEST_H_LEFT,
    H_right: float = TEST_H_RIGHT,
    Q_value: float = TEST_Q_VALUE,
):
    """Return the same deterministic linear reset callback for all controllers."""
    def reset_func(X, Hbar, Qbar, rng):
        del rng
        H0, Q0, _ = generate_test_initial_condition(
            X,
            Hbar,
            Qbar,
            H_left=H_left,
            H_right=H_right,
            Q_value=Q_value,
        )
        return H0, Q0

    return reset_func

# ============================================================
# Hybrid action diagnostics
# ============================================================

def normalized_action_to_physical_ul(a):
    """Map normalized action a in [-1, 1] to physical downstream gate U_L."""
    a = float(np.clip(float(a), -1.0, 1.0))
    return float(min_control_value + 0.5 * (a + 1.0) * (max_control_value - min_control_value))


def physical_ul_to_normalized_action(u_physical):
    """Map physical downstream-gate command U_L back to normalized action [-1, 1]."""
    scale = float(max_control_value - min_control_value)
    if scale <= 0.0:
        raise ValueError("max_control_value must be greater than min_control_value.")
    u_physical = float(np.clip(float(u_physical), min_control_value, max_control_value))
    action = 2.0 * (u_physical - min_control_value) / scale - 1.0
    return float(np.clip(action, -1.0, 1.0))


def lowpass_filter_rl_control(action_raw, previous_u_filtered, alpha=RL_CONTROL_FILTER_ALPHA):
    """Apply a first-order low-pass filter to one scalar RL gate command.

    The policy output is first mapped to the physical downstream-gate command,
    filtered in physical units, and then mapped back to the normalized action
    expected by the environment. Because the action mapping is affine, this is
    equivalent to filtering the normalized action, while keeping the physical
    meaning explicit.

    Returns
    -------
    action_filtered : np.ndarray, shape (1,)
        Filtered normalized action passed to env.step().
    u_command_raw : float
        Unfiltered physical gate command generated by the RL policy.
    u_filtered : float
        Filtered physical gate command actually applied to the environment.
    """
    alpha = float(alpha)
    if not (0.0 < alpha <= 1.0):
        raise ValueError("RL_CONTROL_FILTER_ALPHA must satisfy 0 < alpha <= 1.")

    a_raw = float(np.asarray(action_raw, dtype=float).reshape(-1)[0])
    a_raw = float(np.clip(a_raw, -1.0, 1.0))
    u_command_raw = normalized_action_to_physical_ul(a_raw)

    previous_u_filtered = float(
        np.clip(float(previous_u_filtered), min_control_value, max_control_value)
    )
    u_filtered = (1.0 - alpha) * previous_u_filtered + alpha * u_command_raw
    u_filtered = float(np.clip(u_filtered, min_control_value, max_control_value))

    a_filtered = physical_ul_to_normalized_action(u_filtered)
    action_filtered = np.asarray([a_filtered], dtype=np.float32)
    return action_filtered, u_command_raw, u_filtered


def get_hybrid_action_components(model, obs, action_hybrid):
    """
    Extract DeepONet, RL, and hybrid normalized actions for the loaded hybrid SAC policy.

    The actor actually outputs
        a_hybrid = (1-alpha) * a_deep + alpha * a_RL.
    During testing, model.predict(obs) returns a_hybrid. We compute a_deep by calling
    the actor feature extractor and infer a_RL from the affine mixing formula.
    """
    actor = model.policy.actor
    device = model.device

    obs_tensor = torch.as_tensor(obs, dtype=torch.float32, device=device)
    if obs_tensor.dim() == 1:
        obs_tensor = obs_tensor.unsqueeze(0)

    with torch.no_grad():
        # This call updates actor.features_extractor.u_deep_norm.
        _ = actor.features_extractor(obs_tensor)
        u_deep = getattr(actor.features_extractor, "u_deep_norm", None)

    if u_deep is None:
        a_deep = np.nan
    else:
        a_deep = float(u_deep.detach().cpu().numpy().reshape(-1)[0])
        a_deep = float(np.clip(a_deep, -1.0, 1.0))

    a_hybrid = float(np.asarray(action_hybrid).reshape(-1)[0])
    a_hybrid = float(np.clip(a_hybrid, -1.0, 1.0))

    alpha = float(getattr(actor, "alpha", np.nan))

    if np.isfinite(alpha) and alpha > 1e-8 and np.isfinite(a_deep):
        a_rl = (a_hybrid - (1.0 - alpha) * a_deep) / alpha
        a_rl = float(np.clip(a_rl, -1.0, 1.0))
    else:
        a_rl = np.nan

    return {
        "alpha": alpha,
        "a_deep": a_deep,
        "a_rl": a_rl,
        "a_hybrid": a_hybrid,
        "U_deep": normalized_action_to_physical_ul(a_deep) if np.isfinite(a_deep) else np.nan,
        "U_rl": normalized_action_to_physical_ul(a_rl) if np.isfinite(a_rl) else np.nan,
        "U_hybrid": normalized_action_to_physical_ul(a_hybrid),
    }


# ============================================================
# Reward class
# ============================================================

class CACTrackingSmoothReward(BaseReward):
    """
    Discrete reward at the control instant t_k:

        r_k = -alpha ||y_tilde_k-y_tilde_star||^2
              -beta  ||(y_tilde_k-y_tilde_{k-1})/Delta_t||^2
              -gamma ||U_tilde_k||^2
              -delta ||U_tilde_k-U_tilde_{k-1}||^2.

    Here:
        y_tilde_star = 0,
        H_tilde_error = (H-H_eq)/H_scale,
        Q_tilde_error = (Q-Q_eq)/Q_scale,
        U_tilde = (U-U_eq)/U_scale.

    Spatial means are used instead of unnormalized sums so that the reward scale
    does not grow with the number of grid points N.
    """

    def __init__(
        self,
        H_eq,
        Q_eq,
        delta_t=control_sample_rate,
        H_scale=0.10,
        Q_scale=3.00,
        U_scale=0.10,
        alpha=1.0,
        beta=0.10,
        gamma=0.05,
        delta=0.10,
        w_H=1.0,
        w_Q=1.0,
        U_ref=0.3,
        terminal_tracking_weight=10.0,
        truncate_penalty=-200.0,
    ):
        self.H_eq = np.asarray(H_eq, dtype=float)
        self.Q_eq = np.asarray(Q_eq, dtype=float)
        self.N = len(self.H_eq)

        self.delta_t = float(delta_t)
        self.H_scale = float(H_scale)
        self.Q_scale = float(Q_scale)
        self.U_scale = float(U_scale)

        self.alpha = float(alpha)
        self.beta = float(beta)
        self.gamma = float(gamma)
        self.delta = float(delta)
        self.w_H = float(w_H)
        self.w_Q = float(w_Q)

        self.U_ref = float(U_ref)
        self.terminal_tracking_weight = float(terminal_tracking_weight)
        self.truncate_penalty = float(truncate_penalty)

        # Previous values at the preceding reward/control call.
        self.prev_H: Optional[np.ndarray] = None
        self.prev_Q: Optional[np.ndarray] = None
        self.prev_action: Optional[float] = None

        # Diagnostics.
        self.last_terms = {
            "tracking_cost": 0.0,
            "state_rate_cost": 0.0,
            "control_cost": 0.0,
            "delta_control_cost": 0.0,
        }

    def _reset_memory(self):
        self.prev_H = None
        self.prev_Q = None
        self.prev_action = None

    def _split_state(self, state_vec):
        state_vec = np.asarray(state_vec, dtype=float).reshape(-1)
        if state_vec.size < 2 * self.N:
            raise ValueError(
                f"PDE state length is {state_vec.size}, expected at least {2 * self.N}."
            )
        H = state_vec[:self.N]
        Q = state_vec[self.N:2 * self.N]
        return H, Q

    def _action_value(self, action):
        # The original environment passes the physical downstream gate to reward.
        if action is None:
            return self.U_ref
        value = float(np.asarray(action).reshape(-1)[0])
        return float(
            np.nan_to_num(
                value,
                nan=self.U_ref,
                posinf=self.U_ref,
                neginf=self.U_ref,
            )
        )

    def reward(
        self,
        uVec=None,
        time_index=None,
        terminate=None,
        truncate=None,
        action=None,
        control_sample_rate=None,
    ):
        if uVec is None or time_index is None:
            return 0.0

        # A new episode normally starts with a small time index.
        if int(time_index) <= 1:
            self._reset_memory()

        if truncate:
            self._reset_memory()
            return float(self.truncate_penalty)

        Hk, Qk = self._split_state(uVec[time_index])

        if (
            not np.all(np.isfinite(Hk))
            or not np.all(np.isfinite(Qk))
            or np.any(Hk <= 0.0)
        ):
            self._reset_memory()
            return float(self.truncate_penalty)

        # ----------------------------------------------------
        # 1) Tracking term
        # ----------------------------------------------------
        H_error_tilde = (Hk - self.H_eq) / self.H_scale
        Q_error_tilde = (Qk - self.Q_eq) / self.Q_scale

        tracking_cost = (
            self.w_H * np.mean(H_error_tilde ** 2)
            + self.w_Q * np.mean(Q_error_tilde ** 2)
        )

        # ----------------------------------------------------
        # 2) State-rate term
        # ----------------------------------------------------
        if self.prev_H is None or self.prev_Q is None:
            state_rate_cost = 0.0
        else:
            delta_t_k = (
                float(control_sample_rate)
                if control_sample_rate is not None
                else self.delta_t
            )
            delta_t_k = max(delta_t_k, 1e-12)

            dH_tilde = (Hk - self.prev_H) / (delta_t_k * self.H_scale)
            dQ_tilde = (Qk - self.prev_Q) / (delta_t_k * self.Q_scale)

            state_rate_cost = (
                self.w_H * np.mean(dH_tilde ** 2)
                + self.w_Q * np.mean(dQ_tilde ** 2)
            )

        # ----------------------------------------------------
        # 3) Control-magnitude term
        # ----------------------------------------------------
        Uk = self._action_value(action)
        U_tilde = (Uk - self.U_ref) / self.U_scale
        control_cost = U_tilde ** 2

        # ----------------------------------------------------
        # 4) Control-increment term
        # ----------------------------------------------------
        if self.prev_action is None:
            delta_control_cost = 0.0
        else:
            U_prev_tilde = (self.prev_action - self.U_ref) / self.U_scale
            delta_control_cost = (U_tilde - U_prev_tilde) ** 2

        reward_value = (
            -self.alpha * tracking_cost
            -self.beta * state_rate_cost
            -self.gamma * control_cost
            -self.delta * delta_control_cost
        )

        if terminate:
            reward_value -= self.terminal_tracking_weight * tracking_cost

        self.prev_H = Hk.copy()
        self.prev_Q = Qk.copy()
        self.prev_action = Uk

        self.last_terms = {
            "tracking_cost": float(tracking_cost),
            "state_rate_cost": float(state_rate_cost),
            "control_cost": float(control_cost),
            "delta_control_cost": float(delta_control_cost),
        }

        return float(reward_value)




# ============================================================
# Environments
# ============================================================

def _make_reward(Hbar, Qbar):
    return CACTrackingSmoothReward(
        H_eq=Hbar,
        Q_eq=Qbar,
        delta_t=control_sample_rate,
        H_scale=H_scale,
        Q_scale=Q_scale,
        U_scale=U_scale,
        alpha=reward_alpha,
        beta=reward_beta,
        gamma=reward_gamma,
        delta=reward_delta,
        w_H=reward_w_H,
        w_Q=reward_w_Q,
        U_ref=U_ref,
        terminal_tracking_weight=terminal_tracking_weight,
        truncate_penalty=truncate_penalty,
    )


def make_env_sac(
    H_left=TEST_H_LEFT,
    H_right=TEST_H_RIGHT,
    Q_value=TEST_Q_VALUE,
    seed=123,
    dH_source=None,
):
    """Create the exact evaluation environment for the raw-HQ learned model.

    The policy receives raw physical [H,Q]. Every controller is reset to the
    same deterministic linear H profile and constant Q profile.
    """
    p, X, dx, Hbar, Qbar = build_collection_matched_reference()
    H_init, Q_init = make_fixed_initial_condition(
        X,
        Hbar,
        Qbar,
        H_left=H_left,
        H_right=H_right,
        Q_value=Q_value,
    )

    env_kwargs = dict(
        params=p,
        X_grid=X,
        Hbar=Hbar,
        Qbar=Qbar,
        H_init=H_init,
        Q_init=Q_init,
        sensing_noise_func=lambda observation: observation,
        reset_init_condition_func=make_test_initial_condition_func(
            H_left=H_left,
            H_right=H_right,
            Q_value=Q_value,
        ),
        sensing_loc="full",
        control_type="Gate",
        limit_pde_state_size=True,
        max_state_value=max_state_value,
        min_control_value=min_control_value,
        max_control_value=max_control_value,
        control_sample_rate=control_sample_rate,
        reward_class=_make_reward(Hbar, Qbar),
        normalize=True,
        observation_mode="raw",
        H_scale=H_scale,
        Q_scale=Q_scale,
        obs_clip=obs_clip,
        T=SAC_T,
        dt=dt,
        cfl=cfl,
        check_cfl=check_cfl,
        well_balanced=well_balanced,
    )

    if dH_source is not None:
        env_kwargs["dH_source"] = float(dH_source)

    env = CACSaintVenantPDE1D(**env_kwargs)

    env.reference_X = X
    env.reference_H = Hbar
    env.reference_Q = Qbar
    env.reference_dx = dx

    return env, X, Hbar, Qbar, p


def make_env_sac_error_state(
    H_left=TEST_H_LEFT,
    H_right=TEST_H_RIGHT,
    Q_value=TEST_Q_VALUE,
    seed=123,
    dH_source=None,
):
    """Create the evaluation environment for the old error-state SAC model.

    The physical test problem is unchanged: the ODE equilibrium, linear H0,
    constant Q0, PDE parameters, action range, time step, and test horizon are
    identical to the raw-HQ controllers. Only the policy observation changes to

        [(H-Hbar)/H_scale, (Q-Qbar)/Q_scale].

    This matches the old SAC checkpoint trained with
    observation_mode="normalized_error".
    """
    p, X, dx, Hbar, Qbar = build_collection_matched_reference()
    H_init, Q_init = make_fixed_initial_condition(
        X,
        Hbar,
        Qbar,
        H_left=H_left,
        H_right=H_right,
        Q_value=Q_value,
    )

    env_kwargs = dict(
        params=p,
        X_grid=X,
        Hbar=Hbar,
        Qbar=Qbar,
        H_init=H_init,
        Q_init=Q_init,
        sensing_noise_func=lambda observation: observation,
        reset_init_condition_func=make_test_initial_condition_func(
            H_left=H_left,
            H_right=H_right,
            Q_value=Q_value,
        ),
        sensing_loc="full",
        control_type="Gate",
        limit_pde_state_size=True,
        max_state_value=max_state_value,
        min_control_value=min_control_value,
        max_control_value=max_control_value,
        control_sample_rate=control_sample_rate,
        reward_class=_make_reward(Hbar, Qbar),
        normalize=True,
        observation_mode="normalized_error",
        H_scale=H_scale,
        Q_scale=Q_scale,
        obs_clip=obs_clip,
        T=SAC_T,
        dt=dt,
        cfl=cfl,
        check_cfl=check_cfl,
        well_balanced=well_balanced,
    )

    if dH_source is not None:
        env_kwargs["dH_source"] = float(dH_source)

    env = CACSaintVenantPDE1D(**env_kwargs)
    env.reference_X = X
    env.reference_H = Hbar
    env.reference_Q = Qbar
    env.reference_dx = dx
    env.policy_observation_mode = "normalized_error"

    return env, X, Hbar, Qbar, p


def make_env_backstepping(
    H_left=TEST_H_LEFT,
    H_right=TEST_H_RIGHT,
    Q_value=TEST_Q_VALUE,
    seed=123,
    dH_source=None,
):
    """Create Backstepping with the same ODE reference and linear initial state."""
    p, X, dx, Hbar, Qbar = build_collection_matched_reference()
    H_init, Q_init = make_fixed_initial_condition(
        X,
        Hbar,
        Qbar,
        H_left=H_left,
        H_right=H_right,
        Q_value=Q_value,
    )

    env_kwargs = dict(
        params=p,
        X_grid=X,
        Hbar=Hbar,
        Qbar=Qbar,
        H_init=H_init,
        Q_init=Q_init,
        sensing_noise_func=lambda state: state,
        reset_init_condition_func=make_test_initial_condition_func(
            H_left=H_left,
            H_right=H_right,
            Q_value=Q_value,
        ),
        sensing_loc="full",
        control_type="Gate",
        limit_pde_state_size=True,
        max_state_value=max_state_value,
        min_control_value=min_control_value,
        max_control_value=max_control_value,
        control_sample_rate=control_sample_rate,
        reward_class=_make_reward(Hbar, Qbar),
        normalize=False,
        observation_mode="raw",
        H_scale=H_scale,
        Q_scale=Q_scale,
        obs_clip=obs_clip,
        T=BS_T,
        dt=dt,
        cfl=cfl,
        check_cfl=check_cfl,
        well_balanced=well_balanced,
    )

    if dH_source is not None:
        env_kwargs["dH_source"] = float(dH_source)

    env = CACSaintVenantPDE1D(**env_kwargs)

    env.reference_X = X
    env.reference_H = Hbar
    env.reference_Q = Qbar
    env.reference_dx = dx

    return env, X, Hbar, Qbar, p

# ============================================================
# Backstepping functions copied from the standalone Backstepping script
# ============================================================

def left_eigenvector_matrix(H: float, Q: float, p: CACParams) -> Tuple[np.ndarray, float, float]:
    H = max(float(H), p.H_min)
    V = Q / (p.B * H)
    c = np.sqrt(p.g * H)
    lp = V + c
    lm = V - c

    R = np.array([[1.0, 1.0], [p.B * lp, p.B * lm]], dtype=float)
    Tmat = np.linalg.inv(R)
    return Tmat, lp, lm


def characteristic_variables(H: np.ndarray, Q: np.ndarray, Hbar: np.ndarray, p: CACParams) -> Tuple[np.ndarray, np.ndarray]:
    u = np.zeros_like(H)
    v = np.zeros_like(H)

    for i in range(len(H)):
        Tmat, _, _ = left_eigenvector_matrix(Hbar[i], p.Q_bar, p)
        z = Tmat @ np.array([H[i] - Hbar[i], Q[i] - p.Q_bar])
        u[i] = z[0]
        v[i] = z[1]

    return u, v


def linearized_matrices_profiles(X: np.ndarray, Hbar: np.ndarray, p: CACParams) -> Dict[str, np.ndarray]:
    Np = len(X)
    Qbar = p.Q_bar
    Hx = np.gradient(Hbar, X, edge_order=2)

    T_arr = np.zeros((Np, 2, 2))
    R_arr = np.zeros((Np, 2, 2))
    A_arr = np.zeros((Np, 2, 2))
    lam_plus = np.zeros(Np)
    lam_minus = np.zeros(Np)

    for i in range(Np):
        H = float(Hbar[i])
        Q = Qbar

        Tmat, lp, lm = left_eigenvector_matrix(H, Q, p)
        T_arr[i] = Tmat
        R_arr[i] = np.linalg.inv(Tmat)

        A_arr[i] = np.array(
            [[0.0, 1.0 / p.B],
             [p.g * p.B * H - Q**2 / (p.B * H**2), 2.0 * Q / (p.B * H)]],
            dtype=float,
        )

        lam_plus[i] = lp
        lam_minus[i] = lm

    R_x = np.gradient(R_arr, X, axis=0, edge_order=2)

    def G_of_state(Hval: float, Qval: float, Hx_val: float) -> np.ndarray:
        A = np.array(
            [[0.0, 1.0 / p.B],
             [p.g * p.B * Hval - Qval**2 / (p.B * Hval**2), 2.0 * Qval / (p.B * Hval)]],
            dtype=float,
        )
        Xbar_x = np.array([Hx_val, 0.0])
        F = source(np.array([Hval]), np.array([Qval]), p)[0]
        return A @ Xbar_x - F

    Bstar = np.zeros((Np, 2, 2))

    for i in range(Np):
        H = float(Hbar[i])
        Q = Qbar
        hx = float(Hx[i])

        dh = max(1e-6, abs(H) * 1e-6)
        dq = max(1e-6, abs(Q) * 1e-6)

        dG_dH = (G_of_state(H + dh, Q, hx) - G_of_state(H - dh, Q, hx)) / (2.0 * dh)
        dG_dQ = (G_of_state(H, Q + dq, hx) - G_of_state(H, Q - dq, hx)) / (2.0 * dq)

        Bstar[i, :, 0] = dG_dH
        Bstar[i, :, 1] = dG_dQ

    M = np.zeros((Np, 2, 2))
    for i in range(Np):
        M[i] = T_arr[i] @ (A_arr[i] @ R_x[i] + Bstar[i] @ R_arr[i])

    eps1 = lam_plus.copy()
    eps2 = -lam_minus.copy()

    int1 = np.zeros(Np)
    int2 = np.zeros(Np)

    f1 = M[:, 0, 0] / eps1
    f2 = -M[:, 1, 1] / eps2

    for i in range(1, Np):
        h = X[i] - X[i - 1]
        int1[i] = int1[i - 1] + 0.5 * h * (f1[i] + f1[i - 1])
        int2[i] = int2[i - 1] + 0.5 * h * (f2[i] + f2[i - 1])

    phi1 = np.exp(int1)
    phi2 = np.exp(int2)

    c1 = -(phi1 / phi2) * M[:, 0, 1]
    c2 = -(phi2 / phi1) * M[:, 1, 0]

    Hbar0 = float(Hbar[0])
    kappa0 = -p.U0_bar * p.B * p.mu0 * p.g / np.sqrt(2.0 * p.g * (p.z_up - Hbar0))

    T0 = T_arr[0]
    q0 = (T0[0, 0] + T0[0, 1] * kappa0) / (T0[1, 0] + T0[1, 1] * kappa0)

    return {
        "eps1": eps1,
        "eps2": eps2,
        "eps1_x": np.gradient(eps1, X, edge_order=2),
        "eps2_x": np.gradient(eps2, X, edge_order=2),
        "c1": c1,
        "c2": c2,
        "q0": float(q0),
    }


def compute_variable_kernels(X: np.ndarray, coeff: Dict[str, np.ndarray]) -> Tuple[np.ndarray, np.ndarray]:
    Np = len(X)
    dx = X[1] - X[0]

    e1 = coeff["eps1"]
    e2 = coeff["eps2"]
    e1x = coeff["eps1_x"]
    e2x = coeff["eps2_x"]
    c1 = coeff["c1"]
    c2 = coeff["c2"]
    q0 = coeff["q0"]

    ratio = q0 * e1[0] / e2[0]

    Kvu = np.zeros((Np, Np))
    Kvv = np.zeros((Np, Np))

    for i in range(Np):
        Kvu[i, i] = -c2[i] / (e1[i] + e2[i])

    for i in range(1, Np):
        A = (e2[i] + e1[0]) / dx - e1x[0]
        rhs = e2[i] * Kvu[i - 1, 0] / dx + e1[0] * Kvu[i, 1] / dx
        denom = A - c2[0] * ratio

        Kvu[i, 0] = rhs / denom
        Kvv[i, 0] = ratio * Kvu[i, 0]

        for j in range(1, i):
            A = (e2[i] + e1[j]) / dx - e1x[j]
            B = (e2[i] + e2[j]) / dx + e2x[j]

            rhsK = e2[i] * Kvu[i - 1, j] / dx + e1[j] * Kvu[i, j + 1] / dx
            rhsL = e2[i] * Kvv[i - 1, j] / dx + e2[j] * Kvv[i, j - 1] / dx

            mat = np.array([[A, -c2[j]], [c1[j], B]], dtype=float)
            sol = np.linalg.solve(mat, np.array([rhsK, rhsL], dtype=float))

            Kvu[i, j] = sol[0]
            Kvv[i, j] = sol[1]

        Bdiag = e2[i] / dx + e2x[i]
        rhsdiag = e2[i] * Kvv[i - 1, i - 1] / dx - c1[i] * Kvu[i, i]
        Kvv[i, i] = rhsdiag / Bdiag

    return Kvu, Kvv


def backstepping_action(env: CACSaintVenantPDE1D, Kvu_L: np.ndarray, Kvv_L: np.ndarray, gain: float = 1.0):
    p = env.params
    H, Q = env.get_current_HQ()
    Hbar = env.Hbar
    X = env.X_grid

    u, v = characteristic_variables(H, Q, Hbar, p)
    Ubs = gain * float(np.trapezoid(Kvu_L * u + Kvv_L * v, X))

    T_L, _, _ = left_eigenvector_matrix(float(Hbar[-1]), p.Q_bar, p)
    t21, t22 = T_L[1, 0], T_L[1, 1]

    QL = float(Q[-1])
    Hbs_L = float(Hbar[-1]) + (Ubs - t22 * (QL - p.Q_bar)) / t21

    UL = Hbs_L - p.h_s - (QL**2 / (2.0 * p.g * p.B**2 * p.muL**2)) ** (1.0 / 3.0)

    return float(UL), float(Ubs)




# ============================================================
# SB3 evaluation-only loading helper
# ============================================================

def strip_optimizer_states_from_sb3_model(model_path: str) -> str:
    """
    Create an evaluation-only SB3 zip by removing optimizer states.

    This fixes errors such as:
        ValueError: loaded state dict has a different number of parameter groups

    It is safe for evaluation because optimizer states are only needed when
    continuing training. The policy/actor/critic weights are kept unchanged.

    Return value is a path without the .zip suffix, matching SB3's usual API.
    """
    model_path = Path(model_path)

    if model_path.suffix == ".zip":
        src_zip = model_path
    else:
        src_zip = model_path.with_suffix(".zip")

    if not src_zip.exists():
        raise FileNotFoundError(f"Cannot find SB3 model zip file: {src_zip}")

    dst_zip = src_zip.with_name(src_zip.stem + "_eval_only.zip")

    if dst_zip.exists():
        print(f"Using existing eval-only model: {dst_zip}")
        return str(dst_zip.with_suffix(""))

    with zipfile.ZipFile(src_zip, "r") as zin:
        with zipfile.ZipFile(dst_zip, "w", compression=zipfile.ZIP_DEFLATED) as zout:
            for item in zin.infolist():
                name = item.filename
                # Remove all optimizer states. These can have different parameter-group
                # structures after using DeepONet-specific learning rates.
                if "optimizer" in name.lower():
                    print(f"Skip optimizer state: {name}")
                    continue
                zout.writestr(item, zin.read(name))

    print(f"Created eval-only model: {dst_zip}")
    return str(dst_zip.with_suffix(""))


# ============================================================
# Rollouts
# ============================================================

def run_sac_exact(
    model_path,
    seed=123,
    H_left=TEST_H_LEFT,
    H_right=TEST_H_RIGHT,
    Q_value=TEST_Q_VALUE,
    dH_source=None,
):
    env, X, Hbar, Qbar, p = make_env_sac(
        H_left=H_left,
        H_right=H_right,
        Q_value=Q_value,
        seed=seed,
        dH_source=dH_source,
    )

    # Directly load the trained HybridSAC model.
    # EvalHybridSAC only rebuilds the optimizer structure required by SB3 loading;
    # the actual closed-loop control still comes directly from model.predict().
    # Because this training version saves alpha as actor.alpha_tensor, no manual
    # alpha restoration is needed here.
    model = EvalHybridSAC.load(model_path, env=env)
    obs, _ = env.reset(seed=seed)

    H_hist, Q_hist, U_hist, t_hist, reward_hist = [], [], [], [], []
    U_raw_hist = []

    # The physical actuator/filter starts from the nominal equilibrium gate.
    # Only RL commands are filtered; the plant states are never post-filtered.
    previous_u_filtered = float(U_ref)

    done = False
    while not done:
        action_raw, _ = model.predict(obs, deterministic=True)
        action_filtered, u_command_raw, previous_u_filtered = lowpass_filter_rl_control(
            action_raw,
            previous_u_filtered,
            alpha=RL_CONTROL_FILTER_ALPHA,
        )
        obs, reward, terminated, truncated, info = env.step(action_filtered)

        H, Q = env.get_current_HQ()

        H_hist.append(H.copy())
        Q_hist.append(Q.copy())
        U_hist.append(float(env.last_control))
        U_raw_hist.append(float(u_command_raw))
        t_hist.append(float(info["time"]))
        reward_hist.append(float(reward))

        done = terminated or truncated

    env.close()

    H_hist = np.asarray(H_hist)
    Q_hist = np.asarray(Q_hist)
    U_hist = np.asarray(U_hist)
    U_raw_hist = np.asarray(U_raw_hist)
    t_hist = np.asarray(t_hist)
    reward_hist = np.asarray(reward_hist)

    H_err = H_hist - Hbar[None, :]
    Q_err = Q_hist - Qbar[None, :]

    return {
        "X": X,
        "Hbar": Hbar,
        "Qbar": Qbar,
        "t": t_hist,
        "H": H_hist,
        "Q": Q_hist,
        "U": U_hist,
        "U_raw": U_raw_hist,
        "reward": reward_hist,
        "H_l2": np.sqrt(np.mean(H_err ** 2, axis=1)),
        "Q_l2": np.sqrt(np.mean(Q_err ** 2, axis=1)),
        "final_H_minus_Hbar": H_err[-1],
        "final_Q_minus_Qbar": Q_err[-1],
        "total_reward": float(np.sum(reward_hist)),
    }


def run_backstepping_exact(
    seed=123,
    H_left=TEST_H_LEFT,
    H_right=TEST_H_RIGHT,
    Q_value=TEST_Q_VALUE,
    dH_source=None,
):
    """
    Backstepping rollout unified to the SAC standalone recording style:
        action -> env.step(action) -> record H,Q,U,t,reward.

    Therefore this is no longer the original Backstepping standalone recording mode;
    it is intentionally aligned with the SAC plot.
    """
    env, X, Hbar, Qbar, p = make_env_backstepping(
        H_left=H_left,
        H_right=H_right,
        Q_value=Q_value,
        seed=seed,
        dH_source=dH_source,
    )

    coeff = linearized_matrices_profiles(X, Hbar, p)
    Kvu, Kvv = compute_variable_kernels(X, coeff)
    Kvu_L = Kvu[-1, :]
    Kvv_L = Kvv[-1, :]

    obs, _ = env.reset(seed=seed)

    H_hist, Q_hist, U_hist, t_hist, reward_hist = [], [], [], [], []

    # Record the common prescribed initial state at t=0, exactly as for the
    # learned controller. The initial control is undefined and stored as NaN.
    H0, Q0 = env.get_current_HQ()
    H_hist.append(H0.copy())
    Q_hist.append(Q0.copy())
    U_hist.append(np.nan)
    t_hist.append(0.0)
    reward_hist.append(0.0)

    done = False
    while not done:
        UL_raw, Ubs = backstepping_action(env, Kvu_L, Kvv_L, gain=BACKSTEPPING_GAIN)
        action_env = float(np.clip(UL_raw, min_control_value, max_control_value))

        obs, reward, terminated, truncated, info = env.step(np.array([action_env], dtype=np.float32))

        # SAC plotting style: record AFTER env.step().
        H, Q = env.get_current_HQ()
        H_hist.append(H.copy())
        Q_hist.append(Q.copy())
        U_hist.append(action_env)
        t_hist.append(float(info["time"]))
        reward_hist.append(float(reward))

        done = terminated or truncated

    env.close()

    H_hist = np.asarray(H_hist)
    Q_hist = np.asarray(Q_hist)
    U_hist = np.asarray(U_hist)
    t_hist = np.asarray(t_hist)
    reward_hist = np.asarray(reward_hist)

    H_err = H_hist - Hbar[None, :]
    Q_err = Q_hist - Qbar[None, :]

    return {
        "X": X,
        "Hbar": Hbar,
        "Qbar": Qbar,
        "t": t_hist,
        "H": H_hist,
        "Q": Q_hist,
        "U": U_hist,
        "reward": reward_hist,
        "H_l2": np.sqrt(np.mean(H_err ** 2, axis=1)),
        "Q_l2": np.sqrt(np.mean(Q_err ** 2, axis=1)),
        "final_H_minus_Hbar": H_err[-1],
        "final_Q_minus_Qbar": Q_err[-1],
        "total_reward": float(np.sum(reward_hist)),
    }


# ============================================================
# Plot / save
# ============================================================









def _compute_spatial_l2_over_time(res, state_key):
    """
    Compute spatial L2 norm over time:
        ||H-Hbar||_L2 = sqrt( integral_0^1 (H(x,t)-Hbar(x))^2 ds )
        ||Q-Qbar||_L2 = sqrt( integral_0^1 (Q(x,t)-Qbar(x))^2 ds )
    where s=x/L is the normalized spatial coordinate.
    """
    X = res["X"]
    s = X / X[-1]

    if state_key == "H":
        err = res["H"] - res["Hbar"][None, :]
    elif state_key == "Q":
        err = res["Q"] - res["Qbar"][None, :]
    else:
        raise ValueError("state_key must be 'H' or 'Q'.")

    return np.sqrt(np.trapezoid(err ** 2, s, axis=1))

















# ============================================================
# Controller-comparison comparison utilities
# ============================================================

LABEL_BACKSTEPPING = "BS"
LABEL_NEW_MODEL = "TDPSAC"

# Retain the previous label constants so the common style helper can be reused
# when additional checkpoints are added later.
LABEL_SAC_CURVE = "SAC"
LABEL_FP_SAC = "SAC-FPC"
LABEL_SAC = "DO-SAC"
LABEL_HYBRID = "DPSAC"
LABEL_DFP_DON_SAC = "TDPSAC-FPC"
LABEL_HYBRID_FREEZE = LABEL_NEW_MODEL

# Fixed order for 2D legends, terminal summaries, evidence tables, and saved
# controller arrays.  This is independent of the order in which models are run.
CONTROLLER_DISPLAY_ORDER = [
    LABEL_SAC_CURVE,
    LABEL_FP_SAC,
    LABEL_SAC,
    LABEL_HYBRID,
    LABEL_DFP_DON_SAC,
    LABEL_NEW_MODEL,
]


def _ordered_results(results: Dict[str, dict]) -> Dict[str, dict]:
    """Return a dictionary in the requested display/table order."""
    ordered = {
        label: results[label]
        for label in CONTROLLER_DISPLAY_ORDER
        if label in results
    }
    # Preserve any future additional controllers after the six fixed entries.
    for label, res in results.items():
        if label not in ordered:
            ordered[label] = res
    return ordered


def _safe_load_hybrid_model(model_path: str, env):
    """
    Load a HybridSAC checkpoint for evaluation.

    The HybridSAC training script may save optimizers with special parameter
    groups for DeepONet. EvalHybridSAC rebuilds the optimizer structure. If the
    optimizer groups still do not match, an eval-only copy without optimizer
    states is created and loaded. This does not change the policy weights.
    """
    try:
        return EvalHybridSAC.load(model_path, env=env)
    except Exception as e:
        msg = str(e)
        if ("parameter group" in msg) or ("Names of parameters" in msg) or ("optimizer" in msg):
            print("Hybrid load with optimizer states failed; trying eval-only checkpoint.")
            print("Original error:", repr(e))
            eval_path = strip_optimizer_states_from_sb3_model(model_path)
            return EvalHybridSAC.load(eval_path, env=env)
        raise



def _checkpoint_exists(model_path: str) -> bool:
    """
    Return True only when model_path resolves to a valid SB3 ZIP checkpoint.

    This deliberately rejects arbitrary existing files such as Jupyter's
    kernel-*.json connection file. Stable-Baselines3 checkpoints are ZIP files,
    although callers may pass the path either with or without the .zip suffix.
    """
    if model_path is None or str(model_path).strip() == "":
        return False

    path = Path(str(model_path)).expanduser()

    candidates = [path]
    if path.suffix.lower() != ".zip":
        candidates.append(path.with_suffix(".zip"))

    for candidate in candidates:
        if candidate.is_file() and zipfile.is_zipfile(candidate):
            return True

    return False


def _safe_load_generic_sac_model(model_path: str, env):
    """
    Load a custom SAC-family checkpoint without assuming one exact policy layout.

    This is used for TDPSAC-FPC and SAC-FPC because their policy
    architectures may differ. The loader first asks SB3 to reconstruct the
    policy class stored in the checkpoint. If optimizer parameter groups are
    incompatible during evaluation, it creates an evaluation-only copy with
    optimizer states removed and retries.

    Policy, actor, critic, normalization, and saved alpha weights are preserved.
    """
    errors = []

    try:
        return SAC.load(model_path, env=env)
    except Exception as exc:
        errors.append(("SAC.load", exc))

    try:
        eval_path = strip_optimizer_states_from_sb3_model(model_path)
        return SAC.load(eval_path, env=env)
    except Exception as exc:
        errors.append(("SAC.load(eval-only)", exc))

    # Final fallback for checkpoints trained with the HybridSAC subclass used
    # in this file.
    try:
        return _safe_load_hybrid_model(model_path, env=env)
    except Exception as exc:
        errors.append(("_safe_load_hybrid_model", exc))

    details = "\n".join(f"  {name}: {error!r}" for name, error in errors)
    raise RuntimeError(
        f"Unable to load SAC-family checkpoint: {model_path}\n"
        f"Tried the generic and HybridSAC evaluation loaders:\n{details}"
    )

def run_policy_exact(
    model_path: str,
    model_kind: str,
    label: str,
    seed: int = 123,
    H_left: float = TEST_H_LEFT,
    H_right: float = TEST_H_RIGHT,
    Q_value: float = TEST_Q_VALUE,
    dH_source=None,
):
    """
    Run one learned controller in the CAC environment.

    model_kind:
        "sac"       -> ordinary raw-HQ SAC.load(...)
        "sac_error" -> old SAC checkpoint using normalized H/Q errors
        "hybrid"    -> EvalHybridSAC.load(...), for the existing hybrid checkpoints
        "auto"      -> reconstruct the saved SAC policy class automatically; used
                       for TDPSAC-FPC and SAC-FPC

    Every learned controller sees the same physical PDE reference, initial
    state, and raw physical H/Q observation interface in the current comparison.
    The reset state is recorded first at t=0, then later states are recorded
    after env.step().
    """
    model_kind_lower = model_kind.lower()
    if model_kind_lower == "sac_error":
        env, X, Hbar, Qbar, p = make_env_sac_error_state(
            H_left=H_left,
            H_right=H_right,
            Q_value=Q_value,
            seed=seed,
            dH_source=dH_source,
        )
    else:
        env, X, Hbar, Qbar, p = make_env_sac(
            H_left=H_left,
            H_right=H_right,
            Q_value=Q_value,
            seed=seed,
            dH_source=dH_source,
        )

    if model_kind_lower in {"sac", "sac_error"}:
        model = SAC.load(model_path, env=env)
    elif model_kind_lower == "hybrid":
        model = _safe_load_hybrid_model(model_path, env=env)
    elif model_kind_lower == "auto":
        model = _safe_load_generic_sac_model(model_path, env=env)
    else:
        raise ValueError(
            f"Unknown model_kind: {model_kind}. Use 'sac', 'sac_error', "
            "'hybrid', or 'auto'."
        )

    obs, _ = env.reset(seed=seed)
    obs = np.asarray(obs, dtype=np.float32).reshape(-1)
    if obs.size != RAW_STATE_DIM:
        raise RuntimeError(
            f"The reset observation has dimension {obs.size}; expected "
            f"{RAW_STATE_DIM} for the H/Q or H/Q-error interface."
        )

    observation_mode_used = (
        "normalized_error" if model_kind_lower == "sac_error" else "raw"
    )
    print(
        f"{label} observation mode: {observation_mode_used}, "
        f"dimension={obs.size}"
    )

    H_hist, Q_hist, U_hist, t_hist, reward_hist = [], [], [], [], []
    U_raw_hist = []

    # Record the reset state before any controller acts.
    # This makes the learned controllers start from
    # exactly the same L2 error at t=0.
    H0, Q0 = env.get_current_HQ()
    H_hist.append(H0.copy())
    Q_hist.append(Q0.copy())
    U_hist.append(np.nan)
    U_raw_hist.append(np.nan)
    t_hist.append(0.0)
    reward_hist.append(0.0)

    # Initialize the first-order actuator filter at the nominal equilibrium gate.
    # The same filter is used for SAC, SAC-FPC, DO-SAC, DPSAC, TDPSAC-FPC,
    # and TDPSAC so that the RL comparison remains fair.
    previous_u_filtered = float(U_ref)

    done = False
    while not done:
        action_raw, _ = model.predict(obs, deterministic=True)
        action_filtered, u_command_raw, previous_u_filtered = lowpass_filter_rl_control(
            action_raw,
            previous_u_filtered,
            alpha=RL_CONTROL_FILTER_ALPHA,
        )
        obs, reward, terminated, truncated, info = env.step(action_filtered)

        H, Q = env.get_current_HQ()

        H_hist.append(H.copy())
        Q_hist.append(Q.copy())
        U_hist.append(float(env.last_control))
        U_raw_hist.append(float(u_command_raw))
        t_hist.append(float(info["time"]))
        reward_hist.append(float(reward))

        done = terminated or truncated

    env.close()

    H_hist = np.asarray(H_hist)
    Q_hist = np.asarray(Q_hist)
    U_hist = np.asarray(U_hist)
    U_raw_hist = np.asarray(U_raw_hist)
    t_hist = np.asarray(t_hist)
    reward_hist = np.asarray(reward_hist)

    H_err = H_hist - Hbar[None, :]
    Q_err = Q_hist - Qbar[None, :]

    return {
        "label": label,
        "model_kind": model_kind,
        "observation_mode": observation_mode_used,
        "X": X,
        "Hbar": Hbar,
        "Qbar": Qbar,
        "t": t_hist,
        "H": H_hist,
        "Q": Q_hist,
        "U": U_hist,
        "U_raw": U_raw_hist,
        "reward": reward_hist,
        "H_l2": np.sqrt(np.mean(H_err ** 2, axis=1)),
        "Q_l2": np.sqrt(np.mean(Q_err ** 2, axis=1)),
        "final_H_minus_Hbar": H_err[-1],
        "final_Q_minus_Qbar": Q_err[-1],
        "total_reward": float(np.sum(reward_hist)),
    }


def run_backstepping_for_four(
    seed: int = 123,
    H_left: float = TEST_H_LEFT,
    H_right: float = TEST_H_RIGHT,
    Q_value: float = TEST_Q_VALUE,
    dH_source=None,
):
    res = run_backstepping_exact(
        seed=seed,
        H_left=H_left,
        H_right=H_right,
        Q_value=Q_value,
        dH_source=dH_source,
    )
    res["label"] = LABEL_BACKSTEPPING
    res["model_kind"] = "backstepping"
    return res


def _time_min(res):
    return np.asarray(res["t"], dtype=float) / 60.0


def _get_l2(res, key):
    return _compute_spatial_l2_over_time(res, key)


def _get_linf(res, key):
    """Return the spatial L-infinity tracking-error norm at each time instant."""
    state = np.asarray(res[key], dtype=float)
    reference_key = "Hbar" if key == "H" else "Qbar"
    reference = np.asarray(res[reference_key], dtype=float).reshape(1, -1)
    if state.ndim != 2 or reference.shape[1] != state.shape[1]:
        return np.asarray([], dtype=float)
    return np.max(np.abs(state - reference), axis=1)


def set_size(width, fraction=1, ncols=1, nrows=1, height_add=0):
    """Figure size helper copied from the third comparison-program style."""
    inches_per_pt = 1 / 72.27
    fig_width = width * inches_per_pt * fraction
    fig_height = fig_width * 0.618 * (nrows / ncols) + height_add
    return fig_width, fig_height



def _line_style(label):
    """Return the fixed colors and line styles used by the old comparison program."""
    if label == LABEL_BACKSTEPPING:  # BS
        return dict(color="orange", linestyle=":", linewidth=2.0, alpha=0.8)
    if label == LABEL_SAC_CURVE:  # SAC
        return dict(color="purple", linestyle="-", linewidth=1.5, alpha=0.8)
    if label == LABEL_FP_SAC:  # SAC-FPC
        return dict(
            color="saddlebrown",
            linestyle=(0, (6, 2, 1.5, 2)),
            linewidth=2.2,
            alpha=0.95,
        )
    if label == LABEL_SAC:  # DO-SAC
        return dict(color="blue", linestyle="-.", linewidth=2.0, alpha=0.8)
    if label == LABEL_HYBRID:  # DPSAC
        return dict(color="green", linestyle="--", linewidth=2.0, alpha=0.8)
    if label == LABEL_DFP_DON_SAC:  # TDPSAC-FPC
        return dict(color="black", linestyle="--", linewidth=2.2, alpha=0.95)
    if label == LABEL_HYBRID_FREEZE:  # TDPSAC
        return dict(color="red", linestyle="-", linewidth=3.0, alpha=0.8)
    return dict(linewidth=2, alpha=0.8)


def _line_style_2d(label):
    """2D-only line widths requested by the user; colors/styles stay unchanged."""
    style = dict(_line_style(label))

    if label == LABEL_HYBRID_FREEZE:      # TDPSAC, red
        style["linewidth"] = 2.3
    elif label == LABEL_DFP_DON_SAC:      # TDPSAC-FPC, black
        style["linewidth"] = 2.3
    elif label == LABEL_SAC_CURVE:        # SAC, purple
        style["linewidth"] = 1.5
    elif label != LABEL_BACKSTEPPING:
        # SAC-FPC / DO-SAC / DPSAC and any other learned-controller curve
        style["linewidth"] = 2.0

    return style


def _ordered_results_for_2d_drawing(results: Dict[str, dict]) -> Dict[str, dict]:
    """2D drawing order only: red second-to-last, black last.

    Legend order is unchanged because _apply_ordered_two_column_legend()
    still uses CONTROLLER_DISPLAY_ORDER.
    """
    ordered = {}

    # Draw every non-red/non-black curve first, preserving the current order.
    for label, res in results.items():
        if label not in (LABEL_HYBRID_FREEZE, LABEL_DFP_DON_SAC):
            ordered[label] = res

    # Red: second-to-last.
    if LABEL_HYBRID_FREEZE in results:
        ordered[LABEL_HYBRID_FREEZE] = results[LABEL_HYBRID_FREEZE]

    # Black: last.
    if LABEL_DFP_DON_SAC in results:
        ordered[LABEL_DFP_DON_SAC] = results[LABEL_DFP_DON_SAC]

    return ordered




def _legend_kwargs_two_columns(loc="upper center", bbox_to_anchor=None):
    """
    Shared legend style for the displayed methods.
    """
    kwargs = dict(
        fontsize=11,
        frameon=True,
        shadow=True,
        ncol=2,
        columnspacing=1.2,
        handlelength=2.8,
        borderaxespad=0.6,
    )
    kwargs["loc"] = loc
    if bbox_to_anchor is not None:
        kwargs["bbox_to_anchor"] = bbox_to_anchor
    return kwargs


def _apply_ordered_two_column_legend(
    ax,
    *,
    loc="best",
    fontsize=11,
    frameon=True,
    shadow=True,
    columnspacing=1.2,
    handlelength=2.8,
    borderaxespad=0.6,
):
    """Draw a two-column legend ordered from top to bottom by column.

    The displayed order is read vertically rather than horizontally:

        left column:  SAC, SAC-FPC, DO-SAC
        right column: DPSAC, TDPSAC-FPC, TDPSAC
    """
    handles, labels = ax.get_legend_handles_labels()
    if not handles:
        return None

    pair_by_label = {label: (handle, label) for handle, label in zip(handles, labels)}
    ordered_pairs = [
        pair_by_label[label]
        for label in CONTROLLER_DISPLAY_ORDER
        if label in pair_by_label
    ]
    ordered_pairs.extend(
        (handle, label)
        for handle, label in zip(handles, labels)
        if label not in CONTROLLER_DISPLAY_ORDER
    )

    ordered_handles = [pair[0] for pair in ordered_pairs]
    ordered_labels = [pair[1] for pair in ordered_pairs]

    # Matplotlib fills a multi-column legend from top to bottom in each column.
    # Passing CONTROLLER_DISPLAY_ORDER directly therefore gives the requested
    # vertical ordering without row-wise repacking.
    return ax.legend(
        ordered_handles,
        ordered_labels,
        fontsize=fontsize,
        frameon=frameon,
        shadow=shadow,
        loc=loc,
        ncol=2,
        columnspacing=columnspacing,
        handlelength=handlelength,
        borderaxespad=borderaxespad,
    )


def _style_axes_second(ax, xlabel=None, ylabel=None, legend=True, legend_loc="best"):
    """2D axis style matched to the first program."""
    if xlabel is not None:
        ax.set_xlabel(xlabel, fontsize=16, fontweight="bold", labelpad=10)
    if ylabel is not None:
        ax.set_ylabel(ylabel, fontsize=16, fontweight="bold", labelpad=8)
    ax.tick_params(axis="both", labelsize=16)
    for tick in ax.get_xticklabels() + ax.get_yticklabels():
        tick.set_fontweight("bold")
    ax.grid(True, linestyle="--", alpha=0.8)
    if legend:
        _apply_ordered_two_column_legend(ax, loc=legend_loc, fontsize=11)


# Keep old function name for compatibility with the rest of the script.
def _style_axes(ax, xlabel=None, ylabel=None, legend=True, legend_loc="best"):
    _style_axes_second(ax, xlabel=xlabel, ylabel=ylabel, legend=legend, legend_loc=legend_loc)


def _safe_label_for_plot(label):
    """Keep algorithm names exactly the same as the second program."""
    return label


def _common_time_end_min(results: Dict[str, dict]) -> float:
    """Return the common end time in minutes across all results."""
    ends = []
    for res in results.values():
        t = _time_min(res)
        if len(t) > 0 and np.any(np.isfinite(t)):
            ends.append(float(np.nanmax(t)))
    return min(ends) if ends else 0.0


def _style_inset_axes(ax_inset):
    """Inset style copied from the second program."""
    ax_inset.grid(True, linestyle="--", alpha=0.8)
    ax_inset.tick_params(axis="both", labelsize=14)


def _add_second_style_inset(
    ax,
    results: Dict[str, dict],
    y_getter,
    ref_value=None,
    zoom_ratio: float = 0.75,
    bounds=(0.55, 0.15, 0.40, 0.40),
):
    """
    Add a local zoom inset with the same style as the second program:
        ax.inset_axes([...]) + ax.indicate_inset_zoom(...).

    This function only changes plotting. It does not change model loading,
    controller output, reward, rollout data, or saved numerical results.
    """
    min_t_end = _common_time_end_min(results)
    if min_t_end <= 0:
        return None

    zoom_start = float(zoom_ratio) * min_t_end
    ax_inset = ax.inset_axes(list(bounds))

    y_all = []
    results_draw = _ordered_results_for_2d_drawing(results)
    for label, res in results_draw.items():
        style = _line_style_2d(label)
        t = _time_min(res)
        y = np.asarray(y_getter(res), dtype=float).reshape(-1)
        n = min(len(t), len(y))
        tt = t[:n]
        yy = y[:n]
        mask = (tt >= zoom_start) & (tt <= min_t_end)
        if np.any(mask):
            ax_inset.plot(tt[mask], yy[mask], **style)
            y_all.extend(yy[mask][np.isfinite(yy[mask])].tolist())

    if ref_value is not None:
        ax_inset.axhline(ref_value, linestyle="--", linewidth=1.4, color="dimgray", alpha=0.95)
        if np.isfinite(ref_value):
            y_all.append(float(ref_value))

    if y_all:
        y_min, y_max = float(np.min(y_all)), float(np.max(y_all))
        margin = (y_max - y_min) * 0.1 if y_max != y_min else 0.1
        ax_inset.set_ylim(y_min - margin, y_max + margin)
    ax_inset.set_xlim(zoom_start, min_t_end)
    _style_inset_axes(ax_inset)

    # Same guide-line style as the second program.
    try:
        ax.indicate_inset_zoom(ax_inset, edgecolor="black", linewidth=1.5)
    except Exception:
        pass
    return ax_inset


# Backward-compatible alias for old calls.
def _add_end_time_inset(
    ax,
    results: Dict[str, dict],
    y_getter,
    ref_value=None,
    zoom_ratio: float = 0.75,
    bounds=(0.55, 0.15, 0.40, 0.40),
):
    return _add_second_style_inset(
        ax,
        results,
        y_getter,
        ref_value=ref_value,
        zoom_ratio=zoom_ratio,
        bounds=bounds,
    )


def plot_l2_norm_four(results: Dict[str, dict], save_dir: str, tag: str):
    """Controller-comparison L2 plots with the same 2D style as the second program.

    H-Hbar and Q-Qbar are drawn in two separate figures. Other logic remains unchanged.
    """
    results = _ordered_results(results)
    results_draw = _ordered_results_for_2d_drawing(results)
    # --------------------------------------------------------
    # Figure 1: ||H-Hbar||_L2
    # --------------------------------------------------------
    fig_h, ax_h = plt.subplots(
        figsize=set_size(433, 0.99, ncols=1, nrows=1, height_add=1.0),
        dpi=600,
    )

    for label, res in results_draw.items():
        style = _line_style_2d(label)
        plot_label = _safe_label_for_plot(label)
        ax_h.plot(_time_min(res), _get_l2(res, "H"), label=plot_label, **style)

    ax_h.axhline(y=0, color="dimgray", linestyle="--", alpha=0.95, linewidth=1.8)
    _style_axes_second(
        ax_h,
        xlabel=r"$Time$ [min]",
        ylabel=r"$\Vert H-\bar H\Vert_{L^2}$ [m]",
        legend=True,
    )

    fig_h.tight_layout()
    path_h = os.path.join(
        save_dir,
        f"{tag}_six_algorithms_H_L2_norm_error_second_style.png",
    )
    fig_h.savefig(path_h, dpi=600, bbox_inches="tight")
    print(f"Saved six-controller H L2 figure to: {path_h}")
    plt.show()

    # --------------------------------------------------------
    # Figure 2: ||Q-Qbar||_L2
    # --------------------------------------------------------
    fig_q, ax_q = plt.subplots(
        figsize=set_size(433, 0.99, ncols=1, nrows=1, height_add=1.0),
        dpi=600,
    )

    for label, res in results_draw.items():
        style = _line_style_2d(label)
        plot_label = _safe_label_for_plot(label)
        ax_q.plot(_time_min(res), _get_l2(res, "Q"), label=plot_label, **style)

    ax_q.axhline(y=0, color="dimgray", linestyle="--", alpha=0.95, linewidth=1.8)
    _style_axes_second(
        ax_q,
        xlabel=r"$Time$ [min]",
        ylabel=r"$\Vert Q-\bar Q\Vert_{L^2}$ [m$^3$/s]",
        legend=True,
    )

    # Only Q-Qbar keeps local zoom, as previously requested.
    _add_second_style_inset(
        ax_q,
        results,
        lambda res: _get_l2(res, "Q"),
        ref_value=0.0,
        zoom_ratio=0.75,
        bounds=(0.55, 0.15, 0.40, 0.40),
    )

    fig_q.tight_layout()
    path_q = os.path.join(
        save_dir,
        f"{tag}_six_algorithms_Q_L2_norm_error_second_style.png",
    )
    fig_q.savefig(path_q, dpi=600, bbox_inches="tight")
    print(f"Saved six-controller Q L2 figure to: {path_q}")
    plt.show()




def plot_control_input_four(results: Dict[str, dict], save_dir: str, tag: str):
    """U_L(t) plot with the same 2D style as the second program; no inset."""
    results = _ordered_results(results)
    results_draw = _ordered_results_for_2d_drawing(results)
    fig, ax = plt.subplots(
        figsize=set_size(433, 0.99, ncols=1, nrows=1, height_add=1.0),
        dpi=600,
    )

    min_t_end = min(float(np.nanmax(_time_min(res))) for res in results.values())
    for label, res in results_draw.items():
        style = _line_style_2d(label)
        t = _time_min(res)
        U = np.asarray(res["U"], dtype=float)
        n = min(len(t), len(U))
        ax.plot(t[:n], U[:n], label=_safe_label_for_plot(label), **style)

    ax.axhline(y=U_ref, color="dimgray", linestyle="--", alpha=0.95, linewidth=1.8)
    ax.set_xlim(0, min_t_end)
    _style_axes_second(ax, xlabel=r"$Time$ [min]", ylabel=r"$U_L(t)$ [m]", legend=True)

    fig.tight_layout()
    path = os.path.join(
        save_dir,
        f"{tag}_six_algorithms_control_input_second_style.png",
    )
    fig.savefig(path, dpi=600, bbox_inches="tight")
    print(f"Saved six-controller control input figure to: {path}")
    plt.show()










def _format_three_ticklabels(values):
    """Format the three x/y tick labels used by the 3D figures."""
    values = np.asarray(values, dtype=float)
    labels = []
    for value in values:
        if abs(value) >= 10:
            labels.append(f"{value:.0f}")
        elif abs(value) >= 1:
            labels.append(f"{value:.1f}")
        else:
            labels.append(f"{value:.2f}")
    return labels


# Maximum number of recorded time slices used only for 3D rendering.
# The original simulated H/Q trajectories are not filtered or modified.
MAX_3D_TIME_POINTS = 51


def _subsample_3d_time_rows(t_values, Z_values, max_points=MAX_3D_TIME_POINTS):
    """Uniformly subsample recorded time rows for 3D rendering only.

    The first and last recorded times are always retained.
    No smoothing, interpolation, or H/Q post-filtering is performed.
    plot_surface continues to use rstride=1 to avoid coarse-stride artifacts.
    """
    t_values = np.asarray(t_values, dtype=float).reshape(-1)
    Z_values = np.asarray(Z_values, dtype=float)

    if Z_values.ndim != 2:
        raise ValueError("3D surface data must be a two-dimensional array.")

    n = min(t_values.size, Z_values.shape[0])
    if n <= 0:
        raise ValueError("3D surface data contain no time rows.")

    t_values = t_values[:n]
    Z_values = Z_values[:n, :]

    max_points = max(2, int(max_points))
    if n <= max_points:
        return t_values, Z_values

    indices = np.linspace(0, n - 1, max_points, dtype=int)
    indices = np.unique(indices)

    return t_values[indices], Z_values[indices, :]


def _format_z_ticklabels(values):
    """Format z-axis tick labels used by the 3D figures."""
    values = np.asarray(values, dtype=float)
    span = float(np.nanmax(values) - np.nanmin(values)) if len(values) > 0 else 0.0
    if span >= 10:
        return [f"{v:.0f}" for v in values]
    if span >= 1:
        return [f"{v:.1f}" for v in values]
    return [f"{v:.2f}" for v in values]


def _surface_plot_second_style(res, var: str, title: str, save_path: str):
    """Draw one independent colored 3D surface with the previous 3D style.

    A uniform subset of recorded time rows is used only for 3D rendering.
    The simulated H/Q data are not smoothed or post-filtered.
    """
    X = np.asarray(res["X"], dtype=float)
    t_min = _time_min(res)
    Z = np.asarray(res[var], dtype=float)

    # Reduce only the number of rendered time rows.
    # Keep rstride=1 below so adjacent displayed rows remain contiguous.
    t_min, Z = _subsample_3d_time_rows(t_min, Z)

    spatial = X / 1000.0
    meshx, mesht = np.meshgrid(spatial, t_min)

    fig = plt.figure(figsize=(8.8, 6.6), dpi=600)
    subfigs = fig.subfigures(nrows=1, ncols=1, hspace=0)
    subfig = subfigs
    subfig.subplots_adjust(left=0.18, bottom=0.08, right=0.98, top=0.96)

    axes = subfig.subplots(
        nrows=1,
        ncols=1,
        subplot_kw={"projection": "3d", "computed_zorder": False},
    )

    for axis in [axes.xaxis, axes.yaxis, axes.zaxis]:
        axis._axinfo["axisline"]["linewidth"] = 10
        axis._axinfo["axisline"]["color"] = "b"
        axis._axinfo["grid"]["linewidth"] = 0.8
        axis._axinfo["grid"]["linestyle"] = "--"
        axis._axinfo["grid"]["color"] = "#d1d1d1"
        axis.set_pane_color((1, 1, 1))

    # Keep exactly the same viewing angle as the previous 3D figures.
    axes.view_init(10, 15)

    axes.set_xlabel(r"$x$ [km]", fontsize=16, labelpad=5, fontweight="bold")
    axes.set_ylabel(r"$Time$ [min]", fontsize=16, labelpad=5, fontweight="bold")

    if var == "H":
        zlabel = r"$H(t,x)$ [m]"
        surface_cmap = "viridis"
    else:
        zlabel = r"$Q(t,x)$ [m$^3$/s]"
        surface_cmap = "plasma"

    axes.set_zlabel(
        zlabel,
        rotation=90,
        fontsize=16,
        labelpad=4,
        fontweight="bold",
    )
    axes.zaxis.set_rotate_label(False)
    axes.zaxis.label.set_clip_on(False)

    x_ticks = np.array(
        [spatial[0], 0.5 * (spatial[0] + spatial[-1]), spatial[-1]],
        dtype=float,
    )
    y_ticks = np.array(
        [t_min[0], 0.5 * (t_min[0] + t_min[-1]), t_min[-1]],
        dtype=float,
    )
    z_min, z_max = float(np.nanmin(Z)), float(np.nanmax(Z))
    z_ticks = np.linspace(z_min, z_max, 5)

    axes.set_xticks(x_ticks)
    axes.set_yticks(y_ticks)
    axes.set_zticks(z_ticks)

    axes.set_xticklabels(
        _format_three_ticklabels(x_ticks),
        fontweight="bold",
        fontsize=16,
    )
    axes.set_yticklabels(
        _format_three_ticklabels(y_ticks),
        fontweight="bold",
        fontsize=16,
    )
    axes.set_zticklabels(
        _format_z_ticklabels(z_ticks),
        fontweight="bold",
        fontsize=16,
    )

    # Use every row of the already subsampled 3D data.
    # rstride=1 avoids large time-direction polygons and artificial spikes.
    rstride = 1
    axes.plot_surface(
        meshx,
        mesht,
        Z,
        edgecolor="black",
        lw=0.8,
        rstride=rstride,
        cstride=5,
        alpha=1,
        cmap=surface_cmap,
        shade=False,
        rasterized=True,
        antialiased=True,
    )

    axes.set_title(_safe_label_for_plot(title), fontsize=14)

    if save_path is not None:
        fig.savefig(
            save_path,
            dpi=600,
            bbox_inches="tight",
            pad_inches=0.35,
        )
        print(f"3D surface saved to: {save_path}")

    plt.show()
    plt.close(fig)


def plot_3d_surfaces_with_backstepping(
    learned_results: Dict[str, dict],
    backstepping_result: dict,
    save_dir: str,
    tag: str,
):
    """Generate separate H/Q 3D figures for six learned methods plus Backstepping.

    Important:
        Backstepping is included here ONLY for the 3D outputs.
        The retained 2D comparison functions continue to receive learned_results
        only, so no Backstepping curve is added to any 2D figure.
    """
    learned_results = _ordered_results(learned_results)

    results_3d: Dict[str, dict] = {
        LABEL_BACKSTEPPING: backstepping_result,
    }
    for label, res in learned_results.items():
        results_3d[label] = res

    # Produce H and Q independently for every controller.
    for var in ["H", "Q"]:
        for label, res in results_3d.items():
            safe_label = (
                label.replace("+", "_plus_")
                .replace(" ", "_")
                .replace("/", "_")
            )
            path = os.path.join(
                save_dir,
                f"{tag}_{safe_label}_{var}_3D_surface_second_style.png",
            )
            _surface_plot_second_style(res, var, label, path)


def save_npz_four(results: Dict[str, dict], save_dir: str, tag: str):
    results = _ordered_results(results)
    arrays = {}
    for label, res in results.items():
        safe = label.replace("+", "_plus_").replace(" ", "_").replace("/", "_")
        arrays[f"t_{safe}"] = res["t"]
        arrays[f"H_{safe}"] = res["H"]
        arrays[f"Q_{safe}"] = res["Q"]
        arrays[f"U_{safe}"] = res["U"]
        if "U_raw" in res:
            arrays[f"U_raw_{safe}"] = res["U_raw"]
        arrays[f"reward_{safe}"] = res["reward"]
        arrays[f"total_reward_{safe}"] = np.asarray(res["total_reward"])

    first = next(iter(results.values()))
    arrays["X"] = first["X"]
    arrays["Hbar"] = first["Hbar"]
    arrays["Qbar"] = first["Qbar"]

    e_H0_rel, e_Q0_rel = _initial_relative_deviation_percentages_from_result(first)
    arrays["e_H0_rel_pct"] = np.asarray(e_H0_rel)
    arrays["e_Q0_rel_pct"] = np.asarray(e_Q0_rel)

    path = os.path.join(save_dir, f"{tag}_controller_comparison.npz")
    np.savez(path, **arrays)
    print(f"Saved six-controller data to: {path}")



def print_final_summary_four(results: Dict[str, dict]):
    results = _ordered_results(results)
    print("\n" + "=" * 90)
    print("Final summary of controller comparison")
    print("=" * 90)
    for label, res in results.items():
        H = np.asarray(res["H"])
        Q = np.asarray(res["Q"])
        Hbar = np.asarray(res["Hbar"])
        Qbar = np.asarray(res["Qbar"])
        H_l2_final = _get_l2(res, "H")[-1]
        Q_l2_final = _get_l2(res, "Q")[-1]

        print(f"\n[{label}]")
        print(f"final H(0) = {H[-1,0]:.6f}, target = {Hbar[0]:.6f}, error = {H[-1,0]-Hbar[0]:.6f}")
        print(f"final H(L) = {H[-1,-1]:.6f}, target = {Hbar[-1]:.6f}, error = {H[-1,-1]-Hbar[-1]:.6f}")
        print(f"final Q(0) = {Q[-1,0]:.6f}, target = {Qbar[0]:.6f}, error = {Q[-1,0]-Qbar[0]:.6f}")
        print(f"final Q(L) = {Q[-1,-1]:.6f}, target = {Qbar[-1]:.6f}, error = {Q[-1,-1]-Qbar[-1]:.6f}")
        print(f"final ||H-Hbar||_L2 = {H_l2_final:.6f} m")
        print(f"final ||Q-Qbar||_L2 = {Q_l2_final:.6f} m^3/s")
        print(f"total_reward = {res['total_reward']:.6f}")



# ============================================================
# Shared initial-condition validation helpers
# ============================================================

def _initial_relative_deviation_percentages_from_arrays(
    X: np.ndarray,
    H0: np.ndarray,
    Q0: np.ndarray,
    Hbar: np.ndarray,
    Qbar: np.ndarray,
) -> Tuple[float, float]:
    """
    Compute the shared initial relative deviations:

        e_H0_rel = ||H0-Hbar||_L2 / ||Hbar||_L2 * 100%
        e_Q0_rel = ||Q0-Qbar||_L2 / ||Qbar||_L2 * 100%.

    The same normalized spatial coordinate s=x/L is used in every norm.
    Because numerator and denominator use the same convention, the ratio is
    independent of whether one writes the normalized or unnormalized L2 norm.
    """
    X = np.asarray(X, dtype=float).reshape(-1)
    H0 = np.asarray(H0, dtype=float).reshape(-1)
    Q0 = np.asarray(Q0, dtype=float).reshape(-1)
    Hbar = np.asarray(Hbar, dtype=float).reshape(-1)
    Qbar = np.asarray(Qbar, dtype=float).reshape(-1)

    if (
        X.size < 2
        or H0.size != X.size
        or Q0.size != X.size
        or Hbar.size != X.size
        or Qbar.size != X.size
    ):
        return np.nan, np.nan

    s = X / float(X[-1])

    H_err_norm = float(np.sqrt(np.trapezoid((H0 - Hbar) ** 2, s)))
    Q_err_norm = float(np.sqrt(np.trapezoid((Q0 - Qbar) ** 2, s)))
    H_ref_norm = float(np.sqrt(np.trapezoid(Hbar ** 2, s)))
    Q_ref_norm = float(np.sqrt(np.trapezoid(Qbar ** 2, s)))

    e_H0_rel = (
        100.0 * H_err_norm / H_ref_norm
        if np.isfinite(H_ref_norm) and H_ref_norm > 1e-12
        else np.nan
    )
    e_Q0_rel = (
        100.0 * Q_err_norm / Q_ref_norm
        if np.isfinite(Q_ref_norm) and Q_ref_norm > 1e-12
        else np.nan
    )
    return float(e_H0_rel), float(e_Q0_rel)


def _initial_relative_deviation_percentages_from_result(
    res: dict,
) -> Tuple[float, float]:
    """Read the common t=0 state from one rollout result and compute both percentages."""
    H = np.asarray(res.get("H", []), dtype=float)
    Q = np.asarray(res.get("Q", []), dtype=float)
    if H.ndim != 2 or Q.ndim != 2 or H.shape[0] == 0 or Q.shape[0] == 0:
        return np.nan, np.nan
    return _initial_relative_deviation_percentages_from_arrays(
        X=np.asarray(res.get("X", []), dtype=float),
        H0=H[0],
        Q0=Q[0],
        Hbar=np.asarray(res.get("Hbar", []), dtype=float),
        Qbar=np.asarray(res.get("Qbar", []), dtype=float),
    )



def validate_shared_reference_and_initial_state(
    results: Dict[str, dict],
    X_expected: np.ndarray,
    Hbar_expected: np.ndarray,
    Qbar_expected: np.ndarray,
    H0_expected: np.ndarray,
    Q0_expected: np.ndarray,
    save_dir: str,
    tag: str,
    atol: float = 1e-10,
) -> Dict[str, Dict[str, float]]:
    """Verify that every controller uses exactly the same test problem.

    This check is intentionally strict.  It compares, controller by controller,
    the spatial grid, ODE equilibrium, reset state, and the initial relative-L2
    percentages against the single profiles generated in ``main``.

    A mismatch raises ``RuntimeError`` before figures are produced, preventing a
    visually plausible but unfair comparison from being saved.
    """
    X_expected = np.asarray(X_expected, dtype=float).reshape(-1)
    Hbar_expected = np.asarray(Hbar_expected, dtype=float).reshape(-1)
    Qbar_expected = np.asarray(Qbar_expected, dtype=float).reshape(-1)
    H0_expected = np.asarray(H0_expected, dtype=float).reshape(-1)
    Q0_expected = np.asarray(Q0_expected, dtype=float).reshape(-1)

    expected_eH, expected_eQ = _initial_relative_deviation_percentages_from_arrays(
        X=X_expected,
        H0=H0_expected,
        Q0=Q0_expected,
        Hbar=Hbar_expected,
        Qbar=Qbar_expected,
    )

    report: Dict[str, Dict[str, float]] = {}
    violations = []

    for label, res in results.items():
        X = np.asarray(res.get("X", []), dtype=float).reshape(-1)
        Hbar = np.asarray(res.get("Hbar", []), dtype=float).reshape(-1)
        Qbar = np.asarray(res.get("Qbar", []), dtype=float).reshape(-1)
        H = np.asarray(res.get("H", []), dtype=float)
        Q = np.asarray(res.get("Q", []), dtype=float)
        t = np.asarray(res.get("t", []), dtype=float).reshape(-1)

        if (
            X.shape != X_expected.shape
            or Hbar.shape != Hbar_expected.shape
            or Qbar.shape != Qbar_expected.shape
            or H.ndim != 2
            or Q.ndim != 2
            or H.shape[0] == 0
            or Q.shape[0] == 0
            or H.shape[1] != X_expected.size
            or Q.shape[1] != X_expected.size
            or t.size == 0
        ):
            violations.append(f"{label}: incompatible/missing result-array shapes")
            continue

        metrics = {
            "max_abs_X_difference": float(np.max(np.abs(X - X_expected))),
            "max_abs_Hbar_difference": float(np.max(np.abs(Hbar - Hbar_expected))),
            "max_abs_Qbar_difference": float(np.max(np.abs(Qbar - Qbar_expected))),
            "max_abs_H0_difference": float(np.max(np.abs(H[0] - H0_expected))),
            "max_abs_Q0_difference": float(np.max(np.abs(Q[0] - Q0_expected))),
            "initial_time_s": float(t[0]),
        }
        eH, eQ = _initial_relative_deviation_percentages_from_arrays(
            X=X,
            H0=H[0],
            Q0=Q[0],
            Hbar=Hbar,
            Qbar=Qbar,
        )
        metrics["e_H0_rel_pct"] = float(eH)
        metrics["e_Q0_rel_pct"] = float(eQ)
        metrics["e_H0_rel_difference_pct"] = float(abs(eH - expected_eH))
        metrics["e_Q0_rel_difference_pct"] = float(abs(eQ - expected_eQ))
        report[label] = metrics

        for key in (
            "max_abs_X_difference",
            "max_abs_Hbar_difference",
            "max_abs_Qbar_difference",
            "max_abs_H0_difference",
            "max_abs_Q0_difference",
        ):
            if not np.isfinite(metrics[key]) or metrics[key] > atol:
                violations.append(f"{label}: {key}={metrics[key]:.3e} > {atol:.3e}")

        if abs(metrics["initial_time_s"]) > atol:
            violations.append(
                f"{label}: first recorded time is {metrics['initial_time_s']:.12g} s, not 0 s"
            )
        if metrics["e_H0_rel_difference_pct"] > 1e-9:
            violations.append(
                f"{label}: e_H0_rel differs by {metrics['e_H0_rel_difference_pct']:.3e}%"
            )
        if metrics["e_Q0_rel_difference_pct"] > 1e-9:
            violations.append(
                f"{label}: e_Q0_rel differs by {metrics['e_Q0_rel_difference_pct']:.3e}%"
            )

    # Figure-only mode:
    # retain the strict validation and RuntimeError on mismatch,
    # but do not save a validation TXT file or print a validation summary.

    if violations:
        raise RuntimeError(
            "The controller comparison does not use an identical reference/initial state:\n"
            + "\n".join(f"- {item}" for item in violations)
        )

    return report


def save_backstepping_diagnostic_data(
    res_bs: dict,
    save_dir: str,
    tag: str,
) -> None:
    """Save the BS trajectory separately, independent of multi-curve autoscaling.

    The NPZ makes it straightforward to compare this run numerically with an
    earlier two-controller run.  If the same environment file, seed, reference,
    and initial-condition parameters are used, the arrays should match up to
    floating-point roundoff.
    """
    path = Path(save_dir) / f"{tag}_BS_only_diagnostic.npz"
    np.savez(
        path,
        X=np.asarray(res_bs["X"]),
        Hbar=np.asarray(res_bs["Hbar"]),
        Qbar=np.asarray(res_bs["Qbar"]),
        t=np.asarray(res_bs["t"]),
        H=np.asarray(res_bs["H"]),
        Q=np.asarray(res_bs["Q"]),
        U=np.asarray(res_bs["U"]),
        H_l2=_get_l2(res_bs, "H"),
        Q_l2=_get_l2(res_bs, "Q"),
    )
    print(f"Backstepping-only diagnostic data saved to: {path}")






















# ============================================================
# Main: six-controller comparison
# ============================================================


DEFAULT_NEW_MODEL_PATH = (
     "./sactx/seed_42/"
    "sac_cac_hybrid_yul_ode_cheb_raw_HQ_100000_steps"
) 
DEFAULT_TDPSAC_FPC_MODEL_PATH = (
    "./sacddx/seed_42/"
    "sac_cac_hybrid_fixed_control_yul_ode_cheb_raw_HQ_100000_steps" 
)
DEFAULT_DPSAC_MODEL_PATH = (
    "./saccontrolx/seed_42/"
    "sac_cac_new_hadamard_hybrid_all_trainable_raw_HQ_100000_steps"
)
DEFAULT_DO_SAC_MODEL_PATH = (
    "./sactrainx/seed_42/"
    "sac_cac_deeponet_feature_raw_hq_ode_cheb_100000_steps"
)
DEFAULT_SAC_FPC_MODEL_PATH = (
    "./sacnoxv/seed_42/"
    "sac_fnn_frozen_new_deeponet_prior_raw_HQ_100000_steps"
)
# Newly trained ordinary SAC checkpoint using raw physical H/Q observations.
DEFAULT_SAC_MODEL_PATH = (
    "./saczx/seed_42/"
    "sac_cac_raw_hq_ode_cheb_100000_steps")

DEFAULT_SAVE_DIR_COMPARISON = "./test_results_six_algorithms_new_hadamard_linear_IC"

# Display/run order follows the old multi-controller comparison hierarchy.
LEARNED_CONTROLLER_SPECS = [
    {
        "label": LABEL_SAC_CURVE,          # SAC: newly trained raw-HQ model
        "model_kind": "sac",
        "required": True,
    },
    {
        "label": LABEL_FP_SAC,             # SAC-FPC
        "model_kind": "auto",
        "required": True,
    },
    {
        "label": LABEL_SAC,                # DO-SAC
        "model_kind": "auto",
        "required": True,
    },
    {
        "label": LABEL_HYBRID,             # DPSAC
        "model_kind": "auto",
        "required": True,
    },
    {
        "label": LABEL_DFP_DON_SAC,        # TDPSAC-FPC
        "model_kind": "auto",
        "required": True,
    },
    {
        "label": LABEL_NEW_MODEL,           # TDPSAC
        "model_kind": "hybrid",
        "required": True,
    },
]


def _run_optional_controller(
    results: Dict[str, dict],
    *,
    label: str,
    model_path: str,
    model_kind: str,
    seed: int,
    H_left: float,
    H_right: float,
    Q_value: float,
    dH_source,
    required: bool = False,
):
    """Run one learned controller and keep the common result-data interface."""
    if not _checkpoint_exists(model_path):
        message = (
            f"No valid SB3 ZIP checkpoint was found for {label}.\n"
            f"Current path: {model_path!r}\n"
            "Expected either '/path/model' with '/path/model.zip' present, "
            "or a direct '/path/model.zip' path."
        )
        if required:
            raise FileNotFoundError(message)
        print(f"\n[SKIP] {message}")
        return

    print(f"\nRunning {label}...")
    results[label] = run_policy_exact(
        model_path=model_path,
        model_kind=model_kind,
        label=label,
        seed=seed,
        H_left=H_left,
        H_right=H_right,
        Q_value=Q_value,
        dH_source=dH_source,
    )


def _clean_runtime_argv(argv=None):
    """Remove Jupyter/IPython kernel connection arguments before argparse."""
    raw = list(sys.argv[1:] if argv is None else argv)
    cleaned = []
    i = 0

    while i < len(raw):
        arg = str(raw[i])

        if arg in {"-f", "--f"} and i + 1 < len(raw):
            candidate = str(raw[i + 1])
            if candidate.endswith(".json") and (
                "jupyter" in candidate.lower()
                or "kernel-" in Path(candidate).name.lower()
            ):
                i += 2
                continue

        if arg.startswith(("--f=", "-f=")):
            candidate = arg.split("=", 1)[1]
            if candidate.endswith(".json") and (
                "jupyter" in candidate.lower()
                or "kernel-" in Path(candidate).name.lower()
            ):
                i += 1
                continue

        if arg.endswith(".json") and (
            "jupyter" in arg.lower()
            or "kernel-" in Path(arg).name.lower()
        ):
            i += 1
            continue

        cleaned.append(arg)
        i += 1

    return cleaned



def main():
    parser = argparse.ArgumentParser(
        allow_abbrev=False,
        description=(
            "Compare new raw-HQ SAC, SAC-FPC, DO-SAC, DPSAC, "
            "TDPSAC-FPC, and TDPSAC under one shared ODE-equilibrium/linear-IC test."
        ),
    )
    parser.add_argument(
        "--new-model",
        "--tdpsac-model",
        dest="new_model",
        type=str,
        default=DEFAULT_NEW_MODEL_PATH,
        help="Checkpoint of the latest partially frozen TDPSAC model.",
    )
    parser.add_argument(
        "--tdpsac-fpc-model",
        type=str,
        default=DEFAULT_TDPSAC_FPC_MODEL_PATH,
        help="Checkpoint of TDPSAC-FPC with an independent frozen control prior.",
    )
    parser.add_argument(
        "--dpsac-model",
        type=str,
        default=DEFAULT_DPSAC_MODEL_PATH,
        help="Checkpoint of the all-trainable Hadamard DeepONet-SAC model.",
    )
    parser.add_argument(
        "--do-sac-model",
        type=str,
        default=DEFAULT_DO_SAC_MODEL_PATH,
        help="Checkpoint of the DeepONet-feature-only SAC model.",
    )
    parser.add_argument(
        "--sac-fpc-model",
        type=str,
        default=DEFAULT_SAC_FPC_MODEL_PATH,
        help="Checkpoint of ordinary SAC with a frozen DeepONet action prior.",
    )
    parser.add_argument(
        "--sac-model",
        type=str,
        default=DEFAULT_SAC_MODEL_PATH,
        help=(
            "Checkpoint of the newly trained ordinary SAC model using "
            "raw physical H/Q observations."
        ),
    )
    parser.add_argument(
        "--out-dir",
        type=str,
        default=DEFAULT_SAVE_DIR_COMPARISON,
    )
    parser.add_argument(
        "--seed",
        type=int,
        default=123,
        help="Shared deterministic environment/model evaluation seed.",
    )
    parser.add_argument(
        "--H-left",
        type=float,
        default=TEST_H_LEFT,
        help="Initial upstream water depth H(0,0) [m].",
    )
    parser.add_argument(
        "--H-right",
        type=float,
        default=TEST_H_RIGHT,
        help="Initial downstream water depth H(L,0) [m].",
    )
    parser.add_argument(
        "--Q-value",
        type=float,
        default=TEST_Q_VALUE,
        help="Spatially constant initial discharge Q(x,0) [m^3/s].",
    )
    parser.add_argument("--dH-source", type=float, default=None)
    args, unknown_args = parser.parse_known_args(_clean_runtime_argv())
    if unknown_args:
        print(f"Ignoring unrecognized runtime arguments: {unknown_args}")

    save_dir = Path(args.out_dir)
    save_dir.mkdir(parents=True, exist_ok=True)
    globals()["SAVE_DIR"] = str(save_dir)

    dH_tag = "default" if args.dH_source is None else f"{args.dH_source:.0e}"
    tag = (
        f"case_linear_H{args.H_left:g}_{args.H_right:g}_"
        f"Q{args.Q_value:g}_seed{args.seed}_src{dH_tag}_"
        "six_algorithm_comparison"
    )
    tag = tag.replace(".", "p").replace("-", "m").replace("+", "")

    # Build and report the exact ODE reference and the one common initial state.
    p_ref, X_ref, dx_ref, Hbar_ref, Qbar_ref = build_collection_matched_reference()
    H0_ref, Q0_ref, ic_details = generate_test_initial_condition(
        X_ref,
        Hbar_ref,
        Qbar_ref,
        H_left=args.H_left,
        H_right=args.H_right,
        Q_value=args.Q_value,
    )
    e_H0_rel, e_Q0_rel = _initial_relative_deviation_percentages_from_arrays(
        X=X_ref,
        H0=H0_ref,
        Q0=Q0_ref,
        Hbar=Hbar_ref,
        Qbar=Qbar_ref,
    )

    print("=" * 110)
    print("Six-controller Saint-Venant comparison")
    print("=" * 110)
    print(f"1. {LABEL_SAC_CURVE}: {args.sac_model}")
    print(f"2. {LABEL_FP_SAC}: {args.sac_fpc_model}")
    print(f"3. {LABEL_SAC}: {args.do_sac_model}")
    print(f"4. {LABEL_HYBRID}: {args.dpsac_model}")
    print(f"5. {LABEL_DFP_DON_SAC}: {args.tdpsac_fpc_model}")
    print(f"6. {LABEL_NEW_MODEL}: {args.new_model}")
    print(f"Test horizon: {SAC_T} s")
    print(
        "RL control low-pass filter: "
        f"alpha={RL_CONTROL_FILTER_ALPHA:.3f} "
        "(Backstepping unfiltered)"
    )
    print(
        "Policy observations: all learned controllers use raw physical "
        f"[H,Q]. Dimension={RAW_STATE_DIM}."
    )
    print(
        "ODE equilibrium: "
        f"Hbar(0)={Hbar_ref[0]:.9f}, "
        f"Hbar(L)={Hbar_ref[-1]:.9f}, "
        f"Qbar={Qbar_ref[0]:.9f}, dx={dx_ref:.6f}"
    )
    print(
        "Linear test initial condition: "
        f"H(x,0)={args.H_left:g}+({args.H_right:g}-{args.H_left:g})x/L m, "
        f"Q(x,0)={args.Q_value:g} m^3/s"
    )
    print(
        "Maximum initial deviations from the ODE equilibrium: "
        f"max|H0-Hbar|={ic_details['H_realized_max_deviation']:.9f} m, "
        f"max|Q0-Qbar|={ic_details['Q_realized_max_deviation']:.9f} m^3/s"
    )
    print(
        "Initial-state ranges: "
        f"H0 in [{np.min(H0_ref):.9f}, {np.max(H0_ref):.9f}] m, "
        f"Q0 in [{np.min(Q0_ref):.9f}, {np.max(Q0_ref):.9f}] m^3/s"
    )
    print(
        "Initial relative deviations: "
        f"e_H,0^rel={e_H0_rel:.9f}%, "
        f"e_Q,0^rel={e_Q0_rel:.9f}%"
    )
    print(f"Source disturbance: {args.dH_source}")
    print(f"Output directory: {save_dir}")
    print("=" * 110)

    results: Dict[str, dict] = {}

    controller_paths = {
        LABEL_NEW_MODEL: args.new_model,
        LABEL_DFP_DON_SAC: args.tdpsac_fpc_model,
        LABEL_HYBRID: args.dpsac_model,
        LABEL_FP_SAC: args.sac_fpc_model,
        LABEL_SAC: args.do_sac_model,
        LABEL_SAC_CURVE: args.sac_model,
    }

    # Run and insert the six learned controllers in the exact order requested
    # for the retained 2D legends.
    for spec in LEARNED_CONTROLLER_SPECS:
        _run_optional_controller(
            results,
            label=spec["label"],
            model_path=controller_paths[spec["label"]],
            model_kind=spec["model_kind"],
            required=bool(spec.get("required", False)),
            seed=args.seed,
            H_left=args.H_left,
            H_right=args.H_right,
            Q_value=args.Q_value,
            dH_source=args.dH_source,
        )

    results = _ordered_results(results)

    # Strictly verify that every learned controller uses the same physical
    # test problem, ODE reference, and initial state.
    validate_shared_reference_and_initial_state(
        results=results,
        X_expected=X_ref,
        Hbar_expected=Hbar_ref,
        Qbar_expected=Qbar_ref,
        H0_expected=H0_ref,
        Q0_expected=Q0_ref,
        save_dir=str(save_dir),
        tag=tag,
    )
    # Figure-only mode: no terminal summary and no NPZ/DAT/TXT result files.

    # --------------------------------------------------------
    # 2D figures: six learned controllers ONLY.
    # Backstepping is intentionally not passed to these functions.
    # --------------------------------------------------------
    plot_control_input_four(results, str(save_dir), tag)
    plot_l2_norm_four(results, str(save_dir), tag)

    # --------------------------------------------------------
    # 3D figures: six learned controllers + analytical Backstepping.
    # Backstepping is run only here so it cannot enter the 2D figures.
    # --------------------------------------------------------
    print(f"\nRunning {LABEL_BACKSTEPPING} for 3D surfaces only...")
    res_backstepping_3d = run_backstepping_for_four(
        seed=args.seed,
        H_left=args.H_left,
        H_right=args.H_right,
        Q_value=args.Q_value,
        dH_source=args.dH_source,
    )

    plot_3d_surfaces_with_backstepping(
        learned_results=results,
        backstepping_result=res_backstepping_3d,
        save_dir=str(save_dir),
        tag=tag,
    )



if __name__ == "__main__":
    # Suppress stdout/stderr messages emitted during model/environment setup.
    # Matplotlib figures are still saved and displayed with plt.show().
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
        main()


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
# [MODIFIED] Keep 2D plots as tracking-error L2 plots; add/standardize equilibrium lines.
# [MODIFIED] M_H/M_Q use classical full-state maximum positive overshoot; all other settings unchanged.
"""
Unified controller comparison for Saint-Venant canal regulation.

Monte-Carlo setting in this version: 50 trials, with 25 cases using H above/Q below
and 25 cases using H below/Q above the target. The realized relative-L2 initial
deviations satisfy delta_H,0 in [1%,2%] and delta_Q,0 in [16%,17%].

Purpose:
    Compare seven controllers under one shared Saint-Venant test condition:
    TDPSAC, TDPSAC-FPC, DPSAC, DO-SAC, SAC-FPC, SAC, and analytical
    Backstepping. Only the final SS_H, SS_Q, M_H, and M_Q summary table is output.

Method mapping:
    1) TDPSAC:
       the latest partially frozen Hadamard DeepONet-SAC controller.
    2) TDPSAC-FPC:
       DeepONet actor/critic features plus an independent frozen prior controller.
    3) DPSAC:
       Hadamard DeepONet-SAC with all DeepONet parameters fine-tuned.
    4) DO-SAC:
       Hadamard DeepONet is used only as the SAC feature extractor.
    5) SAC-FPC:
       ordinary SAC actor/critic plus an independent frozen DeepONet prior.
    6) SAC:
       newly trained ordinary SAC without DeepONet, using the raw physical H/Q state.
    7) BS:
       analytical Backstepping controller.

All learned controllers receive the raw physical state
    [H_1,...,H_N,Q_1,...,Q_N],
with the same ODE equilibrium and the same physical initial condition.

Reference and test initial condition:
    - Qbar(x) is fixed at 11.93 m^3/s.
    - Hbar(L) is obtained from the downstream equilibrium gate relation.
    - Hbar(x) is obtained by solving the steady Saint-Venant ODE backward from
      x=L to x=0 with scipy.integrate.solve_ivp.
    - Every controller uses the same deterministic linear initial condition:

          H(x,0) = 3.75 + (4.65 - 3.75) x/L,
          Q(x,0) = 10.0.

      The initial condition is independent of the random seed. The seed is kept
      only for deterministic environment/model evaluation.

Extension:
    Additional learned controllers can be added later through
    LEARNED_CONTROLLER_SPECS without changing the common plotting and table
    functions.
"""

from __future__ import annotations

import argparse
import contextlib
import csv
import io
import os
import sys
import warnings
import zipfile
from pathlib import Path
from typing import Tuple, Dict, Optional

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from gymnasium import spaces
from scipy.integrate import solve_ivp
from stable_baselines3 import SAC
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from stable_baselines3.sac.policies import Actor, SACPolicy

from cac_saint_venant_env_no_bs import (
    CACParams,
    CACSaintVenantPDE1D,
    source,
)

try:
    from pde_control_gym.src.rewards.base_reward import BaseReward
except Exception:
    class BaseReward:
        pass


# ============================================================
# Common physical / reward settings
# ============================================================

N = 102
dt = 5.0
control_sample_rate = 400.0

max_state_value = 1e6

# The learned controller emits a normalized action in [-1,1]. The environment
# maps it to this physical downstream-gate interval. These limits are identical
# to the corresponding raw-HQ training program.
min_control_value = 0.15
max_control_value = 0.40

# H and Q remain raw physical states in the policy observation. These scales are
# used only inside the reward calculation.
H_scale = 0.30
Q_scale = 3.00
U_scale = 0.10
obs_clip = 50.0

cfl = 0.45
check_cfl = True
well_balanced = True

# Collection-matched equilibrium.
Q_BAR_FIXED = 11.93
REFERENCE_MODE = "ode"
U_ref = 0.30

# First-order low-pass filter applied to every learned/RL controller output.
# The filtered command is passed to env.step(), so it is the actual
# closed-loop control input. Analytical Backstepping remains unfiltered.
#
#   U_k^f = (1-alpha_f) U_{k-1}^f + alpha_f U_k^RL
#
RL_CONTROL_FILTER_ALPHA = 0.70

# Shared deterministic linear test initial condition.
TEST_H_LEFT = 3.75
TEST_H_RIGHT = 4.65
TEST_Q_VALUE = 10

# Reward used by the new training program:
# r_k = -alpha*tracking - beta*state_rate
#       -gamma*control - delta*delta_control.
reward_alpha = 1.0
reward_beta = 0.10
reward_gamma = 0.05
reward_delta = 0.10
reward_w_H = 1.0
reward_w_Q = 1.0
terminal_tracking_weight = 0
truncate_penalty = -200.0

# Keep the test horizon and plotting/table workflow consistent with the previous
# unified comparison program.
SAC_T = 80000.0
BS_T = SAC_T

BACKSTEPPING_GAIN = 1.0

learning_rate = 1e-4
deeponet_learning_rate = 1e-6

LATENT_DIM = 102
RAW_STATE_DIM = 2 * N
SAC_FEATURE_DIM = LATENT_DIM

# This file is only needed while the policy object is being constructed. The
# trained SB3 checkpoint subsequently restores all actor/critic/DeepONet weights.
DEEPNET_PRETRAINED_PATH = "modedlNet10000QqQqQ14444_hadamard.pt"

# ============================================================
# New Hadamard DeepONet controller and Hybrid SAC modules
# ============================================================

def preprocess_obs(obs: torch.Tensor) -> torch.Tensor:
    if not isinstance(obs, torch.Tensor):
        obs = torch.as_tensor(obs)
    return obs.float()


class BranchNet(nn.Module):
    def __init__(self, shape=N, output_dim=LATENT_DIM):
        super().__init__()
        self.shape = int(shape)
        self.output_dim = int(output_dim)

        self.conv1 = nn.Conv1d(
            in_channels=2,
            out_channels=16,
            kernel_size=5,
            stride=2,
        )
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv1d(
            in_channels=16,
            out_channels=32,
            kernel_size=5,
            stride=2,
        )

        with torch.no_grad():
            tmp = torch.zeros(1, 2, self.shape)
            tmp = self.relu(self.conv1(tmp))
            tmp = self.relu(self.conv2(tmp))
            flat_dim = tmp.reshape(1, -1).shape[1]

        self.fc1 = nn.Linear(flat_dim, self.output_dim)

    def forward(self, x):
        x = preprocess_obs(x)
        if x.dim() == 1:
            x = x.unsqueeze(0)

        if x.shape[1] != 2 * self.shape:
            raise ValueError(
                f"Branch input dimension is {x.shape[1]}, expected {2 * self.shape}."
            )

        x = x.reshape(x.shape[0], 2, self.shape)
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = torch.flatten(x, start_dim=1)
        x = self.relu(self.fc1(x))
        return x


class TrunkNet(nn.Module):
    def __init__(self, dim_x=1, width=LATENT_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim_x, 16),
            nn.ReLU(),
            nn.Linear(16, 32),
            nn.ReLU(),
            nn.Linear(32, width),
            nn.ReLU(),
        )

    def forward(self, x):
        return self.net(x.float())


class DeepONetModified(nn.Module):
    """
    Hadamard-fusion DeepONet controller.

        b(s) in R^p,
        t(x) in R^p,
        z(s,x) = b(s) * t(x) in R^p,
        U_prior = f_ctrl(z) in R.
    """

    def __init__(self, shape=N, latent_dim=LATENT_DIM):
        super().__init__()
        self.latent_dim = int(latent_dim)

        self.branch = BranchNet(shape, output_dim=self.latent_dim)
        self.trunk = TrunkNet(dim_x=1, width=self.latent_dim)

        # Hadamard fusion preserves latent_dim features.
        self.fc3 = nn.Linear(self.latent_dim, 32)
        self.relu = nn.ReLU()
        self.fc4 = nn.Linear(32, 16)
        self.fc5 = nn.Linear(16, 1)

    def extract_latent(self, obs, grid):
        b = self.branch(obs)       # [batch, latent_dim]
        t = self.trunk(grid)       # [n_query, latent_dim]

        if t.shape[0] == 1:
            t = t.expand(b.shape[0], -1)
        elif t.shape[0] != b.shape[0]:
            raise ValueError(
                f"Branch batch {b.shape[0]} and trunk batch {t.shape[0]} do not match."
            )

        z = b * t                  # Hadamard product
        return z

    def control_head(self, z):
        y = self.relu(self.fc3(z))
        y = self.relu(self.fc4(y))
        y = self.fc5(y)
        return y

    def forward(self, obs, grid, return_latent=False):
        z = self.extract_latent(obs, grid)
        u_prior_physical = self.control_head(z)

        if return_latent:
            return u_prior_physical, z

        return u_prior_physical


class HybridCACDeepONetFeatureExtractor(BaseFeaturesExtractor):
    """
    Return the Hadamard-fused DeepONet latent feature.

    Input observation and DeepONet input:
        [H_1,...,H_N,Q_1,...,Q_N]

    SAC feature:
        [z_1,...,z_p]
    """

    def __init__(
        self,
        observation_space: spaces.Box,
        features_dim: int = SAC_FEATURE_DIM,
        pretrained_path: str = DEEPNET_PRETRAINED_PATH,
        query_point: float = 1.0,
        freeze_deeponet: bool = False,
    ):
        if features_dim != SAC_FEATURE_DIM:
            raise ValueError(
                f"features_dim must be {SAC_FEATURE_DIM}, got {features_dim}."
            )

        super().__init__(observation_space, features_dim)

        self.query_point = float(query_point)
        self.net1 = DeepONetModified(N, latent_dim=LATENT_DIM)

        # Normalized action used by the hybrid SAC actor.
        self.u_deep_norm: Optional[torch.Tensor] = None

        # Physical prior gate retained for diagnostics.
        self.u_prior_physical: Optional[torch.Tensor] = None

        self.register_buffer(
            "grid",
            torch.tensor([[self.query_point]], dtype=torch.float32),
        )

        self._load_pretrained(pretrained_path)
        self._apply_freeze(freeze_all=freeze_deeponet)

    def _apply_freeze(self, freeze_all=False):
        if freeze_all:
            for param in self.net1.parameters():
                param.requires_grad = False
            print("All DeepONet parameters are frozen.")
            return

        # Keep the same transfer-learning structure used during training.
        for param in self.net1.branch.conv1.parameters():
            param.requires_grad = False
        for param in self.net1.branch.conv2.parameters():
            param.requires_grad = False

        for param in self.net1.branch.fc1.parameters():
            param.requires_grad = True

        for param in self.net1.trunk.parameters():
            param.requires_grad = False

        last_linear = None
        for module in reversed(list(self.net1.trunk.net)):
            if isinstance(module, nn.Linear):
                last_linear = module
                break

        if last_linear is not None:
            for param in last_linear.parameters():
                param.requires_grad = True

        for layer in [self.net1.fc3, self.net1.fc4, self.net1.fc5]:
            for param in layer.parameters():
                param.requires_grad = True

    @staticmethod
    def _clean_state_dict_keys(state_dict):
        cleaned = {}
        prefixes = ["module.", "model.", "net1.", "deeponet."]

        for key, value in state_dict.items():
            new_key = key
            changed = True
            while changed:
                changed = False
                for prefix in prefixes:
                    if new_key.startswith(prefix):
                        new_key = new_key[len(prefix):]
                        changed = True
            cleaned[new_key] = value

        return cleaned

    def _load_pretrained(self, pretrained_path: str):
        path = Path(pretrained_path)

        if not path.exists():
            raise FileNotFoundError(
                f"Pretrained Hadamard DeepONet checkpoint not found: {path}"
            )

        state_dict = torch.load(str(path), map_location="cpu")

        if isinstance(state_dict, dict) and "state_dict" in state_dict:
            state_dict = state_dict["state_dict"]

        if not isinstance(state_dict, dict):
            raise TypeError(
                f"Unsupported checkpoint type: {type(state_dict).__name__}"
            )

        cleaned = self._clean_state_dict_keys(state_dict)

        try:
            self.net1.load_state_dict(cleaned, strict=True)
        except RuntimeError as exc:
            raise RuntimeError(
                "The checkpoint is incompatible with the Hadamard-fusion "
                "DeepONet. Re-pretrain DeepONet with fc3=Linear(102,32) and "
                "z=b*t before running this test script.\n"
                f"Original error:\n{exc}"
            ) from exc

        print(f"Loaded pretrained Hadamard DeepONet: {path}")

    @staticmethod
    def _physical_gate_to_normalized_action(u_physical):
        scale = max_control_value - min_control_value
        if scale <= 0.0:
            raise ValueError(
                "max_control_value must be greater than min_control_value."
            )
        return 2.0 * (u_physical - min_control_value) / scale - 1.0

    def forward(self, observations: torch.Tensor) -> torch.Tensor:
        obs = preprocess_obs(observations)
        if obs.dim() == 1:
            obs = obs.unsqueeze(0)

        if obs.shape[1] != RAW_STATE_DIM:
            raise ValueError(
                f"Raw observation dimension is {obs.shape[1]}, "
                f"expected {RAW_STATE_DIM}."
            )

        u_prior_physical, z = self.net1(
            obs,
            self.grid.to(obs.device),
            return_latent=True,
        )

        # Clip the physical DeepONet prior before mapping it to [-1,1].
        u_prior_physical = torch.clamp(
            u_prior_physical,
            min_control_value,
            max_control_value,
        )
        self.u_prior_physical = u_prior_physical

        u_deep_norm = self._physical_gate_to_normalized_action(u_prior_physical)
        self.u_deep_norm = torch.clamp(u_deep_norm, -1.0, 1.0)

        # The SAC representation contains only the fused DeepONet feature z.
        return z.view(z.size(0), -1)


class HybridCACActor(Actor):
    """SAC actor whose normalized output is blended with DeepONet prior action."""

    def __init__(self, *args, alpha: float = 0.4, **kwargs):
        super().__init__(*args, **kwargs)
        init_alpha = float(np.clip(alpha, 0.0, 1.0))

        self.register_buffer(
            "alpha_tensor",
            torch.tensor(init_alpha, dtype=torch.float32),
        )

        self.last_deep_action = None
        self.last_rl_action = None
        self.last_hybrid_action = None

    @property
    def alpha(self) -> float:
        return float(self.alpha_tensor.detach().cpu().item())

    def set_alpha(self, alpha: float):
        alpha_value = float(np.clip(alpha, 0.0, 1.0))
        with torch.no_grad():
            self.alpha_tensor.fill_(alpha_value)

    def _deep_action(self, obs: torch.Tensor) -> torch.Tensor:
        u_deep = getattr(self.features_extractor, "u_deep_norm", None)

        if u_deep is None or u_deep.shape[0] != obs.shape[0]:
            _ = self.features_extractor(obs)
            u_deep = self.features_extractor.u_deep_norm

        return torch.clamp(u_deep, -1.0, 1.0)

    def _hybridize(self, rl_action: torch.Tensor, obs: torch.Tensor) -> torch.Tensor:
        u_deep = self._deep_action(obs)

        alpha = self.alpha_tensor.to(
            device=rl_action.device,
            dtype=rl_action.dtype,
        )

        hybrid = (1.0 - alpha) * u_deep + alpha * rl_action
        hybrid = torch.clamp(hybrid, -1.0, 1.0)

        self.last_deep_action = u_deep.detach()
        self.last_rl_action = rl_action.detach()
        self.last_hybrid_action = hybrid.detach()

        return hybrid

    def forward(self, obs: torch.Tensor, deterministic: bool = False):
        rl_action = super().forward(obs, deterministic=deterministic)
        return self._hybridize(rl_action, obs)

    def action_log_prob(self, obs: torch.Tensor):
        # The log probability is that of the SAC stochastic component.
        # The deterministic affine hybrid mapping is not included in log_prob.
        rl_action, log_prob = super().action_log_prob(obs)
        hybrid_action = self._hybridize(rl_action, obs)
        return hybrid_action, log_prob


class HybridCACCritic(nn.Module):
    """Twin-Q critic using the Hadamard-fused DeepONet feature."""

    def __init__(
        self,
        observation_space: spaces.Box,
        action_space: spaces.Box,
        hidden_dim: int = 256,
        features_dim: int = SAC_FEATURE_DIM,
        pretrained_path: str = DEEPNET_PRETRAINED_PATH,
    ):
        super().__init__()

        self.features_extractor = HybridCACDeepONetFeatureExtractor(
            observation_space,
            features_dim=features_dim,
            pretrained_path=pretrained_path,
            query_point=1.0,
            freeze_deeponet=False,
        )

        action_dim = int(np.prod(action_space.shape))
        input_dim = features_dim + action_dim

        self.q1 = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )
        self.q2 = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )

    def set_training_mode(self, mode: bool):
        self.train(mode)

    def forward(self, obs: torch.Tensor, actions: torch.Tensor):
        features = self.features_extractor(obs)
        q_input = torch.cat([features, actions], dim=1)
        return self.q1(q_input), self.q2(q_input)


class HybridCACSACPolicy(SACPolicy):
    """SAC policy with Hadamard DeepONet representation/control transfer."""

    def __init__(
        self,
        *args,
        hybrid_alpha_initial: float = 0.4,
        deeponet_pretrained_path: str = DEEPNET_PRETRAINED_PATH,
        **kwargs,
    ):
        self.hybrid_alpha_initial = float(hybrid_alpha_initial)
        self.deeponet_pretrained_path = deeponet_pretrained_path

        kwargs.setdefault(
            "features_extractor_class",
            HybridCACDeepONetFeatureExtractor,
        )
        kwargs.setdefault(
            "features_extractor_kwargs",
            dict(
                features_dim=SAC_FEATURE_DIM,
                pretrained_path=deeponet_pretrained_path,
                query_point=1.0,
                freeze_deeponet=False,
            ),
        )

        super().__init__(*args, **kwargs)

    def make_actor(self, features_extractor=None):
        kwargs = self._update_features_extractor(
            self.actor_kwargs,
            features_extractor,
        )
        return HybridCACActor(
            **kwargs,
            alpha=self.hybrid_alpha_initial,
        ).to(self.device)

    def make_critic(self, features_extractor=None):
        return HybridCACCritic(
            self.observation_space,
            self.action_space,
            hidden_dim=256,
            features_dim=SAC_FEATURE_DIM,
            pretrained_path=self.deeponet_pretrained_path,
        ).to(self.device)

    def set_alpha(self, alpha: float):
        self.actor.set_alpha(alpha)


# ============================================================
# 9. Optimizer helper: separate DeepONet learning rate
# ============================================================

def _make_param_groups_with_deeponet_lr(module: nn.Module):
    deeponet_params = []
    other_params = []

    for name, param in module.named_parameters():
        if not param.requires_grad:
            continue

        if "features_extractor.net1" in name or name.startswith("net1."):
            deeponet_params.append(param)
        else:
            other_params.append(param)

    groups = []

    if other_params:
        groups.append(
            {
                "params": other_params,
                "lr": learning_rate,
                "is_deeponet_group": False,
            }
        )

    if deeponet_params:
        groups.append(
            {
                "params": deeponet_params,
                "lr": deeponet_learning_rate,
                "is_deeponet_group": True,
            }
        )

    return groups


def set_deeponet_small_lr_optimizers(model):
    actor_groups = _make_param_groups_with_deeponet_lr(model.actor)
    critic_groups = _make_param_groups_with_deeponet_lr(model.critic)

    model.actor.optimizer = torch.optim.Adam(actor_groups, eps=1e-5)
    model.critic.optimizer = torch.optim.Adam(critic_groups, eps=1e-5)

    n_actor_deep = sum(
        p.numel()
        for group in actor_groups
        if group.get("is_deeponet_group", False)
        for p in group["params"]
    )
    n_actor_other = sum(
        p.numel()
        for group in actor_groups
        if not group.get("is_deeponet_group", False)
        for p in group["params"]
    )
    n_critic_deep = sum(
        p.numel()
        for group in critic_groups
        if group.get("is_deeponet_group", False)
        for p in group["params"]
    )
    n_critic_other = sum(
        p.numel()
        for group in critic_groups
        if not group.get("is_deeponet_group", False)
        for p in group["params"]
    )

    print("Custom optimizers with separate DeepONet learning rate are set.")
    print(f"Actor params: DeepONet={n_actor_deep}, other={n_actor_other}")
    print(f"Critic params: DeepONet={n_critic_deep}, other={n_critic_other}")
    print(
        f"Base SAC learning_rate={learning_rate}, "
        f"DeepONet learning_rate={deeponet_learning_rate}"
    )


class HybridSAC(SAC):
    """Preserve the smaller learning rate of DeepONet parameter groups."""

    def _update_learning_rate(self, optimizers):
        super()._update_learning_rate(optimizers)

        if not isinstance(optimizers, (list, tuple)):
            optimizers = [optimizers]

        current_lr = self.lr_schedule(self._current_progress_remaining)

        for optimizer in optimizers:
            for group in optimizer.param_groups:
                if group.get("is_deeponet_group", False):
                    group["lr"] = deeponet_learning_rate
                elif "is_deeponet_group" in group:
                    group["lr"] = current_lr




class EvalHybridSAC(HybridSAC):
    """Evaluation loader compatible with DeepONet-specific optimizer groups."""

    def _setup_model(self):
        super()._setup_model()
        set_deeponet_small_lr_optimizers(self)

    def set_parameters(self, load_path_or_dict, exact_match: bool = True, device="auto"):
        try:
            return super().set_parameters(
                load_path_or_dict,
                exact_match=exact_match,
                device=device,
            )
        except ValueError as exc:
            message = str(exc)
            optimizer_error = (
                "parameter group" in message
                or "Names of parameters" in message
                or "optimizer" in message.lower()
            )

            if isinstance(load_path_or_dict, dict) and optimizer_error:
                print(
                    "Evaluation load: incompatible optimizer states are skipped; "
                    "policy, actor, critic, DeepONet, and alpha weights are retained."
                )
                filtered = {
                    name: value
                    for name, value in load_path_or_dict.items()
                    if "optimizer" not in name.lower()
                }
                return super().set_parameters(
                    filtered,
                    exact_match=False,
                    device=device,
                )
            raise


# ============================================================
# Physical setup: ODE equilibrium + shared linear test initial condition
# ============================================================

def make_cac_table_params(
    q_bar: float = Q_BAR_FIXED,
    h_bar_0: float = 3.80,
    h_bar_L: float = 4.70,
) -> CACParams:
    """Create CAC parameters consistent with the new training program."""
    return CACParams(
        L=11239.0,
        B=40.0,
        g=9.81,
        I=7.92e-5,
        K=33.0,
        mu0=0.4,
        muL=0.4,
        z_up=5.00,
        U0_bar=0.1548,
        U_L_bar=U_ref,
        h_s=4.094,
        Q_bar=float(q_bar),
        H_bar_0=float(h_bar_0),
        H_bar_L=float(h_bar_L),
        H_min=0.20,
        max_state_value=1e10,
    )


def make_grid(p: CACParams, n_grid: int) -> Tuple[np.ndarray, float]:
    X = np.linspace(0.0, p.L, n_grid)
    dx = p.L / (n_grid - 1)
    return X, dx


def downstream_equilibrium_depth_from_q(Q: float, p: CACParams) -> float:
    """Compute Hbar(L) from the downstream equilibrium gate relation."""
    Q = float(Q)
    if not np.isfinite(Q) or Q <= 0.0:
        raise ValueError("The equilibrium discharge must be positive and finite.")

    kinetic_head = (
        Q**2 / (2.0 * p.g * p.B**2 * p.muL**2)
    ) ** (1.0 / 3.0)
    return float(p.h_s + p.U_L_bar + kinetic_head)


def integrate_steady_fixed_q_ode(
    X: np.ndarray,
    p: CACParams,
    Q_candidate: float,
) -> Tuple[np.ndarray, np.ndarray]:
    """Integrate the steady Saint-Venant ODE backward from x=L to x=0."""
    X = np.asarray(X, dtype=float)
    if X.ndim != 1 or X.size < 2:
        raise ValueError("X must be a one-dimensional grid with at least two points.")
    if not np.all(np.diff(X) > 0.0):
        raise ValueError("X must be strictly increasing.")

    Q_candidate = float(Q_candidate)
    if not np.isfinite(Q_candidate) or Q_candidate <= 0.0:
        raise ValueError("Q_candidate must be positive and finite.")

    Hbar_L = downstream_equilibrium_depth_from_q(Q_candidate, p)

    def rhs(_x: float, state: np.ndarray) -> np.ndarray:
        H = float(state[0])
        Q = float(state[1])

        if not np.isfinite(H) or not np.isfinite(Q):
            raise FloatingPointError("Non-finite state in steady reference ODE.")
        if H <= float(p.H_min):
            raise FloatingPointError("Steady reference ODE violates minimum depth.")

        area = p.B * H
        hydraulic_radius = p.B * H / (p.B + 2.0 * H)
        friction_slope = Q * abs(Q) / (
            p.K**2 * area**2 * hydraulic_radius ** (4.0 / 3.0)
        )

        denominator = p.g * p.B * H - Q**2 / (p.B * H**2)
        if abs(denominator) <= 1e-10:
            raise FloatingPointError(
                "Steady ODE approaches the critical-flow singularity."
            )

        dHdx = p.g * p.B * H * (p.I - friction_slope) / denominator
        if not np.isfinite(dHdx):
            raise FloatingPointError("Non-finite dHbar/dx in steady ODE.")

        return np.asarray([dHdx, 0.0], dtype=float)

    sol = solve_ivp(
        fun=rhs,
        t_span=(float(X[-1]), float(X[0])),
        y0=np.asarray([Hbar_L, Q_candidate], dtype=float),
        t_eval=X[::-1],
        rtol=1e-9,
        atol=1e-11,
        max_step=p.L / 1000.0,
    )

    if not sol.success:
        raise RuntimeError(f"Failed to solve steady ODE: {sol.message}")
    if sol.y.shape[1] != X.size:
        raise RuntimeError("Steady ODE did not return all requested grid points.")

    Hbar = np.asarray(sol.y[0][::-1], dtype=float)
    Qbar = np.asarray(sol.y[1][::-1], dtype=float)

    if not np.all(np.isfinite(Hbar)) or not np.all(np.isfinite(Qbar)):
        raise FloatingPointError("Steady reference contains NaN or Inf.")
    if float(np.min(Hbar)) <= float(p.H_min):
        raise RuntimeError("Steady Hbar profile violates the minimum depth.")

    return Hbar, Qbar


def solve_steady_reference_ode(
    X: np.ndarray,
    p: CACParams,
    q_bar_fixed: float = Q_BAR_FIXED,
) -> Tuple[np.ndarray, np.ndarray, float]:
    """Solve Hbar(x) for the prescribed constant Qbar=11.93 m^3/s."""
    q_bar_fixed = float(q_bar_fixed)
    Hbar, _ = integrate_steady_fixed_q_ode(X, p, q_bar_fixed)
    Qbar = np.full_like(X, q_bar_fixed, dtype=float)
    return Hbar, Qbar, q_bar_fixed


def make_reference_profiles(
    X: np.ndarray,
    p: CACParams,
    reference: str = REFERENCE_MODE,
    q_bar_fixed: float = Q_BAR_FIXED,
) -> Tuple[np.ndarray, np.ndarray, float]:
    if reference in {"ode", "strict"}:
        return solve_steady_reference_ode(X, p, q_bar_fixed=q_bar_fixed)
    if reference == "linear":
        Hbar = p.H_bar_0 + (p.H_bar_L - p.H_bar_0) * (X / p.L)
        Qbar = np.full_like(X, q_bar_fixed, dtype=float)
        return Hbar, Qbar, float(q_bar_fixed)
    raise ValueError("reference must be 'ode', 'strict', or 'linear'.")


def build_collection_matched_reference():
    """Return one p, X, dx, Hbar, Qbar tuple shared by every controller."""
    nominal_p = make_cac_table_params(q_bar=Q_BAR_FIXED)
    X, dx = make_grid(nominal_p, N)
    Hbar, Qbar, q_equilibrium = make_reference_profiles(
        X,
        nominal_p,
        reference=REFERENCE_MODE,
        q_bar_fixed=Q_BAR_FIXED,
    )

    p = make_cac_table_params(
        q_bar=q_equilibrium,
        h_bar_0=float(Hbar[0]),
        h_bar_L=float(Hbar[-1]),
    )
    return p, X, dx, Hbar, Qbar


def generate_test_initial_condition(
    X: np.ndarray,
    Hbar: np.ndarray,
    Qbar: np.ndarray,
    H_left: float = TEST_H_LEFT,
    H_right: float = TEST_H_RIGHT,
    Q_value: float = TEST_Q_VALUE,
) -> Tuple[np.ndarray, np.ndarray, Dict[str, float]]:
    """Generate the shared deterministic linear test initial condition.

    H(x,0) = H_left + (H_right-H_left) x/L,
    Q(x,0) = Q_value.

    Hbar and Qbar are accepted so the function can also report the exact
    initial deviations from the ODE equilibrium used in the test.
    """
    X = np.asarray(X, dtype=float)
    Hbar = np.asarray(Hbar, dtype=float)
    Qbar = np.asarray(Qbar, dtype=float)

    if X.ndim != 1 or Hbar.shape != X.shape or Qbar.shape != X.shape:
        raise ValueError(
            "X, Hbar, and Qbar must be one-dimensional arrays of equal length."
        )
    L = float(X[-1])
    if not np.isfinite(L) or L <= 0.0:
        raise ValueError("The spatial-domain length must be positive and finite.")

    H_left = float(H_left)
    H_right = float(H_right)
    Q_value = float(Q_value)
    if not all(np.isfinite(v) for v in (H_left, H_right, Q_value)):
        raise ValueError("H_left, H_right, and Q_value must be finite.")

    H0 = H_left + (H_right - H_left) * (X / L)
    Q0 = np.full_like(X, Q_value, dtype=float)

    if float(np.min(H0)) <= 0.20:
        raise ValueError("The linear initial H profile violates H_min=0.20 m.")
    if float(np.min(Q0)) <= 0.0:
        raise ValueError("The initial discharge must be positive.")

    details = {
        "H_left": H_left,
        "H_right": H_right,
        "Q_value": Q_value,
        "H_realized_max_deviation": float(np.max(np.abs(H0 - Hbar))),
        "Q_realized_max_deviation": float(np.max(np.abs(Q0 - Qbar))),
    }
    return H0, Q0, details


def make_fixed_initial_condition(
    X: np.ndarray,
    Hbar: np.ndarray,
    Qbar: np.ndarray,
    H_left: float = TEST_H_LEFT,
    H_right: float = TEST_H_RIGHT,
    Q_value: float = TEST_Q_VALUE,
) -> Tuple[np.ndarray, np.ndarray]:
    """Return the shared deterministic linear test initial state."""
    H0, Q0, _ = generate_test_initial_condition(
        X,
        Hbar,
        Qbar,
        H_left=H_left,
        H_right=H_right,
        Q_value=Q_value,
    )
    return H0, Q0


def make_test_initial_condition_func(
    H_left: float = TEST_H_LEFT,
    H_right: float = TEST_H_RIGHT,
    Q_value: float = TEST_Q_VALUE,
):
    """Return the same deterministic linear reset callback for all controllers."""
    def reset_func(X, Hbar, Qbar, rng):
        del rng
        H0, Q0, _ = generate_test_initial_condition(
            X,
            Hbar,
            Qbar,
            H_left=H_left,
            H_right=H_right,
            Q_value=Q_value,
        )
        return H0, Q0

    return reset_func

# ============================================================
# Hybrid action diagnostics
# ============================================================

def normalized_action_to_physical_ul(a):
    """Map normalized action a in [-1, 1] to physical downstream gate U_L."""
    a = float(np.clip(float(a), -1.0, 1.0))
    return float(min_control_value + 0.5 * (a + 1.0) * (max_control_value - min_control_value))


def physical_ul_to_normalized_action(u_physical):
    """Map physical downstream-gate command U_L back to normalized action [-1, 1]."""
    scale = float(max_control_value - min_control_value)
    if scale <= 0.0:
        raise ValueError("max_control_value must be greater than min_control_value.")

    u_physical = float(
        np.clip(float(u_physical), min_control_value, max_control_value)
    )
    action = 2.0 * (u_physical - min_control_value) / scale - 1.0
    return float(np.clip(action, -1.0, 1.0))


def lowpass_filter_rl_control(
    action_raw,
    previous_u_filtered,
    alpha=RL_CONTROL_FILTER_ALPHA,
):
    """Apply a first-order low-pass filter to one scalar RL gate command."""
    alpha = float(alpha)
    if not (0.0 < alpha <= 1.0):
        raise ValueError("RL_CONTROL_FILTER_ALPHA must satisfy 0 < alpha <= 1.")

    a_raw = float(np.asarray(action_raw, dtype=float).reshape(-1)[0])
    a_raw = float(np.clip(a_raw, -1.0, 1.0))
    u_command_raw = normalized_action_to_physical_ul(a_raw)

    previous_u_filtered = float(
        np.clip(float(previous_u_filtered), min_control_value, max_control_value)
    )

    u_filtered = (
        (1.0 - alpha) * previous_u_filtered
        + alpha * u_command_raw
    )
    u_filtered = float(
        np.clip(u_filtered, min_control_value, max_control_value)
    )

    a_filtered = physical_ul_to_normalized_action(u_filtered)
    action_filtered = np.asarray([a_filtered], dtype=np.float32)

    return action_filtered, u_command_raw, u_filtered


def get_hybrid_action_components(model, obs, action_hybrid):
    """
    Extract DeepONet, RL, and hybrid normalized actions for the loaded hybrid SAC policy.

    The actor actually outputs
        a_hybrid = (1-alpha) * a_deep + alpha * a_RL.
    During testing, model.predict(obs) returns a_hybrid. We compute a_deep by calling
    the actor feature extractor and infer a_RL from the affine mixing formula.
    """
    actor = model.policy.actor
    device = model.device

    obs_tensor = torch.as_tensor(obs, dtype=torch.float32, device=device)
    if obs_tensor.dim() == 1:
        obs_tensor = obs_tensor.unsqueeze(0)

    with torch.no_grad():
        # This call updates actor.features_extractor.u_deep_norm.
        _ = actor.features_extractor(obs_tensor)
        u_deep = getattr(actor.features_extractor, "u_deep_norm", None)

    if u_deep is None:
        a_deep = np.nan
    else:
        a_deep = float(u_deep.detach().cpu().numpy().reshape(-1)[0])
        a_deep = float(np.clip(a_deep, -1.0, 1.0))

    a_hybrid = float(np.asarray(action_hybrid).reshape(-1)[0])
    a_hybrid = float(np.clip(a_hybrid, -1.0, 1.0))

    alpha = float(getattr(actor, "alpha", np.nan))

    if np.isfinite(alpha) and alpha > 1e-8 and np.isfinite(a_deep):
        a_rl = (a_hybrid - (1.0 - alpha) * a_deep) / alpha
        a_rl = float(np.clip(a_rl, -1.0, 1.0))
    else:
        a_rl = np.nan

    return {
        "alpha": alpha,
        "a_deep": a_deep,
        "a_rl": a_rl,
        "a_hybrid": a_hybrid,
        "U_deep": normalized_action_to_physical_ul(a_deep) if np.isfinite(a_deep) else np.nan,
        "U_rl": normalized_action_to_physical_ul(a_rl) if np.isfinite(a_rl) else np.nan,
        "U_hybrid": normalized_action_to_physical_ul(a_hybrid),
    }


# ============================================================
# Reward class
# ============================================================

class CACTrackingSmoothReward(BaseReward):
    """
    Discrete reward at the control instant t_k:

        r_k = -alpha ||y_tilde_k-y_tilde_star||^2
              -beta  ||(y_tilde_k-y_tilde_{k-1})/Delta_t||^2
              -gamma ||U_tilde_k||^2
              -delta ||U_tilde_k-U_tilde_{k-1}||^2.

    Here:
        y_tilde_star = 0,
        H_tilde_error = (H-H_eq)/H_scale,
        Q_tilde_error = (Q-Q_eq)/Q_scale,
        U_tilde = (U-U_eq)/U_scale.

    Spatial means are used instead of unnormalized sums so that the reward scale
    does not grow with the number of grid points N.
    """

    def __init__(
        self,
        H_eq,
        Q_eq,
        delta_t=control_sample_rate,
        H_scale=0.10,
        Q_scale=3.00,
        U_scale=0.10,
        alpha=1.0,
        beta=0.10,
        gamma=0.05,
        delta=0.10,
        w_H=1.0,
        w_Q=1.0,
        U_ref=0.3,
        terminal_tracking_weight=10.0,
        truncate_penalty=-200.0,
    ):
        self.H_eq = np.asarray(H_eq, dtype=float)
        self.Q_eq = np.asarray(Q_eq, dtype=float)
        self.N = len(self.H_eq)

        self.delta_t = float(delta_t)
        self.H_scale = float(H_scale)
        self.Q_scale = float(Q_scale)
        self.U_scale = float(U_scale)

        self.alpha = float(alpha)
        self.beta = float(beta)
        self.gamma = float(gamma)
        self.delta = float(delta)
        self.w_H = float(w_H)
        self.w_Q = float(w_Q)

        self.U_ref = float(U_ref)
        self.terminal_tracking_weight = float(terminal_tracking_weight)
        self.truncate_penalty = float(truncate_penalty)

        # Previous values at the preceding reward/control call.
        self.prev_H: Optional[np.ndarray] = None
        self.prev_Q: Optional[np.ndarray] = None
        self.prev_action: Optional[float] = None

        # Diagnostics.
        self.last_terms = {
            "tracking_cost": 0.0,
            "state_rate_cost": 0.0,
            "control_cost": 0.0,
            "delta_control_cost": 0.0,
        }

    def _reset_memory(self):
        self.prev_H = None
        self.prev_Q = None
        self.prev_action = None

    def _split_state(self, state_vec):
        state_vec = np.asarray(state_vec, dtype=float).reshape(-1)
        if state_vec.size < 2 * self.N:
            raise ValueError(
                f"PDE state length is {state_vec.size}, expected at least {2 * self.N}."
            )
        H = state_vec[:self.N]
        Q = state_vec[self.N:2 * self.N]
        return H, Q

    def _action_value(self, action):
        # The original environment passes the physical downstream gate to reward.
        if action is None:
            return self.U_ref
        value = float(np.asarray(action).reshape(-1)[0])
        return float(
            np.nan_to_num(
                value,
                nan=self.U_ref,
                posinf=self.U_ref,
                neginf=self.U_ref,
            )
        )

    def reward(
        self,
        uVec=None,
        time_index=None,
        terminate=None,
        truncate=None,
        action=None,
        control_sample_rate=None,
    ):
        if uVec is None or time_index is None:
            return 0.0

        # A new episode normally starts with a small time index.
        if int(time_index) <= 1:
            self._reset_memory()

        if truncate:
            self._reset_memory()
            return float(self.truncate_penalty)

        Hk, Qk = self._split_state(uVec[time_index])

        if (
            not np.all(np.isfinite(Hk))
            or not np.all(np.isfinite(Qk))
            or np.any(Hk <= 0.0)
        ):
            self._reset_memory()
            return float(self.truncate_penalty)

        # ----------------------------------------------------
        # 1) Tracking term
        # ----------------------------------------------------
        H_error_tilde = (Hk - self.H_eq) / self.H_scale
        Q_error_tilde = (Qk - self.Q_eq) / self.Q_scale

        tracking_cost = (
            self.w_H * np.mean(H_error_tilde ** 2)
            + self.w_Q * np.mean(Q_error_tilde ** 2)
        )

        # ----------------------------------------------------
        # 2) State-rate term
        # ----------------------------------------------------
        if self.prev_H is None or self.prev_Q is None:
            state_rate_cost = 0.0
        else:
            delta_t_k = (
                float(control_sample_rate)
                if control_sample_rate is not None
                else self.delta_t
            )
            delta_t_k = max(delta_t_k, 1e-12)

            dH_tilde = (Hk - self.prev_H) / (delta_t_k * self.H_scale)
            dQ_tilde = (Qk - self.prev_Q) / (delta_t_k * self.Q_scale)

            state_rate_cost = (
                self.w_H * np.mean(dH_tilde ** 2)
                + self.w_Q * np.mean(dQ_tilde ** 2)
            )

        # ----------------------------------------------------
        # 3) Control-magnitude term
        # ----------------------------------------------------
        Uk = self._action_value(action)
        U_tilde = (Uk - self.U_ref) / self.U_scale
        control_cost = U_tilde ** 2

        # ----------------------------------------------------
        # 4) Control-increment term
        # ----------------------------------------------------
        if self.prev_action is None:
            delta_control_cost = 0.0
        else:
            U_prev_tilde = (self.prev_action - self.U_ref) / self.U_scale
            delta_control_cost = (U_tilde - U_prev_tilde) ** 2

        reward_value = (
            -self.alpha * tracking_cost
            -self.beta * state_rate_cost
            -self.gamma * control_cost
            -self.delta * delta_control_cost
        )

        if terminate:
            reward_value -= self.terminal_tracking_weight * tracking_cost

        self.prev_H = Hk.copy()
        self.prev_Q = Qk.copy()
        self.prev_action = Uk

        self.last_terms = {
            "tracking_cost": float(tracking_cost),
            "state_rate_cost": float(state_rate_cost),
            "control_cost": float(control_cost),
            "delta_control_cost": float(delta_control_cost),
        }

        return float(reward_value)




# ============================================================
# Environments
# ============================================================

def _make_reward(Hbar, Qbar):
    return CACTrackingSmoothReward(
        H_eq=Hbar,
        Q_eq=Qbar,
        delta_t=control_sample_rate,
        H_scale=H_scale,
        Q_scale=Q_scale,
        U_scale=U_scale,
        alpha=reward_alpha,
        beta=reward_beta,
        gamma=reward_gamma,
        delta=reward_delta,
        w_H=reward_w_H,
        w_Q=reward_w_Q,
        U_ref=U_ref,
        terminal_tracking_weight=terminal_tracking_weight,
        truncate_penalty=truncate_penalty,
    )


def make_env_sac(
    H_left=TEST_H_LEFT,
    H_right=TEST_H_RIGHT,
    Q_value=TEST_Q_VALUE,
    seed=123,
    dH_source=None,
):
    """Create the exact evaluation environment for the raw-HQ learned model.

    The policy receives raw physical [H,Q]. Every controller is reset to the
    same deterministic linear H profile and constant Q profile.
    """
    p, X, dx, Hbar, Qbar = build_collection_matched_reference()
    H_init, Q_init = make_fixed_initial_condition(
        X,
        Hbar,
        Qbar,
        H_left=H_left,
        H_right=H_right,
        Q_value=Q_value,
    )

    env_kwargs = dict(
        params=p,
        X_grid=X,
        Hbar=Hbar,
        Qbar=Qbar,
        H_init=H_init,
        Q_init=Q_init,
        sensing_noise_func=lambda observation: observation,
        reset_init_condition_func=make_test_initial_condition_func(
            H_left=H_left,
            H_right=H_right,
            Q_value=Q_value,
        ),
        sensing_loc="full",
        control_type="Gate",
        limit_pde_state_size=True,
        max_state_value=max_state_value,
        min_control_value=min_control_value,
        max_control_value=max_control_value,
        control_sample_rate=control_sample_rate,
        reward_class=_make_reward(Hbar, Qbar),
        normalize=True,
        observation_mode="raw",
        H_scale=H_scale,
        Q_scale=Q_scale,
        obs_clip=obs_clip,
        T=SAC_T,
        dt=dt,
        cfl=cfl,
        check_cfl=check_cfl,
        well_balanced=well_balanced,
    )

    if dH_source is not None:
        env_kwargs["dH_source"] = float(dH_source)

    env = CACSaintVenantPDE1D(**env_kwargs)

    env.reference_X = X
    env.reference_H = Hbar
    env.reference_Q = Qbar
    env.reference_dx = dx

    return env, X, Hbar, Qbar, p


def make_env_sac_error_state(
    H_left=TEST_H_LEFT,
    H_right=TEST_H_RIGHT,
    Q_value=TEST_Q_VALUE,
    seed=123,
    dH_source=None,
):
    """Create the evaluation environment for the old error-state SAC model.

    The physical test problem is unchanged: the ODE equilibrium, linear H0,
    constant Q0, PDE parameters, action range, time step, and test horizon are
    identical to the raw-HQ controllers. Only the policy observation changes to

        [(H-Hbar)/H_scale, (Q-Qbar)/Q_scale].

    This matches the old SAC checkpoint trained with
    observation_mode="normalized_error".
    """
    p, X, dx, Hbar, Qbar = build_collection_matched_reference()
    H_init, Q_init = make_fixed_initial_condition(
        X,
        Hbar,
        Qbar,
        H_left=H_left,
        H_right=H_right,
        Q_value=Q_value,
    )

    env_kwargs = dict(
        params=p,
        X_grid=X,
        Hbar=Hbar,
        Qbar=Qbar,
        H_init=H_init,
        Q_init=Q_init,
        sensing_noise_func=lambda observation: observation,
        reset_init_condition_func=make_test_initial_condition_func(
            H_left=H_left,
            H_right=H_right,
            Q_value=Q_value,
        ),
        sensing_loc="full",
        control_type="Gate",
        limit_pde_state_size=True,
        max_state_value=max_state_value,
        min_control_value=min_control_value,
        max_control_value=max_control_value,
        control_sample_rate=control_sample_rate,
        reward_class=_make_reward(Hbar, Qbar),
        normalize=True,
        observation_mode="normalized_error",
        H_scale=H_scale,
        Q_scale=Q_scale,
        obs_clip=obs_clip,
        T=SAC_T,
        dt=dt,
        cfl=cfl,
        check_cfl=check_cfl,
        well_balanced=well_balanced,
    )

    if dH_source is not None:
        env_kwargs["dH_source"] = float(dH_source)

    env = CACSaintVenantPDE1D(**env_kwargs)
    env.reference_X = X
    env.reference_H = Hbar
    env.reference_Q = Qbar
    env.reference_dx = dx
    env.policy_observation_mode = "normalized_error"

    return env, X, Hbar, Qbar, p


def make_env_backstepping(
    H_left=TEST_H_LEFT,
    H_right=TEST_H_RIGHT,
    Q_value=TEST_Q_VALUE,
    seed=123,
    dH_source=None,
):
    """Create Backstepping with the same ODE reference and linear initial state."""
    p, X, dx, Hbar, Qbar = build_collection_matched_reference()
    H_init, Q_init = make_fixed_initial_condition(
        X,
        Hbar,
        Qbar,
        H_left=H_left,
        H_right=H_right,
        Q_value=Q_value,
    )

    env_kwargs = dict(
        params=p,
        X_grid=X,
        Hbar=Hbar,
        Qbar=Qbar,
        H_init=H_init,
        Q_init=Q_init,
        sensing_noise_func=lambda state: state,
        reset_init_condition_func=make_test_initial_condition_func(
            H_left=H_left,
            H_right=H_right,
            Q_value=Q_value,
        ),
        sensing_loc="full",
        control_type="Gate",
        limit_pde_state_size=True,
        max_state_value=max_state_value,
        min_control_value=min_control_value,
        max_control_value=max_control_value,
        control_sample_rate=control_sample_rate,
        reward_class=_make_reward(Hbar, Qbar),
        normalize=False,
        observation_mode="raw",
        H_scale=H_scale,
        Q_scale=Q_scale,
        obs_clip=obs_clip,
        T=BS_T,
        dt=dt,
        cfl=cfl,
        check_cfl=check_cfl,
        well_balanced=well_balanced,
    )

    if dH_source is not None:
        env_kwargs["dH_source"] = float(dH_source)

    env = CACSaintVenantPDE1D(**env_kwargs)

    env.reference_X = X
    env.reference_H = Hbar
    env.reference_Q = Qbar
    env.reference_dx = dx

    return env, X, Hbar, Qbar, p

# ============================================================
# Backstepping functions copied from the standalone Backstepping script
# ============================================================

def left_eigenvector_matrix(H: float, Q: float, p: CACParams) -> Tuple[np.ndarray, float, float]:
    H = max(float(H), p.H_min)
    V = Q / (p.B * H)
    c = np.sqrt(p.g * H)
    lp = V + c
    lm = V - c

    R = np.array([[1.0, 1.0], [p.B * lp, p.B * lm]], dtype=float)
    Tmat = np.linalg.inv(R)
    return Tmat, lp, lm


def characteristic_variables(H: np.ndarray, Q: np.ndarray, Hbar: np.ndarray, p: CACParams) -> Tuple[np.ndarray, np.ndarray]:
    u = np.zeros_like(H)
    v = np.zeros_like(H)

    for i in range(len(H)):
        Tmat, _, _ = left_eigenvector_matrix(Hbar[i], p.Q_bar, p)
        z = Tmat @ np.array([H[i] - Hbar[i], Q[i] - p.Q_bar])
        u[i] = z[0]
        v[i] = z[1]

    return u, v


def linearized_matrices_profiles(X: np.ndarray, Hbar: np.ndarray, p: CACParams) -> Dict[str, np.ndarray]:
    Np = len(X)
    Qbar = p.Q_bar
    Hx = np.gradient(Hbar, X, edge_order=2)

    T_arr = np.zeros((Np, 2, 2))
    R_arr = np.zeros((Np, 2, 2))
    A_arr = np.zeros((Np, 2, 2))
    lam_plus = np.zeros(Np)
    lam_minus = np.zeros(Np)

    for i in range(Np):
        H = float(Hbar[i])
        Q = Qbar

        Tmat, lp, lm = left_eigenvector_matrix(H, Q, p)
        T_arr[i] = Tmat
        R_arr[i] = np.linalg.inv(Tmat)

        A_arr[i] = np.array(
            [[0.0, 1.0 / p.B],
             [p.g * p.B * H - Q**2 / (p.B * H**2), 2.0 * Q / (p.B * H)]],
            dtype=float,
        )

        lam_plus[i] = lp
        lam_minus[i] = lm

    R_x = np.gradient(R_arr, X, axis=0, edge_order=2)

    def G_of_state(Hval: float, Qval: float, Hx_val: float) -> np.ndarray:
        A = np.array(
            [[0.0, 1.0 / p.B],
             [p.g * p.B * Hval - Qval**2 / (p.B * Hval**2), 2.0 * Qval / (p.B * Hval)]],
            dtype=float,
        )
        Xbar_x = np.array([Hx_val, 0.0])
        F = source(np.array([Hval]), np.array([Qval]), p)[0]
        return A @ Xbar_x - F

    Bstar = np.zeros((Np, 2, 2))

    for i in range(Np):
        H = float(Hbar[i])
        Q = Qbar
        hx = float(Hx[i])

        dh = max(1e-6, abs(H) * 1e-6)
        dq = max(1e-6, abs(Q) * 1e-6)

        dG_dH = (G_of_state(H + dh, Q, hx) - G_of_state(H - dh, Q, hx)) / (2.0 * dh)
        dG_dQ = (G_of_state(H, Q + dq, hx) - G_of_state(H, Q - dq, hx)) / (2.0 * dq)

        Bstar[i, :, 0] = dG_dH
        Bstar[i, :, 1] = dG_dQ

    M = np.zeros((Np, 2, 2))
    for i in range(Np):
        M[i] = T_arr[i] @ (A_arr[i] @ R_x[i] + Bstar[i] @ R_arr[i])

    eps1 = lam_plus.copy()
    eps2 = -lam_minus.copy()

    int1 = np.zeros(Np)
    int2 = np.zeros(Np)

    f1 = M[:, 0, 0] / eps1
    f2 = -M[:, 1, 1] / eps2

    for i in range(1, Np):
        h = X[i] - X[i - 1]
        int1[i] = int1[i - 1] + 0.5 * h * (f1[i] + f1[i - 1])
        int2[i] = int2[i - 1] + 0.5 * h * (f2[i] + f2[i - 1])

    phi1 = np.exp(int1)
    phi2 = np.exp(int2)

    c1 = -(phi1 / phi2) * M[:, 0, 1]
    c2 = -(phi2 / phi1) * M[:, 1, 0]

    Hbar0 = float(Hbar[0])
    kappa0 = -p.U0_bar * p.B * p.mu0 * p.g / np.sqrt(2.0 * p.g * (p.z_up - Hbar0))

    T0 = T_arr[0]
    q0 = (T0[0, 0] + T0[0, 1] * kappa0) / (T0[1, 0] + T0[1, 1] * kappa0)

    return {
        "eps1": eps1,
        "eps2": eps2,
        "eps1_x": np.gradient(eps1, X, edge_order=2),
        "eps2_x": np.gradient(eps2, X, edge_order=2),
        "c1": c1,
        "c2": c2,
        "q0": float(q0),
    }


def compute_variable_kernels(X: np.ndarray, coeff: Dict[str, np.ndarray]) -> Tuple[np.ndarray, np.ndarray]:
    Np = len(X)
    dx = X[1] - X[0]

    e1 = coeff["eps1"]
    e2 = coeff["eps2"]
    e1x = coeff["eps1_x"]
    e2x = coeff["eps2_x"]
    c1 = coeff["c1"]
    c2 = coeff["c2"]
    q0 = coeff["q0"]

    ratio = q0 * e1[0] / e2[0]

    Kvu = np.zeros((Np, Np))
    Kvv = np.zeros((Np, Np))

    for i in range(Np):
        Kvu[i, i] = -c2[i] / (e1[i] + e2[i])

    for i in range(1, Np):
        A = (e2[i] + e1[0]) / dx - e1x[0]
        rhs = e2[i] * Kvu[i - 1, 0] / dx + e1[0] * Kvu[i, 1] / dx
        denom = A - c2[0] * ratio

        Kvu[i, 0] = rhs / denom
        Kvv[i, 0] = ratio * Kvu[i, 0]

        for j in range(1, i):
            A = (e2[i] + e1[j]) / dx - e1x[j]
            B = (e2[i] + e2[j]) / dx + e2x[j]

            rhsK = e2[i] * Kvu[i - 1, j] / dx + e1[j] * Kvu[i, j + 1] / dx
            rhsL = e2[i] * Kvv[i - 1, j] / dx + e2[j] * Kvv[i, j - 1] / dx

            mat = np.array([[A, -c2[j]], [c1[j], B]], dtype=float)
            sol = np.linalg.solve(mat, np.array([rhsK, rhsL], dtype=float))

            Kvu[i, j] = sol[0]
            Kvv[i, j] = sol[1]

        Bdiag = e2[i] / dx + e2x[i]
        rhsdiag = e2[i] * Kvv[i - 1, i - 1] / dx - c1[i] * Kvu[i, i]
        Kvv[i, i] = rhsdiag / Bdiag

    return Kvu, Kvv


def backstepping_action(env: CACSaintVenantPDE1D, Kvu_L: np.ndarray, Kvv_L: np.ndarray, gain: float = 1.0):
    p = env.params
    H, Q = env.get_current_HQ()
    Hbar = env.Hbar
    X = env.X_grid

    u, v = characteristic_variables(H, Q, Hbar, p)
    Ubs = gain * float(np.trapezoid(Kvu_L * u + Kvv_L * v, X))

    T_L, _, _ = left_eigenvector_matrix(float(Hbar[-1]), p.Q_bar, p)
    t21, t22 = T_L[1, 0], T_L[1, 1]

    QL = float(Q[-1])
    Hbs_L = float(Hbar[-1]) + (Ubs - t22 * (QL - p.Q_bar)) / t21

    UL = Hbs_L - p.h_s - (QL**2 / (2.0 * p.g * p.B**2 * p.muL**2)) ** (1.0 / 3.0)

    return float(UL), float(Ubs)




# ============================================================
# SB3 evaluation-only loading helper
# ============================================================

def strip_optimizer_states_from_sb3_model(model_path: str) -> str:
    """
    Create an evaluation-only SB3 zip by removing optimizer states.

    This fixes errors such as:
        ValueError: loaded state dict has a different number of parameter groups

    It is safe for evaluation because optimizer states are only needed when
    continuing training. The policy/actor/critic weights are kept unchanged.

    Return value is a path without the .zip suffix, matching SB3's usual API.
    """
    model_path = Path(model_path)

    if model_path.suffix == ".zip":
        src_zip = model_path
    else:
        src_zip = model_path.with_suffix(".zip")

    if not src_zip.exists():
        raise FileNotFoundError(f"Cannot find SB3 model zip file: {src_zip}")

    dst_zip = src_zip.with_name(src_zip.stem + "_eval_only.zip")

    if dst_zip.exists():
        print(f"Using existing eval-only model: {dst_zip}")
        return str(dst_zip.with_suffix(""))

    with zipfile.ZipFile(src_zip, "r") as zin:
        with zipfile.ZipFile(dst_zip, "w", compression=zipfile.ZIP_DEFLATED) as zout:
            for item in zin.infolist():
                name = item.filename
                # Remove all optimizer states. These can have different parameter-group
                # structures after using DeepONet-specific learning rates.
                if "optimizer" in name.lower():
                    print(f"Skip optimizer state: {name}")
                    continue
                zout.writestr(item, zin.read(name))

    print(f"Created eval-only model: {dst_zip}")
    return str(dst_zip.with_suffix(""))


# ============================================================
# Rollouts
# ============================================================

def run_sac_exact(
    model_path,
    seed=123,
    H_left=TEST_H_LEFT,
    H_right=TEST_H_RIGHT,
    Q_value=TEST_Q_VALUE,
    dH_source=None,
):
    env, X, Hbar, Qbar, p = make_env_sac(
        H_left=H_left,
        H_right=H_right,
        Q_value=Q_value,
        seed=seed,
        dH_source=dH_source,
    )

    # Directly load the trained HybridSAC model.
    # EvalHybridSAC only rebuilds the optimizer structure required by SB3 loading;
    # the actual closed-loop control still comes directly from model.predict().
    # Because this training version saves alpha as actor.alpha_tensor, no manual
    # alpha restoration is needed here.
    model = EvalHybridSAC.load(model_path, env=env)
    obs, _ = env.reset(seed=seed)

    H_hist, Q_hist, U_hist, t_hist, reward_hist = [], [], [], [], []

    previous_u_filtered = float(U_ref)

    done = False
    while not done:
        # Raw trained-controller output.
        action, _ = model.predict(obs, deterministic=True)

        # Same online first-order low-pass filter as the previous program.
        action_filtered, u_command_raw, previous_u_filtered = lowpass_filter_rl_control(
            action,
            previous_u_filtered,
            alpha=RL_CONTROL_FILTER_ALPHA,
        )

        # The filtered command is the actual closed-loop input.
        obs, reward, terminated, truncated, info = env.step(action_filtered)

        H, Q = env.get_current_HQ()

        H_hist.append(H.copy())
        Q_hist.append(Q.copy())
        U_hist.append(float(env.last_control))
        t_hist.append(float(info["time"]))
        reward_hist.append(float(reward))

        done = terminated or truncated

    env.close()

    H_hist = np.asarray(H_hist)
    Q_hist = np.asarray(Q_hist)
    U_hist = np.asarray(U_hist)
    t_hist = np.asarray(t_hist)
    reward_hist = np.asarray(reward_hist)

    H_err = H_hist - Hbar[None, :]
    Q_err = Q_hist - Qbar[None, :]

    return {
        "X": X,
        "Hbar": Hbar,
        "Qbar": Qbar,
        "t": t_hist,
        "H": H_hist,
        "Q": Q_hist,
        "U": U_hist,
        "reward": reward_hist,
        "H_l2": np.sqrt(np.mean(H_err ** 2, axis=1)),
        "Q_l2": np.sqrt(np.mean(Q_err ** 2, axis=1)),
        "final_H_minus_Hbar": H_err[-1],
        "final_Q_minus_Qbar": Q_err[-1],
        "total_reward": float(np.sum(reward_hist)),
    }


def run_backstepping_exact(
    seed=123,
    H_left=TEST_H_LEFT,
    H_right=TEST_H_RIGHT,
    Q_value=TEST_Q_VALUE,
    dH_source=None,
):
    """
    Backstepping rollout unified to the SAC standalone recording style:
        action -> env.step(action) -> record H,Q,U,t,reward.

    Therefore this is no longer the original Backstepping standalone recording mode;
    it is intentionally aligned with the SAC plot.
    """
    env, X, Hbar, Qbar, p = make_env_backstepping(
        H_left=H_left,
        H_right=H_right,
        Q_value=Q_value,
        seed=seed,
        dH_source=dH_source,
    )

    coeff = linearized_matrices_profiles(X, Hbar, p)
    Kvu, Kvv = compute_variable_kernels(X, coeff)
    Kvu_L = Kvu[-1, :]
    Kvv_L = Kvv[-1, :]

    obs, _ = env.reset(seed=seed)

    H_hist, Q_hist, U_hist, t_hist, reward_hist = [], [], [], [], []

    # Record the common prescribed initial state at t=0, exactly as for the
    # learned controller. The initial control is undefined and stored as NaN.
    H0, Q0 = env.get_current_HQ()
    H_hist.append(H0.copy())
    Q_hist.append(Q0.copy())
    U_hist.append(np.nan)
    t_hist.append(0.0)
    reward_hist.append(0.0)

    done = False
    while not done:
        UL_raw, Ubs = backstepping_action(env, Kvu_L, Kvv_L, gain=BACKSTEPPING_GAIN)
        action_env = float(np.clip(UL_raw, min_control_value, max_control_value))

        obs, reward, terminated, truncated, info = env.step(np.array([action_env], dtype=np.float32))

        # SAC plotting style: record AFTER env.step().
        H, Q = env.get_current_HQ()
        H_hist.append(H.copy())
        Q_hist.append(Q.copy())
        U_hist.append(action_env)
        t_hist.append(float(info["time"]))
        reward_hist.append(float(reward))

        done = terminated or truncated

    env.close()

    H_hist = np.asarray(H_hist)
    Q_hist = np.asarray(Q_hist)
    U_hist = np.asarray(U_hist)
    t_hist = np.asarray(t_hist)
    reward_hist = np.asarray(reward_hist)

    H_err = H_hist - Hbar[None, :]
    Q_err = Q_hist - Qbar[None, :]

    return {
        "X": X,
        "Hbar": Hbar,
        "Qbar": Qbar,
        "t": t_hist,
        "H": H_hist,
        "Q": Q_hist,
        "U": U_hist,
        "reward": reward_hist,
        "H_l2": np.sqrt(np.mean(H_err ** 2, axis=1)),
        "Q_l2": np.sqrt(np.mean(Q_err ** 2, axis=1)),
        "final_H_minus_Hbar": H_err[-1],
        "final_Q_minus_Qbar": Q_err[-1],
        "total_reward": float(np.sum(reward_hist)),
    }


# ============================================================
# Plot / save
# ============================================================

def plot_boundary_comparison(res_sac, res_bs, save_dir, tag):
    t_sac = res_sac["t"] / 60.0
    t_bs = res_bs["t"] / 60.0

    Hbar = res_sac["Hbar"]
    Qbar = res_sac["Qbar"]

    fig, ax = plt.subplots(4, 1, figsize=(8, 10), dpi=300, sharex=True)

    ax[0].plot(t_sac, res_sac["H"][:, 0], label="SAC H(0,t)")
    ax[0].plot(t_bs, res_bs["H"][:, 0], "--", label="BS H(0,t)")
    ax[0].axhline(Hbar[0], linestyle=":", linewidth=1, label="Hbar(0)")
    ax[0].set_ylabel("H(0,t) [m]", fontsize=18, fontweight="bold", labelpad=8)
    ax[0].legend(**_legend_kwargs_two_columns(loc="best")); ax[0].grid(True, alpha=0.3)

    ax[1].plot(t_sac, res_sac["H"][:, -1], label="SAC H(L,t)")
    ax[1].plot(t_bs, res_bs["H"][:, -1], "--", label="BS H(L,t)")
    ax[1].axhline(Hbar[-1], linestyle=":", linewidth=1, label="Hbar(L)")
    ax[1].set_ylabel("H(L,t) [m]", fontsize=18, fontweight="bold", labelpad=8)
    ax[1].legend(**_legend_kwargs_two_columns(loc="best")); ax[1].grid(True, alpha=0.3)

    ax[2].plot(t_sac, res_sac["Q"][:, 0], label="SAC Q(0,t)")
    ax[2].plot(t_bs, res_bs["Q"][:, 0], "--", label="BS Q(0,t)")
    ax[2].axhline(Qbar[0], linestyle=":", linewidth=1, label="Qbar")
    ax[2].set_ylabel("Q(0,t) [m^3/s]", fontsize=18, fontweight="bold", labelpad=8)
    ax[2].legend(**_legend_kwargs_two_columns(loc="best")); ax[2].grid(True, alpha=0.3)

    ax[3].plot(t_sac, res_sac["Q"][:, -1], label="SAC Q(L,t)")
    ax[3].plot(t_bs, res_bs["Q"][:, -1], "--", label="BS Q(L,t)")
    ax[3].axhline(Qbar[-1], linestyle=":", linewidth=1, label="Qbar")
    ax[3].set_ylabel("Q(L,t) [m^3/s]", fontsize=18, fontweight="bold", labelpad=8)
    ax[3].set_xlabel("Time [min]", fontsize=18, fontweight="bold", labelpad=10)
    ax[3].legend(**_legend_kwargs_two_columns(loc="best")); ax[3].grid(True, alpha=0.3)

    fig.suptitle("SAC-style boundary comparison: SAC vs Backstepping")
    fig.tight_layout()

    path = os.path.join(save_dir, f"{tag}_SAC_vs_Backstepping_boundary_only.png")
    fig.savefig(path, dpi=500, bbox_inches="tight")
    print(f"Saved comparison figure to: {path}")
    plt.show()


def plot_hybrid_action_diagnostics(res_sac, save_dir, tag):
    """Plot normalized actions and physical gate commands of DeepONet/RL/hybrid."""
    required = ["A_deep", "A_hybrid", "alpha", "U_deep", "U_hybrid"]
    if any(k not in res_sac for k in required):
        print("Hybrid diagnostics are not available in res_sac.")
        return

    t = np.asarray(res_sac["t"], dtype=float) / 60.0
    a_deep = np.asarray(res_sac["A_deep"], dtype=float)
    a_rl = np.asarray(res_sac["A_rl"], dtype=float)
    a_hybrid = np.asarray(res_sac["A_hybrid"], dtype=float)
    alpha = np.asarray(res_sac["alpha"], dtype=float)
    U_deep = np.asarray(res_sac["U_deep"], dtype=float)
    U_rl = np.asarray(res_sac["U_rl"], dtype=float)
    U_hybrid = np.asarray(res_sac["U_hybrid"], dtype=float)

    fig, ax = plt.subplots(3, 1, figsize=(8, 8), dpi=300, sharex=True)

    ax[0].plot(t, a_deep, label=r"$a_{DeepONet}$")
    if np.any(np.isfinite(a_rl)):
        ax[0].plot(t, a_rl, label=r"$a_{RL}$")
    ax[0].plot(t, a_hybrid, label=r"$a_{hybrid}$")
    ax[0].axhline(1.0, linestyle=":", linewidth=1)
    ax[0].axhline(-1.0, linestyle=":", linewidth=1)
    ax[0].set_ylabel("normalized action", fontsize=18, fontweight="bold", labelpad=8)
    ax[0].legend(**_legend_kwargs_two_columns(loc="best"))
    ax[0].grid(True, alpha=0.3)

    ax[1].plot(t, U_deep, label=r"$U_{DeepONet}$")
    if np.any(np.isfinite(U_rl)):
        ax[1].plot(t, U_rl, label=r"$U_{RL}$")
    ax[1].plot(t, U_hybrid, label=r"$U_{hybrid}$")
    ax[1].axhline(U_ref, linestyle=":", linewidth=1, label=r"$\bar U_L$")
    ax[1].axhline(max_control_value, linestyle=":", linewidth=1, label=r"$U_{max}$")
    ax[1].set_ylabel(r"$U_L$ [m]", fontsize=18, fontweight="bold", labelpad=8)
    ax[1].legend(**_legend_kwargs_two_columns(loc="best"))
    ax[1].grid(True, alpha=0.3)

    ax[2].plot(t, alpha, label=r"$\alpha$")
    ax[2].set_ylabel(r"RL weight $\alpha$", fontsize=18, fontweight="bold", labelpad=8)
    ax[2].set_xlabel("Time [min]", fontsize=18, fontweight="bold", labelpad=10)
    ax[2].set_ylim([-0.05, 1.05])
    ax[2].legend(**_legend_kwargs_two_columns(loc="best"))
    ax[2].grid(True, alpha=0.3)

    fig.suptitle("Hybrid DeepONet-SAC action diagnostics")
    fig.tight_layout()

    path = os.path.join(save_dir, f"{tag}_hybrid_action_diagnostics.png")
    fig.savefig(path, dpi=500, bbox_inches="tight")
    print(f"Saved hybrid action diagnostic figure to: {path}")
    plt.show()


def plot_control_input_comparison(res_sac, res_bs, save_dir, tag):
    """
    Plot downstream gate input U_L(t) for SAC and Backstepping.

    Upper panel:
        U_L^{SAC}(t) and U_L^{Backstepping}(t)

    Lower panel:
        Delta U_L(t) = U_L^{SAC}(t) - U_L^{Backstepping}(t)

    This figure is used to show how the learned SAC controller changes the
    physical downstream gate command compared with the nominal backstepping
    controller.
    """
    t_sac = np.asarray(res_sac["t"], dtype=float) / 60.0
    t_bs = np.asarray(res_bs["t"], dtype=float) / 60.0

    U_sac = np.asarray(res_sac["U"], dtype=float)
    U_bs = np.asarray(res_bs["U"], dtype=float)

    # Robust length alignment in case one rollout records one extra sample.
    n_sac = min(len(t_sac), len(U_sac))
    n_bs = min(len(t_bs), len(U_bs))
    t_sac = t_sac[:n_sac]
    U_sac = U_sac[:n_sac]
    t_bs = t_bs[:n_bs]
    U_bs = U_bs[:n_bs]

    # Interpolate Backstepping U onto the SAC time grid for Delta U.
    if len(t_bs) >= 2 and len(t_sac) >= 1:
        U_bs_on_sac = np.interp(t_sac, t_bs, U_bs)
        dU = U_sac - U_bs_on_sac
        t_delta = t_sac
    else:
        dU = np.array([])
        t_delta = np.array([])

    fig, ax = plt.subplots(2, 1, figsize=(8, 6), dpi=300, sharex=True)

    ax[0].plot(t_sac, U_sac, label=r"SAC $U_L(t)$")
    ax[0].plot(t_bs, U_bs, "--", label=r"BS $U_L(t)$")
    ax[0].axhline(U_ref, linestyle=":", linewidth=1, label=r"$\bar U_L$")
    ax[0].set_ylabel(r"$U_L(t)$ [m]", fontsize=18, fontweight="bold", labelpad=8)
    ax[0].legend(**_legend_kwargs_two_columns(loc="best"))
    ax[0].grid(True, alpha=0.3)

    if len(dU) > 0:
        ax[1].plot(t_delta, dU, label=r"$U_L^{SAC}(t)-U_L^{Backstepping}(t)$")
        ax[1].axhline(0.0, linestyle=":", linewidth=1)
    ax[1].set_ylabel(r"$\Delta U_L(t)$ [m]", fontsize=18, fontweight="bold", labelpad=8)
    ax[1].set_xlabel("Time [min]", fontsize=18, fontweight="bold", labelpad=10)
    ax[1].legend(**_legend_kwargs_two_columns(loc="best"))
    ax[1].grid(True, alpha=0.3)

    fig.suptitle("Control input comparison: SAC vs Backstepping")
    fig.tight_layout()

    path = os.path.join(save_dir, f"{tag}_SAC_vs_Backstepping_control_input.png")
    fig.savefig(path, dpi=500, bbox_inches="tight")
    print(f"Saved control input figure to: {path}")
    plt.show()


def _resolve_output_dir():
    """Return the active output directory used by this script."""
    if "SAVE_DIR" in globals():
        out = globals()["SAVE_DIR"]
    elif "DEFAULT_SAVE_DIR" in globals():
        out = globals()["DEFAULT_SAVE_DIR"]
    elif "save_dir" in globals():
        out = globals()["save_dir"]
    elif "out_dir" in globals():
        out = globals()["out_dir"]
    else:
        out = "./test_results_sac_style_backstepping_vs_sac_with_l2_3d"
    out = Path(out)
    out.mkdir(parents=True, exist_ok=True)
    return str(out)


def _compute_spatial_l2_over_time(res, state_key):
    """
    Compute spatial L2 norm over time:
        ||H-Hbar||_L2 = sqrt( integral_0^1 (H(x,t)-Hbar(x))^2 ds )
        ||Q-Qbar||_L2 = sqrt( integral_0^1 (Q(x,t)-Qbar(x))^2 ds )
    where s=x/L is the normalized spatial coordinate.
    """
    X = res["X"]
    s = X / X[-1]

    if state_key == "H":
        err = res["H"] - res["Hbar"][None, :]
    elif state_key == "Q":
        err = res["Q"] - res["Qbar"][None, :]
    else:
        raise ValueError("state_key must be 'H' or 'Q'.")

    return np.sqrt(np.trapezoid(err ** 2, s, axis=1))


def plot_l2_norm_comparison(res_sac, res_bs, tag="case_Q9p5_H3p5_4p4"):
    """
    Plot time evolution of spatial L2 norms:
        ||H-Hbar||_L2
        ||Q-Qbar||_L2
    for SAC and Backstepping.
    """
    t_sac = res_sac["t"] / 60.0
    t_bs = res_bs["t"] / 60.0

    H_l2_sac = _compute_spatial_l2_over_time(res_sac, "H")
    Q_l2_sac = _compute_spatial_l2_over_time(res_sac, "Q")
    H_l2_bs = _compute_spatial_l2_over_time(res_bs, "H")
    Q_l2_bs = _compute_spatial_l2_over_time(res_bs, "Q")

    fig, axes = plt.subplots(2, 1, figsize=(9, 7), sharex=True)

    axes[0].plot(t_sac, H_l2_sac, label=r"SAC $\Vert H-\bar H\Vert_{L_2}$ [m]")
    axes[0].plot(t_bs, H_l2_bs, "--", label=r"BS $\Vert H-\bar H\Vert_{L_2}$ [m]")
    axes[0].set_ylabel(r"$\Vert H-\bar H\Vert_{L_2}$ [m]", fontsize=18, fontweight="bold", labelpad=8)
    axes[0].legend(**_legend_kwargs_two_columns(loc="best"))
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(t_sac, Q_l2_sac, label=r"SAC $\Vert Q-\bar Q\Vert_{L_2}$ [m$^3$/s]")
    axes[1].plot(t_bs, Q_l2_bs, "--", label=r"BS $\Vert Q-\bar Q\Vert_{L_2}$ [m$^3$/s]")
    axes[1].set_ylabel(r"$\Vert Q-\bar Q\Vert_{L_2}$ [m$^3$/s]", fontsize=18, fontweight="bold", labelpad=8)
    axes[1].set_xlabel("Time [min]", fontsize=18, fontweight="bold", labelpad=10)
    axes[1].legend(**_legend_kwargs_two_columns(loc="best"))
    axes[1].grid(True, alpha=0.3)

    fig.suptitle("Spatial L2 norm tracking errors")
    fig.tight_layout()

    fig_path = os.path.join(_resolve_output_dir(), f"{tag}_H_Q_L2_norm_errors.png")
    fig.savefig(fig_path, dpi=300, bbox_inches="tight")
    print(f"L2-norm error figure saved to: {fig_path}")
    plt.show()

    # Save L2 norm time series as .dat files for later plotting.
    np.savetxt(
        os.path.join(_resolve_output_dir(), f"{tag}_SAC_H_L2.dat"),
        np.column_stack([t_sac, H_l2_sac]),
        header="time_min H_L2",
    )
    np.savetxt(
        os.path.join(_resolve_output_dir(), f"{tag}_SAC_Q_L2.dat"),
        np.column_stack([t_sac, Q_l2_sac]),
        header="time_min Q_L2",
    )
    np.savetxt(
        os.path.join(_resolve_output_dir(), f"{tag}_Backstepping_H_L2.dat"),
        np.column_stack([t_bs, H_l2_bs]),
        header="time_min H_L2",
    )
    np.savetxt(
        os.path.join(_resolve_output_dir(), f"{tag}_Backstepping_Q_L2.dat"),
        np.column_stack([t_bs, Q_l2_bs]),
        header="time_min Q_L2",
    )
    print("L2-norm time series saved.")


def plot_surface_single(X, t, Z, zlabel, title, save_path):
    """
    Draw one 3D surface plot for H(x,t) or Q(x,t).
    Plotting style is matched to the first program; all data and calls are unchanged.
    """
    X_km = X / 1000.0
    t_min = t / 60.0

    meshx, mesht = np.meshgrid(X_km, t_min)

    fig = plt.figure(figsize=(8.8, 6.6), dpi=600)
    subfigs = fig.subfigures(nrows=1, ncols=1, hspace=0)
    subfig = subfigs
    subfig.subplots_adjust(left=0.18, bottom=0.08, right=0.98, top=0.96)

    axes = subfig.subplots(
        nrows=1,
        ncols=1,
        subplot_kw={"projection": "3d", "computed_zorder": False},
    )

    for axis in [axes.xaxis, axes.yaxis, axes.zaxis]:
        axis._axinfo["axisline"]["linewidth"] = 10
        axis._axinfo["axisline"]["color"] = "b"
        axis._axinfo["grid"]["linewidth"] = 0.8
        axis._axinfo["grid"]["linestyle"] = "--"
        axis._axinfo["grid"]["color"] = "#d1d1d1"
        axis.set_pane_color((1, 1, 1))

    axes.view_init(10, 15)
    axes.set_xlabel(r"$x$ [km]", fontsize=18, labelpad=5, fontweight="bold")
    axes.set_ylabel(r"$Time$ [min]", fontsize=18, labelpad=5, fontweight="bold")
    axes.set_zlabel(zlabel, rotation=90, fontsize=18, labelpad=4, fontweight="bold")
    axes.zaxis.set_rotate_label(False)
    axes.zaxis.label.set_clip_on(False)

    x_ticks = np.array([X_km[0], 0.5 * (X_km[0] + X_km[-1]), X_km[-1]], dtype=float)
    y_ticks = np.array([t_min[0], 0.5 * (t_min[0] + t_min[-1]), t_min[-1]], dtype=float)
    z_min, z_max = float(np.nanmin(Z)), float(np.nanmax(Z))
    z_ticks = np.linspace(z_min, z_max, 5)

    axes.set_xticks(x_ticks)
    axes.set_yticks(y_ticks)
    axes.set_zticks(z_ticks)
    axes.set_xticklabels(_format_three_ticklabels(x_ticks), fontweight="bold", fontsize=18)
    axes.set_yticklabels(_format_three_ticklabels(y_ticks), fontweight="bold", fontsize=18)
    axes.set_zticklabels(_format_z_ticklabels(z_ticks), fontweight="bold", fontsize=18)

    rstride = max(1, Z.shape[0] // 6)
    axes.plot_surface(
        meshx,
        mesht,
        Z,
        edgecolor="black",
        lw=0.8,
        rstride=rstride,
        cstride=5,
        alpha=1,
        color="white",
        shade=False,
        rasterized=True,
        antialiased=True,
    )

    # Downstream boundary highlight curve remains removed as in the target program.

    axes.set_title(title, fontsize=14)
    fig.savefig(save_path, dpi=600, bbox_inches="tight", pad_inches=0.35)
    print(f"3D surface saved to: {save_path}")
    plt.show()
    plt.close(fig)

def plot_HQ_3d_surfaces(res_sac, res_bs, tag="case_Q9p5_H3p5_4p4"):
    """
    Plot H(x,t) and Q(x,t) 3D surfaces for SAC and Backstepping.
    Four figures are generated:
        SAC H, SAC Q, Backstepping H, Backstepping Q.
    """
    plot_surface_single(
        res_sac["X"],
        res_sac["t"],
        res_sac["H"],
        r"$H(x,t)$ [m]",
        "SAC: $H(x,t)$",
        os.path.join(_resolve_output_dir(), f"{tag}_SAC_H_3d.png"),
    )

    plot_surface_single(
        res_sac["X"],
        res_sac["t"],
        res_sac["Q"],
        r"$Q(x,t)$ [m$^3$/s]",
        "SAC: $Q(x,t)$",
        os.path.join(_resolve_output_dir(), f"{tag}_SAC_Q_3d.png"),
    )

    plot_surface_single(
        res_bs["X"],
        res_bs["t"],
        res_bs["H"],
        r"$H(x,t)$ [m]",
        "Backstepping: $H(x,t)$",
        os.path.join(_resolve_output_dir(), f"{tag}_Backstepping_H_3d.png"),
    )

    plot_surface_single(
        res_bs["X"],
        res_bs["t"],
        res_bs["Q"],
        r"$Q(x,t)$ [m$^3$/s]",
        "Backstepping: $Q(x,t)$",
        os.path.join(_resolve_output_dir(), f"{tag}_Backstepping_Q_3d.png"),
    )


def save_final_error_vectors(res_sac, res_bs, tag="case_Q9p5_H3p5_4p4"):
    """
    Save final H-Hbar and Q-Qbar vectors for both controllers.
    """
    np.savetxt(
        os.path.join(_resolve_output_dir(), f"{tag}_SAC_final_H_minus_Hbar.dat"),
        res_sac["H"][-1] - res_sac["Hbar"],
    )
    np.savetxt(
        os.path.join(_resolve_output_dir(), f"{tag}_SAC_final_Q_minus_Qbar.dat"),
        res_sac["Q"][-1] - res_sac["Qbar"],
    )
    np.savetxt(
        os.path.join(_resolve_output_dir(), f"{tag}_Backstepping_final_H_minus_Hbar.dat"),
        res_bs["H"][-1] - res_bs["Hbar"],
    )
    np.savetxt(
        os.path.join(_resolve_output_dir(), f"{tag}_Backstepping_final_Q_minus_Qbar.dat"),
        res_bs["Q"][-1] - res_bs["Qbar"],
    )
    print("Final H-Hbar and Q-Qbar vectors saved.")

def save_npz(res_sac, res_bs, save_dir, tag):
    path = os.path.join(save_dir, f"{tag}_SAC_vs_Backstepping_exact.npz")
    np.savez(
        path,
        X=res_sac["X"],
        Hbar=res_sac["Hbar"],
        Qbar=res_sac["Qbar"],
        t_sac=res_sac["t"],
        H_sac=res_sac["H"],
        Q_sac=res_sac["Q"],
        U_sac=res_sac["U"],
        reward_sac=res_sac["reward"],
        t_backstepping=res_bs["t"],
        H_backstepping=res_bs["H"],
        Q_backstepping=res_bs["Q"],
        U_backstepping=res_bs["U"],
        reward_backstepping=res_bs["reward"],
        total_reward_sac=res_sac["total_reward"],
        total_reward_backstepping=res_bs["total_reward"],
    )
    print(f"Saved data to: {path}")


def print_final(name, res):
    H = res["H"]
    Q = res["Q"]
    Hbar = res["Hbar"]
    Qbar = res["Qbar"]
    print("")
    print("=" * 90)
    print(f"{name} final")
    print("=" * 90)
    print(f"final H(0) = {H[-1,0]:.6f}, target = {Hbar[0]:.6f}, error = {H[-1,0]-Hbar[0]:.6f}")
    print(f"final H(L) = {H[-1,-1]:.6f}, target = {Hbar[-1]:.6f}, error = {H[-1,-1]-Hbar[-1]:.6f}")
    print(f"final Q(0) = {Q[-1,0]:.6f}, target = {Qbar[0]:.6f}, error = {Q[-1,0]-Qbar[0]:.6f}")
    print(f"final Q(L) = {Q[-1,-1]:.6f}, target = {Qbar[-1]:.6f}, error = {Q[-1,-1]-Qbar[-1]:.6f}")
    print(f"total_reward = {res['total_reward']:.6f}")


def print_final_error_values(name, res):
    """
    Print the final-time values of H-Hbar and Q-Qbar at every spatial grid point.
    """
    X = np.asarray(res["X"], dtype=float)
    H_err_final = np.asarray(res["H"][-1] - res["Hbar"], dtype=float)
    Q_err_final = np.asarray(res["Q"][-1] - res["Qbar"], dtype=float)

    print("")
    print("=" * 90)
    print(f"{name} final H-Hbar and Q-Qbar values")
    print("=" * 90)
    print("index, x[m], H-Hbar[m], Q-Qbar[m^3/s]")
    for i, (x_i, h_i, q_i) in enumerate(zip(X, H_err_final, Q_err_final)):
        print(f"{i:4d}, {x_i:12.6f}, {h_i: .10e}, {q_i: .10e}")


def save_final_error_value_table(res_sac, res_bs, save_dir, tag):
    """
    Save the final-time H-Hbar and Q-Qbar values of SAC and Backstepping
    into one .dat file for direct inspection.
    """
    X = np.asarray(res_sac["X"], dtype=float)

    sac_H_err = np.asarray(res_sac["H"][-1] - res_sac["Hbar"], dtype=float)
    sac_Q_err = np.asarray(res_sac["Q"][-1] - res_sac["Qbar"], dtype=float)

    bs_H_err = np.asarray(res_bs["H"][-1] - res_bs["Hbar"], dtype=float)
    bs_Q_err = np.asarray(res_bs["Q"][-1] - res_bs["Qbar"], dtype=float)

    data = np.column_stack([
        X,
        sac_H_err,
        sac_Q_err,
        bs_H_err,
        bs_Q_err,
    ])

    path = os.path.join(save_dir, f"{tag}_final_H_Q_error_values.dat")
    np.savetxt(
        path,
        data,
        header=(
            "x_m "
            "SAC_H_minus_Hbar_m "
            "SAC_Q_minus_Qbar_m3s "
            "Backstepping_H_minus_Hbar_m "
            "Backstepping_Q_minus_Qbar_m3s"
        ),
    )
    print(f"Final H-Hbar and Q-Qbar value table saved to: {path}")



# ============================================================
# Controller-comparison comparison utilities
# ============================================================

LABEL_BACKSTEPPING = "BS"
LABEL_NEW_MODEL = "TDPSAC"

# Retain the previous label constants so the common style helper can be reused
# when additional checkpoints are added later.
LABEL_SAC_CURVE = "SAC"
LABEL_FP_SAC = "SAC-FPC"
LABEL_SAC = "DO-SAC"
LABEL_HYBRID = "DPSAC"
LABEL_DFP_DON_SAC = "TDPSAC-FPC"
LABEL_HYBRID_FREEZE = LABEL_NEW_MODEL

# Fixed order for 2D legends, terminal summaries, evidence tables, and saved
# controller arrays.  This is independent of the order in which models are run.
CONTROLLER_DISPLAY_ORDER = [
    LABEL_BACKSTEPPING,
    LABEL_SAC_CURVE,
    LABEL_FP_SAC,
    LABEL_SAC,
    LABEL_HYBRID,
    LABEL_DFP_DON_SAC,
    LABEL_NEW_MODEL,
]


def _ordered_results(results: Dict[str, dict]) -> Dict[str, dict]:
    """Return a dictionary in the requested display/table order."""
    ordered = {
        label: results[label]
        for label in CONTROLLER_DISPLAY_ORDER
        if label in results
    }
    # Preserve any future additional controllers after the seven fixed entries.
    for label, res in results.items():
        if label not in ordered:
            ordered[label] = res
    return ordered


def _safe_load_hybrid_model(model_path: str, env):
    """
    Load a HybridSAC checkpoint for evaluation.

    The HybridSAC training script may save optimizers with special parameter
    groups for DeepONet. EvalHybridSAC rebuilds the optimizer structure. If the
    optimizer groups still do not match, an eval-only copy without optimizer
    states is created and loaded. This does not change the policy weights.
    """
    try:
        return EvalHybridSAC.load(model_path, env=env)
    except Exception as e:
        msg = str(e)
        if ("parameter group" in msg) or ("Names of parameters" in msg) or ("optimizer" in msg):
            print("Hybrid load with optimizer states failed; trying eval-only checkpoint.")
            print("Original error:", repr(e))
            eval_path = strip_optimizer_states_from_sb3_model(model_path)
            return EvalHybridSAC.load(eval_path, env=env)
        raise



def _checkpoint_exists(model_path: str) -> bool:
    """
    Return True only when model_path resolves to a valid SB3 ZIP checkpoint.

    This deliberately rejects arbitrary existing files such as Jupyter's
    kernel-*.json connection file. Stable-Baselines3 checkpoints are ZIP files,
    although callers may pass the path either with or without the .zip suffix.
    """
    if model_path is None or str(model_path).strip() == "":
        return False

    path = Path(str(model_path)).expanduser()

    candidates = [path]
    if path.suffix.lower() != ".zip":
        candidates.append(path.with_suffix(".zip"))

    for candidate in candidates:
        if candidate.is_file() and zipfile.is_zipfile(candidate):
            return True

    return False


def _safe_load_generic_sac_model(model_path: str, env):
    """
    Load a custom SAC-family checkpoint without assuming one exact policy layout.

    This is used for TDPSAC-FPC and SAC-FPC because their policy
    architectures may differ. The loader first asks SB3 to reconstruct the
    policy class stored in the checkpoint. If optimizer parameter groups are
    incompatible during evaluation, it creates an evaluation-only copy with
    optimizer states removed and retries.

    Policy, actor, critic, normalization, and saved alpha weights are preserved.
    """
    errors = []

    try:
        return SAC.load(model_path, env=env)
    except Exception as exc:
        errors.append(("SAC.load", exc))

    try:
        eval_path = strip_optimizer_states_from_sb3_model(model_path)
        return SAC.load(eval_path, env=env)
    except Exception as exc:
        errors.append(("SAC.load(eval-only)", exc))

    # Final fallback for checkpoints trained with the HybridSAC subclass used
    # in this file.
    try:
        return _safe_load_hybrid_model(model_path, env=env)
    except Exception as exc:
        errors.append(("_safe_load_hybrid_model", exc))

    details = "\n".join(f"  {name}: {error!r}" for name, error in errors)
    raise RuntimeError(
        f"Unable to load SAC-family checkpoint: {model_path}\n"
        f"Tried the generic and HybridSAC evaluation loaders:\n{details}"
    )

def run_policy_exact(
    model_path: str,
    model_kind: str,
    label: str,
    seed: int = 123,
    H_left: float = TEST_H_LEFT,
    H_right: float = TEST_H_RIGHT,
    Q_value: float = TEST_Q_VALUE,
    dH_source=None,
):
    """
    Run one learned controller in the CAC environment.

    model_kind:
        "sac"       -> ordinary raw-HQ SAC.load(...)
        "sac_error" -> old SAC checkpoint using normalized H/Q errors
        "hybrid"    -> EvalHybridSAC.load(...), for the existing hybrid checkpoints
        "auto"      -> reconstruct the saved SAC policy class automatically; used
                       for TDPSAC-FPC and SAC-FPC

    Every learned controller sees the same physical PDE reference, initial
    state, and raw physical H/Q observation interface in the current comparison.
    The reset state is recorded first at t=0, then later states are recorded
    after env.step().
    """
    model_kind_lower = model_kind.lower()
    if model_kind_lower == "sac_error":
        env, X, Hbar, Qbar, p = make_env_sac_error_state(
            H_left=H_left,
            H_right=H_right,
            Q_value=Q_value,
            seed=seed,
            dH_source=dH_source,
        )
    else:
        env, X, Hbar, Qbar, p = make_env_sac(
            H_left=H_left,
            H_right=H_right,
            Q_value=Q_value,
            seed=seed,
            dH_source=dH_source,
        )

    if model_kind_lower in {"sac", "sac_error"}:
        model = SAC.load(model_path, env=env)
    elif model_kind_lower == "hybrid":
        model = _safe_load_hybrid_model(model_path, env=env)
    elif model_kind_lower == "auto":
        model = _safe_load_generic_sac_model(model_path, env=env)
    else:
        raise ValueError(
            f"Unknown model_kind: {model_kind}. Use 'sac', 'sac_error', "
            "'hybrid', or 'auto'."
        )

    obs, _ = env.reset(seed=seed)
    obs = np.asarray(obs, dtype=np.float32).reshape(-1)
    if obs.size != RAW_STATE_DIM:
        raise RuntimeError(
            f"The reset observation has dimension {obs.size}; expected "
            f"{RAW_STATE_DIM} for the H/Q or H/Q-error interface."
        )

    observation_mode_used = (
        "normalized_error" if model_kind_lower == "sac_error" else "raw"
    )
    print(
        f"{label} observation mode: {observation_mode_used}, "
        f"dimension={obs.size}"
    )

    H_hist, Q_hist, U_hist, t_hist, reward_hist = [], [], [], [], []

    # Record the reset state before any controller acts.
    # This makes the learned controllers start from
    # exactly the same L2 error at t=0.
    H0, Q0 = env.get_current_HQ()
    H_hist.append(H0.copy())
    Q_hist.append(Q0.copy())
    U_hist.append(np.nan)
    t_hist.append(0.0)
    reward_hist.append(0.0)

    previous_u_filtered = float(U_ref)

    done = False
    while not done:
        action, _ = model.predict(obs, deterministic=True)

        action_filtered, u_command_raw, previous_u_filtered = lowpass_filter_rl_control(
            action,
            previous_u_filtered,
            alpha=RL_CONTROL_FILTER_ALPHA,
        )

        obs, reward, terminated, truncated, info = env.step(action_filtered)

        H, Q = env.get_current_HQ()

        H_hist.append(H.copy())
        Q_hist.append(Q.copy())
        U_hist.append(float(env.last_control))
        t_hist.append(float(info["time"]))
        reward_hist.append(float(reward))

        done = terminated or truncated

    env.close()

    H_hist = np.asarray(H_hist)
    Q_hist = np.asarray(Q_hist)
    U_hist = np.asarray(U_hist)
    t_hist = np.asarray(t_hist)
    reward_hist = np.asarray(reward_hist)

    H_err = H_hist - Hbar[None, :]
    Q_err = Q_hist - Qbar[None, :]

    return {
        "label": label,
        "model_kind": model_kind,
        "observation_mode": observation_mode_used,
        "X": X,
        "Hbar": Hbar,
        "Qbar": Qbar,
        "t": t_hist,
        "H": H_hist,
        "Q": Q_hist,
        "U": U_hist,
        "reward": reward_hist,
        "H_l2": np.sqrt(np.mean(H_err ** 2, axis=1)),
        "Q_l2": np.sqrt(np.mean(Q_err ** 2, axis=1)),
        "final_H_minus_Hbar": H_err[-1],
        "final_Q_minus_Qbar": Q_err[-1],
        "total_reward": float(np.sum(reward_hist)),
    }


def run_backstepping_for_four(
    seed: int = 123,
    H_left: float = TEST_H_LEFT,
    H_right: float = TEST_H_RIGHT,
    Q_value: float = TEST_Q_VALUE,
    dH_source=None,
):
    res = run_backstepping_exact(
        seed=seed,
        H_left=H_left,
        H_right=H_right,
        Q_value=Q_value,
        dH_source=dH_source,
    )
    res["label"] = LABEL_BACKSTEPPING
    res["model_kind"] = "backstepping"
    return res


def _time_min(res):
    return np.asarray(res["t"], dtype=float) / 60.0


def _get_l2(res, key):
    return _compute_spatial_l2_over_time(res, key)


def _get_linf(res, key):
    """Return the spatial L-infinity tracking-error norm at each time instant."""
    state = np.asarray(res[key], dtype=float)
    reference_key = "Hbar" if key == "H" else "Qbar"
    reference = np.asarray(res[reference_key], dtype=float).reshape(1, -1)
    if state.ndim != 2 or reference.shape[1] != state.shape[1]:
        return np.asarray([], dtype=float)
    return np.max(np.abs(state - reference), axis=1)


def set_size(width, fraction=1, ncols=1, nrows=1, height_add=0):
    """Figure size helper copied from the third comparison-program style."""
    inches_per_pt = 1 / 72.27
    fig_width = width * inches_per_pt * fraction
    fig_height = fig_width * 0.618 * (nrows / ncols) + height_add
    return fig_width, fig_height



def _line_style(label):
    """Return the fixed colors and line styles used by the old comparison program."""
    if label == LABEL_BACKSTEPPING:  # BS
        return dict(color="orange", linestyle=":", linewidth=2.0, alpha=0.8)
    if label == LABEL_SAC_CURVE:  # SAC
        return dict(color="purple", linestyle="-", linewidth=2.0, alpha=0.8)
    if label == LABEL_FP_SAC:  # SAC-FPC
        return dict(
            color="saddlebrown",
            linestyle=(0, (6, 2, 1.5, 2)),
            linewidth=2.2,
            alpha=0.95,
        )
    if label == LABEL_SAC:  # DO-SAC
        return dict(color="blue", linestyle="-.", linewidth=2.0, alpha=0.8)
    if label == LABEL_HYBRID:  # DPSAC
        return dict(color="green", linestyle="--", linewidth=2.0, alpha=0.8)
    if label == LABEL_DFP_DON_SAC:  # TDPSAC-FPC
        return dict(color="black", linestyle="--", linewidth=2.2, alpha=0.95)
    if label == LABEL_HYBRID_FREEZE:  # TDPSAC
        return dict(color="red", linestyle="-", linewidth=2.0, alpha=0.8)
    return dict(linewidth=2.0, alpha=0.8)




def _legend_kwargs_two_columns(loc="upper center", bbox_to_anchor=None):
    """
    Shared legend style for the displayed methods.
    """
    kwargs = dict(
        fontsize=11,
        frameon=True,
        shadow=True,
        ncol=2,
        columnspacing=1.2,
        handlelength=2.8,
        borderaxespad=0.6,
    )
    kwargs["loc"] = loc
    if bbox_to_anchor is not None:
        kwargs["bbox_to_anchor"] = bbox_to_anchor
    return kwargs


def _apply_ordered_two_column_legend(
    ax,
    *,
    loc="best",
    fontsize=11,
    frameon=True,
    shadow=True,
    columnspacing=1.2,
    handlelength=2.8,
    borderaxespad=0.6,
):
    """Draw a two-column legend in top-to-bottom column order.

    The legend names follow CONTROLLER_DISPLAY_ORDER from top to bottom.
    With seven methods and two columns, the displayed order is:

        left column:  BS, SAC, SAC-FPC, DO-SAC
        right column: DPSAC, TDPSAC-FPC, TDPSAC
    """
    handles, labels = ax.get_legend_handles_labels()
    if not handles:
        return None

    pair_by_label = {
        label: (handle, label)
        for handle, label in zip(handles, labels)
    }

    ordered_pairs = [
        pair_by_label[label]
        for label in CONTROLLER_DISPLAY_ORDER
        if label in pair_by_label
    ]
    ordered_pairs.extend(
        (handle, label)
        for handle, label in zip(handles, labels)
        if label not in CONTROLLER_DISPLAY_ORDER
    )

    # Matplotlib fills a multi-column legend from top to bottom within each
    # column, so pass the requested order directly without row-wise repacking.
    ordered_handles = [pair[0] for pair in ordered_pairs]
    ordered_labels = [pair[1] for pair in ordered_pairs]

    return ax.legend(
        ordered_handles,
        ordered_labels,
        fontsize=fontsize,
        frameon=frameon,
        shadow=shadow,
        loc=loc,
        ncol=2,
        columnspacing=columnspacing,
        handlelength=handlelength,
        borderaxespad=borderaxespad,
    )


def _style_axes_second(ax, xlabel=None, ylabel=None, legend=True, legend_loc="best"):
    """2D axis style matched to the first program."""
    if xlabel is not None:
        ax.set_xlabel(xlabel, fontsize=18, fontweight="bold", labelpad=10)
    if ylabel is not None:
        ax.set_ylabel(ylabel, fontsize=18, fontweight="bold", labelpad=8)
    ax.tick_params(axis="both", labelsize=18)
    for tick in ax.get_xticklabels() + ax.get_yticklabels():
        tick.set_fontweight("bold")
    ax.grid(True, linestyle="--", alpha=0.8)
    if legend:
        _apply_ordered_two_column_legend(ax, loc=legend_loc, fontsize=11)


# Keep old function name for compatibility with the rest of the script.
def _style_axes(ax, xlabel=None, ylabel=None, legend=True, legend_loc="best"):
    _style_axes_second(ax, xlabel=xlabel, ylabel=ylabel, legend=legend, legend_loc=legend_loc)


def _safe_label_for_plot(label):
    """Keep algorithm names exactly the same as the second program."""
    return label


def _common_time_end_min(results: Dict[str, dict]) -> float:
    """Return the common end time in minutes across all results."""
    ends = []
    for res in results.values():
        t = _time_min(res)
        if len(t) > 0 and np.any(np.isfinite(t)):
            ends.append(float(np.nanmax(t)))
    return min(ends) if ends else 0.0


def _style_inset_axes(ax_inset):
    """Inset style copied from the second program."""
    ax_inset.grid(True, linestyle="--", alpha=0.8)
    ax_inset.tick_params(axis="both", labelsize=14)


def _add_second_style_inset(
    ax,
    results: Dict[str, dict],
    y_getter,
    ref_value=None,
    zoom_ratio: float = 0.75,
    bounds=(0.55, 0.15, 0.40, 0.40),
):
    """
    Add a local zoom inset with the same style as the second program:
        ax.inset_axes([...]) + ax.indicate_inset_zoom(...).

    This function only changes plotting. It does not change model loading,
    controller output, reward, rollout data, or saved numerical results.
    """
    min_t_end = _common_time_end_min(results)
    if min_t_end <= 0:
        return None

    zoom_start = float(zoom_ratio) * min_t_end
    ax_inset = ax.inset_axes(list(bounds))

    y_all = []
    for label, res in results.items():
        style = _line_style(label)
        t = _time_min(res)
        y = np.asarray(y_getter(res), dtype=float).reshape(-1)
        n = min(len(t), len(y))
        tt = t[:n]
        yy = y[:n]
        mask = (tt >= zoom_start) & (tt <= min_t_end)
        if np.any(mask):
            ax_inset.plot(tt[mask], yy[mask], **style)
            y_all.extend(yy[mask][np.isfinite(yy[mask])].tolist())

    if ref_value is not None:
        ax_inset.axhline(ref_value, linestyle="--", linewidth=1.4, color="dimgray", alpha=0.95)
        if np.isfinite(ref_value):
            y_all.append(float(ref_value))

    if y_all:
        y_min, y_max = float(np.min(y_all)), float(np.max(y_all))
        margin = (y_max - y_min) * 0.1 if y_max != y_min else 0.1
        ax_inset.set_ylim(y_min - margin, y_max + margin)
    ax_inset.set_xlim(zoom_start, min_t_end)
    _style_inset_axes(ax_inset)

    # Same guide-line style as the second program.
    try:
        ax.indicate_inset_zoom(ax_inset, edgecolor="black", linewidth=1.5)
    except Exception:
        pass
    return ax_inset


# Backward-compatible alias for old calls.
def _add_end_time_inset(
    ax,
    results: Dict[str, dict],
    y_getter,
    ref_value=None,
    zoom_ratio: float = 0.75,
    bounds=(0.55, 0.15, 0.40, 0.40),
):
    return _add_second_style_inset(
        ax,
        results,
        y_getter,
        ref_value=ref_value,
        zoom_ratio=zoom_ratio,
        bounds=bounds,
    )


def plot_l2_norm_four(results: Dict[str, dict], save_dir: str, tag: str):
    """Controller-comparison L2 plots with the same 2D style as the second program.

    H-Hbar and Q-Qbar are drawn in two separate figures. Other logic remains unchanged.
    """
    results = _ordered_results(results)
    # --------------------------------------------------------
    # Figure 1: ||H-Hbar||_L2
    # --------------------------------------------------------
    fig_h, ax_h = plt.subplots(
        figsize=set_size(433, 0.99, ncols=1, nrows=1, height_add=1.0),
        dpi=600,
    )

    for label, res in results.items():
        style = _line_style(label)
        plot_label = _safe_label_for_plot(label)
        ax_h.plot(_time_min(res), _get_l2(res, "H"), label=plot_label, **style)

    ax_h.axhline(y=0.0, color="dimgray", linestyle="--", alpha=0.95, linewidth=1.8, label="_nolegend_")
    _style_axes_second(
        ax_h,
        xlabel=r"$Time$ [min]",
        ylabel=r"$\Vert H-\bar H\Vert_{L_2}$ [m]",
        legend=True,
    )

    fig_h.tight_layout()
    path_h = os.path.join(save_dir, f"{tag}_seven_algorithms_H_L2_norm_error_second_style.png")
    fig_h.savefig(path_h, dpi=600, bbox_inches="tight")
    print(f"Saved seven-controller H L2 figure to: {path_h}")
    plt.show()

    # --------------------------------------------------------
    # Figure 2: ||Q-Qbar||_L2
    # --------------------------------------------------------
    fig_q, ax_q = plt.subplots(
        figsize=set_size(433, 0.99, ncols=1, nrows=1, height_add=1.0),
        dpi=600,
    )

    for label, res in results.items():
        style = _line_style(label)
        plot_label = _safe_label_for_plot(label)
        ax_q.plot(_time_min(res), _get_l2(res, "Q"), label=plot_label, **style)

    ax_q.axhline(y=0.0, color="dimgray", linestyle="--", alpha=0.95, linewidth=1.8, label="_nolegend_")
    _style_axes_second(
        ax_q,
        xlabel=r"$Time$ [min]",
        ylabel=r"$\Vert Q-\bar Q\Vert_{L_2}$ [m$^3$/s]",
        legend=True,
    )

    # Only Q-Qbar keeps local zoom, as previously requested.
    _add_second_style_inset(
        ax_q,
        results,
        lambda res: _get_l2(res, "Q"),
        ref_value=0.0,
        zoom_ratio=0.75,
        bounds=(0.55, 0.15, 0.40, 0.40),
    )

    fig_q.tight_layout()
    path_q = os.path.join(save_dir, f"{tag}_seven_algorithms_Q_L2_norm_error_second_style.png")
    fig_q.savefig(path_q, dpi=600, bbox_inches="tight")
    print(f"Saved seven-controller Q L2 figure to: {path_q}")
    plt.show()

    # Save each L2 time series.
    for label, res in results.items():
        safe_label = label.replace("+", "_plus_").replace(" ", "_").replace("/", "_")
        np.savetxt(
            os.path.join(save_dir, f"{tag}_{safe_label}_H_L2.dat"),
            np.column_stack([_time_min(res), _get_l2(res, "H")]),
            header="time_min H_L2",
        )
        np.savetxt(
            os.path.join(save_dir, f"{tag}_{safe_label}_Q_L2.dat"),
            np.column_stack([_time_min(res), _get_l2(res, "Q")]),
            header="time_min Q_L2",
        )


def _pad_to_length_1d(data, target_len):
    data = np.asarray(data, dtype=float).reshape(-1)
    if len(data) < target_len:
        return np.pad(data, (0, target_len - len(data)), mode="edge")
    return data[:target_len]


def plot_control_input_four(results: Dict[str, dict], save_dir: str, tag: str):
    """U_L(t) plot with the same 2D style as the second program; no inset."""
    results = _ordered_results(results)
    fig, ax = plt.subplots(
        figsize=set_size(433, 0.99, ncols=1, nrows=1, height_add=1.0),
        dpi=600,
    )

    min_t_end = min(float(np.nanmax(_time_min(res))) for res in results.values())
    for label, res in results.items():
        style = _line_style(label)
        t = _time_min(res)
        U = np.asarray(res["U"], dtype=float)
        n = min(len(t), len(U))
        ax.plot(t[:n], U[:n], label=_safe_label_for_plot(label), **style)

    ax.axhline(y=U_ref, color="dimgray", linestyle="--", alpha=0.95, linewidth=1.8, label="_nolegend_")
    ax.set_xlim(0, min_t_end)
    _style_axes_second(ax, xlabel=r"$Time$ [min]", ylabel=r"$U_L(t)$ [m]", legend=True)

    fig.tight_layout()
    path = os.path.join(save_dir, f"{tag}_seven_algorithms_control_input_second_style.png")
    fig.savefig(path, dpi=600, bbox_inches="tight")
    print(f"Saved seven-controller control input figure to: {path}")
    plt.show()


def plot_boundary_comparison_four(results: Dict[str, dict], save_dir: str, tag: str):
    """Boundary H/Q comparison with the same 2D style as the second program."""
    results = _ordered_results(results)
    first = next(iter(results.values()))
    Hbar = first["Hbar"]
    Qbar = first["Qbar"]

    fig, axes = plt.subplots(
        2, 2,
        figsize=set_size(780, 0.99, ncols=2, nrows=2, height_add=1.1),
        dpi=600,
        sharex=True,
    )
    axes = axes.ravel()

    specs = [
        ("H", 0, r"$H(0,t)$ [m]", Hbar[0]),
        ("H", -1, r"$H(L,t)$ [m]", Hbar[-1]),
        ("Q", 0, r"$Q(0,t)$ [m$^3$/s]", Qbar[0]),
        ("Q", -1, r"$Q(L,t)$ [m$^3$/s]", Qbar[-1]),
    ]

    for ax, (var, idx, ylabel, ref) in zip(axes, specs):
        for label, res in results.items():
            style = _line_style(label)
            y = np.asarray(res[var])[:, idx]
            t = _time_min(res)
            n = min(len(t), len(y))
            ax.plot(t[:n], y[:n], label=_safe_label_for_plot(label), **style)

        ax.axhline(y=ref, color="dimgray", linestyle="--", alpha=0.95, linewidth=1.8, label="_nolegend_")
        _style_axes_second(ax, ylabel=ylabel, legend=False)

        # Only Q(L,t) keeps local zoom, as previously requested.
        if var == "Q" and idx == -1:
            _add_second_style_inset(
                ax,
                results,
                lambda res, var=var, idx=idx: np.asarray(res[var])[:, idx],
                ref_value=float(ref),
                zoom_ratio=0.75,
                bounds=(0.55, 0.52, 0.40, 0.40),
            )

    _style_axes_second(axes[2], xlabel=r"$Time$ [min]", ylabel=specs[2][2], legend=False)
    _style_axes_second(axes[3], xlabel=r"$Time$ [min]", ylabel=specs[3][2], legend=False)
    _apply_ordered_two_column_legend(
        axes[0],
        fontsize=14,
        frameon=True,
        shadow=True,
        loc="best",
        columnspacing=1.2,
        handlelength=2.8,
        borderaxespad=0.6,
    )

    fig.tight_layout()
    path = os.path.join(save_dir, f"{tag}_seven_algorithms_boundary_comparison_second_style.png")
    fig.savefig(path, dpi=600, bbox_inches="tight")
    print(f"Saved seven-controller boundary figure to: {path}")
    plt.show()


def _format_three_ticklabels(values):
    """Format three x/y tick labels in the same style as the first program's 3D figures."""
    values = np.asarray(values, dtype=float)
    labels = []
    for value in values:
        if abs(value) >= 10:
            labels.append(f"{value:.0f}")
        elif abs(value) >= 1:
            labels.append(f"{value:.1f}")
        else:
            labels.append(f"{value:.2f}")
    return labels

def _format_z_ticklabels(values):
    """Format z-axis tick labels in the same style as the first program's 3D figures."""
    values = np.asarray(values, dtype=float)
    span = float(np.nanmax(values) - np.nanmin(values)) if len(values) > 0 else 0.0
    if span >= 10:
        return [f"{v:.0f}" for v in values]
    if span >= 1:
        return [f"{v:.1f}" for v in values]
    return [f"{v:.2f}" for v in values]

def _surface_plot_second_style(res, var: str, title: str, save_path: str):
    """Draw one independent 3D surface using the same style as the first program."""
    X = np.asarray(res["X"], dtype=float)
    t_min = _time_min(res)
    Z = np.asarray(res[var], dtype=float)

    spatial = X / 1000.0
    meshx, mesht = np.meshgrid(spatial, t_min)

    fig = plt.figure(figsize=(8.8, 6.6), dpi=600)
    subfigs = fig.subfigures(nrows=1, ncols=1, hspace=0)
    subfig = subfigs
    subfig.subplots_adjust(left=0.18, bottom=0.08, right=0.98, top=0.96)

    axes = subfig.subplots(
        nrows=1,
        ncols=1,
        subplot_kw={"projection": "3d", "computed_zorder": False},
    )

    for axis in [axes.xaxis, axes.yaxis, axes.zaxis]:
        axis._axinfo["axisline"]["linewidth"] = 10
        axis._axinfo["axisline"]["color"] = "b"
        axis._axinfo["grid"]["linewidth"] = 0.8
        axis._axinfo["grid"]["linestyle"] = "--"
        axis._axinfo["grid"]["color"] = "#d1d1d1"
        axis.set_pane_color((1, 1, 1))

    axes.view_init(10, 15)
    axes.set_xlabel(r"$x$ [km]", fontsize=18, labelpad=5, fontweight="bold")
    axes.set_ylabel(r"$Time$ [min]", fontsize=18, labelpad=5, fontweight="bold")
    if var == "H":
        zlabel = r"$H(x,t)$ [m]"
    else:
        zlabel = r"$Q(x,t)$ [m$^3$/s]"
    axes.set_zlabel(zlabel, rotation=90, fontsize=18, labelpad=4, fontweight="bold")
    axes.zaxis.set_rotate_label(False)
    axes.zaxis.label.set_clip_on(False)

    x_ticks = np.array([spatial[0], 0.5 * (spatial[0] + spatial[-1]), spatial[-1]], dtype=float)
    y_ticks = np.array([t_min[0], 0.5 * (t_min[0] + t_min[-1]), t_min[-1]], dtype=float)
    z_min, z_max = float(np.nanmin(Z)), float(np.nanmax(Z))
    z_ticks = np.linspace(z_min, z_max, 5)

    axes.set_xticks(x_ticks)
    axes.set_yticks(y_ticks)
    axes.set_zticks(z_ticks)
    axes.set_xticklabels(_format_three_ticklabels(x_ticks), fontweight="bold", fontsize=18)
    axes.set_yticklabels(_format_three_ticklabels(y_ticks), fontweight="bold", fontsize=18)
    axes.set_zticklabels(_format_z_ticklabels(z_ticks), fontweight="bold", fontsize=18)

    rstride = max(1, Z.shape[0] // 6)
    axes.plot_surface(
        meshx,
        mesht,
        Z,
        edgecolor="black",
        lw=0.8,
        rstride=rstride,
        cstride=5,
        alpha=1,
        color="white",
        shade=False,
        rasterized=True,
        antialiased=True,
    )

    # Red downstream boundary curve remains removed as in the target program.

    axes.set_title(_safe_label_for_plot(title), fontsize=14)
    if save_path is not None:
        fig.savefig(save_path, dpi=600, bbox_inches="tight", pad_inches=0.35)
        print(f"3D surface saved to: {save_path}")
    plt.show()
    plt.close(fig)

def plot_3d_surfaces_four(results: Dict[str, dict], save_dir: str, tag: str):
    """
    Draw independent 3D figures for each algorithm and each state variable,
    using the same style as the second program. No combined 3D figure is drawn.
    """
    results = _ordered_results(results)
    for var in ["H", "Q"]:
        for label, res in results.items():
            safe_label = label.replace("+", "_plus_").replace(" ", "_").replace("/", "_")
            path = os.path.join(save_dir, f"{tag}_{safe_label}_{var}_3D_surface_second_style.png")
            _surface_plot_second_style(res, var, label, path)


def save_npz_four(results: Dict[str, dict], save_dir: str, tag: str):
    results = _ordered_results(results)
    arrays = {}
    for label, res in results.items():
        safe = label.replace("+", "_plus_").replace(" ", "_").replace("/", "_")
        arrays[f"t_{safe}"] = res["t"]
        arrays[f"H_{safe}"] = res["H"]
        arrays[f"Q_{safe}"] = res["Q"]
        arrays[f"U_{safe}"] = res["U"]
        arrays[f"reward_{safe}"] = res["reward"]
        arrays[f"total_reward_{safe}"] = np.asarray(res["total_reward"])

    first = next(iter(results.values()))
    arrays["X"] = first["X"]
    arrays["Hbar"] = first["Hbar"]
    arrays["Qbar"] = first["Qbar"]

    e_H0_rel, e_Q0_rel = _initial_relative_deviation_percentages_from_result(first)
    arrays["e_H0_rel_pct"] = np.asarray(e_H0_rel)
    arrays["e_Q0_rel_pct"] = np.asarray(e_Q0_rel)

    path = os.path.join(save_dir, f"{tag}_controller_comparison.npz")
    np.savez(path, **arrays)
    print(f"Saved seven-controller data to: {path}")



def print_final_summary_four(results: Dict[str, dict]):
    results = _ordered_results(results)
    print("\n" + "=" * 90)
    print("Final summary of controller comparison")
    print("=" * 90)
    for label, res in results.items():
        H = np.asarray(res["H"])
        Q = np.asarray(res["Q"])
        Hbar = np.asarray(res["Hbar"])
        Qbar = np.asarray(res["Qbar"])
        H_l2_final = _get_l2(res, "H")[-1]
        Q_l2_final = _get_l2(res, "Q")[-1]

        print(f"\n[{label}]")
        print(f"final H(0) = {H[-1,0]:.6f}, target = {Hbar[0]:.6f}, error = {H[-1,0]-Hbar[0]:.6f}")
        print(f"final H(L) = {H[-1,-1]:.6f}, target = {Hbar[-1]:.6f}, error = {H[-1,-1]-Hbar[-1]:.6f}")
        print(f"final Q(0) = {Q[-1,0]:.6f}, target = {Qbar[0]:.6f}, error = {Q[-1,0]-Qbar[0]:.6f}")
        print(f"final Q(L) = {Q[-1,-1]:.6f}, target = {Qbar[-1]:.6f}, error = {Q[-1,-1]-Qbar[-1]:.6f}")
        print(f"final ||H-Hbar||_L2 = {H_l2_final:.6f} m")
        print(f"final ||Q-Qbar||_L2 = {Q_l2_final:.6f} m^3/s")
        print(f"total_reward = {res['total_reward']:.6f}")



# ============================================================
# Evidence-summary table for the selected single scenario
# ============================================================

# Keep the steady-state evaluation window identical to the local enlarged plot.
# The inset starts at 0.75 of the common end time, so the table uses the final 25%.
LOCAL_ZOOM_START_RATIO = 0.75

# Settling bands are constructed from the steady-state mean error:
# threshold = steady_state_mean + 5% * |initial_error - steady_state_mean|.
SETTLING_BAND_FRACTION = 0.05


def _safe_metric_value(value):
    """Return a finite float when possible; otherwise return np.nan."""
    try:
        value = float(value)
    except Exception:
        return np.nan
    if not np.isfinite(value):
        return np.nan
    return value


def _format_metric(value, precision=6):
    """Format one metric for terminal table output."""
    value = _safe_metric_value(value)
    if not np.isfinite(value):
        return "—"
    return f"{value:.{precision}f}"


def _time_average_on_interval(
    t_min: np.ndarray,
    values: np.ndarray,
    start_min: float,
    end_min: float,
) -> float:
    """
    Calculate the time average on the same interval displayed by the local inset.

    The mask is intentionally identical to the inset logic:
        start_min <= t <= end_min.
    """
    t_min = np.asarray(t_min, dtype=float).reshape(-1)
    values = np.asarray(values, dtype=float).reshape(-1)

    n = min(t_min.size, values.size)
    if n == 0:
        return np.nan

    t_min = t_min[:n]
    values = values[:n]

    mask = (
        np.isfinite(t_min)
        & np.isfinite(values)
        & (t_min >= float(start_min))
        & (t_min <= float(end_min))
    )

    t_window = t_min[mask]
    value_window = values[mask]

    if value_window.size == 0:
        return np.nan

    if value_window.size == 1:
        return float(value_window[0])

    duration = float(t_window[-1] - t_window[0])
    if duration <= 0.0:
        return float(np.mean(value_window))

    return float(
        np.trapezoid(value_window, t_window)
        / duration
    )



def _normalized_spatial_l2_state_over_time(res: dict, key: str) -> np.ndarray:
    """
    Return the normalized spatial L2 norm of the state itself:

        ||v(.,t)||_{L2,n}
            = sqrt((1/L) * integral_0^L v(x,t)^2 dx)
            = sqrt(integral_0^1 v(s,t)^2 ds),  s=x/L.

    This is different from _get_l2(), which computes the tracking-error norm
    ||v-vbar||_{L2,n}.
    """
    X = np.asarray(res.get("X", []), dtype=float).reshape(-1)
    state = np.asarray(res.get(key, []), dtype=float)

    if X.size < 2 or state.ndim != 2 or state.shape[1] != X.size:
        return np.asarray([], dtype=float)

    s = X / float(X[-1])
    return np.sqrt(np.trapezoid(state ** 2, s, axis=1))


def _initial_relative_deviation_percentages_from_arrays(
    X: np.ndarray,
    H0: np.ndarray,
    Q0: np.ndarray,
    Hbar: np.ndarray,
    Qbar: np.ndarray,
) -> Tuple[float, float]:
    """
    Compute the shared initial relative deviations:

        e_H0_rel = ||H0-Hbar||_L2 / ||Hbar||_L2 * 100%
        e_Q0_rel = ||Q0-Qbar||_L2 / ||Qbar||_L2 * 100%.

    The same normalized spatial coordinate s=x/L is used in every norm.
    Because numerator and denominator use the same convention, the ratio is
    independent of whether one writes the normalized or unnormalized L2 norm.
    """
    X = np.asarray(X, dtype=float).reshape(-1)
    H0 = np.asarray(H0, dtype=float).reshape(-1)
    Q0 = np.asarray(Q0, dtype=float).reshape(-1)
    Hbar = np.asarray(Hbar, dtype=float).reshape(-1)
    Qbar = np.asarray(Qbar, dtype=float).reshape(-1)

    if (
        X.size < 2
        or H0.size != X.size
        or Q0.size != X.size
        or Hbar.size != X.size
        or Qbar.size != X.size
    ):
        return np.nan, np.nan

    s = X / float(X[-1])

    H_err_norm = float(np.sqrt(np.trapezoid((H0 - Hbar) ** 2, s)))
    Q_err_norm = float(np.sqrt(np.trapezoid((Q0 - Qbar) ** 2, s)))
    H_ref_norm = float(np.sqrt(np.trapezoid(Hbar ** 2, s)))
    Q_ref_norm = float(np.sqrt(np.trapezoid(Qbar ** 2, s)))

    e_H0_rel = (
        100.0 * H_err_norm / H_ref_norm
        if np.isfinite(H_ref_norm) and H_ref_norm > 1e-12
        else np.nan
    )
    e_Q0_rel = (
        100.0 * Q_err_norm / Q_ref_norm
        if np.isfinite(Q_ref_norm) and Q_ref_norm > 1e-12
        else np.nan
    )
    return float(e_H0_rel), float(e_Q0_rel)


def _initial_relative_deviation_percentages_from_result(
    res: dict,
) -> Tuple[float, float]:
    """Read the common t=0 state from one rollout result and compute both percentages."""
    H = np.asarray(res.get("H", []), dtype=float)
    Q = np.asarray(res.get("Q", []), dtype=float)
    if H.ndim != 2 or Q.ndim != 2 or H.shape[0] == 0 or Q.shape[0] == 0:
        return np.nan, np.nan
    return _initial_relative_deviation_percentages_from_arrays(
        X=np.asarray(res.get("X", []), dtype=float),
        H0=H[0],
        Q0=Q[0],
        Hbar=np.asarray(res.get("Hbar", []), dtype=float),
        Qbar=np.asarray(res.get("Qbar", []), dtype=float),
    )



def validate_shared_reference_and_initial_state(
    results: Dict[str, dict],
    X_expected: np.ndarray,
    Hbar_expected: np.ndarray,
    Qbar_expected: np.ndarray,
    H0_expected: np.ndarray,
    Q0_expected: np.ndarray,
    save_dir: str,
    tag: str,
    atol: float = 1e-10,
) -> Dict[str, Dict[str, float]]:
    """Verify that every controller uses exactly the same test problem.

    This check is intentionally strict.  It compares, controller by controller,
    the spatial grid, ODE equilibrium, reset state, and the initial relative-L2
    percentages against the single profiles generated in ``main``.

    A mismatch raises ``RuntimeError`` before figures are produced, preventing a
    visually plausible but unfair comparison from being saved.
    """
    X_expected = np.asarray(X_expected, dtype=float).reshape(-1)
    Hbar_expected = np.asarray(Hbar_expected, dtype=float).reshape(-1)
    Qbar_expected = np.asarray(Qbar_expected, dtype=float).reshape(-1)
    H0_expected = np.asarray(H0_expected, dtype=float).reshape(-1)
    Q0_expected = np.asarray(Q0_expected, dtype=float).reshape(-1)

    expected_eH, expected_eQ = _initial_relative_deviation_percentages_from_arrays(
        X=X_expected,
        H0=H0_expected,
        Q0=Q0_expected,
        Hbar=Hbar_expected,
        Qbar=Qbar_expected,
    )

    report: Dict[str, Dict[str, float]] = {}
    violations = []

    for label, res in results.items():
        X = np.asarray(res.get("X", []), dtype=float).reshape(-1)
        Hbar = np.asarray(res.get("Hbar", []), dtype=float).reshape(-1)
        Qbar = np.asarray(res.get("Qbar", []), dtype=float).reshape(-1)
        H = np.asarray(res.get("H", []), dtype=float)
        Q = np.asarray(res.get("Q", []), dtype=float)
        t = np.asarray(res.get("t", []), dtype=float).reshape(-1)

        if (
            X.shape != X_expected.shape
            or Hbar.shape != Hbar_expected.shape
            or Qbar.shape != Qbar_expected.shape
            or H.ndim != 2
            or Q.ndim != 2
            or H.shape[0] == 0
            or Q.shape[0] == 0
            or H.shape[1] != X_expected.size
            or Q.shape[1] != X_expected.size
            or t.size == 0
        ):
            violations.append(f"{label}: incompatible/missing result-array shapes")
            continue

        metrics = {
            "max_abs_X_difference": float(np.max(np.abs(X - X_expected))),
            "max_abs_Hbar_difference": float(np.max(np.abs(Hbar - Hbar_expected))),
            "max_abs_Qbar_difference": float(np.max(np.abs(Qbar - Qbar_expected))),
            "max_abs_H0_difference": float(np.max(np.abs(H[0] - H0_expected))),
            "max_abs_Q0_difference": float(np.max(np.abs(Q[0] - Q0_expected))),
            "initial_time_s": float(t[0]),
        }
        eH, eQ = _initial_relative_deviation_percentages_from_arrays(
            X=X,
            H0=H[0],
            Q0=Q[0],
            Hbar=Hbar,
            Qbar=Qbar,
        )
        metrics["e_H0_rel_pct"] = float(eH)
        metrics["e_Q0_rel_pct"] = float(eQ)
        metrics["e_H0_rel_difference_pct"] = float(abs(eH - expected_eH))
        metrics["e_Q0_rel_difference_pct"] = float(abs(eQ - expected_eQ))
        report[label] = metrics

        for key in (
            "max_abs_X_difference",
            "max_abs_Hbar_difference",
            "max_abs_Qbar_difference",
            "max_abs_H0_difference",
            "max_abs_Q0_difference",
        ):
            if not np.isfinite(metrics[key]) or metrics[key] > atol:
                violations.append(f"{label}: {key}={metrics[key]:.3e} > {atol:.3e}")

        if abs(metrics["initial_time_s"]) > atol:
            violations.append(
                f"{label}: first recorded time is {metrics['initial_time_s']:.12g} s, not 0 s"
            )
        if metrics["e_H0_rel_difference_pct"] > 1e-9:
            violations.append(
                f"{label}: e_H0_rel differs by {metrics['e_H0_rel_difference_pct']:.3e}%"
            )
        if metrics["e_Q0_rel_difference_pct"] > 1e-9:
            violations.append(
                f"{label}: e_Q0_rel differs by {metrics['e_Q0_rel_difference_pct']:.3e}%"
            )

    report_path = Path(save_dir) / f"{tag}_shared_initial_condition_validation.txt"
    with report_path.open("w", encoding="utf-8") as f:
        f.write("Shared reference/initial-state validation\n")
        f.write(f"expected e_H0_rel_pct = {expected_eH:.12f}\n")
        f.write(f"expected e_Q0_rel_pct = {expected_eQ:.12f}\n")
        f.write(f"absolute tolerance = {atol:.3e}\n\n")
        for label, metrics in report.items():
            f.write(f"[{label}]\n")
            for key, value in metrics.items():
                f.write(f"{key} = {value:.12e}\n")
            f.write("\n")
        if violations:
            f.write("VIOLATIONS\n")
            for item in violations:
                f.write(f"- {item}\n")
        else:
            f.write("PASS: every controller uses the identical grid, reference, and t=0 state.\n")

    print("\nShared initial-condition validation")
    print("-" * 78)
    print(f"Expected e_H,0^rel = {expected_eH:.9f}%")
    print(f"Expected e_Q,0^rel = {expected_eQ:.9f}%")
    for label, metrics in report.items():
        print(
            f"{label}: max|dH0|={metrics['max_abs_H0_difference']:.3e}, "
            f"max|dQ0|={metrics['max_abs_Q0_difference']:.3e}, "
            f"eH0={metrics['e_H0_rel_pct']:.9f}%, "
            f"eQ0={metrics['e_Q0_rel_pct']:.9f}%"
        )
    print(f"Validation report saved to: {report_path}")

    if violations:
        raise RuntimeError(
            "The controller comparison does not use an identical reference/initial state:\n"
            + "\n".join(f"- {item}" for item in violations)
        )

    return report


def save_backstepping_diagnostic_data(
    res_bs: dict,
    save_dir: str,
    tag: str,
) -> None:
    """Save the BS trajectory separately, independent of multi-curve autoscaling.

    The NPZ makes it straightforward to compare this run numerically with an
    earlier two-controller run.  If the same environment file, seed, reference,
    and initial-condition parameters are used, the arrays should match up to
    floating-point roundoff.
    """
    path = Path(save_dir) / f"{tag}_BS_only_diagnostic.npz"
    np.savez(
        path,
        X=np.asarray(res_bs["X"]),
        Hbar=np.asarray(res_bs["Hbar"]),
        Qbar=np.asarray(res_bs["Qbar"]),
        t=np.asarray(res_bs["t"]),
        H=np.asarray(res_bs["H"]),
        Q=np.asarray(res_bs["Q"]),
        U=np.asarray(res_bs["U"]),
        H_l2=_get_l2(res_bs, "H"),
        Q_l2=_get_l2(res_bs, "Q"),
    )
    print(f"Backstepping-only diagnostic data saved to: {path}")


def _normalized_l2_norm_overshoot_percentage(
    res: dict,
    key: str,
) -> float:
    r"""
    Compute the normalized-L2-norm overshoot of a distributed state field:

        M_v = 100 * max_{t>0}
              [ (||v(.,t)||_{L2,n} - ||vbar||_{L2,n})
                / ||vbar||_{L2,n} ]_+,

    where v is H or Q and

        ||v(.,t)||_{L2,n}
            = sqrt((1/L) * integral_0^L v(x,t)^2 dx).

    The common initial sample t=0 is excluded. For the spatially constant
    Qbar used here, ||Qbar||_{L2,n}=Qbar, so this reduces exactly to the
    previously requested Q formula.
    """
    if key not in {"H", "Q"}:
        raise ValueError("key must be 'H' or 'Q'.")

    t_min = _time_min(res)
    state_norm = _normalized_spatial_l2_state_over_time(res, key)
    reference_key = "Hbar" if key == "H" else "Qbar"
    reference = np.asarray(res.get(reference_key, []), dtype=float).reshape(-1)
    X = np.asarray(res.get("X", []), dtype=float).reshape(-1)

    if (
        t_min.size == 0
        or state_norm.size == 0
        or reference.size == 0
        or X.size < 2
        or reference.size != X.size
    ):
        return np.nan

    n = min(t_min.size, state_norm.size)
    t_min = t_min[:n]
    state_norm = state_norm[:n]

    finite_time = t_min[np.isfinite(t_min)]
    if finite_time.size == 0:
        return np.nan

    initial_time = float(np.min(finite_time))
    tolerance = max(1e-12, 1e-10 * max(abs(initial_time), 1.0))
    time_mask = np.isfinite(t_min) & (t_min > initial_time + tolerance)
    if not np.any(time_mask):
        return np.nan

    if not np.all(np.isfinite(reference)):
        return np.nan

    domain_length = float(X[-1] - X[0])
    if not np.isfinite(domain_length) or domain_length <= 0.0:
        return np.nan

    # Use s=(x-x0)/L, so this is the normalized spatial L2 norm and has the
    # same physical unit as the state itself.
    s = (X - X[0]) / domain_length
    reference_norm = float(np.sqrt(np.trapezoid(reference ** 2, s)))
    if not np.isfinite(reference_norm) or reference_norm <= 1e-12:
        return np.nan

    relative_excursion = (
        state_norm[time_mask] - reference_norm
    ) / reference_norm
    relative_excursion = relative_excursion[np.isfinite(relative_excursion)]
    if relative_excursion.size == 0:
        return np.nan

    return float(100.0 * max(0.0, np.max(relative_excursion)))


def _h_normalized_l2_norm_overshoot_percentage(res: dict) -> float:
    """Return M_H using the same normalized-L2-norm definition as M_Q."""
    return _normalized_l2_norm_overshoot_percentage(res, "H")


def _q_normalized_l2_norm_overshoot_percentage(res: dict) -> float:
    """Return M_Q; for constant Qbar this equals the requested Q formula."""
    return _normalized_l2_norm_overshoot_percentage(res, "Q")


def _state_overshoot_percentage(
    res: dict,
    key: str,
) -> float:
    r"""
    Return the classical maximum positive overshoot generalized to the full
    distributed state field.

    For v = H or Q,

        M_v = 100 * max_{t,x} [ (v(x,t)-vbar(x)) / |vbar(x)| ]_+,

    where [z]_+ = max(z,0). Thus the metric is the largest positive
    pointwise overshoot anywhere in space and at any sampled time, including
    the prescribed initial instant t=0. It is the direct distributed-state
    analogue of the classical scalar maximum-overshoot definition.
    """
    t_min = _time_min(res)
    state = np.asarray(res.get(key, []), dtype=float)
    reference_key = "Hbar" if key == "H" else "Qbar"
    reference = np.asarray(res.get(reference_key, []), dtype=float).reshape(-1)

    if state.ndim != 2 or reference.size != state.shape[1] or t_min.size == 0:
        return np.nan

    n = min(t_min.size, state.shape[0])
    t_min = t_min[:n]
    state = state[:n]

    # Classical maximum overshoot uses the complete response, including t=0.
    time_mask = np.isfinite(t_min)
    if not np.any(time_mask):
        return np.nan

    denominator = np.abs(reference)
    valid_reference = np.isfinite(reference) & (denominator > 1e-12)
    if not np.any(valid_reference):
        return np.nan

    state_window = state[time_mask][:, valid_reference]
    ref_window = reference[valid_reference].reshape(1, -1)
    denominator = denominator[valid_reference].reshape(1, -1)

    relative_positive_excursion = (state_window - ref_window) / denominator
    relative_positive_excursion = relative_positive_excursion[
        np.isfinite(relative_positive_excursion)
    ]
    if relative_positive_excursion.size == 0:
        return np.nan

    return float(100.0 * max(0.0, np.max(relative_positive_excursion)))


def _peak_spatial_linf(res: dict, key: str) -> float:
    """
    Return the peak spatial L-infinity tracking error over the full simulation.

    At each time instant, the spatial L-infinity norm is
        max_x |state(t, x) - reference(x)|.
    The reported peak then takes the maximum of this norm over all simulated
    time instants, including the initial state:
        max_t max_x |state(t, x) - reference(x)|.
    """
    linf_over_time = np.asarray(_get_linf(res, key), dtype=float).reshape(-1)
    finite_values = linf_over_time[np.isfinite(linf_over_time)]
    if finite_values.size == 0:
        return np.nan
    return float(np.max(finite_values))


def _joint_settling_time(
    t_min: np.ndarray,
    H_l2: np.ndarray,
    Q_l2: np.ndarray,
    steady_H_l2: float,
    steady_Q_l2: float,
    common_end_min: float,
    band_fraction: float = SETTLING_BAND_FRACTION,
) -> float:
    """
    Find the earliest time after which both H and Q L2 errors remain in their
    respective steady-state bands through the common comparison end time.

    H threshold:
        steady_H_l2 + band_fraction * |H_l2(0) - steady_H_l2|

    Q threshold:
        steady_Q_l2 + band_fraction * |Q_l2(0) - steady_Q_l2|
    """
    t_min = np.asarray(t_min, dtype=float).reshape(-1)
    H_l2 = np.asarray(H_l2, dtype=float).reshape(-1)
    Q_l2 = np.asarray(Q_l2, dtype=float).reshape(-1)

    n = min(t_min.size, H_l2.size, Q_l2.size)
    if n == 0:
        return np.nan

    t_min = t_min[:n]
    H_l2 = H_l2[:n]
    Q_l2 = Q_l2[:n]

    valid = (
        np.isfinite(t_min)
        & np.isfinite(H_l2)
        & np.isfinite(Q_l2)
        & (t_min <= float(common_end_min))
    )

    t_eval = t_min[valid]
    H_eval = H_l2[valid]
    Q_eval = Q_l2[valid]

    if (
        t_eval.size == 0
        or not np.isfinite(steady_H_l2)
        or not np.isfinite(steady_Q_l2)
    ):
        return np.nan

    H_threshold = float(
        steady_H_l2
        + float(band_fraction)
        * abs(float(H_eval[0]) - steady_H_l2)
    )
    Q_threshold = float(
        steady_Q_l2
        + float(band_fraction)
        * abs(float(Q_eval[0]) - steady_Q_l2)
    )

    inside_both_bands = (
        (H_eval <= H_threshold)
        & (Q_eval <= Q_threshold)
    )

    # suffix_all[i] is True only when every sample from i to the end remains
    # inside both steady-state bands.
    suffix_all = np.logical_and.accumulate(
        inside_both_bands[::-1]
    )[::-1]

    settled_indices = np.flatnonzero(suffix_all)
    if settled_indices.size == 0:
        return np.nan

    return float(t_eval[settled_indices[0]])










# ============================================================
# Main: seven-controller comparison
# ============================================================


DEFAULT_NEW_MODEL_PATH = (
     "./sactx/seed_42/"
    "sac_cac_hybrid_yul_ode_cheb_raw_HQ_100000_steps"
) 
DEFAULT_TDPSAC_FPC_MODEL_PATH = (
    "./sacddx/seed_42/"
    "sac_cac_hybrid_fixed_control_yul_ode_cheb_raw_HQ_100000_steps" 
)
DEFAULT_DPSAC_MODEL_PATH = (
    "./saccontrolx/seed_42/"
    "sac_cac_new_hadamard_hybrid_all_trainable_raw_HQ_100000_steps"
)
DEFAULT_DO_SAC_MODEL_PATH = (
    "./sactrainx/seed_42/"
    "sac_cac_deeponet_feature_raw_hq_ode_cheb_100000_steps"
)
DEFAULT_SAC_FPC_MODEL_PATH = (
    "./sacnoxv/seed_42/"
    "sac_fnn_frozen_new_deeponet_prior_raw_HQ_100000_steps"
)
# Newly trained ordinary SAC checkpoint using raw physical H/Q observations.
DEFAULT_SAC_MODEL_PATH = (
    "./saczx/seed_42/"
    "sac_cac_raw_hq_ode_cheb_100000_steps")
DEFAULT_SAVE_DIR_COMPARISON = "./test_results_seven_algorithms_MC50_H1to2pct_Q16to17pct_mean_std"

# Display/run order follows the old multi-controller comparison hierarchy.
LEARNED_CONTROLLER_SPECS = [
    {
        "label": LABEL_SAC_CURVE,          # SAC: newly trained raw-HQ model
        "model_kind": "sac",
        "required": True,
    },
    {
        "label": LABEL_FP_SAC,             # SAC-FPC
        "model_kind": "auto",
        "required": True,
    },
    {
        "label": LABEL_SAC,                # DO-SAC
        "model_kind": "auto",
        "required": True,
    },
    {
        "label": LABEL_HYBRID,             # DPSAC
        "model_kind": "auto",
        "required": True,
    },
    {
        "label": LABEL_DFP_DON_SAC,        # TDPSAC-FPC
        "model_kind": "auto",
        "required": True,
    },
    {
        "label": LABEL_NEW_MODEL,           # TDPSAC
        "model_kind": "hybrid",
        "required": True,
    },
]


def _run_optional_controller(
    results: Dict[str, dict],
    *,
    label: str,
    model_path: str,
    model_kind: str,
    seed: int,
    H_left: float,
    H_right: float,
    Q_value: float,
    dH_source,
    required: bool = False,
):
    """Run one learned controller and keep the common result-data interface."""
    if not _checkpoint_exists(model_path):
        message = (
            f"No valid SB3 ZIP checkpoint was found for {label}.\n"
            f"Current path: {model_path!r}\n"
            "Expected either '/path/model' with '/path/model.zip' present, "
            "or a direct '/path/model.zip' path."
        )
        if required:
            raise FileNotFoundError(message)
        print(f"\n[SKIP] {message}")
        return

    print(f"\nRunning {label}...")
    results[label] = run_policy_exact(
        model_path=model_path,
        model_kind=model_kind,
        label=label,
        seed=seed,
        H_left=H_left,
        H_right=H_right,
        Q_value=Q_value,
        dH_source=dH_source,
    )


def _clean_runtime_argv(argv=None):
    """Remove Jupyter/IPython kernel connection arguments before argparse."""
    raw = list(sys.argv[1:] if argv is None else argv)
    cleaned = []
    i = 0

    while i < len(raw):
        arg = str(raw[i])

        if arg in {"-f", "--f"} and i + 1 < len(raw):
            candidate = str(raw[i + 1])
            if candidate.endswith(".json") and (
                "jupyter" in candidate.lower()
                or "kernel-" in Path(candidate).name.lower()
            ):
                i += 2
                continue

        if arg.startswith(("--f=", "-f=")):
            candidate = arg.split("=", 1)[1]
            if candidate.endswith(".json") and (
                "jupyter" in candidate.lower()
                or "kernel-" in Path(candidate).name.lower()
            ):
                i += 1
                continue

        if arg.endswith(".json") and (
            "jupyter" in arg.lower()
            or "kernel-" in Path(arg).name.lower()
        ):
            i += 1
            continue

        cleaned.append(arg)
        i += 1

    return cleaned




# ============================================================
# Monte-Carlo robustness test: 50 linear initial conditions
# 25 above the target equilibrium and 25 below it
# ============================================================

MC_METRIC_KEYS = [
    "SS mean L2(H) [m]",
    "SS mean L2(Q) [m^3/s]",
    "M_H max overshoot [%]",
    "M_Q max overshoot [%]",
]

MC_SHORT_NAMES = {
    "SS mean L2(H) [m]": "SS_H",
    "SS mean L2(Q) [m^3/s]": "SS_Q",
    "M_H max overshoot [%]": "M_H",
    "M_Q max overshoot [%]": "M_Q",
}


def generate_symmetric_linear_ic_cases(
    X: np.ndarray,
    Hbar: np.ndarray,
    Qbar: np.ndarray,
    *,
    n_total: int = 50,
    n_above: int = 25,
    dH_pct_min: float = 1.0,
    dH_pct_max: float = 2.0,
    dQ_pct_min: float = 16.0,
    dQ_pct_max: float = 17.0,
    seed: int = 20260810,
):
    """
    Generate the SAME fixed Monte-Carlo initial-condition list for all controllers.

    The near-nominal initial-condition bands are defined using the realized
    relative spatial-L2 deviations

        delta_H,0 in [1%, 2%],
        delta_Q,0 in [16%, 17%].

    The water-depth profile keeps the previously used linear form

        H(x,0) = H_left + (H_right-H_left) x/L,

    while Q(x,0) remains spatially constant.

    The water-depth and discharge deviations are forced to lie on opposite sides
    of the target. For trials 1,...,n_above, both H_left and H_right are above
    their equilibrium values while Q_value is below its equilibrium value. For
    the remaining trials, both H_left and H_right are below equilibrium while
    Q_value is above equilibrium. H_left and H_right therefore always remain on
    the same side of the equilibrium profile. Each candidate is accepted only
    when its realized relative-L2 deviations lie inside the prescribed bands.
    """
    X = np.asarray(X, dtype=float).reshape(-1)
    Hbar = np.asarray(Hbar, dtype=float).reshape(-1)
    Qbar = np.asarray(Qbar, dtype=float).reshape(-1)

    if X.size < 2 or Hbar.shape != X.shape or Qbar.shape != X.shape:
        raise ValueError("X, Hbar, and Qbar must have the same one-dimensional shape.")
    if n_total <= 0:
        raise ValueError("n_total must be positive.")
    if n_above < 0 or n_above > n_total:
        raise ValueError("n_above must satisfy 0 <= n_above <= n_total.")
    if not (0.0 < dH_pct_min < dH_pct_max):
        raise ValueError("Require 0 < dH_pct_min < dH_pct_max.")
    if not (0.0 < dQ_pct_min < dQ_pct_max):
        raise ValueError("Require 0 < dQ_pct_min < dQ_pct_max.")

    rng = np.random.default_rng(seed)
    H_left_ref = float(Hbar[0])
    H_right_ref = float(Hbar[-1])
    Q_ref = float(Qbar[0])
    L = float(X[-1])

    cases = []
    max_attempts_per_case = 100000

    for i in range(n_total):
        # ``side`` denotes the side of the water-depth profile. The discharge
        # is always placed on the opposite side of its equilibrium value.
        side = "above" if i < n_above else "below"
        sign = 1.0 if side == "above" else -1.0
        q_sign = -sign

        accepted = False
        for _ in range(max_attempts_per_case):
            pH_left = float(rng.uniform(dH_pct_min, dH_pct_max))
            pH_right = float(rng.uniform(dH_pct_min, dH_pct_max))
            pQ = float(rng.uniform(dQ_pct_min, dQ_pct_max))

            H_left = H_left_ref * (1.0 + sign * pH_left / 100.0)
            H_right = H_right_ref * (1.0 + sign * pH_right / 100.0)
            Q_value = Q_ref * (1.0 + q_sign * pQ / 100.0)

            if H_left <= 0.20 or H_right <= 0.20 or Q_value <= 0.0:
                continue

            H0 = H_left + (H_right - H_left) * (X / L)
            Q0 = np.full_like(X, Q_value, dtype=float)

            eH_rel, eQ_rel = _initial_relative_deviation_percentages_from_arrays(
                X=X,
                H0=H0,
                Q0=Q0,
                Hbar=Hbar,
                Qbar=Qbar,
            )

            if not np.isfinite(eH_rel) or not (dH_pct_min <= eH_rel <= dH_pct_max):
                continue
            if not np.isfinite(eQ_rel) or not (dQ_pct_min <= eQ_rel <= dQ_pct_max):
                continue

            dH_left = H_left - H_left_ref
            dH_right = H_right - H_right_ref
            dQ = Q_value - Q_ref

            cases.append({
                "trial": i + 1,
                "side": side,
                "H_left": float(H_left),
                "H_right": float(H_right),
                "Q_value": float(Q_value),
                "delta_H_left": float(dH_left),
                "delta_H_right": float(dH_right),
                "delta_Q": float(dQ),
                "delta_H_left_pct": float(sign * pH_left),
                "delta_H_right_pct": float(sign * pH_right),
                "delta_Q_pct": float(q_sign * pQ),
                "delta_H_L2_pct": float(eH_rel),
                "delta_Q_L2_pct": float(eQ_rel),
            })
            accepted = True
            break

        if not accepted:
            raise RuntimeError(
                f"Could not generate trial {i+1} satisfying the requested near-nominal IC bands."
            )

    return cases



def _make_eval_env_by_kind(
    model_kind: str,
    *,
    H_left: float,
    H_right: float,
    Q_value: float,
    seed: int,
    dH_source=None,
):
    kind = model_kind.lower()
    if kind == "sac_error":
        return make_env_sac_error_state(
            H_left=H_left,
            H_right=H_right,
            Q_value=Q_value,
            seed=seed,
            dH_source=dH_source,
        )
    return make_env_sac(
        H_left=H_left,
        H_right=H_right,
        Q_value=Q_value,
        seed=seed,
        dH_source=dH_source,
    )


def load_learned_controller_once(
    model_path: str,
    model_kind: str,
    *,
    first_case: dict,
    seed: int,
    dH_source=None,
):
    """Load one learned controller once; later trials only replace its environment."""
    env, _, _, _, _ = _make_eval_env_by_kind(
        model_kind,
        H_left=first_case["H_left"],
        H_right=first_case["H_right"],
        Q_value=first_case["Q_value"],
        seed=seed,
        dH_source=dH_source,
    )

    kind = model_kind.lower()
    try:
        if kind in {"sac", "sac_error"}:
            model = SAC.load(model_path, env=env)
        elif kind == "hybrid":
            model = _safe_load_hybrid_model(model_path, env=env)
        elif kind == "auto":
            model = _safe_load_generic_sac_model(model_path, env=env)
        else:
            raise ValueError(
                f"Unknown model_kind={model_kind!r}; expected sac/sac_error/hybrid/auto."
            )
    except Exception:
        env.close()
        raise

    # The first environment is only used to reconstruct/check the saved policy.
    env.close()
    return model


def run_loaded_policy_exact(
    model,
    model_kind: str,
    label: str,
    *,
    seed: int,
    H_left: float,
    H_right: float,
    Q_value: float,
    dH_source=None,
):
    """Evaluate an already-loaded SAC-family controller on one linear IC."""
    env, X, Hbar, Qbar, _ = _make_eval_env_by_kind(
        model_kind,
        H_left=H_left,
        H_right=H_right,
        Q_value=Q_value,
        seed=seed,
        dH_source=dH_source,
    )

    model.set_env(env)
    obs, _ = env.reset(seed=seed)
    obs = np.asarray(obs, dtype=np.float32).reshape(-1)
    if obs.size != RAW_STATE_DIM:
        env.close()
        raise RuntimeError(
            f"{label}: reset observation dimension={obs.size}, expected={RAW_STATE_DIM}."
        )

    observation_mode_used = (
        "normalized_error" if model_kind.lower() == "sac_error" else "raw"
    )

    H_hist, Q_hist, U_hist, t_hist, reward_hist = [], [], [], [], []
    H0, Q0 = env.get_current_HQ()
    H_hist.append(H0.copy())
    Q_hist.append(Q0.copy())
    U_hist.append(np.nan)
    t_hist.append(0.0)
    reward_hist.append(0.0)

    previous_u_filtered = float(U_ref)

    done = False
    while not done:
        action, _ = model.predict(obs, deterministic=True)

        action_filtered, u_command_raw, previous_u_filtered = lowpass_filter_rl_control(
            action,
            previous_u_filtered,
            alpha=RL_CONTROL_FILTER_ALPHA,
        )

        obs, reward, terminated, truncated, info = env.step(action_filtered)
        H, Q = env.get_current_HQ()

        H_hist.append(H.copy())
        Q_hist.append(Q.copy())
        U_hist.append(float(env.last_control))
        t_hist.append(float(info["time"]))
        reward_hist.append(float(reward))
        done = bool(terminated or truncated)

    env.close()

    H_hist = np.asarray(H_hist)
    Q_hist = np.asarray(Q_hist)
    U_hist = np.asarray(U_hist)
    t_hist = np.asarray(t_hist)
    reward_hist = np.asarray(reward_hist)

    H_err = H_hist - Hbar[None, :]
    Q_err = Q_hist - Qbar[None, :]

    return {
        "label": label,
        "model_kind": model_kind,
        "observation_mode": observation_mode_used,
        "X": X,
        "Hbar": Hbar,
        "Qbar": Qbar,
        "t": t_hist,
        "H": H_hist,
        "Q": Q_hist,
        "U": U_hist,
        "reward": reward_hist,
        "H_l2": np.sqrt(np.mean(H_err ** 2, axis=1)),
        "Q_l2": np.sqrt(np.mean(Q_err ** 2, axis=1)),
        "final_H_minus_Hbar": H_err[-1],
        "final_Q_minus_Qbar": Q_err[-1],
        "total_reward": float(np.sum(reward_hist)),
    }


def extract_one_mc_row(label: str, result: dict):
    """Extract only SS_H, SS_Q, M_H, and M_Q for one trial."""
    t = np.asarray(result.get("t", []), dtype=float).reshape(-1)
    H = np.asarray(result.get("H", []), dtype=float)
    Q = np.asarray(result.get("Q", []), dtype=float)
    H_l2 = np.asarray(_get_l2(result, "H"), dtype=float).reshape(-1)
    Q_l2 = np.asarray(_get_l2(result, "Q"), dtype=float).reshape(-1)

    has_nonfinite = (
        t.size == 0
        or H.ndim != 2
        or Q.ndim != 2
        or H_l2.size == 0
        or Q_l2.size == 0
        or not np.all(np.isfinite(t))
        or not np.all(np.isfinite(H))
        or not np.all(np.isfinite(Q))
        or not np.all(np.isfinite(H_l2))
        or not np.all(np.isfinite(Q_l2))
    )
    early_stop = bool(t.size > 0 and t[-1] < float(SAC_T) - 1e-9)

    if has_nonfinite or early_stop:
        return {
            "Controller": label,
            "SS mean L2(H) [m]": np.nan,
            "SS mean L2(Q) [m^3/s]": np.nan,
            "M_H max overshoot [%]": np.nan,
            "M_Q max overshoot [%]": np.nan,
        }

    t_min = t / 60.0
    common_end_min = float(np.max(t_min))
    steady_start_min = LOCAL_ZOOM_START_RATIO * common_end_min

    SS_H = _time_average_on_interval(
        t_min, H_l2, steady_start_min, common_end_min
    )
    SS_Q = _time_average_on_interval(
        t_min, Q_l2, steady_start_min, common_end_min
    )

    # Preserve the original classical full-state maximum positive overshoot:
    # M_v = 100*max_{t,x}[(v(x,t)-vbar(x))/|vbar(x)|]_+.
    M_H = _state_overshoot_percentage(result, "H")
    M_Q = _state_overshoot_percentage(result, "Q")

    return {
        "Controller": label,
        "SS mean L2(H) [m]": SS_H,
        "SS mean L2(Q) [m^3/s]": SS_Q,
        "M_H max overshoot [%]": M_H,
        "M_Q max overshoot [%]": M_Q,
    }



def _mean_std(values):
    values = np.asarray(values, dtype=float).reshape(-1)
    values = values[np.isfinite(values)]
    if values.size == 0:
        return np.nan, np.nan, 0
    mean = float(np.mean(values))
    std = float(np.std(values, ddof=1)) if values.size > 1 else 0.0
    return mean, std, int(values.size)


def _mean_std_string(mean, std, precision=6):
    if not np.isfinite(mean) or not np.isfinite(std):
        return "--"
    return f"{mean:.{precision}g} +/- {std:.{precision}g}"


def aggregate_mc_rows(raw_rows, controller_order, n_total):
    """Aggregate mean and sample standard deviation for the four retained metrics."""
    del n_total
    summary = []

    for label in controller_order:
        rows = [r for r in raw_rows if r.get("Controller") == label]
        if not rows:
            continue

        item = {"Controller": label}
        for key in MC_METRIC_KEYS:
            mean, std, n_valid = _mean_std([r.get(key, np.nan) for r in rows])
            short = MC_SHORT_NAMES[key]
            item[f"{short}_mean"] = mean
            item[f"{short}_std"] = std
            item[f"{short}_n"] = n_valid
            item[f"{short}_mean_std"] = _mean_std_string(mean, std)

        summary.append(item)

    return summary



def print_mc_summary_table(summary, output_stream=None, save_path=None):
    """Print and optionally save the final MC table only.

    output_stream is explicitly supplied by main() so the final table is not
    swallowed by the temporary stdout redirection used to silence SB3/DeepONet.
    If save_path is given, exactly the same formatted table is saved as one TXT file.
    """
    if output_stream is None:
        output_stream = sys.stdout

    headers = [
        "Controller",
        "SS_H mean+/-std",
        "SS_Q mean+/-std",
        "M_H mean+/-std [%]",
        "M_Q mean+/-std [%]",
    ]

    rows = [
        [
            s["Controller"],
            s["SS_H_mean_std"],
            s["SS_Q_mean_std"],
            s["M_H_mean_std"],
            s["M_Q_mean_std"],
        ]
        for s in summary
    ]

    widths = [
        max(len(headers[i]), max((len(r[i]) for r in rows), default=0))
        for i in range(len(headers))
    ]

    header_line = " | ".join(
        headers[i].ljust(widths[i]) for i in range(len(headers))
    )
    sep_line = "-+-".join("-" * widths[i] for i in range(len(headers)))
    row_lines = [
        " | ".join(row[i].ljust(widths[i]) for i in range(len(headers)))
        for row in rows
    ]

    # Visible terminal/Jupyter output.
    print(header_line, file=output_stream, flush=True)
    print(sep_line, file=output_stream, flush=True)
    for line in row_lines:
        print(line, file=output_stream, flush=True)

    # Save ONLY the final table, with no additional logs or result files.
    if save_path is not None:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        with save_path.open("w", encoding="utf-8") as f:
            f.write(header_line + "\n")
            f.write(sep_line + "\n")
            for line in row_lines:
                f.write(line + "\n")

    return headers, rows





def save_mc_summary(summary, save_dir: Path):
    csv_path = save_dir / "MC50_table_SS_H_SS_Q_M_H_M_Q.csv"
    csv_fields = [
        "Controller",
        "SS_H_mean_std",
        "SS_Q_mean_std",
        "M_H_mean_std",
        "M_Q_mean_std",
    ]

    with csv_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=csv_fields, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(summary)

    print_mc_summary_table(summary)
    return csv_path



def main():
    # Keep the notebook/terminal output stream before temporarily silencing
    # model/environment messages. The final table is written directly here.
    final_table_stream = sys.stdout

    parser = argparse.ArgumentParser(
        allow_abbrev=False,
        description=(
            "Monte-Carlo table-only comparison of BS, SAC, SAC-FPC, DO-SAC, "
            "DPSAC, TDPSAC-FPC, and TDPSAC."
        ),
    )
    parser.add_argument(
        "--new-model", "--tdpsac-model", dest="new_model", type=str,
        default=DEFAULT_NEW_MODEL_PATH,
    )
    parser.add_argument(
        "--tdpsac-fpc-model", type=str, default=DEFAULT_TDPSAC_FPC_MODEL_PATH,
    )
    parser.add_argument(
        "--dpsac-model", type=str, default=DEFAULT_DPSAC_MODEL_PATH,
    )
    parser.add_argument(
        "--do-sac-model", type=str, default=DEFAULT_DO_SAC_MODEL_PATH,
    )
    parser.add_argument(
        "--sac-fpc-model", type=str, default=DEFAULT_SAC_FPC_MODEL_PATH,
    )
    parser.add_argument(
        "--sac-model", type=str, default=DEFAULT_SAC_MODEL_PATH,
    )
    parser.add_argument(
        "--out-dir", type=str,
        default="./test_results_MC50_table_SS_H_SS_Q_M_H_M_Q",
    )
    parser.add_argument(
        "--seed", type=int, default=123,
        help="Base evaluation seed; trial i uses seed+i-1 for every controller.",
    )
    parser.add_argument(
        "--ic-seed", type=int, default=20260810,
        help="Seed used once to generate the shared 50 initial conditions.",
    )
    parser.add_argument("--n-trials", type=int, default=50)
    parser.add_argument("--n-above", type=int, default=25)
    parser.add_argument("--dH-pct-min", type=float, default=1.0)
    parser.add_argument("--dH-pct-max", type=float, default=2.0)
    parser.add_argument("--dQ-pct-min", type=float, default=9.0)
    parser.add_argument("--dQ-pct-max", type=float, default=10.0)
    parser.add_argument("--dH-source", type=float, default=None)

    args, _ = parser.parse_known_args(_clean_runtime_argv())

    save_dir = Path(args.out_dir)
    save_dir.mkdir(parents=True, exist_ok=True)
    globals()["SAVE_DIR"] = str(save_dir)

    # Preserve the original common equilibrium and initial-condition generation.
    _, X_ref, _, Hbar_ref, Qbar_ref = build_collection_matched_reference()
    cases = generate_symmetric_linear_ic_cases(
        X_ref,
        Hbar_ref,
        Qbar_ref,
        n_total=args.n_trials,
        n_above=args.n_above,
        dH_pct_min=args.dH_pct_min,
        dH_pct_max=args.dH_pct_max,
        dQ_pct_min=args.dQ_pct_min,
        dQ_pct_max=args.dQ_pct_max,
        seed=args.ic_seed,
    )

    controller_paths = {
        LABEL_NEW_MODEL: args.new_model,
        LABEL_DFP_DON_SAC: args.tdpsac_fpc_model,
        LABEL_HYBRID: args.dpsac_model,
        LABEL_FP_SAC: args.sac_fpc_model,
        LABEL_SAC: args.do_sac_model,
        LABEL_SAC_CURVE: args.sac_model,
    }

    for spec in LEARNED_CONTROLLER_SPECS:
        label = spec["label"]
        path = controller_paths[label]
        if not _checkpoint_exists(path):
            raise FileNotFoundError(
                f"No valid checkpoint for {label}: {path!r}. "
                "Fix all model paths before starting the Monte-Carlo comparison."
            )

    raw_rows = []
    controller_order = list(CONTROLLER_DISPLAY_ORDER)

    # Suppress all model/environment/progress output, including
    # Stable-Baselines3 wrapper messages, DeepONet loading/freezing messages,
    # and warnings. Backstepping is still fully evaluated and retained.
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            # --------------------------------------------------------
            # 1) Backstepping: retained
            # --------------------------------------------------------
            label = LABEL_BACKSTEPPING
            for case in cases:
                trial_seed = args.seed + case["trial"] - 1
                base = {"Controller": label}
                try:
                    result = run_backstepping_for_four(
                        seed=trial_seed,
                        H_left=case["H_left"],
                        H_right=case["H_right"],
                        Q_value=case["Q_value"],
                        dH_source=args.dH_source,
                    )
                    base.update(extract_one_mc_row(label, result))
                except Exception:
                    for key in MC_METRIC_KEYS:
                        base[key] = np.nan
                raw_rows.append(base)

            # --------------------------------------------------------
            # 2) Six learned controllers
            # --------------------------------------------------------
            for spec in LEARNED_CONTROLLER_SPECS:
                label = spec["label"]
                model_kind = spec["model_kind"]
                model_path = controller_paths[label]

                model = load_learned_controller_once(
                    model_path,
                    model_kind,
                    first_case=cases[0],
                    seed=args.seed,
                    dH_source=args.dH_source,
                )

                for case in cases:
                    trial_seed = args.seed + case["trial"] - 1
                    base = {"Controller": label}
                    try:
                        result = run_loaded_policy_exact(
                            model,
                            model_kind,
                            label,
                            seed=trial_seed,
                            H_left=case["H_left"],
                            H_right=case["H_right"],
                            Q_value=case["Q_value"],
                            dH_source=args.dH_source,
                        )
                        base.update(extract_one_mc_row(label, result))
                    except Exception:
                        for key in MC_METRIC_KEYS:
                            base[key] = np.nan
                    raw_rows.append(base)

                del model
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

    summary = aggregate_mc_rows(
        raw_rows,
        controller_order=controller_order,
        n_total=args.n_trials,
    )

    # The ONLY visible/output result is the final table.
    # No CSV/TXT/figure file is written here.
    table_path = save_dir / "MC50_final_table.txt"
    print_mc_summary_table(
        summary,
        output_stream=final_table_stream,
        save_path=table_path,
    )



if __name__ == "__main__":
    main()


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
# [DIAGNOSTIC] Adds directional/non-convergence diagnostics only; controller logic and all test settings are unchanged.
"""
Unified controller comparison for Saint-Venant canal regulation.

Purpose:
    Compare six learned controllers under a 50-trial large-initial-condition Monte-Carlo test:
    SAC, SAC-FPC, DO-SAC, DPSAC, TDPSAC-FPC, and TDPSAC. This version
    generates no figures; it saves the shared initial-condition list, raw metrics,
    and a mean +/- sample-std table with convergence success rates.

Method mapping:
    1) TDPSAC:
       the latest partially frozen Hadamard DeepONet-SAC controller.
    2) TDPSAC-FPC:
       DeepONet actor/critic features plus an independent frozen prior controller.
    3) DPSAC:
       Hadamard DeepONet-SAC with all DeepONet parameters fine-tuned.
    4) DO-SAC:
       Hadamard DeepONet is used only as the SAC feature extractor.
    5) SAC-FPC:
       ordinary SAC actor/critic plus an independent frozen DeepONet prior.
    6) SAC:
       newly trained ordinary SAC without DeepONet, using the raw physical H/Q state.

All learned controllers receive the raw physical state
    [H_1,...,H_N,Q_1,...,Q_N],
with the same ODE equilibrium and the same physical initial condition.

Reference and test initial condition:
    - Qbar(x) is fixed at 11.93 m^3/s.
    - Hbar(L) is obtained from the downstream equilibrium gate relation.
    - Hbar(x) is obtained by solving the steady Saint-Venant ODE backward from
      x=L to x=0 with scipy.integrate.solve_ivp.
    - Every controller uses the same deterministic linear initial condition:

          H(x,0) = 3.75 + (4.65 - 3.75) x/L,
          Q(x,0) = 10.0.

      The initial condition is independent of the random seed. The seed is kept
      only for deterministic environment/model evaluation.

Extension:
    Additional learned controllers can be added later through
    LEARNED_CONTROLLER_SPECS without changing the common plotting and table
    functions.
"""

from __future__ import annotations

import argparse
import builtins
import contextlib
import csv
import io
import os
import sys
import warnings
import zipfile
from pathlib import Path
from typing import Tuple, Dict, Optional

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from gymnasium import spaces
from scipy.integrate import solve_ivp
from stable_baselines3 import SAC
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from stable_baselines3.sac.policies import Actor, SACPolicy

from cac_saint_venant_env_no_bs import (
    CACParams,
    CACSaintVenantPDE1D,
)

try:
    from pde_control_gym.src.rewards.base_reward import BaseReward
except Exception:
    class BaseReward:
        pass


# ============================================================
# Common physical / reward settings
# ============================================================

N = 102
dt = 5.0
control_sample_rate = 400.0

max_state_value = 1e6

# The learned controller emits a normalized action in [-1,1]. The environment
# maps it to this physical downstream-gate interval. These limits are identical
# to the corresponding raw-HQ training program.
min_control_value = 0.15
max_control_value = 0.40

# H and Q remain raw physical states in the policy observation. These scales are
# used only inside the reward calculation.
H_scale = 0.30
Q_scale = 3.00
U_scale = 0.10
obs_clip = 50.0

cfl = 0.45
check_cfl = True
well_balanced = True

# Collection-matched equilibrium.
Q_BAR_FIXED = 11.93
REFERENCE_MODE = "ode"
U_ref = 0.30

# Online first-order low-pass filter for every learned/RL controller.
# The filtered action is the ACTUAL closed-loop command passed to env.step().
#
#   U_k^f = (1-alpha_f) U_{k-1}^f + alpha_f U_k^RL
#
RL_CONTROL_FILTER_ALPHA = 0.70

# Shared deterministic linear test initial condition.
TEST_H_LEFT = 3.5
TEST_H_RIGHT = 4.4
TEST_Q_VALUE = 9.5

# Reward used by the new training program:
# r_k = -alpha*tracking - beta*state_rate
#       -gamma*control - delta*delta_control.
reward_alpha = 1.0
reward_beta = 0.10
reward_gamma = 0.05
reward_delta = 0.10
reward_w_H = 1.0
reward_w_Q = 1.0
terminal_tracking_weight = 0
truncate_penalty = -200.0

# Keep the test horizon and plotting/table workflow consistent with the previous
# unified comparison program.
SAC_T = 80000.0


learning_rate = 1e-4
deeponet_learning_rate = 1e-6

LATENT_DIM = 102
RAW_STATE_DIM = 2 * N
SAC_FEATURE_DIM = LATENT_DIM

# This file is only needed while the policy object is being constructed. The
# trained SB3 checkpoint subsequently restores all actor/critic/DeepONet weights.
DEEPNET_PRETRAINED_PATH = "modedlNet10000QqQqQ14444_hadamard.pt"

# ============================================================
# New Hadamard DeepONet controller and Hybrid SAC modules
# ============================================================

def preprocess_obs(obs: torch.Tensor) -> torch.Tensor:
    if not isinstance(obs, torch.Tensor):
        obs = torch.as_tensor(obs)
    return obs.float()


class BranchNet(nn.Module):
    def __init__(self, shape=N, output_dim=LATENT_DIM):
        super().__init__()
        self.shape = int(shape)
        self.output_dim = int(output_dim)

        self.conv1 = nn.Conv1d(
            in_channels=2,
            out_channels=16,
            kernel_size=5,
            stride=2,
        )
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv1d(
            in_channels=16,
            out_channels=32,
            kernel_size=5,
            stride=2,
        )

        with torch.no_grad():
            tmp = torch.zeros(1, 2, self.shape)
            tmp = self.relu(self.conv1(tmp))
            tmp = self.relu(self.conv2(tmp))
            flat_dim = tmp.reshape(1, -1).shape[1]

        self.fc1 = nn.Linear(flat_dim, self.output_dim)

    def forward(self, x):
        x = preprocess_obs(x)
        if x.dim() == 1:
            x = x.unsqueeze(0)

        if x.shape[1] != 2 * self.shape:
            raise ValueError(
                f"Branch input dimension is {x.shape[1]}, expected {2 * self.shape}."
            )

        x = x.reshape(x.shape[0], 2, self.shape)
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = torch.flatten(x, start_dim=1)
        x = self.relu(self.fc1(x))
        return x


class TrunkNet(nn.Module):
    def __init__(self, dim_x=1, width=LATENT_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim_x, 16),
            nn.ReLU(),
            nn.Linear(16, 32),
            nn.ReLU(),
            nn.Linear(32, width),
            nn.ReLU(),
        )

    def forward(self, x):
        return self.net(x.float())


class DeepONetModified(nn.Module):
    """
    Hadamard-fusion DeepONet controller.

        b(s) in R^p,
        t(x) in R^p,
        z(s,x) = b(s) * t(x) in R^p,
        U_prior = f_ctrl(z) in R.
    """

    def __init__(self, shape=N, latent_dim=LATENT_DIM):
        super().__init__()
        self.latent_dim = int(latent_dim)

        self.branch = BranchNet(shape, output_dim=self.latent_dim)
        self.trunk = TrunkNet(dim_x=1, width=self.latent_dim)

        # Hadamard fusion preserves latent_dim features.
        self.fc3 = nn.Linear(self.latent_dim, 32)
        self.relu = nn.ReLU()
        self.fc4 = nn.Linear(32, 16)
        self.fc5 = nn.Linear(16, 1)

    def extract_latent(self, obs, grid):
        b = self.branch(obs)       # [batch, latent_dim]
        t = self.trunk(grid)       # [n_query, latent_dim]

        if t.shape[0] == 1:
            t = t.expand(b.shape[0], -1)
        elif t.shape[0] != b.shape[0]:
            raise ValueError(
                f"Branch batch {b.shape[0]} and trunk batch {t.shape[0]} do not match."
            )

        z = b * t                  # Hadamard product
        return z

    def control_head(self, z):
        y = self.relu(self.fc3(z))
        y = self.relu(self.fc4(y))
        y = self.fc5(y)
        return y

    def forward(self, obs, grid, return_latent=False):
        z = self.extract_latent(obs, grid)
        u_prior_physical = self.control_head(z)

        if return_latent:
            return u_prior_physical, z

        return u_prior_physical


class HybridCACDeepONetFeatureExtractor(BaseFeaturesExtractor):
    """
    Return the Hadamard-fused DeepONet latent feature.

    Input observation and DeepONet input:
        [H_1,...,H_N,Q_1,...,Q_N]

    SAC feature:
        [z_1,...,z_p]
    """

    def __init__(
        self,
        observation_space: spaces.Box,
        features_dim: int = SAC_FEATURE_DIM,
        pretrained_path: str = DEEPNET_PRETRAINED_PATH,
        query_point: float = 1.0,
        freeze_deeponet: bool = False,
    ):
        if features_dim != SAC_FEATURE_DIM:
            raise ValueError(
                f"features_dim must be {SAC_FEATURE_DIM}, got {features_dim}."
            )

        super().__init__(observation_space, features_dim)

        self.query_point = float(query_point)
        self.net1 = DeepONetModified(N, latent_dim=LATENT_DIM)

        # Normalized action used by the hybrid SAC actor.
        self.u_deep_norm: Optional[torch.Tensor] = None

        # Physical prior gate retained for diagnostics.
        self.u_prior_physical: Optional[torch.Tensor] = None

        self.register_buffer(
            "grid",
            torch.tensor([[self.query_point]], dtype=torch.float32),
        )

        self._load_pretrained(pretrained_path)
        self._apply_freeze(freeze_all=freeze_deeponet)

    def _apply_freeze(self, freeze_all=False):
        if freeze_all:
            for param in self.net1.parameters():
                param.requires_grad = False
            print("All DeepONet parameters are frozen.")
            return

        # Keep the same transfer-learning structure used during training.
        for param in self.net1.branch.conv1.parameters():
            param.requires_grad = False
        for param in self.net1.branch.conv2.parameters():
            param.requires_grad = False

        for param in self.net1.branch.fc1.parameters():
            param.requires_grad = True

        for param in self.net1.trunk.parameters():
            param.requires_grad = False

        last_linear = None
        for module in reversed(list(self.net1.trunk.net)):
            if isinstance(module, nn.Linear):
                last_linear = module
                break

        if last_linear is not None:
            for param in last_linear.parameters():
                param.requires_grad = True

        for layer in [self.net1.fc3, self.net1.fc4, self.net1.fc5]:
            for param in layer.parameters():
                param.requires_grad = True

    @staticmethod
    def _clean_state_dict_keys(state_dict):
        cleaned = {}
        prefixes = ["module.", "model.", "net1.", "deeponet."]

        for key, value in state_dict.items():
            new_key = key
            changed = True
            while changed:
                changed = False
                for prefix in prefixes:
                    if new_key.startswith(prefix):
                        new_key = new_key[len(prefix):]
                        changed = True
            cleaned[new_key] = value

        return cleaned

    def _load_pretrained(self, pretrained_path: str):
        path = Path(pretrained_path)

        if not path.exists():
            raise FileNotFoundError(
                f"Pretrained Hadamard DeepONet checkpoint not found: {path}"
            )

        state_dict = torch.load(str(path), map_location="cpu")

        if isinstance(state_dict, dict) and "state_dict" in state_dict:
            state_dict = state_dict["state_dict"]

        if not isinstance(state_dict, dict):
            raise TypeError(
                f"Unsupported checkpoint type: {type(state_dict).__name__}"
            )

        cleaned = self._clean_state_dict_keys(state_dict)

        try:
            self.net1.load_state_dict(cleaned, strict=True)
        except RuntimeError as exc:
            raise RuntimeError(
                "The checkpoint is incompatible with the Hadamard-fusion "
                "DeepONet. Re-pretrain DeepONet with fc3=Linear(102,32) and "
                "z=b*t before running this test script.\n"
                f"Original error:\n{exc}"
            ) from exc

        print(f"Loaded pretrained Hadamard DeepONet: {path}")

    @staticmethod
    def _physical_gate_to_normalized_action(u_physical):
        scale = max_control_value - min_control_value
        if scale <= 0.0:
            raise ValueError(
                "max_control_value must be greater than min_control_value."
            )
        return 2.0 * (u_physical - min_control_value) / scale - 1.0

    def forward(self, observations: torch.Tensor) -> torch.Tensor:
        obs = preprocess_obs(observations)
        if obs.dim() == 1:
            obs = obs.unsqueeze(0)

        if obs.shape[1] != RAW_STATE_DIM:
            raise ValueError(
                f"Raw observation dimension is {obs.shape[1]}, "
                f"expected {RAW_STATE_DIM}."
            )

        u_prior_physical, z = self.net1(
            obs,
            self.grid.to(obs.device),
            return_latent=True,
        )

        # Clip the physical DeepONet prior before mapping it to [-1,1].
        u_prior_physical = torch.clamp(
            u_prior_physical,
            min_control_value,
            max_control_value,
        )
        self.u_prior_physical = u_prior_physical

        u_deep_norm = self._physical_gate_to_normalized_action(u_prior_physical)
        self.u_deep_norm = torch.clamp(u_deep_norm, -1.0, 1.0)

        # The SAC representation contains only the fused DeepONet feature z.
        return z.view(z.size(0), -1)


class HybridCACActor(Actor):
    """SAC actor whose normalized output is blended with DeepONet prior action."""

    def __init__(self, *args, alpha: float = 0.4, **kwargs):
        super().__init__(*args, **kwargs)
        init_alpha = float(np.clip(alpha, 0.0, 1.0))

        self.register_buffer(
            "alpha_tensor",
            torch.tensor(init_alpha, dtype=torch.float32),
        )

        self.last_deep_action = None
        self.last_rl_action = None
        self.last_hybrid_action = None

    @property
    def alpha(self) -> float:
        return float(self.alpha_tensor.detach().cpu().item())

    def set_alpha(self, alpha: float):
        alpha_value = float(np.clip(alpha, 0.0, 1.0))
        with torch.no_grad():
            self.alpha_tensor.fill_(alpha_value)

    def _deep_action(self, obs: torch.Tensor) -> torch.Tensor:
        u_deep = getattr(self.features_extractor, "u_deep_norm", None)

        if u_deep is None or u_deep.shape[0] != obs.shape[0]:
            _ = self.features_extractor(obs)
            u_deep = self.features_extractor.u_deep_norm

        return torch.clamp(u_deep, -1.0, 1.0)

    def _hybridize(self, rl_action: torch.Tensor, obs: torch.Tensor) -> torch.Tensor:
        u_deep = self._deep_action(obs)

        alpha = self.alpha_tensor.to(
            device=rl_action.device,
            dtype=rl_action.dtype,
        )

        hybrid = (1.0 - alpha) * u_deep + alpha * rl_action
        hybrid = torch.clamp(hybrid, -1.0, 1.0)

        self.last_deep_action = u_deep.detach()
        self.last_rl_action = rl_action.detach()
        self.last_hybrid_action = hybrid.detach()

        return hybrid

    def forward(self, obs: torch.Tensor, deterministic: bool = False):
        rl_action = super().forward(obs, deterministic=deterministic)
        return self._hybridize(rl_action, obs)

    def action_log_prob(self, obs: torch.Tensor):
        # The log probability is that of the SAC stochastic component.
        # The deterministic affine hybrid mapping is not included in log_prob.
        rl_action, log_prob = super().action_log_prob(obs)
        hybrid_action = self._hybridize(rl_action, obs)
        return hybrid_action, log_prob


class HybridCACCritic(nn.Module):
    """Twin-Q critic using the Hadamard-fused DeepONet feature."""

    def __init__(
        self,
        observation_space: spaces.Box,
        action_space: spaces.Box,
        hidden_dim: int = 256,
        features_dim: int = SAC_FEATURE_DIM,
        pretrained_path: str = DEEPNET_PRETRAINED_PATH,
    ):
        super().__init__()

        self.features_extractor = HybridCACDeepONetFeatureExtractor(
            observation_space,
            features_dim=features_dim,
            pretrained_path=pretrained_path,
            query_point=1.0,
            freeze_deeponet=False,
        )

        action_dim = int(np.prod(action_space.shape))
        input_dim = features_dim + action_dim

        self.q1 = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )
        self.q2 = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )

    def set_training_mode(self, mode: bool):
        self.train(mode)

    def forward(self, obs: torch.Tensor, actions: torch.Tensor):
        features = self.features_extractor(obs)
        q_input = torch.cat([features, actions], dim=1)
        return self.q1(q_input), self.q2(q_input)


class HybridCACSACPolicy(SACPolicy):
    """SAC policy with Hadamard DeepONet representation/control transfer."""

    def __init__(
        self,
        *args,
        hybrid_alpha_initial: float = 0.4,
        deeponet_pretrained_path: str = DEEPNET_PRETRAINED_PATH,
        **kwargs,
    ):
        self.hybrid_alpha_initial = float(hybrid_alpha_initial)
        self.deeponet_pretrained_path = deeponet_pretrained_path

        kwargs.setdefault(
            "features_extractor_class",
            HybridCACDeepONetFeatureExtractor,
        )
        kwargs.setdefault(
            "features_extractor_kwargs",
            dict(
                features_dim=SAC_FEATURE_DIM,
                pretrained_path=deeponet_pretrained_path,
                query_point=1.0,
                freeze_deeponet=False,
            ),
        )

        super().__init__(*args, **kwargs)

    def make_actor(self, features_extractor=None):
        kwargs = self._update_features_extractor(
            self.actor_kwargs,
            features_extractor,
        )
        return HybridCACActor(
            **kwargs,
            alpha=self.hybrid_alpha_initial,
        ).to(self.device)

    def make_critic(self, features_extractor=None):
        return HybridCACCritic(
            self.observation_space,
            self.action_space,
            hidden_dim=256,
            features_dim=SAC_FEATURE_DIM,
            pretrained_path=self.deeponet_pretrained_path,
        ).to(self.device)

    def set_alpha(self, alpha: float):
        self.actor.set_alpha(alpha)


# ============================================================
# 9. Optimizer helper: separate DeepONet learning rate
# ============================================================

def _make_param_groups_with_deeponet_lr(module: nn.Module):
    deeponet_params = []
    other_params = []

    for name, param in module.named_parameters():
        if not param.requires_grad:
            continue

        if "features_extractor.net1" in name or name.startswith("net1."):
            deeponet_params.append(param)
        else:
            other_params.append(param)

    groups = []

    if other_params:
        groups.append(
            {
                "params": other_params,
                "lr": learning_rate,
                "is_deeponet_group": False,
            }
        )

    if deeponet_params:
        groups.append(
            {
                "params": deeponet_params,
                "lr": deeponet_learning_rate,
                "is_deeponet_group": True,
            }
        )

    return groups


def set_deeponet_small_lr_optimizers(model):
    actor_groups = _make_param_groups_with_deeponet_lr(model.actor)
    critic_groups = _make_param_groups_with_deeponet_lr(model.critic)

    model.actor.optimizer = torch.optim.Adam(actor_groups, eps=1e-5)
    model.critic.optimizer = torch.optim.Adam(critic_groups, eps=1e-5)

    n_actor_deep = sum(
        p.numel()
        for group in actor_groups
        if group.get("is_deeponet_group", False)
        for p in group["params"]
    )
    n_actor_other = sum(
        p.numel()
        for group in actor_groups
        if not group.get("is_deeponet_group", False)
        for p in group["params"]
    )
    n_critic_deep = sum(
        p.numel()
        for group in critic_groups
        if group.get("is_deeponet_group", False)
        for p in group["params"]
    )
    n_critic_other = sum(
        p.numel()
        for group in critic_groups
        if not group.get("is_deeponet_group", False)
        for p in group["params"]
    )

    print("Custom optimizers with separate DeepONet learning rate are set.")
    print(f"Actor params: DeepONet={n_actor_deep}, other={n_actor_other}")
    print(f"Critic params: DeepONet={n_critic_deep}, other={n_critic_other}")
    print(
        f"Base SAC learning_rate={learning_rate}, "
        f"DeepONet learning_rate={deeponet_learning_rate}"
    )


class HybridSAC(SAC):
    """Preserve the smaller learning rate of DeepONet parameter groups."""

    def _update_learning_rate(self, optimizers):
        super()._update_learning_rate(optimizers)

        if not isinstance(optimizers, (list, tuple)):
            optimizers = [optimizers]

        current_lr = self.lr_schedule(self._current_progress_remaining)

        for optimizer in optimizers:
            for group in optimizer.param_groups:
                if group.get("is_deeponet_group", False):
                    group["lr"] = deeponet_learning_rate
                elif "is_deeponet_group" in group:
                    group["lr"] = current_lr




class EvalHybridSAC(HybridSAC):
    """Evaluation loader compatible with DeepONet-specific optimizer groups."""

    def _setup_model(self):
        super()._setup_model()
        set_deeponet_small_lr_optimizers(self)

    def set_parameters(self, load_path_or_dict, exact_match: bool = True, device="auto"):
        try:
            return super().set_parameters(
                load_path_or_dict,
                exact_match=exact_match,
                device=device,
            )
        except ValueError as exc:
            message = str(exc)
            optimizer_error = (
                "parameter group" in message
                or "Names of parameters" in message
                or "optimizer" in message.lower()
            )

            if isinstance(load_path_or_dict, dict) and optimizer_error:
                print(
                    "Evaluation load: incompatible optimizer states are skipped; "
                    "policy, actor, critic, DeepONet, and alpha weights are retained."
                )
                filtered = {
                    name: value
                    for name, value in load_path_or_dict.items()
                    if "optimizer" not in name.lower()
                }
                return super().set_parameters(
                    filtered,
                    exact_match=False,
                    device=device,
                )
            raise


# ============================================================
# Physical setup: ODE equilibrium + shared linear test initial condition
# ============================================================

def make_cac_table_params(
    q_bar: float = Q_BAR_FIXED,
    h_bar_0: float = 3.80,
    h_bar_L: float = 4.70,
) -> CACParams:
    """Create CAC parameters consistent with the new training program."""
    return CACParams(
        L=11239.0,
        B=40.0,
        g=9.81,
        I=7.92e-5,
        K=33.0,
        mu0=0.4,
        muL=0.4,
        z_up=5.00,
        U0_bar=0.1548,
        U_L_bar=U_ref,
        h_s=4.094,
        Q_bar=float(q_bar),
        H_bar_0=float(h_bar_0),
        H_bar_L=float(h_bar_L),
        H_min=0.20,
        max_state_value=1e10,
    )


def make_grid(p: CACParams, n_grid: int) -> Tuple[np.ndarray, float]:
    X = np.linspace(0.0, p.L, n_grid)
    dx = p.L / (n_grid - 1)
    return X, dx


def downstream_equilibrium_depth_from_q(Q: float, p: CACParams) -> float:
    """Compute Hbar(L) from the downstream equilibrium gate relation."""
    Q = float(Q)
    if not np.isfinite(Q) or Q <= 0.0:
        raise ValueError("The equilibrium discharge must be positive and finite.")

    kinetic_head = (
        Q**2 / (2.0 * p.g * p.B**2 * p.muL**2)
    ) ** (1.0 / 3.0)
    return float(p.h_s + p.U_L_bar + kinetic_head)


def integrate_steady_fixed_q_ode(
    X: np.ndarray,
    p: CACParams,
    Q_candidate: float,
) -> Tuple[np.ndarray, np.ndarray]:
    """Integrate the steady Saint-Venant ODE backward from x=L to x=0."""
    X = np.asarray(X, dtype=float)
    if X.ndim != 1 or X.size < 2:
        raise ValueError("X must be a one-dimensional grid with at least two points.")
    if not np.all(np.diff(X) > 0.0):
        raise ValueError("X must be strictly increasing.")

    Q_candidate = float(Q_candidate)
    if not np.isfinite(Q_candidate) or Q_candidate <= 0.0:
        raise ValueError("Q_candidate must be positive and finite.")

    Hbar_L = downstream_equilibrium_depth_from_q(Q_candidate, p)

    def rhs(_x: float, state: np.ndarray) -> np.ndarray:
        H = float(state[0])
        Q = float(state[1])

        if not np.isfinite(H) or not np.isfinite(Q):
            raise FloatingPointError("Non-finite state in steady reference ODE.")
        if H <= float(p.H_min):
            raise FloatingPointError("Steady reference ODE violates minimum depth.")

        area = p.B * H
        hydraulic_radius = p.B * H / (p.B + 2.0 * H)
        friction_slope = Q * abs(Q) / (
            p.K**2 * area**2 * hydraulic_radius ** (4.0 / 3.0)
        )

        denominator = p.g * p.B * H - Q**2 / (p.B * H**2)
        if abs(denominator) <= 1e-10:
            raise FloatingPointError(
                "Steady ODE approaches the critical-flow singularity."
            )

        dHdx = p.g * p.B * H * (p.I - friction_slope) / denominator
        if not np.isfinite(dHdx):
            raise FloatingPointError("Non-finite dHbar/dx in steady ODE.")

        return np.asarray([dHdx, 0.0], dtype=float)

    sol = solve_ivp(
        fun=rhs,
        t_span=(float(X[-1]), float(X[0])),
        y0=np.asarray([Hbar_L, Q_candidate], dtype=float),
        t_eval=X[::-1],
        rtol=1e-9,
        atol=1e-11,
        max_step=p.L / 1000.0,
    )

    if not sol.success:
        raise RuntimeError(f"Failed to solve steady ODE: {sol.message}")
    if sol.y.shape[1] != X.size:
        raise RuntimeError("Steady ODE did not return all requested grid points.")

    Hbar = np.asarray(sol.y[0][::-1], dtype=float)
    Qbar = np.asarray(sol.y[1][::-1], dtype=float)

    if not np.all(np.isfinite(Hbar)) or not np.all(np.isfinite(Qbar)):
        raise FloatingPointError("Steady reference contains NaN or Inf.")
    if float(np.min(Hbar)) <= float(p.H_min):
        raise RuntimeError("Steady Hbar profile violates the minimum depth.")

    return Hbar, Qbar


def solve_steady_reference_ode(
    X: np.ndarray,
    p: CACParams,
    q_bar_fixed: float = Q_BAR_FIXED,
) -> Tuple[np.ndarray, np.ndarray, float]:
    """Solve Hbar(x) for the prescribed constant Qbar=11.93 m^3/s."""
    q_bar_fixed = float(q_bar_fixed)
    Hbar, _ = integrate_steady_fixed_q_ode(X, p, q_bar_fixed)
    Qbar = np.full_like(X, q_bar_fixed, dtype=float)
    return Hbar, Qbar, q_bar_fixed


def make_reference_profiles(
    X: np.ndarray,
    p: CACParams,
    reference: str = REFERENCE_MODE,
    q_bar_fixed: float = Q_BAR_FIXED,
) -> Tuple[np.ndarray, np.ndarray, float]:
    if reference in {"ode", "strict"}:
        return solve_steady_reference_ode(X, p, q_bar_fixed=q_bar_fixed)
    if reference == "linear":
        Hbar = p.H_bar_0 + (p.H_bar_L - p.H_bar_0) * (X / p.L)
        Qbar = np.full_like(X, q_bar_fixed, dtype=float)
        return Hbar, Qbar, float(q_bar_fixed)
    raise ValueError("reference must be 'ode', 'strict', or 'linear'.")


def build_collection_matched_reference():
    """Return one p, X, dx, Hbar, Qbar tuple shared by every controller."""
    nominal_p = make_cac_table_params(q_bar=Q_BAR_FIXED)
    X, dx = make_grid(nominal_p, N)
    Hbar, Qbar, q_equilibrium = make_reference_profiles(
        X,
        nominal_p,
        reference=REFERENCE_MODE,
        q_bar_fixed=Q_BAR_FIXED,
    )

    p = make_cac_table_params(
        q_bar=q_equilibrium,
        h_bar_0=float(Hbar[0]),
        h_bar_L=float(Hbar[-1]),
    )
    return p, X, dx, Hbar, Qbar


def generate_test_initial_condition(
    X: np.ndarray,
    Hbar: np.ndarray,
    Qbar: np.ndarray,
    H_left: float = TEST_H_LEFT,
    H_right: float = TEST_H_RIGHT,
    Q_value: float = TEST_Q_VALUE,
) -> Tuple[np.ndarray, np.ndarray, Dict[str, float]]:
    """Generate the shared deterministic linear test initial condition.

    The water-depth profile is linearly interpolated between the prescribed
    upstream and downstream endpoint values:

        H(x,0) = H_left + (H_right-H_left) x/L,

    while the discharge profile remains spatially constant:

        Q(x,0) = Q_value.

    Hbar and Qbar are used only to report deviations from the ODE equilibrium.
    """
    X = np.asarray(X, dtype=float)
    Hbar = np.asarray(Hbar, dtype=float)
    Qbar = np.asarray(Qbar, dtype=float)

    if X.ndim != 1 or Hbar.shape != X.shape or Qbar.shape != X.shape:
        raise ValueError(
            "X, Hbar, and Qbar must be one-dimensional arrays of equal length."
        )

    L = float(X[-1])
    if not np.isfinite(L) or L <= 0.0:
        raise ValueError("The spatial-domain length must be positive and finite.")

    H_left = float(H_left)
    H_right = float(H_right)
    Q_value = float(Q_value)
    if not all(np.isfinite(v) for v in (H_left, H_right, Q_value)):
        raise ValueError("H_left, H_right, and Q_value must be finite.")

    H0 = H_left + (H_right - H_left) * (X / L)
    Q0 = np.full_like(X, Q_value, dtype=float)

    if float(np.min(H0)) <= 0.20:
        raise ValueError("The linear initial H profile violates H_min=0.20 m.")
    if float(np.min(Q0)) <= 0.0:
        raise ValueError("The initial discharge must be positive.")

    details = {
        "H_left": float(H_left),
        "H_right": float(H_right),
        "Q_value": Q_value,
        "H_realized_max_deviation": float(np.max(np.abs(H0 - Hbar))),
        "Q_realized_max_deviation": float(np.max(np.abs(Q0 - Qbar))),
    }
    return H0, Q0, details

def make_fixed_initial_condition(
    X: np.ndarray,
    Hbar: np.ndarray,
    Qbar: np.ndarray,
    H_left: float = TEST_H_LEFT,
    H_right: float = TEST_H_RIGHT,
    Q_value: float = TEST_Q_VALUE,
) -> Tuple[np.ndarray, np.ndarray]:
    """Return the shared deterministic linear-interpolation test initial state."""
    H0, Q0, _ = generate_test_initial_condition(
        X,
        Hbar,
        Qbar,
        H_left=H_left,
        H_right=H_right,
        Q_value=Q_value,
    )
    return H0, Q0


def make_test_initial_condition_func(
    H_left: float = TEST_H_LEFT,
    H_right: float = TEST_H_RIGHT,
    Q_value: float = TEST_Q_VALUE,
):
    """Return the same deterministic linear-interpolation reset callback for all controllers."""
    def reset_func(X, Hbar, Qbar, rng):
        del rng
        H0, Q0, _ = generate_test_initial_condition(
            X,
            Hbar,
            Qbar,
            H_left=H_left,
            H_right=H_right,
            Q_value=Q_value,
        )
        return H0, Q0

    return reset_func

# ============================================================
# Hybrid action diagnostics
# ============================================================

def normalized_action_to_physical_ul(a):
    """Map normalized action a in [-1, 1] to physical downstream gate U_L."""
    a = float(np.clip(float(a), -1.0, 1.0))
    return float(min_control_value + 0.5 * (a + 1.0) * (max_control_value - min_control_value))


def physical_ul_to_normalized_action(u_physical):
    """Map physical downstream-gate command U_L back to normalized action [-1,1]."""
    scale = float(max_control_value - min_control_value)
    if scale <= 0.0:
        raise ValueError("max_control_value must be greater than min_control_value.")

    u_physical = float(
        np.clip(float(u_physical), min_control_value, max_control_value)
    )
    action = 2.0 * (u_physical - min_control_value) / scale - 1.0
    return float(np.clip(action, -1.0, 1.0))


def lowpass_filter_rl_control(
    action_raw,
    previous_u_filtered,
    alpha=RL_CONTROL_FILTER_ALPHA,
):
    """Apply a first-order low-pass filter to one scalar learned-controller command."""
    alpha = float(alpha)
    if not (0.0 < alpha <= 1.0):
        raise ValueError("RL_CONTROL_FILTER_ALPHA must satisfy 0 < alpha <= 1.")

    a_raw = float(np.asarray(action_raw, dtype=float).reshape(-1)[0])
    a_raw = float(np.clip(a_raw, -1.0, 1.0))
    u_raw = normalized_action_to_physical_ul(a_raw)

    previous_u_filtered = float(
        np.clip(float(previous_u_filtered), min_control_value, max_control_value)
    )
    u_filtered = (
        (1.0 - alpha) * previous_u_filtered
        + alpha * u_raw
    )
    u_filtered = float(
        np.clip(u_filtered, min_control_value, max_control_value)
    )

    a_filtered = physical_ul_to_normalized_action(u_filtered)
    action_filtered = np.asarray([a_filtered], dtype=np.float32)

    return action_filtered, u_raw, u_filtered


def get_hybrid_action_components(model, obs, action_hybrid):
    """
    Extract DeepONet, RL, and hybrid normalized actions for the loaded hybrid SAC policy.

    The actor actually outputs
        a_hybrid = (1-alpha) * a_deep + alpha * a_RL.
    During testing, model.predict(obs) returns a_hybrid. We compute a_deep by calling
    the actor feature extractor and infer a_RL from the affine mixing formula.
    """
    actor = model.policy.actor
    device = model.device

    obs_tensor = torch.as_tensor(obs, dtype=torch.float32, device=device)
    if obs_tensor.dim() == 1:
        obs_tensor = obs_tensor.unsqueeze(0)

    with torch.no_grad():
        # This call updates actor.features_extractor.u_deep_norm.
        _ = actor.features_extractor(obs_tensor)
        u_deep = getattr(actor.features_extractor, "u_deep_norm", None)

    if u_deep is None:
        a_deep = np.nan
    else:
        a_deep = float(u_deep.detach().cpu().numpy().reshape(-1)[0])
        a_deep = float(np.clip(a_deep, -1.0, 1.0))

    a_hybrid = float(np.asarray(action_hybrid).reshape(-1)[0])
    a_hybrid = float(np.clip(a_hybrid, -1.0, 1.0))

    alpha = float(getattr(actor, "alpha", np.nan))

    if np.isfinite(alpha) and alpha > 1e-8 and np.isfinite(a_deep):
        a_rl = (a_hybrid - (1.0 - alpha) * a_deep) / alpha
        a_rl = float(np.clip(a_rl, -1.0, 1.0))
    else:
        a_rl = np.nan

    return {
        "alpha": alpha,
        "a_deep": a_deep,
        "a_rl": a_rl,
        "a_hybrid": a_hybrid,
        "U_deep": normalized_action_to_physical_ul(a_deep) if np.isfinite(a_deep) else np.nan,
        "U_rl": normalized_action_to_physical_ul(a_rl) if np.isfinite(a_rl) else np.nan,
        "U_hybrid": normalized_action_to_physical_ul(a_hybrid),
    }


# ============================================================
# Reward class
# ============================================================

class CACTrackingSmoothReward(BaseReward):
    """
    Discrete reward at the control instant t_k:

        r_k = -alpha ||y_tilde_k-y_tilde_star||^2
              -beta  ||(y_tilde_k-y_tilde_{k-1})/Delta_t||^2
              -gamma ||U_tilde_k||^2
              -delta ||U_tilde_k-U_tilde_{k-1}||^2.

    Here:
        y_tilde_star = 0,
        H_tilde_error = (H-H_eq)/H_scale,
        Q_tilde_error = (Q-Q_eq)/Q_scale,
        U_tilde = (U-U_eq)/U_scale.

    Spatial means are used instead of unnormalized sums so that the reward scale
    does not grow with the number of grid points N.
    """

    def __init__(
        self,
        H_eq,
        Q_eq,
        delta_t=control_sample_rate,
        H_scale=0.10,
        Q_scale=3.00,
        U_scale=0.10,
        alpha=1.0,
        beta=0.10,
        gamma=0.05,
        delta=0.10,
        w_H=1.0,
        w_Q=1.0,
        U_ref=0.3,
        terminal_tracking_weight=10.0,
        truncate_penalty=-200.0,
    ):
        self.H_eq = np.asarray(H_eq, dtype=float)
        self.Q_eq = np.asarray(Q_eq, dtype=float)
        self.N = len(self.H_eq)

        self.delta_t = float(delta_t)
        self.H_scale = float(H_scale)
        self.Q_scale = float(Q_scale)
        self.U_scale = float(U_scale)

        self.alpha = float(alpha)
        self.beta = float(beta)
        self.gamma = float(gamma)
        self.delta = float(delta)
        self.w_H = float(w_H)
        self.w_Q = float(w_Q)

        self.U_ref = float(U_ref)
        self.terminal_tracking_weight = float(terminal_tracking_weight)
        self.truncate_penalty = float(truncate_penalty)

        # Previous values at the preceding reward/control call.
        self.prev_H: Optional[np.ndarray] = None
        self.prev_Q: Optional[np.ndarray] = None
        self.prev_action: Optional[float] = None

        # Diagnostics.
        self.last_terms = {
            "tracking_cost": 0.0,
            "state_rate_cost": 0.0,
            "control_cost": 0.0,
            "delta_control_cost": 0.0,
        }

    def _reset_memory(self):
        self.prev_H = None
        self.prev_Q = None
        self.prev_action = None

    def _split_state(self, state_vec):
        state_vec = np.asarray(state_vec, dtype=float).reshape(-1)
        if state_vec.size < 2 * self.N:
            raise ValueError(
                f"PDE state length is {state_vec.size}, expected at least {2 * self.N}."
            )
        H = state_vec[:self.N]
        Q = state_vec[self.N:2 * self.N]
        return H, Q

    def _action_value(self, action):
        # The original environment passes the physical downstream gate to reward.
        if action is None:
            return self.U_ref
        value = float(np.asarray(action).reshape(-1)[0])
        return float(
            np.nan_to_num(
                value,
                nan=self.U_ref,
                posinf=self.U_ref,
                neginf=self.U_ref,
            )
        )

    def reward(
        self,
        uVec=None,
        time_index=None,
        terminate=None,
        truncate=None,
        action=None,
        control_sample_rate=None,
    ):
        if uVec is None or time_index is None:
            return 0.0

        # A new episode normally starts with a small time index.
        if int(time_index) <= 1:
            self._reset_memory()

        if truncate:
            self._reset_memory()
            return float(self.truncate_penalty)

        Hk, Qk = self._split_state(uVec[time_index])

        if (
            not np.all(np.isfinite(Hk))
            or not np.all(np.isfinite(Qk))
            or np.any(Hk <= 0.0)
        ):
            self._reset_memory()
            return float(self.truncate_penalty)

        # ----------------------------------------------------
        # 1) Tracking term
        # ----------------------------------------------------
        H_error_tilde = (Hk - self.H_eq) / self.H_scale
        Q_error_tilde = (Qk - self.Q_eq) / self.Q_scale

        tracking_cost = (
            self.w_H * np.mean(H_error_tilde ** 2)
            + self.w_Q * np.mean(Q_error_tilde ** 2)
        )

        # ----------------------------------------------------
        # 2) State-rate term
        # ----------------------------------------------------
        if self.prev_H is None or self.prev_Q is None:
            state_rate_cost = 0.0
        else:
            delta_t_k = (
                float(control_sample_rate)
                if control_sample_rate is not None
                else self.delta_t
            )
            delta_t_k = max(delta_t_k, 1e-12)

            dH_tilde = (Hk - self.prev_H) / (delta_t_k * self.H_scale)
            dQ_tilde = (Qk - self.prev_Q) / (delta_t_k * self.Q_scale)

            state_rate_cost = (
                self.w_H * np.mean(dH_tilde ** 2)
                + self.w_Q * np.mean(dQ_tilde ** 2)
            )

        # ----------------------------------------------------
        # 3) Control-magnitude term
        # ----------------------------------------------------
        Uk = self._action_value(action)
        U_tilde = (Uk - self.U_ref) / self.U_scale
        control_cost = U_tilde ** 2

        # ----------------------------------------------------
        # 4) Control-increment term
        # ----------------------------------------------------
        if self.prev_action is None:
            delta_control_cost = 0.0
        else:
            U_prev_tilde = (self.prev_action - self.U_ref) / self.U_scale
            delta_control_cost = (U_tilde - U_prev_tilde) ** 2

        reward_value = (
            -self.alpha * tracking_cost
            -self.beta * state_rate_cost
            -self.gamma * control_cost
            -self.delta * delta_control_cost
        )

        if terminate:
            reward_value -= self.terminal_tracking_weight * tracking_cost

        self.prev_H = Hk.copy()
        self.prev_Q = Qk.copy()
        self.prev_action = Uk

        self.last_terms = {
            "tracking_cost": float(tracking_cost),
            "state_rate_cost": float(state_rate_cost),
            "control_cost": float(control_cost),
            "delta_control_cost": float(delta_control_cost),
        }

        return float(reward_value)




# ============================================================
# Environments
# ============================================================

def _make_reward(Hbar, Qbar):
    return CACTrackingSmoothReward(
        H_eq=Hbar,
        Q_eq=Qbar,
        delta_t=control_sample_rate,
        H_scale=H_scale,
        Q_scale=Q_scale,
        U_scale=U_scale,
        alpha=reward_alpha,
        beta=reward_beta,
        gamma=reward_gamma,
        delta=reward_delta,
        w_H=reward_w_H,
        w_Q=reward_w_Q,
        U_ref=U_ref,
        terminal_tracking_weight=terminal_tracking_weight,
        truncate_penalty=truncate_penalty,
    )


def make_env_sac(
    H_left=TEST_H_LEFT,
    H_right=TEST_H_RIGHT,
    Q_value=TEST_Q_VALUE,
    seed=123,
    dH_source=None,
):
    """Create the exact evaluation environment for the raw-HQ learned model.

    The policy receives raw physical [H,Q]. Every controller is reset to the
    same deterministic linear-interpolation H profile and constant Q profile.
    """
    p, X, dx, Hbar, Qbar = build_collection_matched_reference()
    H_init, Q_init = make_fixed_initial_condition(
        X,
        Hbar,
        Qbar,
        H_left=H_left,
        H_right=H_right,
        Q_value=Q_value,
    )

    env_kwargs = dict(
        params=p,
        X_grid=X,
        Hbar=Hbar,
        Qbar=Qbar,
        H_init=H_init,
        Q_init=Q_init,
        sensing_noise_func=lambda observation: observation,
        reset_init_condition_func=make_test_initial_condition_func(
            H_left=H_left,
            H_right=H_right,
            Q_value=Q_value,
        ),
        sensing_loc="full",
        control_type="Gate",
        limit_pde_state_size=True,
        max_state_value=max_state_value,
        min_control_value=min_control_value,
        max_control_value=max_control_value,
        control_sample_rate=control_sample_rate,
        reward_class=_make_reward(Hbar, Qbar),
        normalize=True,
        observation_mode="raw",
        H_scale=H_scale,
        Q_scale=Q_scale,
        obs_clip=obs_clip,
        T=SAC_T,
        dt=dt,
        cfl=cfl,
        check_cfl=check_cfl,
        well_balanced=well_balanced,
    )

    if dH_source is not None:
        env_kwargs["dH_source"] = float(dH_source)

    env = CACSaintVenantPDE1D(**env_kwargs)

    env.reference_X = X
    env.reference_H = Hbar
    env.reference_Q = Qbar
    env.reference_dx = dx

    return env, X, Hbar, Qbar, p


def make_env_sac_error_state(
    H_left=TEST_H_LEFT,
    H_right=TEST_H_RIGHT,
    Q_value=TEST_Q_VALUE,
    seed=123,
    dH_source=None,
):
    """Create the evaluation environment for the old error-state SAC model.

    The physical test problem is unchanged: the ODE equilibrium, linear-interpolation H0,
    constant Q0, PDE parameters, action range, time step, and test horizon are
    identical to the raw-HQ controllers. Only the policy observation changes to

        [(H-Hbar)/H_scale, (Q-Qbar)/Q_scale].

    This matches the old SAC checkpoint trained with
    observation_mode="normalized_error".
    """
    p, X, dx, Hbar, Qbar = build_collection_matched_reference()
    H_init, Q_init = make_fixed_initial_condition(
        X,
        Hbar,
        Qbar,
        H_left=H_left,
        H_right=H_right,
        Q_value=Q_value,
    )

    env_kwargs = dict(
        params=p,
        X_grid=X,
        Hbar=Hbar,
        Qbar=Qbar,
        H_init=H_init,
        Q_init=Q_init,
        sensing_noise_func=lambda observation: observation,
        reset_init_condition_func=make_test_initial_condition_func(
            H_left=H_left,
            H_right=H_right,
            Q_value=Q_value,
        ),
        sensing_loc="full",
        control_type="Gate",
        limit_pde_state_size=True,
        max_state_value=max_state_value,
        min_control_value=min_control_value,
        max_control_value=max_control_value,
        control_sample_rate=control_sample_rate,
        reward_class=_make_reward(Hbar, Qbar),
        normalize=True,
        observation_mode="normalized_error",
        H_scale=H_scale,
        Q_scale=Q_scale,
        obs_clip=obs_clip,
        T=SAC_T,
        dt=dt,
        cfl=cfl,
        check_cfl=check_cfl,
        well_balanced=well_balanced,
    )

    if dH_source is not None:
        env_kwargs["dH_source"] = float(dH_source)

    env = CACSaintVenantPDE1D(**env_kwargs)
    env.reference_X = X
    env.reference_H = Hbar
    env.reference_Q = Qbar
    env.reference_dx = dx
    env.policy_observation_mode = "normalized_error"

    return env, X, Hbar, Qbar, p
















# ============================================================
# SB3 evaluation-only loading helper
# ============================================================

def strip_optimizer_states_from_sb3_model(model_path: str) -> str:
    """
    Create an evaluation-only SB3 zip by removing optimizer states.

    This fixes errors such as:
        ValueError: loaded state dict has a different number of parameter groups

    It is safe for evaluation because optimizer states are only needed when
    continuing training. The policy/actor/critic weights are kept unchanged.

    Return value is a path without the .zip suffix, matching SB3's usual API.
    """
    model_path = Path(model_path)

    if model_path.suffix == ".zip":
        src_zip = model_path
    else:
        src_zip = model_path.with_suffix(".zip")

    if not src_zip.exists():
        raise FileNotFoundError(f"Cannot find SB3 model zip file: {src_zip}")

    dst_zip = src_zip.with_name(src_zip.stem + "_eval_only.zip")

    if dst_zip.exists():
        print(f"Using existing eval-only model: {dst_zip}")
        return str(dst_zip.with_suffix(""))

    with zipfile.ZipFile(src_zip, "r") as zin:
        with zipfile.ZipFile(dst_zip, "w", compression=zipfile.ZIP_DEFLATED) as zout:
            for item in zin.infolist():
                name = item.filename
                # Remove all optimizer states. These can have different parameter-group
                # structures after using DeepONet-specific learning rates.
                if "optimizer" in name.lower():
                    print(f"Skip optimizer state: {name}")
                    continue
                zout.writestr(item, zin.read(name))

    print(f"Created eval-only model: {dst_zip}")
    return str(dst_zip.with_suffix(""))


# ============================================================
# Rollouts
# ============================================================

def run_sac_exact(
    model_path,
    seed=123,
    H_left=TEST_H_LEFT,
    H_right=TEST_H_RIGHT,
    Q_value=TEST_Q_VALUE,
    dH_source=None,
):
    env, X, Hbar, Qbar, p = make_env_sac(
        H_left=H_left,
        H_right=H_right,
        Q_value=Q_value,
        seed=seed,
        dH_source=dH_source,
    )

    # Directly load the trained HybridSAC model.
    # EvalHybridSAC only rebuilds the optimizer structure required by SB3 loading;
    # the actual closed-loop control still comes directly from model.predict().
    # Because this training version saves alpha as actor.alpha_tensor, no manual
    # alpha restoration is needed here.
    model = EvalHybridSAC.load(model_path, env=env)
    obs, _ = env.reset(seed=seed)

    H_hist, Q_hist, U_hist, t_hist, reward_hist = [], [], [], [], []

    previous_u_filtered = float(U_ref)

    done = False
    while not done:
        action, _ = model.predict(obs, deterministic=True)

        action_filtered, u_raw, previous_u_filtered = lowpass_filter_rl_control(
            action,
            previous_u_filtered,
            alpha=RL_CONTROL_FILTER_ALPHA,
        )

        obs, reward, terminated, truncated, info = env.step(action_filtered)

        H, Q = env.get_current_HQ()

        H_hist.append(H.copy())
        Q_hist.append(Q.copy())
        U_hist.append(float(env.last_control))
        t_hist.append(float(info["time"]))
        reward_hist.append(float(reward))

        done = terminated or truncated

    env.close()

    H_hist = np.asarray(H_hist)
    Q_hist = np.asarray(Q_hist)
    U_hist = np.asarray(U_hist)
    t_hist = np.asarray(t_hist)
    reward_hist = np.asarray(reward_hist)

    H_err = H_hist - Hbar[None, :]
    Q_err = Q_hist - Qbar[None, :]

    return {
        "X": X,
        "Hbar": Hbar,
        "Qbar": Qbar,
        "t": t_hist,
        "H": H_hist,
        "Q": Q_hist,
        "U": U_hist,
        "reward": reward_hist,
        "H_l2": np.sqrt(np.mean(H_err ** 2, axis=1)),
        "Q_l2": np.sqrt(np.mean(Q_err ** 2, axis=1)),
        "final_H_minus_Hbar": H_err[-1],
        "final_Q_minus_Qbar": Q_err[-1],
        "total_reward": float(np.sum(reward_hist)),
    }




# ============================================================
# Plot / save
# ============================================================



def plot_hybrid_action_diagnostics(res_sac, save_dir, tag):
    """Plot normalized actions and physical gate commands of DeepONet/RL/hybrid."""
    required = ["A_deep", "A_hybrid", "alpha", "U_deep", "U_hybrid"]
    if any(k not in res_sac for k in required):
        print("Hybrid diagnostics are not available in res_sac.")
        return

    t = np.asarray(res_sac["t"], dtype=float) / 60.0
    a_deep = np.asarray(res_sac["A_deep"], dtype=float)
    a_rl = np.asarray(res_sac["A_rl"], dtype=float)
    a_hybrid = np.asarray(res_sac["A_hybrid"], dtype=float)
    alpha = np.asarray(res_sac["alpha"], dtype=float)
    U_deep = np.asarray(res_sac["U_deep"], dtype=float)
    U_rl = np.asarray(res_sac["U_rl"], dtype=float)
    U_hybrid = np.asarray(res_sac["U_hybrid"], dtype=float)

    fig, ax = plt.subplots(3, 1, figsize=(8, 8), dpi=300, sharex=True)

    ax[0].plot(t, a_deep, label=r"$a_{DeepONet}$")
    if np.any(np.isfinite(a_rl)):
        ax[0].plot(t, a_rl, label=r"$a_{RL}$")
    ax[0].plot(t, a_hybrid, label=r"$a_{hybrid}$")
    ax[0].axhline(1.0, linestyle=":", linewidth=1)
    ax[0].axhline(-1.0, linestyle=":", linewidth=1)
    ax[0].set_ylabel("normalized action", fontsize=18, fontweight="bold", labelpad=8)
    ax[0].legend(**_legend_kwargs_two_columns(loc="best"))
    ax[0].grid(True, alpha=0.3)

    ax[1].plot(t, U_deep, label=r"$U_{DeepONet}$")
    if np.any(np.isfinite(U_rl)):
        ax[1].plot(t, U_rl, label=r"$U_{RL}$")
    ax[1].plot(t, U_hybrid, label=r"$U_{hybrid}$")
    ax[1].axhline(U_ref, linestyle=":", linewidth=1, label=r"$\bar U_L$")
    ax[1].axhline(max_control_value, linestyle=":", linewidth=1, label=r"$U_{max}$")
    ax[1].set_ylabel(r"$U_L$ [m]", fontsize=18, fontweight="bold", labelpad=8)
    ax[1].legend(**_legend_kwargs_two_columns(loc="best"))
    ax[1].grid(True, alpha=0.3)

    ax[2].plot(t, alpha, label=r"$\alpha$")
    ax[2].set_ylabel(r"RL weight $\alpha$", fontsize=18, fontweight="bold", labelpad=8)
    ax[2].set_xlabel("Time [min]", fontsize=18, fontweight="bold", labelpad=10)
    ax[2].set_ylim([-0.05, 1.05])
    ax[2].legend(**_legend_kwargs_two_columns(loc="best"))
    ax[2].grid(True, alpha=0.3)

    fig.suptitle("Hybrid DeepONet-SAC action diagnostics")
    fig.tight_layout()

    path = os.path.join(save_dir, f"{tag}_hybrid_action_diagnostics.png")
    fig.savefig(path, dpi=500, bbox_inches="tight")
    print(f"Saved hybrid action diagnostic figure to: {path}")
    plt.show()




def _resolve_output_dir():
    """Return the active output directory used by this script."""
    if "SAVE_DIR" in globals():
        out = globals()["SAVE_DIR"]
    elif "DEFAULT_SAVE_DIR" in globals():
        out = globals()["DEFAULT_SAVE_DIR"]
    elif "save_dir" in globals():
        out = globals()["save_dir"]
    elif "out_dir" in globals():
        out = globals()["out_dir"]
    else:
        out = "./test_results_sac_style_learned_controllers"
    out = Path(out)
    out.mkdir(parents=True, exist_ok=True)
    return str(out)


def _compute_spatial_l2_over_time(res, state_key):
    """
    Compute spatial L2 norm over time:
        ||H-Hbar||_L2 = sqrt( integral_0^1 (H(x,t)-Hbar(x))^2 ds )
        ||Q-Qbar||_L2 = sqrt( integral_0^1 (Q(x,t)-Qbar(x))^2 ds )
    where s=x/L is the normalized spatial coordinate.
    """
    X = res["X"]
    s = X / X[-1]

    if state_key == "H":
        err = res["H"] - res["Hbar"][None, :]
    elif state_key == "Q":
        err = res["Q"] - res["Qbar"][None, :]
    else:
        raise ValueError("state_key must be 'H' or 'Q'.")

    return np.sqrt(np.trapezoid(err ** 2, s, axis=1))




def plot_surface_single(X, t, Z, zlabel, title, save_path):
    """
    Draw one 3D surface plot for H(x,t) or Q(x,t).
    Plotting style is matched to the first program; all data and calls are unchanged.
    """
    X_km = X / 1000.0
    t_min = t / 60.0

    meshx, mesht = np.meshgrid(X_km, t_min)

    fig = plt.figure(figsize=(8.8, 6.6), dpi=600)
    subfigs = fig.subfigures(nrows=1, ncols=1, hspace=0)
    subfig = subfigs
    subfig.subplots_adjust(left=0.18, bottom=0.08, right=0.98, top=0.96)

    axes = subfig.subplots(
        nrows=1,
        ncols=1,
        subplot_kw={"projection": "3d", "computed_zorder": False},
    )

    for axis in [axes.xaxis, axes.yaxis, axes.zaxis]:
        axis._axinfo["axisline"]["linewidth"] = 10
        axis._axinfo["axisline"]["color"] = "b"
        axis._axinfo["grid"]["linewidth"] = 0.8
        axis._axinfo["grid"]["linestyle"] = "--"
        axis._axinfo["grid"]["color"] = "#d1d1d1"
        axis.set_pane_color((1, 1, 1))

    axes.view_init(10, 15)
    axes.set_xlabel(r"$x$ [km]", fontsize=18, labelpad=5, fontweight="bold")
    axes.set_ylabel(r"$Time$ [min]", fontsize=18, labelpad=5, fontweight="bold")
    axes.set_zlabel(zlabel, rotation=90, fontsize=18, labelpad=4, fontweight="bold")
    axes.zaxis.set_rotate_label(False)
    axes.zaxis.label.set_clip_on(False)

    x_ticks = np.array([X_km[0], 0.5 * (X_km[0] + X_km[-1]), X_km[-1]], dtype=float)
    y_ticks = np.array([t_min[0], 0.5 * (t_min[0] + t_min[-1]), t_min[-1]], dtype=float)
    z_min, z_max = float(np.nanmin(Z)), float(np.nanmax(Z))
    z_ticks = np.linspace(z_min, z_max, 5)

    axes.set_xticks(x_ticks)
    axes.set_yticks(y_ticks)
    axes.set_zticks(z_ticks)
    axes.set_xticklabels(_format_three_ticklabels(x_ticks), fontweight="bold", fontsize=18)
    axes.set_yticklabels(_format_three_ticklabels(y_ticks), fontweight="bold", fontsize=18)
    axes.set_zticklabels(_format_z_ticklabels(z_ticks), fontweight="bold", fontsize=18)

    rstride = max(1, Z.shape[0] // 6)
    axes.plot_surface(
        meshx,
        mesht,
        Z,
        edgecolor="black",
        lw=0.8,
        rstride=rstride,
        cstride=5,
        alpha=1,
        color="white",
        shade=False,
        rasterized=True,
        antialiased=True,
    )

    # Downstream boundary highlight curve remains removed as in the target program.

    axes.set_title(title, fontsize=14)
    fig.savefig(save_path, dpi=600, bbox_inches="tight", pad_inches=0.35)
    print(f"3D surface saved to: {save_path}")
    plt.show()
    plt.close(fig)






def print_final(name, res):
    H = res["H"]
    Q = res["Q"]
    Hbar = res["Hbar"]
    Qbar = res["Qbar"]
    print("")
    print("=" * 90)
    print(f"{name} final")
    print("=" * 90)
    print(f"final H(0) = {H[-1,0]:.6f}, target = {Hbar[0]:.6f}, error = {H[-1,0]-Hbar[0]:.6f}")
    print(f"final H(L) = {H[-1,-1]:.6f}, target = {Hbar[-1]:.6f}, error = {H[-1,-1]-Hbar[-1]:.6f}")
    print(f"final Q(0) = {Q[-1,0]:.6f}, target = {Qbar[0]:.6f}, error = {Q[-1,0]-Qbar[0]:.6f}")
    print(f"final Q(L) = {Q[-1,-1]:.6f}, target = {Qbar[-1]:.6f}, error = {Q[-1,-1]-Qbar[-1]:.6f}")
    print(f"total_reward = {res['total_reward']:.6f}")


def print_final_error_values(name, res):
    """
    Print the final-time values of H-Hbar and Q-Qbar at every spatial grid point.
    """
    X = np.asarray(res["X"], dtype=float)
    H_err_final = np.asarray(res["H"][-1] - res["Hbar"], dtype=float)
    Q_err_final = np.asarray(res["Q"][-1] - res["Qbar"], dtype=float)

    print("")
    print("=" * 90)
    print(f"{name} final H-Hbar and Q-Qbar values")
    print("=" * 90)
    print("index, x[m], H-Hbar[m], Q-Qbar[m^3/s]")
    for i, (x_i, h_i, q_i) in enumerate(zip(X, H_err_final, Q_err_final)):
        print(f"{i:4d}, {x_i:12.6f}, {h_i: .10e}, {q_i: .10e}")





# ============================================================
# Controller-comparison comparison utilities
# ============================================================

LABEL_NEW_MODEL = "TDPSAC"

# Retain the previous label constants so the common style helper can be reused
# when additional checkpoints are added later.
LABEL_SAC_CURVE = "SAC"
LABEL_FP_SAC = "SAC-FPC"
LABEL_SAC = "DO-SAC"
LABEL_HYBRID = "DPSAC"
LABEL_DFP_DON_SAC = "TDPSAC-FPC"
LABEL_HYBRID_FREEZE = LABEL_NEW_MODEL

# Fixed order for 2D legends, terminal summaries, evidence tables, and saved
# controller arrays.  This is independent of the order in which models are run.
CONTROLLER_DISPLAY_ORDER = [
    LABEL_NEW_MODEL,
    LABEL_DFP_DON_SAC,
    LABEL_HYBRID,
    LABEL_SAC,
    LABEL_FP_SAC,
    LABEL_SAC_CURVE,
]


def _ordered_results(results: Dict[str, dict]) -> Dict[str, dict]:
    """Return a dictionary in the requested display/table order."""
    ordered = {
        label: results[label]
        for label in CONTROLLER_DISPLAY_ORDER
        if label in results
    }
    # Preserve any future additional controllers after the six fixed entries.
    for label, res in results.items():
        if label not in ordered:
            ordered[label] = res
    return ordered


def _safe_load_hybrid_model(model_path: str, env):
    """
    Load a HybridSAC checkpoint for evaluation.

    The HybridSAC training script may save optimizers with special parameter
    groups for DeepONet. EvalHybridSAC rebuilds the optimizer structure. If the
    optimizer groups still do not match, an eval-only copy without optimizer
    states is created and loaded. This does not change the policy weights.
    """
    try:
        return EvalHybridSAC.load(model_path, env=env)
    except Exception as e:
        msg = str(e)
        if ("parameter group" in msg) or ("Names of parameters" in msg) or ("optimizer" in msg):
            print("Hybrid load with optimizer states failed; trying eval-only checkpoint.")
            print("Original error:", repr(e))
            eval_path = strip_optimizer_states_from_sb3_model(model_path)
            return EvalHybridSAC.load(eval_path, env=env)
        raise



def _checkpoint_exists(model_path: str) -> bool:
    """
    Return True only when model_path resolves to a valid SB3 ZIP checkpoint.

    This deliberately rejects arbitrary existing files such as Jupyter's
    kernel-*.json connection file. Stable-Baselines3 checkpoints are ZIP files,
    although callers may pass the path either with or without the .zip suffix.
    """
    if model_path is None or str(model_path).strip() == "":
        return False

    path = Path(str(model_path)).expanduser()

    candidates = [path]
    if path.suffix.lower() != ".zip":
        candidates.append(path.with_suffix(".zip"))

    for candidate in candidates:
        if candidate.is_file() and zipfile.is_zipfile(candidate):
            return True

    return False


def _safe_load_generic_sac_model(model_path: str, env):
    """
    Load a custom SAC-family checkpoint without assuming one exact policy layout.

    This is used for TDPSAC-FPC and SAC-FPC because their policy
    architectures may differ. The loader first asks SB3 to reconstruct the
    policy class stored in the checkpoint. If optimizer parameter groups are
    incompatible during evaluation, it creates an evaluation-only copy with
    optimizer states removed and retries.

    Policy, actor, critic, normalization, and saved alpha weights are preserved.
    """
    errors = []

    try:
        return SAC.load(model_path, env=env)
    except Exception as exc:
        errors.append(("SAC.load", exc))

    try:
        eval_path = strip_optimizer_states_from_sb3_model(model_path)
        return SAC.load(eval_path, env=env)
    except Exception as exc:
        errors.append(("SAC.load(eval-only)", exc))

    # Final fallback for checkpoints trained with the HybridSAC subclass used
    # in this file.
    try:
        return _safe_load_hybrid_model(model_path, env=env)
    except Exception as exc:
        errors.append(("_safe_load_hybrid_model", exc))

    details = "\n".join(f"  {name}: {error!r}" for name, error in errors)
    raise RuntimeError(
        f"Unable to load SAC-family checkpoint: {model_path}\n"
        f"Tried the generic and HybridSAC evaluation loaders:\n{details}"
    )

def run_policy_exact(
    model_path: str,
    model_kind: str,
    label: str,
    seed: int = 123,
    H_left: float = TEST_H_LEFT,
    H_right: float = TEST_H_RIGHT,
    Q_value: float = TEST_Q_VALUE,
    dH_source=None,
):
    """
    Run one learned controller in the CAC environment.

    model_kind:
        "sac"       -> ordinary raw-HQ SAC.load(...)
        "sac_error" -> old SAC checkpoint using normalized H/Q errors
        "hybrid"    -> EvalHybridSAC.load(...), for the existing hybrid checkpoints
        "auto"      -> reconstruct the saved SAC policy class automatically; used
                       for TDPSAC-FPC and SAC-FPC

    Every learned controller sees the same physical PDE reference, initial
    state, and raw physical H/Q observation interface in the current comparison.
    The reset state is recorded first at t=0, then later states are recorded
    after env.step().
    """
    model_kind_lower = model_kind.lower()
    if model_kind_lower == "sac_error":
        env, X, Hbar, Qbar, p = make_env_sac_error_state(
            H_left=H_left,
            H_right=H_right,
            Q_value=Q_value,
            seed=seed,
            dH_source=dH_source,
        )
    else:
        env, X, Hbar, Qbar, p = make_env_sac(
            H_left=H_left,
            H_right=H_right,
            Q_value=Q_value,
            seed=seed,
            dH_source=dH_source,
        )

    if model_kind_lower in {"sac", "sac_error"}:
        model = SAC.load(model_path, env=env)
    elif model_kind_lower == "hybrid":
        model = _safe_load_hybrid_model(model_path, env=env)
    elif model_kind_lower == "auto":
        model = _safe_load_generic_sac_model(model_path, env=env)
    else:
        raise ValueError(
            f"Unknown model_kind: {model_kind}. Use 'sac', 'sac_error', "
            "'hybrid', or 'auto'."
        )

    obs, _ = env.reset(seed=seed)
    obs = np.asarray(obs, dtype=np.float32).reshape(-1)
    if obs.size != RAW_STATE_DIM:
        raise RuntimeError(
            f"The reset observation has dimension {obs.size}; expected "
            f"{RAW_STATE_DIM} for the H/Q or H/Q-error interface."
        )

    observation_mode_used = (
        "normalized_error" if model_kind_lower == "sac_error" else "raw"
    )
    print(
        f"{label} observation mode: {observation_mode_used}, "
        f"dimension={obs.size}"
    )

    H_hist, Q_hist, U_hist, t_hist, reward_hist = [], [], [], [], []

    # Record the reset state before any controller acts.
    # This makes the learned controllers start from
    # exactly the same L2 error at t=0.
    H0, Q0 = env.get_current_HQ()
    H_hist.append(H0.copy())
    Q_hist.append(Q0.copy())
    U_hist.append(np.nan)
    t_hist.append(0.0)
    reward_hist.append(0.0)

    previous_u_filtered = float(U_ref)

    done = False
    while not done:
        action, _ = model.predict(obs, deterministic=True)

        action_filtered, u_raw, previous_u_filtered = lowpass_filter_rl_control(
            action,
            previous_u_filtered,
            alpha=RL_CONTROL_FILTER_ALPHA,
        )

        obs, reward, terminated, truncated, info = env.step(action_filtered)

        H, Q = env.get_current_HQ()

        H_hist.append(H.copy())
        Q_hist.append(Q.copy())
        U_hist.append(float(env.last_control))
        t_hist.append(float(info["time"]))
        reward_hist.append(float(reward))

        done = terminated or truncated

    env.close()

    H_hist = np.asarray(H_hist)
    Q_hist = np.asarray(Q_hist)
    U_hist = np.asarray(U_hist)
    t_hist = np.asarray(t_hist)
    reward_hist = np.asarray(reward_hist)

    H_err = H_hist - Hbar[None, :]
    Q_err = Q_hist - Qbar[None, :]

    return {
        "label": label,
        "model_kind": model_kind,
        "observation_mode": observation_mode_used,
        "X": X,
        "Hbar": Hbar,
        "Qbar": Qbar,
        "t": t_hist,
        "H": H_hist,
        "Q": Q_hist,
        "U": U_hist,
        "reward": reward_hist,
        "H_l2": np.sqrt(np.mean(H_err ** 2, axis=1)),
        "Q_l2": np.sqrt(np.mean(Q_err ** 2, axis=1)),
        "final_H_minus_Hbar": H_err[-1],
        "final_Q_minus_Qbar": Q_err[-1],
        "total_reward": float(np.sum(reward_hist)),
    }




def _time_min(res):
    return np.asarray(res["t"], dtype=float) / 60.0


def _get_l2(res, key):
    return _compute_spatial_l2_over_time(res, key)


def _get_linf(res, key):
    """Return the spatial L-infinity tracking-error norm at each time instant."""
    state = np.asarray(res[key], dtype=float)
    reference_key = "Hbar" if key == "H" else "Qbar"
    reference = np.asarray(res[reference_key], dtype=float).reshape(1, -1)
    if state.ndim != 2 or reference.shape[1] != state.shape[1]:
        return np.asarray([], dtype=float)
    return np.max(np.abs(state - reference), axis=1)


def set_size(width, fraction=1, ncols=1, nrows=1, height_add=0):
    """Figure size helper copied from the third comparison-program style."""
    inches_per_pt = 1 / 72.27
    fig_width = width * inches_per_pt * fraction
    fig_height = fig_width * 0.618 * (nrows / ncols) + height_add
    return fig_width, fig_height



def _line_style(label):
    """Return the fixed colors and line styles used by the old comparison program."""
    if label == LABEL_SAC_CURVE:  # SAC
        return dict(color="purple", linestyle="-", linewidth=2.0, alpha=0.8)
    if label == LABEL_FP_SAC:  # SAC-FPC
        return dict(
            color="saddlebrown",
            linestyle=(0, (6, 2, 1.5, 2)),
            linewidth=2.2,
            alpha=0.95,
        )
    if label == LABEL_SAC:  # DO-SAC
        return dict(color="blue", linestyle="-.", linewidth=2.0, alpha=0.8)
    if label == LABEL_HYBRID:  # DPSAC
        return dict(color="green", linestyle="--", linewidth=2.0, alpha=0.8)
    if label == LABEL_DFP_DON_SAC:  # TDPSAC-FPC
        return dict(color="black", linestyle="--", linewidth=2.2, alpha=0.95)
    if label == LABEL_HYBRID_FREEZE:  # TDPSAC
        return dict(color="red", linestyle="-", linewidth=2.0, alpha=0.8)
    return dict(linewidth=2.0, alpha=0.8)




def _legend_kwargs_two_columns(loc="upper center", bbox_to_anchor=None):
    """
    Shared legend style for the displayed methods.
    """
    kwargs = dict(
        fontsize=11,
        frameon=True,
        shadow=True,
        ncol=2,
        columnspacing=1.2,
        handlelength=2.8,
        borderaxespad=0.6,
    )
    kwargs["loc"] = loc
    if bbox_to_anchor is not None:
        kwargs["bbox_to_anchor"] = bbox_to_anchor
    return kwargs


def _apply_ordered_two_column_legend(
    ax,
    *,
    loc="best",
    fontsize=11,
    frameon=True,
    shadow=True,
    columnspacing=1.2,
    handlelength=2.8,
    borderaxespad=0.6,
):
    """Draw a two-column legend ordered from top to bottom by column.

    The displayed order is read vertically rather than horizontally:

        left column:  SAC, SAC-FPC, DO-SAC
        right column: DPSAC, TDPSAC-FPC, TDPSAC
    """
    handles, labels = ax.get_legend_handles_labels()
    if not handles:
        return None

    pair_by_label = {label: (handle, label) for handle, label in zip(handles, labels)}
    ordered_pairs = [
        pair_by_label[label]
        for label in CONTROLLER_DISPLAY_ORDER
        if label in pair_by_label
    ]
    ordered_pairs.extend(
        (handle, label)
        for handle, label in zip(handles, labels)
        if label not in CONTROLLER_DISPLAY_ORDER
    )

    ordered_handles = [pair[0] for pair in ordered_pairs]
    ordered_labels = [pair[1] for pair in ordered_pairs]

    # Matplotlib fills a multi-column legend from top to bottom in each column.
    # Passing CONTROLLER_DISPLAY_ORDER directly therefore gives the requested
    # vertical ordering without row-wise repacking.
    return ax.legend(
        ordered_handles,
        ordered_labels,
        fontsize=fontsize,
        frameon=frameon,
        shadow=shadow,
        loc=loc,
        ncol=2,
        columnspacing=columnspacing,
        handlelength=handlelength,
        borderaxespad=borderaxespad,
    )


def _style_axes_second(ax, xlabel=None, ylabel=None, legend=True, legend_loc="best"):
    """2D axis style matched to the first program."""
    if xlabel is not None:
        ax.set_xlabel(xlabel, fontsize=18, fontweight="bold", labelpad=10)
    if ylabel is not None:
        ax.set_ylabel(ylabel, fontsize=18, fontweight="bold", labelpad=8)
    ax.tick_params(axis="both", labelsize=18)
    for tick in ax.get_xticklabels() + ax.get_yticklabels():
        tick.set_fontweight("bold")
    ax.grid(True, linestyle="--", alpha=0.8)
    if legend:
        _apply_ordered_two_column_legend(ax, loc=legend_loc, fontsize=11)


# Keep old function name for compatibility with the rest of the script.
def _style_axes(ax, xlabel=None, ylabel=None, legend=True, legend_loc="best"):
    _style_axes_second(ax, xlabel=xlabel, ylabel=ylabel, legend=legend, legend_loc=legend_loc)


def _safe_label_for_plot(label):
    """Keep algorithm names exactly the same as the second program."""
    return label


def _common_time_end_min(results: Dict[str, dict]) -> float:
    """Return the common end time in minutes across all results."""
    ends = []
    for res in results.values():
        t = _time_min(res)
        if len(t) > 0 and np.any(np.isfinite(t)):
            ends.append(float(np.nanmax(t)))
    return min(ends) if ends else 0.0


def _style_inset_axes(ax_inset):
    """Inset style copied from the second program."""
    ax_inset.grid(True, linestyle="--", alpha=0.8)
    ax_inset.tick_params(axis="both", labelsize=14)


def _add_second_style_inset(
    ax,
    results: Dict[str, dict],
    y_getter,
    ref_value=None,
    zoom_ratio: float = 0.75,
    bounds=(0.55, 0.15, 0.40, 0.40),
):
    """
    Add a local zoom inset with the same style as the second program:
        ax.inset_axes([...]) + ax.indicate_inset_zoom(...).

    This function only changes plotting. It does not change model loading,
    controller output, reward, rollout data, or saved numerical results.
    """
    min_t_end = _common_time_end_min(results)
    if min_t_end <= 0:
        return None

    zoom_start = float(zoom_ratio) * min_t_end
    ax_inset = ax.inset_axes(list(bounds))

    y_all = []
    for label, res in results.items():
        style = _line_style(label)
        t = _time_min(res)
        y = np.asarray(y_getter(res), dtype=float).reshape(-1)
        n = min(len(t), len(y))
        tt = t[:n]
        yy = y[:n]
        mask = (tt >= zoom_start) & (tt <= min_t_end)
        if np.any(mask):
            ax_inset.plot(tt[mask], yy[mask], **style)
            y_all.extend(yy[mask][np.isfinite(yy[mask])].tolist())

    if ref_value is not None:
        ax_inset.axhline(ref_value, linestyle="--", linewidth=1.4, color="dimgray", alpha=0.95)
        if np.isfinite(ref_value):
            y_all.append(float(ref_value))

    if y_all:
        y_min, y_max = float(np.min(y_all)), float(np.max(y_all))
        margin = (y_max - y_min) * 0.1 if y_max != y_min else 0.1
        ax_inset.set_ylim(y_min - margin, y_max + margin)
    ax_inset.set_xlim(zoom_start, min_t_end)
    _style_inset_axes(ax_inset)

    # Same guide-line style as the second program.
    try:
        ax.indicate_inset_zoom(ax_inset, edgecolor="black", linewidth=1.5)
    except Exception:
        pass
    return ax_inset


# Backward-compatible alias for old calls.
def _add_end_time_inset(
    ax,
    results: Dict[str, dict],
    y_getter,
    ref_value=None,
    zoom_ratio: float = 0.75,
    bounds=(0.55, 0.15, 0.40, 0.40),
):
    return _add_second_style_inset(
        ax,
        results,
        y_getter,
        ref_value=ref_value,
        zoom_ratio=zoom_ratio,
        bounds=bounds,
    )


def plot_l2_norm_four(results: Dict[str, dict], save_dir: str, tag: str):
    """Controller-comparison L2 plots with the same 2D style as the second program.

    H-Hbar and Q-Qbar are drawn in two separate figures. Other logic remains unchanged.
    """
    results = _ordered_results(results)
    # --------------------------------------------------------
    # Figure 1: ||H-Hbar||_L2
    # --------------------------------------------------------
    fig_h, ax_h = plt.subplots(
        figsize=set_size(433, 0.99, ncols=1, nrows=1, height_add=1.0),
        dpi=600,
    )

    for label, res in results.items():
        style = _line_style(label)
        plot_label = _safe_label_for_plot(label)
        ax_h.plot(_time_min(res), _get_l2(res, "H"), label=plot_label, **style)

    ax_h.axhline(y=0, color="dimgray", linestyle="--", alpha=0.95, linewidth=1.8)
    _style_axes_second(
        ax_h,
        xlabel=r"$Time$ [min]",
        ylabel=r"$\Vert H-\bar H\Vert_{L_2}$ [m]",
        legend=True,
    )

    fig_h.tight_layout()
    path_h = os.path.join(save_dir, f"{tag}_six_algorithms_H_L2_norm_error_second_style.png")
    fig_h.savefig(path_h, dpi=600, bbox_inches="tight")
    print(f"Saved six-controller H L2 figure to: {path_h}")
    plt.show()

    # --------------------------------------------------------
    # Figure 2: ||Q-Qbar||_L2
    # --------------------------------------------------------
    fig_q, ax_q = plt.subplots(
        figsize=set_size(433, 0.99, ncols=1, nrows=1, height_add=1.0),
        dpi=600,
    )

    for label, res in results.items():
        style = _line_style(label)
        plot_label = _safe_label_for_plot(label)
        ax_q.plot(_time_min(res), _get_l2(res, "Q"), label=plot_label, **style)

    ax_q.axhline(y=0, color="dimgray", linestyle="--", alpha=0.95, linewidth=1.8)
    _style_axes_second(
        ax_q,
        xlabel=r"$Time$ [min]",
        ylabel=r"$\Vert Q-\bar Q\Vert_{L_2}$ [m$^3$/s]",
        legend=True,
    )

    # Only Q-Qbar keeps local zoom, as previously requested.
    _add_second_style_inset(
        ax_q,
        results,
        lambda res: _get_l2(res, "Q"),
        ref_value=0.0,
        zoom_ratio=0.75,
        bounds=(0.55, 0.15, 0.40, 0.40),
    )

    fig_q.tight_layout()
    path_q = os.path.join(save_dir, f"{tag}_six_algorithms_Q_L2_norm_error_second_style.png")
    fig_q.savefig(path_q, dpi=600, bbox_inches="tight")
    print(f"Saved six-controller Q L2 figure to: {path_q}")
    plt.show()

    # Save each L2 time series.
    for label, res in results.items():
        safe_label = label.replace("+", "_plus_").replace(" ", "_").replace("/", "_")
        np.savetxt(
            os.path.join(save_dir, f"{tag}_{safe_label}_H_L2.dat"),
            np.column_stack([_time_min(res), _get_l2(res, "H")]),
            header="time_min H_L2",
        )
        np.savetxt(
            os.path.join(save_dir, f"{tag}_{safe_label}_Q_L2.dat"),
            np.column_stack([_time_min(res), _get_l2(res, "Q")]),
            header="time_min Q_L2",
        )


def _pad_to_length_1d(data, target_len):
    data = np.asarray(data, dtype=float).reshape(-1)
    if len(data) < target_len:
        return np.pad(data, (0, target_len - len(data)), mode="edge")
    return data[:target_len]


def plot_control_input_four(results: Dict[str, dict], save_dir: str, tag: str):
    """U_L(t) plot with the same 2D style as the second program; no inset."""
    results = _ordered_results(results)
    fig, ax = plt.subplots(
        figsize=set_size(433, 0.99, ncols=1, nrows=1, height_add=1.0),
        dpi=600,
    )

    min_t_end = min(float(np.nanmax(_time_min(res))) for res in results.values())
    for label, res in results.items():
        style = _line_style(label)
        t = _time_min(res)
        U = np.asarray(res["U"], dtype=float)
        n = min(len(t), len(U))
        ax.plot(t[:n], U[:n], label=_safe_label_for_plot(label), **style)

    ax.axhline(y=U_ref, color="dimgray", linestyle="--", alpha=0.95, linewidth=1.8)
    ax.set_xlim(0, min_t_end)
    _style_axes_second(ax, xlabel=r"$Time$ [min]", ylabel=r"$U_L(t)$ [m]", legend=True)

    fig.tight_layout()
    path = os.path.join(save_dir, f"{tag}_six_algorithms_control_input_second_style.png")
    fig.savefig(path, dpi=600, bbox_inches="tight")
    print(f"Saved six-controller control input figure to: {path}")
    plt.show()


def plot_boundary_comparison_four(results: Dict[str, dict], save_dir: str, tag: str):
    """Boundary H/Q comparison with the same 2D style as the second program."""
    results = _ordered_results(results)
    first = next(iter(results.values()))
    Hbar = first["Hbar"]
    Qbar = first["Qbar"]

    fig, axes = plt.subplots(
        2, 2,
        figsize=set_size(780, 0.99, ncols=2, nrows=2, height_add=1.1),
        dpi=600,
        sharex=True,
    )
    axes = axes.ravel()

    specs = [
        ("H", 0, r"$H(0,t)$ [m]", Hbar[0]),
        ("H", -1, r"$H(L,t)$ [m]", Hbar[-1]),
        ("Q", 0, r"$Q(0,t)$ [m$^3$/s]", Qbar[0]),
        ("Q", -1, r"$Q(L,t)$ [m$^3$/s]", Qbar[-1]),
    ]

    for ax, (var, idx, ylabel, ref) in zip(axes, specs):
        for label, res in results.items():
            style = _line_style(label)
            y = np.asarray(res[var])[:, idx]
            t = _time_min(res)
            n = min(len(t), len(y))
            ax.plot(t[:n], y[:n], label=_safe_label_for_plot(label), **style)

        ax.axhline(y=ref, color="dimgray", linestyle="--", alpha=0.95, linewidth=1.8)
        _style_axes_second(ax, ylabel=ylabel, legend=False)

        # Only Q(L,t) keeps local zoom, as previously requested.
        if var == "Q" and idx == -1:
            _add_second_style_inset(
                ax,
                results,
                lambda res, var=var, idx=idx: np.asarray(res[var])[:, idx],
                ref_value=float(ref),
                zoom_ratio=0.75,
                bounds=(0.55, 0.52, 0.40, 0.40),
            )

    _style_axes_second(axes[2], xlabel=r"$Time$ [min]", ylabel=specs[2][2], legend=False)
    _style_axes_second(axes[3], xlabel=r"$Time$ [min]", ylabel=specs[3][2], legend=False)
    _apply_ordered_two_column_legend(
        axes[0],
        fontsize=14,
        frameon=True,
        shadow=True,
        loc="best",
        columnspacing=1.2,
        handlelength=2.8,
        borderaxespad=0.6,
    )

    fig.tight_layout()
    path = os.path.join(save_dir, f"{tag}_six_algorithms_boundary_comparison_second_style.png")
    fig.savefig(path, dpi=600, bbox_inches="tight")
    print(f"Saved six-controller boundary figure to: {path}")
    plt.show()


def _format_three_ticklabels(values):
    """Format three x/y tick labels in the same style as the first program's 3D figures."""
    values = np.asarray(values, dtype=float)
    labels = []
    for value in values:
        if abs(value) >= 10:
            labels.append(f"{value:.0f}")
        elif abs(value) >= 1:
            labels.append(f"{value:.1f}")
        else:
            labels.append(f"{value:.2f}")
    return labels

def _format_z_ticklabels(values):
    """Format z-axis tick labels in the same style as the first program's 3D figures."""
    values = np.asarray(values, dtype=float)
    span = float(np.nanmax(values) - np.nanmin(values)) if len(values) > 0 else 0.0
    if span >= 10:
        return [f"{v:.0f}" for v in values]
    if span >= 1:
        return [f"{v:.1f}" for v in values]
    return [f"{v:.2f}" for v in values]

def _surface_plot_second_style(res, var: str, title: str, save_path: str):
    """Draw one independent 3D surface using the same style as the first program."""
    X = np.asarray(res["X"], dtype=float)
    t_min = _time_min(res)
    Z = np.asarray(res[var], dtype=float)

    spatial = X / 1000.0
    meshx, mesht = np.meshgrid(spatial, t_min)

    fig = plt.figure(figsize=(8.8, 6.6), dpi=600)
    subfigs = fig.subfigures(nrows=1, ncols=1, hspace=0)
    subfig = subfigs
    subfig.subplots_adjust(left=0.18, bottom=0.08, right=0.98, top=0.96)

    axes = subfig.subplots(
        nrows=1,
        ncols=1,
        subplot_kw={"projection": "3d", "computed_zorder": False},
    )

    for axis in [axes.xaxis, axes.yaxis, axes.zaxis]:
        axis._axinfo["axisline"]["linewidth"] = 10
        axis._axinfo["axisline"]["color"] = "b"
        axis._axinfo["grid"]["linewidth"] = 0.8
        axis._axinfo["grid"]["linestyle"] = "--"
        axis._axinfo["grid"]["color"] = "#d1d1d1"
        axis.set_pane_color((1, 1, 1))

    axes.view_init(10, 15)
    axes.set_xlabel(r"$x$ [km]", fontsize=18, labelpad=5, fontweight="bold")
    axes.set_ylabel(r"$Time$ [min]", fontsize=18, labelpad=5, fontweight="bold")
    if var == "H":
        zlabel = r"$H(x,t)$ [m]"
    else:
        zlabel = r"$Q(x,t)$ [m$^3$/s]"
    axes.set_zlabel(zlabel, rotation=90, fontsize=18, labelpad=4, fontweight="bold")
    axes.zaxis.set_rotate_label(False)
    axes.zaxis.label.set_clip_on(False)

    x_ticks = np.array([spatial[0], 0.5 * (spatial[0] + spatial[-1]), spatial[-1]], dtype=float)
    y_ticks = np.array([t_min[0], 0.5 * (t_min[0] + t_min[-1]), t_min[-1]], dtype=float)
    z_min, z_max = float(np.nanmin(Z)), float(np.nanmax(Z))
    z_ticks = np.linspace(z_min, z_max, 5)

    axes.set_xticks(x_ticks)
    axes.set_yticks(y_ticks)
    axes.set_zticks(z_ticks)
    axes.set_xticklabels(_format_three_ticklabels(x_ticks), fontweight="bold", fontsize=18)
    axes.set_yticklabels(_format_three_ticklabels(y_ticks), fontweight="bold", fontsize=18)
    axes.set_zticklabels(_format_z_ticklabels(z_ticks), fontweight="bold", fontsize=18)

    rstride = max(1, Z.shape[0] // 6)
    axes.plot_surface(
        meshx,
        mesht,
        Z,
        edgecolor="black",
        lw=0.8,
        rstride=rstride,
        cstride=5,
        alpha=1,
        color="white",
        shade=False,
        rasterized=True,
        antialiased=True,
    )

    # Red downstream boundary curve remains removed as in the target program.

    axes.set_title(_safe_label_for_plot(title), fontsize=14)
    if save_path is not None:
        fig.savefig(save_path, dpi=600, bbox_inches="tight", pad_inches=0.35)
        print(f"3D surface saved to: {save_path}")
    plt.show()
    plt.close(fig)

def plot_3d_surfaces_four(results: Dict[str, dict], save_dir: str, tag: str):
    """
    Draw independent 3D figures for each algorithm and each state variable,
    using the same style as the second program. No combined 3D figure is drawn.
    """
    results = _ordered_results(results)
    for var in ["H", "Q"]:
        for label, res in results.items():
            safe_label = label.replace("+", "_plus_").replace(" ", "_").replace("/", "_")
            path = os.path.join(save_dir, f"{tag}_{safe_label}_{var}_3D_surface_second_style.png")
            _surface_plot_second_style(res, var, label, path)


def save_npz_four(results: Dict[str, dict], save_dir: str, tag: str):
    results = _ordered_results(results)
    arrays = {}
    for label, res in results.items():
        safe = label.replace("+", "_plus_").replace(" ", "_").replace("/", "_")
        arrays[f"t_{safe}"] = res["t"]
        arrays[f"H_{safe}"] = res["H"]
        arrays[f"Q_{safe}"] = res["Q"]
        arrays[f"U_{safe}"] = res["U"]
        arrays[f"reward_{safe}"] = res["reward"]
        arrays[f"total_reward_{safe}"] = np.asarray(res["total_reward"])

    first = next(iter(results.values()))
    arrays["X"] = first["X"]
    arrays["Hbar"] = first["Hbar"]
    arrays["Qbar"] = first["Qbar"]

    e_H0_rel, e_Q0_rel = _initial_relative_deviation_percentages_from_result(first)
    arrays["e_H0_rel_pct"] = np.asarray(e_H0_rel)
    arrays["e_Q0_rel_pct"] = np.asarray(e_Q0_rel)

    path = os.path.join(save_dir, f"{tag}_controller_comparison.npz")
    np.savez(path, **arrays)
    print(f"Saved six-controller data to: {path}")



def print_final_summary_four(results: Dict[str, dict]):
    results = _ordered_results(results)
    print("\n" + "=" * 90)
    print("Final summary of controller comparison")
    print("=" * 90)
    for label, res in results.items():
        H = np.asarray(res["H"])
        Q = np.asarray(res["Q"])
        Hbar = np.asarray(res["Hbar"])
        Qbar = np.asarray(res["Qbar"])
        H_l2_final = _get_l2(res, "H")[-1]
        Q_l2_final = _get_l2(res, "Q")[-1]

        print(f"\n[{label}]")
        print(f"final H(0) = {H[-1,0]:.6f}, target = {Hbar[0]:.6f}, error = {H[-1,0]-Hbar[0]:.6f}")
        print(f"final H(L) = {H[-1,-1]:.6f}, target = {Hbar[-1]:.6f}, error = {H[-1,-1]-Hbar[-1]:.6f}")
        print(f"final Q(0) = {Q[-1,0]:.6f}, target = {Qbar[0]:.6f}, error = {Q[-1,0]-Qbar[0]:.6f}")
        print(f"final Q(L) = {Q[-1,-1]:.6f}, target = {Qbar[-1]:.6f}, error = {Q[-1,-1]-Qbar[-1]:.6f}")
        print(f"final ||H-Hbar||_L2 = {H_l2_final:.6f} m")
        print(f"final ||Q-Qbar||_L2 = {Q_l2_final:.6f} m^3/s")
        print(f"total_reward = {res['total_reward']:.6f}")



# ============================================================
# Evidence-summary table for the selected single scenario
# ============================================================

# Keep the steady-state evaluation window identical to the local enlarged plot.
# The inset starts at 0.75 of the common end time, so the table uses the final 25%.
LOCAL_ZOOM_START_RATIO = 0.75

# Unified convergence thresholds are the initial L2 tracking errors:
# E_H(t) <= E_H(0) and E_Q(t) <= E_Q(0).
# Keep this legacy constant only for call-signature compatibility; it is not
# used in the convergence threshold calculation below.
SETTLING_BAND_FRACTION = 0.05


def _safe_metric_value(value):
    """Return a finite float when possible; otherwise return np.nan."""
    try:
        value = float(value)
    except Exception:
        return np.nan
    if not np.isfinite(value):
        return np.nan
    return value


def _format_metric(value, precision=6):
    """Format one metric for terminal table output."""
    value = _safe_metric_value(value)
    if not np.isfinite(value):
        return "—"
    return f"{value:.{precision}f}"


def _time_average_on_interval(
    t_min: np.ndarray,
    values: np.ndarray,
    start_min: float,
    end_min: float,
) -> float:
    """
    Calculate the time average on the same interval displayed by the local inset.

    The mask is intentionally identical to the inset logic:
        start_min <= t <= end_min.
    """
    t_min = np.asarray(t_min, dtype=float).reshape(-1)
    values = np.asarray(values, dtype=float).reshape(-1)

    n = min(t_min.size, values.size)
    if n == 0:
        return np.nan

    t_min = t_min[:n]
    values = values[:n]

    mask = (
        np.isfinite(t_min)
        & np.isfinite(values)
        & (t_min >= float(start_min))
        & (t_min <= float(end_min))
    )

    t_window = t_min[mask]
    value_window = values[mask]

    if value_window.size == 0:
        return np.nan

    if value_window.size == 1:
        return float(value_window[0])

    duration = float(t_window[-1] - t_window[0])
    if duration <= 0.0:
        return float(np.mean(value_window))

    return float(
        np.trapezoid(value_window, t_window)
        / duration
    )



def _normalized_spatial_l2_state_over_time(res: dict, key: str) -> np.ndarray:
    """
    Return the normalized spatial L2 norm of the state itself:

        ||v(.,t)||_{L2,n}
            = sqrt((1/L) * integral_0^L v(x,t)^2 dx)
            = sqrt(integral_0^1 v(s,t)^2 ds),  s=x/L.

    This is different from _get_l2(), which computes the tracking-error norm
    ||v-vbar||_{L2,n}.
    """
    X = np.asarray(res.get("X", []), dtype=float).reshape(-1)
    state = np.asarray(res.get(key, []), dtype=float)

    if X.size < 2 or state.ndim != 2 or state.shape[1] != X.size:
        return np.asarray([], dtype=float)

    s = X / float(X[-1])
    return np.sqrt(np.trapezoid(state ** 2, s, axis=1))


def _initial_relative_deviation_percentages_from_arrays(
    X: np.ndarray,
    H0: np.ndarray,
    Q0: np.ndarray,
    Hbar: np.ndarray,
    Qbar: np.ndarray,
) -> Tuple[float, float]:
    """
    Compute the shared initial relative deviations:

        e_H0_rel = ||H0-Hbar||_L2 / ||Hbar||_L2 * 100%
        e_Q0_rel = ||Q0-Qbar||_L2 / ||Qbar||_L2 * 100%.

    The same normalized spatial coordinate s=x/L is used in every norm.
    Because numerator and denominator use the same convention, the ratio is
    independent of whether one writes the normalized or unnormalized L2 norm.
    """
    X = np.asarray(X, dtype=float).reshape(-1)
    H0 = np.asarray(H0, dtype=float).reshape(-1)
    Q0 = np.asarray(Q0, dtype=float).reshape(-1)
    Hbar = np.asarray(Hbar, dtype=float).reshape(-1)
    Qbar = np.asarray(Qbar, dtype=float).reshape(-1)

    if (
        X.size < 2
        or H0.size != X.size
        or Q0.size != X.size
        or Hbar.size != X.size
        or Qbar.size != X.size
    ):
        return np.nan, np.nan

    s = X / float(X[-1])

    H_err_norm = float(np.sqrt(np.trapezoid((H0 - Hbar) ** 2, s)))
    Q_err_norm = float(np.sqrt(np.trapezoid((Q0 - Qbar) ** 2, s)))
    H_ref_norm = float(np.sqrt(np.trapezoid(Hbar ** 2, s)))
    Q_ref_norm = float(np.sqrt(np.trapezoid(Qbar ** 2, s)))

    e_H0_rel = (
        100.0 * H_err_norm / H_ref_norm
        if np.isfinite(H_ref_norm) and H_ref_norm > 1e-12
        else np.nan
    )
    e_Q0_rel = (
        100.0 * Q_err_norm / Q_ref_norm
        if np.isfinite(Q_ref_norm) and Q_ref_norm > 1e-12
        else np.nan
    )
    return float(e_H0_rel), float(e_Q0_rel)


def _initial_relative_deviation_percentages_from_result(
    res: dict,
) -> Tuple[float, float]:
    """Read the common t=0 state from one rollout result and compute both percentages."""
    H = np.asarray(res.get("H", []), dtype=float)
    Q = np.asarray(res.get("Q", []), dtype=float)
    if H.ndim != 2 or Q.ndim != 2 or H.shape[0] == 0 or Q.shape[0] == 0:
        return np.nan, np.nan
    return _initial_relative_deviation_percentages_from_arrays(
        X=np.asarray(res.get("X", []), dtype=float),
        H0=H[0],
        Q0=Q[0],
        Hbar=np.asarray(res.get("Hbar", []), dtype=float),
        Qbar=np.asarray(res.get("Qbar", []), dtype=float),
    )



def validate_shared_reference_and_initial_state(
    results: Dict[str, dict],
    X_expected: np.ndarray,
    Hbar_expected: np.ndarray,
    Qbar_expected: np.ndarray,
    H0_expected: np.ndarray,
    Q0_expected: np.ndarray,
    save_dir: str,
    tag: str,
    atol: float = 1e-10,
) -> Dict[str, Dict[str, float]]:
    """Verify that every controller uses exactly the same test problem.

    This check is intentionally strict.  It compares, controller by controller,
    the spatial grid, ODE equilibrium, reset state, and the initial relative-L2
    percentages against the single profiles generated in ``main``.

    A mismatch raises ``RuntimeError`` before figures are produced, preventing a
    visually plausible but unfair comparison from being saved.
    """
    X_expected = np.asarray(X_expected, dtype=float).reshape(-1)
    Hbar_expected = np.asarray(Hbar_expected, dtype=float).reshape(-1)
    Qbar_expected = np.asarray(Qbar_expected, dtype=float).reshape(-1)
    H0_expected = np.asarray(H0_expected, dtype=float).reshape(-1)
    Q0_expected = np.asarray(Q0_expected, dtype=float).reshape(-1)

    expected_eH, expected_eQ = _initial_relative_deviation_percentages_from_arrays(
        X=X_expected,
        H0=H0_expected,
        Q0=Q0_expected,
        Hbar=Hbar_expected,
        Qbar=Qbar_expected,
    )

    report: Dict[str, Dict[str, float]] = {}
    violations = []

    for label, res in results.items():
        X = np.asarray(res.get("X", []), dtype=float).reshape(-1)
        Hbar = np.asarray(res.get("Hbar", []), dtype=float).reshape(-1)
        Qbar = np.asarray(res.get("Qbar", []), dtype=float).reshape(-1)
        H = np.asarray(res.get("H", []), dtype=float)
        Q = np.asarray(res.get("Q", []), dtype=float)
        t = np.asarray(res.get("t", []), dtype=float).reshape(-1)

        if (
            X.shape != X_expected.shape
            or Hbar.shape != Hbar_expected.shape
            or Qbar.shape != Qbar_expected.shape
            or H.ndim != 2
            or Q.ndim != 2
            or H.shape[0] == 0
            or Q.shape[0] == 0
            or H.shape[1] != X_expected.size
            or Q.shape[1] != X_expected.size
            or t.size == 0
        ):
            violations.append(f"{label}: incompatible/missing result-array shapes")
            continue

        metrics = {
            "max_abs_X_difference": float(np.max(np.abs(X - X_expected))),
            "max_abs_Hbar_difference": float(np.max(np.abs(Hbar - Hbar_expected))),
            "max_abs_Qbar_difference": float(np.max(np.abs(Qbar - Qbar_expected))),
            "max_abs_H0_difference": float(np.max(np.abs(H[0] - H0_expected))),
            "max_abs_Q0_difference": float(np.max(np.abs(Q[0] - Q0_expected))),
            "initial_time_s": float(t[0]),
        }
        eH, eQ = _initial_relative_deviation_percentages_from_arrays(
            X=X,
            H0=H[0],
            Q0=Q[0],
            Hbar=Hbar,
            Qbar=Qbar,
        )
        metrics["e_H0_rel_pct"] = float(eH)
        metrics["e_Q0_rel_pct"] = float(eQ)
        metrics["e_H0_rel_difference_pct"] = float(abs(eH - expected_eH))
        metrics["e_Q0_rel_difference_pct"] = float(abs(eQ - expected_eQ))
        report[label] = metrics

        for key in (
            "max_abs_X_difference",
            "max_abs_Hbar_difference",
            "max_abs_Qbar_difference",
            "max_abs_H0_difference",
            "max_abs_Q0_difference",
        ):
            if not np.isfinite(metrics[key]) or metrics[key] > atol:
                violations.append(f"{label}: {key}={metrics[key]:.3e} > {atol:.3e}")

        if abs(metrics["initial_time_s"]) > atol:
            violations.append(
                f"{label}: first recorded time is {metrics['initial_time_s']:.12g} s, not 0 s"
            )
        if metrics["e_H0_rel_difference_pct"] > 1e-9:
            violations.append(
                f"{label}: e_H0_rel differs by {metrics['e_H0_rel_difference_pct']:.3e}%"
            )
        if metrics["e_Q0_rel_difference_pct"] > 1e-9:
            violations.append(
                f"{label}: e_Q0_rel differs by {metrics['e_Q0_rel_difference_pct']:.3e}%"
            )

    report_path = Path(save_dir) / f"{tag}_shared_initial_condition_validation.txt"
    with report_path.open("w", encoding="utf-8") as f:
        f.write("Shared reference/initial-state validation\n")
        f.write(f"expected e_H0_rel_pct = {expected_eH:.12f}\n")
        f.write(f"expected e_Q0_rel_pct = {expected_eQ:.12f}\n")
        f.write(f"absolute tolerance = {atol:.3e}\n\n")
        for label, metrics in report.items():
            f.write(f"[{label}]\n")
            for key, value in metrics.items():
                f.write(f"{key} = {value:.12e}\n")
            f.write("\n")
        if violations:
            f.write("VIOLATIONS\n")
            for item in violations:
                f.write(f"- {item}\n")
        else:
            f.write("PASS: every controller uses the identical grid, reference, and t=0 state.\n")

    print("\nShared initial-condition validation")
    print("-" * 78)
    print(f"Expected e_H,0^rel = {expected_eH:.9f}%")
    print(f"Expected e_Q,0^rel = {expected_eQ:.9f}%")
    for label, metrics in report.items():
        print(
            f"{label}: max|dH0|={metrics['max_abs_H0_difference']:.3e}, "
            f"max|dQ0|={metrics['max_abs_Q0_difference']:.3e}, "
            f"eH0={metrics['e_H0_rel_pct']:.9f}%, "
            f"eQ0={metrics['e_Q0_rel_pct']:.9f}%"
        )
    print(f"Validation report saved to: {report_path}")

    if violations:
        raise RuntimeError(
            "The controller comparison does not use an identical reference/initial state:\n"
            + "\n".join(f"- {item}" for item in violations)
        )

    return report




def _normalized_l2_norm_overshoot_percentage(
    res: dict,
    key: str,
) -> float:
    r"""
    Compute the normalized-L2-norm overshoot of a distributed state field:

        M_v = 100 * max_{t>0}
              [ (||v(.,t)||_{L2,n} - ||vbar||_{L2,n})
                / ||vbar||_{L2,n} ]_+,

    where v is H or Q and

        ||v(.,t)||_{L2,n}
            = sqrt((1/L) * integral_0^L v(x,t)^2 dx).

    The common initial sample t=0 is excluded. For the spatially constant
    Qbar used here, ||Qbar||_{L2,n}=Qbar, so this reduces exactly to the
    previously requested Q formula.
    """
    if key not in {"H", "Q"}:
        raise ValueError("key must be 'H' or 'Q'.")

    t_min = _time_min(res)
    state_norm = _normalized_spatial_l2_state_over_time(res, key)
    reference_key = "Hbar" if key == "H" else "Qbar"
    reference = np.asarray(res.get(reference_key, []), dtype=float).reshape(-1)
    X = np.asarray(res.get("X", []), dtype=float).reshape(-1)

    if (
        t_min.size == 0
        or state_norm.size == 0
        or reference.size == 0
        or X.size < 2
        or reference.size != X.size
    ):
        return np.nan

    n = min(t_min.size, state_norm.size)
    t_min = t_min[:n]
    state_norm = state_norm[:n]

    finite_time = t_min[np.isfinite(t_min)]
    if finite_time.size == 0:
        return np.nan

    initial_time = float(np.min(finite_time))
    tolerance = max(1e-12, 1e-10 * max(abs(initial_time), 1.0))
    time_mask = np.isfinite(t_min) & (t_min > initial_time + tolerance)
    if not np.any(time_mask):
        return np.nan

    if not np.all(np.isfinite(reference)):
        return np.nan

    domain_length = float(X[-1] - X[0])
    if not np.isfinite(domain_length) or domain_length <= 0.0:
        return np.nan

    # Use s=(x-x0)/L, so this is the normalized spatial L2 norm and has the
    # same physical unit as the state itself.
    s = (X - X[0]) / domain_length
    reference_norm = float(np.sqrt(np.trapezoid(reference ** 2, s)))
    if not np.isfinite(reference_norm) or reference_norm <= 1e-12:
        return np.nan

    relative_excursion = (
        state_norm[time_mask] - reference_norm
    ) / reference_norm
    relative_excursion = relative_excursion[np.isfinite(relative_excursion)]
    if relative_excursion.size == 0:
        return np.nan

    return float(100.0 * max(0.0, np.max(relative_excursion)))


def _h_normalized_l2_norm_overshoot_percentage(res: dict) -> float:
    """Return M_H using the same normalized-L2-norm definition as M_Q."""
    return _normalized_l2_norm_overshoot_percentage(res, "H")


def _q_normalized_l2_norm_overshoot_percentage(res: dict) -> float:
    """Return M_Q; for constant Qbar this equals the requested Q formula."""
    return _normalized_l2_norm_overshoot_percentage(res, "Q")


def _state_overshoot_percentage(
    res: dict,
    key: str,
) -> float:
    r"""
    Return the classical maximum positive overshoot percentage of the
    distributed state over the complete space-time response.

    For v = H or Q,

        M_v = 100 * max_{t>=0, x in [0,L]}
              [ (v(x,t) - vbar(x)) / |vbar(x)| ]_+,

    where [z]_+ = max(z, 0).

    This is the direct distributed-state extension of the classical scalar
    maximum-overshoot definition.  The complete simulated response is used,
    including the prescribed initial state at t=0.
    """
    t_min = _time_min(res)
    state = np.asarray(res.get(key, []), dtype=float)
    reference_key = "Hbar" if key == "H" else "Qbar"
    reference = np.asarray(res.get(reference_key, []), dtype=float).reshape(-1)

    if state.ndim != 2 or reference.size != state.shape[1] or t_min.size == 0:
        return np.nan

    n = min(t_min.size, state.shape[0])
    t_min = t_min[:n]
    state = state[:n]

    # Classical maximum overshoot is evaluated over the complete response,
    # including the initial sample.
    time_mask = np.isfinite(t_min)
    if not np.any(time_mask):
        return np.nan

    denominator = np.abs(reference)
    valid_reference = np.isfinite(reference) & (denominator > 1e-12)
    if not np.any(valid_reference):
        return np.nan

    state_window = state[time_mask][:, valid_reference]
    ref_window = reference[valid_reference].reshape(1, -1)
    denominator = denominator[valid_reference].reshape(1, -1)

    relative_positive_excursion = (state_window - ref_window) / denominator
    relative_positive_excursion = relative_positive_excursion[
        np.isfinite(relative_positive_excursion)
    ]
    if relative_positive_excursion.size == 0:
        return np.nan

    return float(100.0 * max(0.0, np.max(relative_positive_excursion)))


def _peak_spatial_linf(res: dict, key: str) -> float:
    """
    Return the peak spatial L-infinity tracking error over the full simulation.

    At each time instant, the spatial L-infinity norm is
        max_x |state(t, x) - reference(x)|.
    The reported peak then takes the maximum of this norm over all simulated
    time instants, including the initial state:
        max_t max_x |state(t, x) - reference(x)|.
    """
    linf_over_time = np.asarray(_get_linf(res, key), dtype=float).reshape(-1)
    finite_values = linf_over_time[np.isfinite(linf_over_time)]
    if finite_values.size == 0:
        return np.nan
    return float(np.max(finite_values))


def _joint_settling_time(
    t_min: np.ndarray,
    H_l2: np.ndarray,
    Q_l2: np.ndarray,
    steady_H_l2: float,
    steady_Q_l2: float,
    common_end_min: float,
    band_fraction: float = SETTLING_BAND_FRACTION,
) -> float:
    """
    Find the earliest time after which both H and Q L2 errors remain in their
    respective steady-state bands through the common comparison end time.

    H threshold:
        steady_H_l2 + band_fraction * |H_l2(0) - steady_H_l2|

    Q threshold:
        steady_Q_l2 + band_fraction * |Q_l2(0) - steady_Q_l2|

    With SETTLING_BAND_FRACTION = 0.05, the transient part of each tracking
    error must decay to within 5% of its initial-to-steady-state difference.
    """
    t_min = np.asarray(t_min, dtype=float).reshape(-1)
    H_l2 = np.asarray(H_l2, dtype=float).reshape(-1)
    Q_l2 = np.asarray(Q_l2, dtype=float).reshape(-1)

    n = min(t_min.size, H_l2.size, Q_l2.size)
    if n == 0:
        return np.nan

    t_min = t_min[:n]
    H_l2 = H_l2[:n]
    Q_l2 = Q_l2[:n]

    valid = (
        np.isfinite(t_min)
        & np.isfinite(H_l2)
        & np.isfinite(Q_l2)
        & (t_min <= float(common_end_min))
    )

    t_eval = t_min[valid]
    H_eval = H_l2[valid]
    Q_eval = Q_l2[valid]

    if (
        t_eval.size == 0
        or not np.isfinite(steady_H_l2)
        or not np.isfinite(steady_Q_l2)
    ):
        return np.nan

    H_threshold = float(
        steady_H_l2
        + float(band_fraction)
        * abs(float(H_eval[0]) - steady_H_l2)
    )
    Q_threshold = float(
        steady_Q_l2
        + float(band_fraction)
        * abs(float(Q_eval[0]) - steady_Q_l2)
    )

    inside_both_bands = (
        (H_eval <= H_threshold)
        & (Q_eval <= Q_threshold)
    )

    # suffix_all[i] is True only when every sample from i to the end remains
    # inside both steady-state bands.
    suffix_all = np.logical_and.accumulate(
        inside_both_bands[::-1]
    )[::-1]

    settled_indices = np.flatnonzero(suffix_all)
    if settled_indices.size == 0:
        return np.nan

    return float(t_eval[settled_indices[0]])



def build_selected_case_evidence_rows(
    results: Dict[str, dict],
    expected_T: Optional[float] = None,
):
    """
    Build the compact quantitative table for the selected scenario.

    Measures:
        1) steady-state mean spatial L2 tracking errors for H and Q;
        2) peak spatial L-infinity tracking errors for H and Q;
        3) H classical maximum positive overshoot M_H;
        4) Q classical maximum positive overshoot M_Q;
        5) joint H/Q settling time and convergence status.
    """
    results = _ordered_results(results)
    rows = []

    common_end_min = float(_common_time_end_min(results))
    steady_start_min = (
        LOCAL_ZOOM_START_RATIO * common_end_min
        if common_end_min > 0.0
        else np.nan
    )

    for label, res in results.items():
        t = np.asarray(res.get("t", []), dtype=float)
        t_min = t / 60.0 if t.size > 0 else np.asarray([], dtype=float)

        H_l2 = np.asarray(_get_l2(res, "H"), dtype=float)
        Q_l2 = np.asarray(_get_l2(res, "Q"), dtype=float)
        H_linf = np.asarray(_get_linf(res, "H"), dtype=float)
        Q_linf = np.asarray(_get_linf(res, "Q"), dtype=float)

        has_nonfinite = (
            t.size == 0
            or H_l2.size == 0
            or Q_l2.size == 0
            or H_linf.size == 0
            or Q_linf.size == 0
            or not np.all(np.isfinite(t))
            or not np.all(np.isfinite(H_l2))
            or not np.all(np.isfinite(Q_l2))
            or not np.all(np.isfinite(H_linf))
            or not np.all(np.isfinite(Q_linf))
        )

        early_stop = False
        if expected_T is not None and t.size > 0:
            early_stop = bool(t[-1] < float(expected_T) - 1e-9)

        diverged = bool(has_nonfinite or early_stop)

        if diverged:
            steady_H_l2 = np.nan
            steady_Q_l2 = np.nan
            peak_H_linf = np.nan
            peak_Q_linf = np.nan
            H_norm_overshoot_pct = np.nan
            Q_norm_overshoot_pct = np.nan
            settling_time_min = np.nan
            convergence_status = "Diverged"
        else:
            steady_H_l2 = _time_average_on_interval(
                t_min, H_l2, steady_start_min, common_end_min
            )
            steady_Q_l2 = _time_average_on_interval(
                t_min, Q_l2, steady_start_min, common_end_min
            )
            peak_H_linf = _peak_spatial_linf(res, "H")
            peak_Q_linf = _peak_spatial_linf(res, "Q")

            # Classical maximum positive overshoot over the full space-time state.
            H_norm_overshoot_pct = _state_overshoot_percentage(res, "H")
            Q_norm_overshoot_pct = _state_overshoot_percentage(res, "Q")

            settling_time_min = _joint_settling_time(
                t_min=t_min,
                H_l2=H_l2,
                Q_l2=Q_l2,
                steady_H_l2=steady_H_l2,
                steady_Q_l2=steady_Q_l2,
                common_end_min=common_end_min,
                band_fraction=SETTLING_BAND_FRACTION,
            )

            convergence_status = (
                "Converged" if np.isfinite(settling_time_min) else "Not settled"
            )

        rows.append({
            "Controller": label,
            "SS mean L2(H) [m]": steady_H_l2,
            "SS mean L2(Q) [m^3/s]": steady_Q_l2,
            "Peak Linf(H) [m]": peak_H_linf,
            "Peak Linf(Q) [m^3/s]": peak_Q_linf,
            "M_H norm overshoot [%]": H_norm_overshoot_pct,
            "M_Q norm overshoot [%]": Q_norm_overshoot_pct,
            "Settling time [min]": settling_time_min,
            "Convergence status": convergence_status,
        })

    return rows




def print_selected_case_evidence_summary(
    results: Dict[str, dict],
    H_left: float,
    H_right: float,
    Q_value: float,
    seed: int,
    dH_source=None,
):
    """Print the evidence-summary table for the shared six-controller scenario."""
    rows = build_selected_case_evidence_rows(
        results,
        expected_T=SAC_T,
    )

    headers = [
        "Controller",
        "SS mean L2(H) [m]",
        "SS mean L2(Q) [m^3/s]",
        "Peak Linf(H) [m]",
        "Peak Linf(Q) [m^3/s]",
        "M_H norm overshoot [%]",
        "M_Q norm overshoot [%]",
        "Settling time [min]",
        "Convergence status",
    ]

    common_end_min = float(_common_time_end_min(results))
    steady_start_min = (
        LOCAL_ZOOM_START_RATIO * common_end_min
        if common_end_min > 0.0
        else np.nan
    )
    steady_percentage = 100.0 * (1.0 - LOCAL_ZOOM_START_RATIO)

    first_result = next(iter(results.values())) if results else None
    if first_result is not None:
        H0 = np.asarray(first_result["H"], dtype=float)[0]
        Q0 = np.asarray(first_result["Q"], dtype=float)[0]
        Hbar0 = np.asarray(first_result["Hbar"], dtype=float)
        Qbar0 = np.asarray(first_result["Qbar"], dtype=float)
        actual_H = float(np.max(np.abs(H0 - Hbar0)))
        actual_Q = float(np.max(np.abs(Q0 - Qbar0)))
        e_H0_rel, e_Q0_rel = _initial_relative_deviation_percentages_from_result(
            first_result
        )
    else:
        actual_H = np.nan
        actual_Q = np.nan
        e_H0_rel = np.nan
        e_Q0_rel = np.nan

    scenario = (
        "Selected linear initial condition: "
        f"H(0)={H_left:g} m, H(L)={H_right:g} m, "
        f"Q(x,0)={Q_value:g} m^3/s, seed={seed}, "
        f"max deviations from ODE equilibrium="
        f"({actual_H:.6f} m, {actual_Q:.6f} m^3/s)"
    )
    if dH_source is not None:
        scenario += f", dH_source={dH_source:g}"

    printable_rows = []
    for row in rows:
        printable_rows.append([
            str(row["Controller"]),
            _format_metric(row["SS mean L2(H) [m]"]),
            _format_metric(row["SS mean L2(Q) [m^3/s]"]),
            _format_metric(row["Peak Linf(H) [m]"]),
            _format_metric(row["Peak Linf(Q) [m^3/s]"]),
            _format_metric(row["M_H norm overshoot [%]"], precision=3),
            _format_metric(row["M_Q norm overshoot [%]"], precision=3),
            _format_metric(row["Settling time [min]"]),
            str(row["Convergence status"]),
        ])

    widths = []
    for i, header in enumerate(headers):
        widths.append(
            max(
                len(header),
                max(len(r[i]) for r in printable_rows),
            )
        )

    header_line = " | ".join(
        header.ljust(widths[i])
        for i, header in enumerate(headers)
    )
    sep_line = "-+-".join(
        "-" * widths[i]
        for i in range(len(headers))
    )
    table_width = max(120, len(header_line))

    print("\n" + "=" * table_width)
    print("Closed-loop performance table for the selected single scenario")
    print(scenario)
    print(
        "Shared initial relative deviations: "
        f"e_H,0^rel={e_H0_rel:.6f}%, "
        f"e_Q,0^rel={e_Q0_rel:.6f}%."
    )
    print(
        "Steady-state averaging window "
        f"(same as local zoom): "
        f"[{steady_start_min:.6f}, {common_end_min:.6f}] min, "
        f"final {steady_percentage:.1f}% of the common horizon."
    )
    print(
        "Peak Linf is max_t max_x |state-reference| over the complete "
        "simulation horizon. H and Q use the classical maximum positive "
        "overshoot definition extended to the full distributed state: "
        "M_H=100*max_{t>=0,x}[(H(x,t)-Hbar(x))/|Hbar(x)|]_+ and "
        "M_Q=100*max_{t>=0,x}[(Q(x,t)-Qbar(x))/|Qbar(x)|]_+. "
        "Convergence requires both L2 tracking errors to satisfy "
        "E_H(t)<=E_H(0) and E_Q(t)<=E_Q(0) from some time onward "
        "through the end of the simulation."
    )
    print("=" * table_width)

    print(header_line)
    print(sep_line)
    for row in printable_rows:
        print(
            " | ".join(
                row[i].ljust(widths[i])
                for i in range(len(headers))
            )
        )
    print("=" * table_width)



# ============================================================
# Main: six-controller comparison
# ============================================================

DEFAULT_NEW_MODEL_PATH = (
     "./sactx/seed_42/"
    "sac_cac_hybrid_yul_ode_cheb_raw_HQ_100000_steps"
) 
DEFAULT_TDPSAC_FPC_MODEL_PATH = (
    "./sacddx/seed_42/"
    "sac_cac_hybrid_fixed_control_yul_ode_cheb_raw_HQ_100000_steps" 
)
DEFAULT_DPSAC_MODEL_PATH = (
    "./saccontrolx/seed_42/"
    "sac_cac_new_hadamard_hybrid_all_trainable_raw_HQ_100000_steps"
)
DEFAULT_DO_SAC_MODEL_PATH = (
    "./sactrainx/seed_42/"
    "sac_cac_deeponet_feature_raw_hq_ode_cheb_100000_steps"
)
DEFAULT_SAC_FPC_MODEL_PATH = (
    "./sacnoxv/seed_42/"
    "sac_fnn_frozen_new_deeponet_prior_raw_HQ_100000_steps"
)
# Newly trained ordinary SAC checkpoint using raw physical H/Q observations.
DEFAULT_SAC_MODEL_PATH = (
    "./saczx/seed_42/"
    "sac_cac_raw_hq_ode_cheb_100000_steps")

DEFAULT_SAVE_DIR_COMPARISON = "./test_results_six_algorithms_new_hadamard_linear_IC"

# Display/run order follows the old multi-controller comparison hierarchy.
LEARNED_CONTROLLER_SPECS = [
    {
        "label": LABEL_NEW_MODEL,          # TDPSAC
        "model_kind": "hybrid",
        "required": True,
    },
    {
        "label": LABEL_DFP_DON_SAC,        # TDPSAC-FPC
        "model_kind": "auto",
        "required": True,
    },
    {
        "label": LABEL_HYBRID,             # DPSAC
        "model_kind": "auto",
        "required": True,
    },
    {
        "label": LABEL_SAC,                # DO-SAC
        "model_kind": "auto",
        "required": True,
    },
    {
        "label": LABEL_FP_SAC,             # SAC-FPC
        "model_kind": "auto",
        "required": True,
    },
    {
        "label": LABEL_SAC_CURVE,          # SAC
        "model_kind": "sac",
        "required": True,
    },
]


def _run_optional_controller(
    results: Dict[str, dict],
    *,
    label: str,
    model_path: str,
    model_kind: str,
    seed: int,
    H_left: float,
    H_right: float,
    Q_value: float,
    dH_source,
    required: bool = False,
):
    """Run one learned controller and keep the common result-data interface."""
    if not _checkpoint_exists(model_path):
        message = (
            f"No valid SB3 ZIP checkpoint was found for {label}.\n"
            f"Current path: {model_path!r}\n"
            "Expected either '/path/model' with '/path/model.zip' present, "
            "or a direct '/path/model.zip' path."
        )
        if required:
            raise FileNotFoundError(message)
        print(f"\n[SKIP] {message}")
        return

    print(f"\nRunning {label}...")
    results[label] = run_policy_exact(
        model_path=model_path,
        model_kind=model_kind,
        label=label,
        seed=seed,
        H_left=H_left,
        H_right=H_right,
        Q_value=Q_value,
        dH_source=dH_source,
    )


def _clean_runtime_argv(argv=None):
    """Remove Jupyter/IPython kernel connection arguments before argparse."""
    raw = list(sys.argv[1:] if argv is None else argv)
    cleaned = []
    i = 0

    while i < len(raw):
        arg = str(raw[i])

        if arg in {"-f", "--f"} and i + 1 < len(raw):
            candidate = str(raw[i + 1])
            if candidate.endswith(".json") and (
                "jupyter" in candidate.lower()
                or "kernel-" in Path(candidate).name.lower()
            ):
                i += 2
                continue

        if arg.startswith(("--f=", "-f=")):
            candidate = arg.split("=", 1)[1]
            if candidate.endswith(".json") and (
                "jupyter" in candidate.lower()
                or "kernel-" in Path(candidate).name.lower()
            ):
                i += 1
                continue

        if arg.endswith(".json") and (
            "jupyter" in arg.lower()
            or "kernel-" in Path(arg).name.lower()
        ):
            i += 1
            continue

        cleaned.append(arg)
        i += 1

    return cleaned




# ============================================================
# Monte-Carlo large-initial-condition robustness test
# 50 trials: 25 H-above/Q-below + 25 H-below/Q-above
# Large-IC scale: H around 8%, Q around 20%
# Default non-overlapping bands: H 6%-10%, Q 18%-22%
# ============================================================

MC_METRIC_KEYS = [
    "SS mean L2(H) [m]",
    "SS mean L2(Q) [m^3/s]",
    "Peak Linf(H) [m]",
    "Peak Linf(Q) [m^3/s]",
    "M_H norm overshoot [%]",
    "M_Q norm overshoot [%]",
    "Settling time [min]",
]

MC_SHORT_NAMES = {
    "SS mean L2(H) [m]": "SS_H",
    "SS mean L2(Q) [m^3/s]": "SS_Q",
    "Peak Linf(H) [m]": "Peak_H",
    "Peak Linf(Q) [m^3/s]": "Peak_Q",
    "M_H norm overshoot [%]": "M_H",
    "M_Q norm overshoot [%]": "M_Q",
    "Settling time [min]": "t_settle",
}

MC_CONTROLLER_ORDER = [
    LABEL_NEW_MODEL,
    LABEL_DFP_DON_SAC,
    LABEL_HYBRID,
    LABEL_SAC,
    LABEL_FP_SAC,
    LABEL_SAC_CURVE,
]


def generate_large_linear_ic_cases(
    X: np.ndarray,
    Hbar: np.ndarray,
    Qbar: np.ndarray,
    *,
    n_total: int = 50,
    n_above: int = 25,
    H_pct_min: float = 6.0,
    H_pct_max: float = 10.0,
    Q_pct_min: float = 18.0,
    Q_pct_max: float = 22.0,
    seed: int = 20260810,
):
    """Generate one fixed 50-case list shared by all six learned controllers.

    The H initial profile uses linear interpolation:

        H(x,0) = H_left + (H_right-H_left) x/L.

    The realized relative spatial-L2 H deviation must remain inside the existing
    [H_pct_min, H_pct_max] band.

    Cross-side design is unchanged:
        trials 1..n_above:
            H below equilibrium, Q above equilibrium;
        remaining trials:
            H above equilibrium, Q below equilibrium.

    No additional absolute endpoint bounds are imposed on H_left or H_right.
    The H initial conditions are generated using the existing relative-deviation
    band [H_pct_min, H_pct_max].

    Q(x,0) remains spatially constant and is generated exactly as before.
    """
    X = np.asarray(X, dtype=float).reshape(-1)
    Hbar = np.asarray(Hbar, dtype=float).reshape(-1)
    Qbar = np.asarray(Qbar, dtype=float).reshape(-1)

    if X.size < 2 or Hbar.shape != X.shape or Qbar.shape != X.shape:
        raise ValueError("X, Hbar, and Qbar must have the same one-dimensional shape.")
    if n_total <= 0 or not (0 <= n_above <= n_total):
        raise ValueError("Require n_total>0 and 0<=n_above<=n_total.")
    if not (0.0 < H_pct_min < H_pct_max):
        raise ValueError("Require 0 < H_pct_min < H_pct_max.")
    if not (0.0 < Q_pct_min < Q_pct_max):
        raise ValueError("Require 0 < Q_pct_min < Q_pct_max.")

    rng = np.random.default_rng(seed)
    H_left_ref = float(Hbar[0])
    H_right_ref = float(Hbar[-1])
    Q_ref = float(Qbar[0])
    L = float(X[-1])

    cases = []
    max_attempts_per_case = 100000

    for i in range(n_total):
        H_side = "below" if i < n_above else "above"
        H_sign = -1.0 if H_side == "below" else 1.0
        Q_sign = -H_sign
        side = "H_below_Q_above" if H_sign < 0.0 else "H_above_Q_below"

        accepted = False
        for _ in range(max_attempts_per_case):
            pQ = float(rng.uniform(Q_pct_min, Q_pct_max))

            if H_sign < 0.0:
                # H-below group: use the existing H deviation band without
                # additional absolute endpoint bounds.
                pH_left = float(rng.uniform(H_pct_min, H_pct_max))
                pH_right = float(rng.uniform(H_pct_min, H_pct_max))
                H_left = H_left_ref * (1.0 - pH_left / 100.0)
                H_right = H_right_ref * (1.0 - pH_right / 100.0)
            else:
                # H-above group: retain the existing H deviation band at the
                # two endpoints, then verify the full-profile relative-L2 band.
                pH_left = float(rng.uniform(H_pct_min, H_pct_max))
                pH_right = float(rng.uniform(H_pct_min, H_pct_max))
                H_left = H_left_ref * (1.0 + pH_left / 100.0)
                H_right = H_right_ref * (1.0 + pH_right / 100.0)

            # Q generation is unchanged.
            Q_value = Q_ref * (1.0 + Q_sign * pQ / 100.0)

            if H_left <= 0.20 or H_right <= 0.20 or Q_value <= 0.0:
                continue

            # Linear interpolation of the water-depth initial profile.
            H0 = H_left + (H_right - H_left) * (X / L)
            Q0 = np.full_like(X, Q_value, dtype=float)

            # Preserve the cross-side design over the whole spatial domain.
            if H_sign < 0.0:
                if not np.all(H0 < Hbar):
                    continue
            else:
                if not np.all(H0 > Hbar):
                    continue

            eH_rel, eQ_rel = _initial_relative_deviation_percentages_from_arrays(
                X=X,
                H0=H0,
                Q0=Q0,
                Hbar=Hbar,
                Qbar=Qbar,
            )

            # Keep the existing relative-L2 acceptance checks.
            if not np.isfinite(eH_rel) or not (H_pct_min <= eH_rel <= H_pct_max):
                continue
            if not np.isfinite(eQ_rel) or not (Q_pct_min <= eQ_rel <= Q_pct_max):
                continue

            cases.append({
                "trial": i + 1,
                "side": side,
                "H_left": float(H_left),
                "H_right": float(H_right),
                "Q_value": float(Q_value),
                "H_left_endpoint_pct": float(
                    100.0 * (H_left - H_left_ref) / H_left_ref
                ),
                "H_right_endpoint_pct": float(
                    100.0 * (H_right - H_right_ref) / H_right_ref
                ),
                "Q_endpoint_pct": float(Q_sign * pQ),
                "delta_H_L2_pct": float(eH_rel),
                "delta_Q_L2_pct": float(eQ_rel),
            })
            accepted = True
            break

        if not accepted:
            raise RuntimeError(
                f"Could not generate trial {i+1} satisfying the requested large-IC bands."
            )

    return cases

def save_mc_initial_conditions(cases, save_dir: Path):
    path = save_dir / "MC50_largeIC_initial_conditions.csv"
    fields = [
        "trial", "side", "H_left", "H_right", "Q_value",
        "H_left_endpoint_pct", "H_right_endpoint_pct", "Q_endpoint_pct",
        "delta_H_L2_pct", "delta_Q_L2_pct",
    ]
    with path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        writer.writerows(cases)
    print(f"Saved initial-condition list: {path}")
    return path


def _make_eval_env_by_kind(
    model_kind: str,
    *,
    H_left: float,
    H_right: float,
    Q_value: float,
    seed: int,
    dH_source=None,
):
    kind = str(model_kind).lower()
    if kind == "sac_error":
        return make_env_sac_error_state(
            H_left=H_left,
            H_right=H_right,
            Q_value=Q_value,
            seed=seed,
            dH_source=dH_source,
        )
    return make_env_sac(
        H_left=H_left,
        H_right=H_right,
        Q_value=Q_value,
        seed=seed,
        dH_source=dH_source,
    )


def load_learned_controller_once(
    model_path: str,
    model_kind: str,
    *,
    first_case: dict,
    seed: int,
    dH_source=None,
):
    """Load a learned policy once; later trials only replace the evaluation env."""
    env, _, _, _, _ = _make_eval_env_by_kind(
        model_kind,
        H_left=first_case["H_left"],
        H_right=first_case["H_right"],
        Q_value=first_case["Q_value"],
        seed=seed,
        dH_source=dH_source,
    )

    kind = str(model_kind).lower()
    try:
        if kind in {"sac", "sac_error"}:
            model = SAC.load(model_path, env=env)
        elif kind == "hybrid":
            model = _safe_load_hybrid_model(model_path, env=env)
        elif kind == "auto":
            model = _safe_load_generic_sac_model(model_path, env=env)
        else:
            raise ValueError(
                f"Unknown model_kind={model_kind!r}; expected sac/sac_error/hybrid/auto."
            )
    except Exception:
        env.close()
        raise

    env.close()
    return model


def run_loaded_policy_exact(
    model,
    model_kind: str,
    label: str,
    *,
    seed: int,
    H_left: float,
    H_right: float,
    Q_value: float,
    dH_source=None,
):
    """Evaluate an already-loaded SAC-family controller on one linear IC."""
    env, X, Hbar, Qbar, _ = _make_eval_env_by_kind(
        model_kind,
        H_left=H_left,
        H_right=H_right,
        Q_value=Q_value,
        seed=seed,
        dH_source=dH_source,
    )
    model.set_env(env)

    obs, _ = env.reset(seed=seed)
    obs = np.asarray(obs, dtype=np.float32).reshape(-1)
    if obs.size != RAW_STATE_DIM:
        env.close()
        raise RuntimeError(
            f"{label}: reset observation dimension={obs.size}, expected={RAW_STATE_DIM}."
        )

    observation_mode_used = (
        "normalized_error" if str(model_kind).lower() == "sac_error" else "raw"
    )

    H_hist, Q_hist, U_hist, t_hist, reward_hist = [], [], [], [], []
    H0, Q0 = env.get_current_HQ()
    H_hist.append(H0.copy())
    Q_hist.append(Q0.copy())
    U_hist.append(np.nan)
    t_hist.append(0.0)
    reward_hist.append(0.0)

    previous_u_filtered = float(U_ref)

    done = False
    last_terminated = False
    last_truncated = False
    last_info = {}
    while not done:
        action, _ = model.predict(obs, deterministic=True)

        action_filtered, u_raw, previous_u_filtered = lowpass_filter_rl_control(
            action,
            previous_u_filtered,
            alpha=RL_CONTROL_FILTER_ALPHA,
        )

        obs, reward, terminated, truncated, info = env.step(action_filtered)
        H, Q = env.get_current_HQ()

        H_hist.append(H.copy())
        Q_hist.append(Q.copy())
        U_hist.append(float(env.last_control))
        t_hist.append(float(info["time"]))
        reward_hist.append(float(reward))
        last_terminated = bool(terminated)
        last_truncated = bool(truncated)
        last_info = dict(info) if isinstance(info, dict) else {"info_repr": repr(info)}
        done = bool(terminated or truncated)

    env.close()

    H_hist = np.asarray(H_hist)
    Q_hist = np.asarray(Q_hist)
    U_hist = np.asarray(U_hist)
    t_hist = np.asarray(t_hist)
    reward_hist = np.asarray(reward_hist)

    H_err = H_hist - Hbar[None, :]
    Q_err = Q_hist - Qbar[None, :]

    return {
        "label": label,
        "model_kind": model_kind,
        "observation_mode": observation_mode_used,
        "X": X,
        "Hbar": Hbar,
        "Qbar": Qbar,
        "t": t_hist,
        "H": H_hist,
        "Q": Q_hist,
        "U": U_hist,
        "reward": reward_hist,
        "H_l2": np.sqrt(np.mean(H_err ** 2, axis=1)),
        "Q_l2": np.sqrt(np.mean(Q_err ** 2, axis=1)),
        "final_H_minus_Hbar": H_err[-1],
        "final_Q_minus_Qbar": Q_err[-1],
        "total_reward": float(np.sum(reward_hist)),
        "last_terminated": bool(last_terminated),
        "last_truncated": bool(last_truncated),
        "last_info": last_info,
    }






def extract_one_mc_row(label: str, result: dict):
    # Use one controller at a time so a diverged controller cannot shorten the
    # steady-state evaluation window of any other controller in the same trial.
    return build_selected_case_evidence_rows(
        {label: result},
        expected_T=SAC_T,
    )[0]


def _tdpsac_failure_ss_diagnostic(result: dict, row: dict):
    """
    Return SS_H/SS_Q values only for diagnosing a failed TDPSAC trial.

    For a completed-but-not-settled run, use the standard SS_H/SS_Q already
    computed by the normal evaluation workflow.

    For an early-stopped/diverged run, the standard SS metrics intentionally
    remain NaN in the main table.  For diagnostics only, estimate a tail mean
    from the final 25% of the finite available trajectory.  These diagnostic
    values are never used in MC mean/std aggregation or in the success test.
    """
    ss_h = _safe_metric_value(row.get("SS mean L2(H) [m]", np.nan))
    ss_q = _safe_metric_value(row.get("SS mean L2(Q) [m^3/s]", np.nan))

    if np.isfinite(ss_h) and np.isfinite(ss_q):
        return ss_h, ss_q

    t = np.asarray(result.get("t", []), dtype=float).reshape(-1)
    H_l2 = np.asarray(_get_l2(result, "H"), dtype=float).reshape(-1)
    Q_l2 = np.asarray(_get_l2(result, "Q"), dtype=float).reshape(-1)

    n = min(t.size, H_l2.size, Q_l2.size)
    if n == 0:
        return np.nan, np.nan

    t_min = t[:n] / 60.0
    H_l2 = H_l2[:n]
    Q_l2 = Q_l2[:n]

    valid = np.isfinite(t_min) & np.isfinite(H_l2) & np.isfinite(Q_l2)
    if not np.any(valid):
        return np.nan, np.nan

    t_finite = t_min[valid]
    H_finite = H_l2[valid]
    Q_finite = Q_l2[valid]

    available_end_min = float(np.max(t_finite))
    available_start_min = LOCAL_ZOOM_START_RATIO * available_end_min

    ss_h_diag = _time_average_on_interval(
        t_finite, H_finite, available_start_min, available_end_min
    )
    ss_q_diag = _time_average_on_interval(
        t_finite, Q_finite, available_start_min, available_end_min
    )
    return ss_h_diag, ss_q_diag



def _finite_tail_mean(values, fraction=0.25):
    values = np.asarray(values, dtype=float).reshape(-1)
    values = values[np.isfinite(values)]
    if values.size == 0:
        return np.nan
    n_tail = max(1, int(np.ceil(float(fraction) * values.size)))
    return float(np.mean(values[-n_tail:]))


def _finite_tail_slope(t, values, fraction=0.25):
    t = np.asarray(t, dtype=float).reshape(-1)
    values = np.asarray(values, dtype=float).reshape(-1)
    n = min(t.size, values.size)
    if n < 2:
        return np.nan
    t = t[:n]
    values = values[:n]
    valid = np.isfinite(t) & np.isfinite(values)
    t = t[valid]
    values = values[valid]
    if t.size < 2:
        return np.nan
    n_tail = max(2, int(np.ceil(float(fraction) * t.size)))
    tt = t[-n_tail:]
    yy = values[-n_tail:]
    if np.ptp(tt) <= 0.0:
        return np.nan
    return float(np.polyfit(tt, yy, 1)[0])


def build_nonconvergence_diagnostic(result: dict, row: dict, expected_T: float):
    """Diagnose WHY one trial is not classified as converged.

    This function is diagnostic only.  It does not change the controller,
    simulation, convergence criterion, or the standard MC metrics.
    """
    t = np.asarray(result.get("t", []), dtype=float).reshape(-1)
    H_l2 = np.asarray(_get_l2(result, "H"), dtype=float).reshape(-1)
    Q_l2 = np.asarray(_get_l2(result, "Q"), dtype=float).reshape(-1)
    H = np.asarray(result.get("H", []), dtype=float)
    Q = np.asarray(result.get("Q", []), dtype=float)
    U = np.asarray(result.get("U", []), dtype=float).reshape(-1)

    n = min(t.size, H_l2.size, Q_l2.size)
    t = t[:n]
    H_l2 = H_l2[:n]
    Q_l2 = Q_l2[:n]

    status = str(row.get("Convergence status", "Unknown"))
    t_end = float(t[-1]) if t.size and np.isfinite(t[-1]) else np.nan
    early_stop = bool(np.isfinite(t_end) and t_end < float(expected_T) - 1e-9)

    ss_h = _safe_metric_value(row.get("SS mean L2(H) [m]", np.nan))
    ss_q = _safe_metric_value(row.get("SS mean L2(Q) [m^3/s]", np.nan))

    H_thr = np.nan
    Q_thr = np.nan
    H_final = float(H_l2[-1]) if H_l2.size and np.isfinite(H_l2[-1]) else np.nan
    Q_final = float(Q_l2[-1]) if Q_l2.size and np.isfinite(Q_l2[-1]) else np.nan
    H_min_err = float(np.nanmin(H_l2)) if H_l2.size and np.any(np.isfinite(H_l2)) else np.nan
    Q_min_err = float(np.nanmin(Q_l2)) if Q_l2.size and np.any(np.isfinite(Q_l2)) else np.nan

    valid = np.isfinite(t) & np.isfinite(H_l2) & np.isfinite(Q_l2)
    if np.any(valid) and np.isfinite(ss_h) and np.isfinite(ss_q):
        Hv = H_l2[valid]
        Qv = Q_l2[valid]
        H_thr = float(ss_h + SETTLING_BAND_FRACTION * abs(float(Hv[0]) - ss_h))
        Q_thr = float(ss_q + SETTLING_BAND_FRACTION * abs(float(Qv[0]) - ss_q))

    H_final_inside = bool(np.isfinite(H_final) and np.isfinite(H_thr) and H_final <= H_thr)
    Q_final_inside = bool(np.isfinite(Q_final) and np.isfinite(Q_thr) and Q_final <= Q_thr)

    H_ever_inside = bool(np.isfinite(H_thr) and np.any(np.isfinite(H_l2) & (H_l2 <= H_thr)))
    Q_ever_inside = bool(np.isfinite(Q_thr) and np.any(np.isfinite(Q_l2) & (Q_l2 <= Q_thr)))

    ufinite = U[np.isfinite(U)]
    if ufinite.size:
        u_min = float(np.min(ufinite))
        u_max = float(np.max(ufinite))
        tol = 1e-6
        frac_u_low = float(np.mean(ufinite <= min_control_value + tol))
        frac_u_high = float(np.mean(ufinite >= max_control_value - tol))
    else:
        u_min = u_max = frac_u_low = frac_u_high = np.nan

    min_H_state = float(np.nanmin(H)) if H.size and np.any(np.isfinite(H)) else np.nan
    max_H_state = float(np.nanmax(H)) if H.size and np.any(np.isfinite(H)) else np.nan
    min_Q_state = float(np.nanmin(Q)) if Q.size and np.any(np.isfinite(Q)) else np.nan
    max_Q_state = float(np.nanmax(Q)) if Q.size and np.any(np.isfinite(Q)) else np.nan

    last_info = result.get("last_info", {})
    if isinstance(last_info, dict):
        info_text = "; ".join(f"{k}={v}" for k, v in sorted(last_info.items()))
    else:
        info_text = repr(last_info)

    if status == "Converged":
        reason = "converged"
    elif status == "Diverged":
        if early_stop:
            if bool(result.get("last_truncated", False)):
                reason = "early_stop_truncated"
            elif bool(result.get("last_terminated", False)):
                reason = "early_stop_terminated"
            else:
                reason = "early_stop_or_nonfinite"
        else:
            reason = "nonfinite_or_incomplete_trajectory"
    else:
        # Completed full horizon but failed the joint sustained 5% settling test.
        if (not H_final_inside) and (not Q_final_inside):
            reason = "full_horizon_both_errors_outside_settling_band"
        elif not H_final_inside:
            reason = "full_horizon_H_error_outside_settling_band"
        elif not Q_final_inside:
            reason = "full_horizon_Q_error_outside_settling_band"
        elif not (H_ever_inside and Q_ever_inside):
            reason = "full_horizon_one_error_never_entered_band"
        else:
            reason = "entered_band_but_did_not_remain_there"

    return {
        "diagnostic_reason": reason,
        "diagnostic_t_end_s": t_end,
        "diagnostic_early_stop": early_stop,
        "diagnostic_last_terminated": bool(result.get("last_terminated", False)),
        "diagnostic_last_truncated": bool(result.get("last_truncated", False)),
        "diagnostic_H_threshold": H_thr,
        "diagnostic_Q_threshold": Q_thr,
        "diagnostic_H_final_L2": H_final,
        "diagnostic_Q_final_L2": Q_final,
        "diagnostic_H_min_L2": H_min_err,
        "diagnostic_Q_min_L2": Q_min_err,
        "diagnostic_H_tail_mean_L2": _finite_tail_mean(H_l2),
        "diagnostic_Q_tail_mean_L2": _finite_tail_mean(Q_l2),
        "diagnostic_H_tail_slope_per_s": _finite_tail_slope(t, H_l2),
        "diagnostic_Q_tail_slope_per_s": _finite_tail_slope(t, Q_l2),
        "diagnostic_H_final_inside_band": H_final_inside,
        "diagnostic_Q_final_inside_band": Q_final_inside,
        "diagnostic_H_ever_inside_band": H_ever_inside,
        "diagnostic_Q_ever_inside_band": Q_ever_inside,
        "diagnostic_U_min": u_min,
        "diagnostic_U_max": u_max,
        "diagnostic_U_at_lower_bound_fraction": frac_u_low,
        "diagnostic_U_at_upper_bound_fraction": frac_u_high,
        "diagnostic_min_H_state": min_H_state,
        "diagnostic_max_H_state": max_H_state,
        "diagnostic_min_Q_state": min_Q_state,
        "diagnostic_max_Q_state": max_Q_state,
        "diagnostic_last_info": info_text,
    }


def save_directional_diagnostics(raw_rows, save_dir: Path):
    """Save per-trial failure diagnostics and H/Q cross-direction summaries."""
    diag_csv = save_dir / "MC50_largeIC_nonconvergence_diagnostics.csv"
    diag_fields = [
        "trial", "side", "Controller", "Convergence status", "diagnostic_reason",
        "H_left", "H_right", "Q_value", "delta_H_L2_pct", "delta_Q_L2_pct",
        "diagnostic_t_end_s", "diagnostic_early_stop",
        "diagnostic_last_terminated", "diagnostic_last_truncated",
        "diagnostic_H_threshold", "diagnostic_Q_threshold",
        "diagnostic_H_final_L2", "diagnostic_Q_final_L2",
        "diagnostic_H_min_L2", "diagnostic_Q_min_L2",
        "diagnostic_H_tail_mean_L2", "diagnostic_Q_tail_mean_L2",
        "diagnostic_H_tail_slope_per_s", "diagnostic_Q_tail_slope_per_s",
        "diagnostic_H_final_inside_band", "diagnostic_Q_final_inside_band",
        "diagnostic_H_ever_inside_band", "diagnostic_Q_ever_inside_band",
        "diagnostic_U_min", "diagnostic_U_max",
        "diagnostic_U_at_lower_bound_fraction", "diagnostic_U_at_upper_bound_fraction",
        "diagnostic_min_H_state", "diagnostic_max_H_state",
        "diagnostic_min_Q_state", "diagnostic_max_Q_state",
        "diagnostic_last_info", "error_message",
    ]
    with diag_csv.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=diag_fields, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(raw_rows)

    sides = ["H_above_Q_below", "H_below_Q_above"]
    summary_csv = save_dir / "MC50_largeIC_directional_success_diagnostics.csv"
    summary_rows = []
    for controller in MC_CONTROLLER_ORDER:
        for side in sides:
            rows = [
                r for r in raw_rows
                if r.get("Controller") == controller and r.get("side") == side
            ]
            if not rows:
                continue
            n = len(rows)
            n_conv = sum(r.get("Convergence status") == "Converged" for r in rows)
            n_not = sum(r.get("Convergence status") == "Not settled" for r in rows)
            n_div = sum(r.get("Convergence status") == "Diverged" for r in rows)
            reasons = {}
            for r in rows:
                reason = str(r.get("diagnostic_reason", "unknown"))
                reasons[reason] = reasons.get(reason, 0) + 1
            dominant_reason = max(reasons.items(), key=lambda kv: kv[1])[0] if reasons else "--"
            summary_rows.append({
                "Controller": controller,
                "side": side,
                "n_trials": n,
                "n_converged": n_conv,
                "n_not_settled": n_not,
                "n_diverged": n_div,
                "success_rate_pct": 100.0 * n_conv / n,
                "dominant_failure_reason": dominant_reason,
                "reason_counts": "; ".join(f"{k}:{v}" for k, v in sorted(reasons.items())),
            })

    with summary_csv.open("w", newline="", encoding="utf-8") as f:
        fields = [
            "Controller", "side", "n_trials", "n_converged", "n_not_settled",
            "n_diverged", "success_rate_pct", "dominant_failure_reason", "reason_counts",
        ]
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        writer.writerows(summary_rows)

    txt_path = save_dir / "MC50_largeIC_directional_success_diagnostics.txt"
    with txt_path.open("w", encoding="utf-8") as f:
        f.write("Directional non-convergence diagnostic (diagnostic only; controller logic unchanged)\n")
        f.write("Cross IC groups: H_above_Q_below and H_below_Q_above.\n")
        f.write("Success criterion remains the original joint sustained 5% L2 settling criterion.\n\n")
        header = (
            f"{'Controller':<14} {'Direction':<20} {'Conv':>6} {'NotSet':>7} "
            f"{'Div':>5} {'Success%':>9}  Dominant reason\n"
        )
        f.write(header)
        f.write("-" * 120 + "\n")
        for r in summary_rows:
            f.write(
                f"{r['Controller']:<14} {r['side']:<20} {r['n_converged']:>6d} "
                f"{r['n_not_settled']:>7d} {r['n_diverged']:>5d} "
                f"{r['success_rate_pct']:>8.1f}%  {r['dominant_failure_reason']}\n"
            )

    print("\n" + "=" * 120)
    print("Directional non-convergence diagnostic")
    print("=" * 120)
    for r in summary_rows:
        print(
            f"{r['Controller']:<14} | {r['side']:<20} | "
            f"Converged {r['n_converged']:2d}/{r['n_trials']:2d} | "
            f"Not settled {r['n_not_settled']:2d} | Diverged {r['n_diverged']:2d} | "
            f"dominant reason: {r['dominant_failure_reason']}"
        )
    print("=" * 120)
    print(f"Saved per-trial diagnostics: {diag_csv}")
    print(f"Saved directional summary: {summary_csv}")
    print(f"Saved directional text report: {txt_path}")
    return diag_csv, summary_csv, txt_path


def _mean_std(values):
    values = np.asarray(values, dtype=float).reshape(-1)
    values = values[np.isfinite(values)]
    if values.size == 0:
        return np.nan, np.nan, 0
    mean = float(np.mean(values))
    std = float(np.std(values, ddof=1)) if values.size > 1 else 0.0
    return mean, std, int(values.size)


def _mean_std_string(mean, std, precision=6):
    if not np.isfinite(mean) or not np.isfinite(std):
        return "--"
    return f"{mean:.{precision}g} +/- {std:.{precision}g}"


def aggregate_mc_rows(raw_rows, controller_order, n_total):
    """Aggregate 50 trials into mean +/- sample-std and convergence success rate."""
    summary = []
    for label in controller_order:
        rows = [r for r in raw_rows if r.get("Controller") == label]
        if not rows:
            continue

        n_success = sum(r.get("Convergence status") == "Converged" for r in rows)
        n_diverged = sum(r.get("Convergence status") == "Diverged" for r in rows)
        n_not_settled = sum(r.get("Convergence status") == "Not settled" for r in rows)
        success_rate = 100.0 * n_success / float(n_total)

        item = {
            "Controller": label,
            "n_success": n_success,
            "n_total": n_total,
            "Success rate [%]": success_rate,
            "n_diverged": n_diverged,
            "n_not_settled": n_not_settled,
        }

        for key in MC_METRIC_KEYS:
            mean, std, n_valid = _mean_std([r.get(key, np.nan) for r in rows])
            short = MC_SHORT_NAMES[key]
            item[f"{short}_mean"] = mean
            item[f"{short}_std"] = std
            item[f"{short}_n"] = n_valid
            item[f"{short}_mean_std"] = _mean_std_string(mean, std)

        summary.append(item)
    return summary


def print_mc_summary_table(summary, save_path=None):
    """Print only SS_H, SS_Q, M_H, and M_Q, and optionally save the same table."""
    headers = [
        "Controller",
        "SS_H",
        "SS_Q",
        "M_H",
        "M_Q",
    ]

    rows = []
    for s in summary:
        rows.append([
            s["Controller"],
            s["SS_H_mean_std"],
            s["SS_Q_mean_std"],
            s["M_H_mean_std"],
            s["M_Q_mean_std"],
        ])

    widths = [
        max(len(headers[i]), max((len(r[i]) for r in rows), default=0))
        for i in range(len(headers))
    ]

    header_line = " | ".join(
        headers[i].ljust(widths[i]) for i in range(len(headers))
    )
    sep_line = "-+-".join("-" * widths[i] for i in range(len(headers)))
    row_lines = [
        " | ".join(r[i].ljust(widths[i]) for i in range(len(headers)))
        for r in rows
    ]

    print(header_line)
    print(sep_line)
    for line in row_lines:
        print(line)

    if save_path is not None:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        with save_path.open("w", encoding="utf-8") as f:
            f.write(header_line + "\n")
            f.write(sep_line + "\n")
            for line in row_lines:
                f.write(line + "\n")

    return headers, rows


def save_mc_raw_rows(raw_rows, save_dir: Path):
    path = save_dir / "MC50_largeIC_raw_metrics.csv"
    fields = [
        "trial", "side", "seed", "Controller",
        "H_left", "H_right", "Q_value",
        "delta_H_L2_pct", "delta_Q_L2_pct",
        *MC_METRIC_KEYS,
        "TDPSAC failure diagnostic SS_H [m]",
        "TDPSAC failure diagnostic SS_Q [m^3/s]",
        "diagnostic_reason", "diagnostic_t_end_s", "diagnostic_early_stop",
        "diagnostic_last_terminated", "diagnostic_last_truncated",
        "diagnostic_H_threshold", "diagnostic_Q_threshold",
        "diagnostic_H_final_L2", "diagnostic_Q_final_L2",
        "diagnostic_H_min_L2", "diagnostic_Q_min_L2",
        "diagnostic_H_tail_mean_L2", "diagnostic_Q_tail_mean_L2",
        "diagnostic_H_tail_slope_per_s", "diagnostic_Q_tail_slope_per_s",
        "diagnostic_H_final_inside_band", "diagnostic_Q_final_inside_band",
        "diagnostic_H_ever_inside_band", "diagnostic_Q_ever_inside_band",
        "diagnostic_U_min", "diagnostic_U_max",
        "diagnostic_U_at_lower_bound_fraction", "diagnostic_U_at_upper_bound_fraction",
        "diagnostic_min_H_state", "diagnostic_max_H_state",
        "diagnostic_min_Q_state", "diagnostic_max_Q_state",
        "diagnostic_last_info",
        "Convergence status", "error_message",
    ]
    with path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fields, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(raw_rows)
    print(f"Saved raw trial metrics: {path}")
    return path


def save_mc_summary(summary, save_dir: Path):
    csv_path = save_dir / "MC50_largeIC_mean_std_success.csv"
    txt_path = save_dir / "MC50_largeIC_mean_std_success.txt"

    csv_fields = [
        "Controller",
        "SS_H_mean_std", "SS_Q_mean_std",
        "Peak_H_mean_std", "Peak_Q_mean_std",
        "M_H_mean_std", "M_Q_mean_std",
        "t_settle_mean_std",
        "n_success", "n_total", "Success rate [%]",
        "n_diverged", "n_not_settled",
    ]
    with csv_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=csv_fields, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(summary)

    headers, rows = print_mc_summary_table(summary)
    widths = [
        max(len(headers[i]), max((len(r[i]) for r in rows), default=0))
        for i in range(len(headers))
    ]
    header_line = " | ".join(headers[i].ljust(widths[i]) for i in range(len(headers)))
    sep_line = "-+-".join("-" * widths[i] for i in range(len(headers)))

    with txt_path.open("w", encoding="utf-8") as f:
        f.write("MC50 large-initial-condition robustness table\n")
        f.write("H band: 6%-10% centered at 8%; Q band: 18%-22% centered at 20%.\n")
        f.write("Mean +/- std uses sample std (ddof=1).\n")
        f.write("Success means Convergence status == Converged.\n")
        f.write("Metric statistics use finite values only.\n\n")
        f.write(header_line + "\n")
        f.write(sep_line + "\n")
        for r in rows:
            f.write(" | ".join(r[i].ljust(widths[i]) for i in range(len(headers))) + "\n")

    print(f"Saved summary CSV: {csv_path}")
    print(f"Saved summary TXT: {txt_path}")
    return csv_path, txt_path



@contextlib.contextmanager
def _silent_external_output():
    """Suppress third-party stdout/stderr and warnings during evaluation."""
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            yield


def main():
    global print
    _real_print = builtins.print
    print = lambda *args, **kwargs: None

    parser = argparse.ArgumentParser(
        allow_abbrev=False,
        description=(
            "MC50 table-only comparison of SAC, SAC-FPC, DO-SAC, DPSAC, "
            "TDPSAC-FPC, and TDPSAC under large linear initial-condition deviations."
        ),
    )
    parser.add_argument(
        "--new-model", "--tdpsac-model", dest="new_model", type=str,
        default=DEFAULT_NEW_MODEL_PATH,
    )
    parser.add_argument(
        "--tdpsac-fpc-model", type=str, default=DEFAULT_TDPSAC_FPC_MODEL_PATH,
    )
    parser.add_argument(
        "--dpsac-model", type=str, default=DEFAULT_DPSAC_MODEL_PATH,
    )
    parser.add_argument(
        "--do-sac-model", type=str, default=DEFAULT_DO_SAC_MODEL_PATH,
    )
    parser.add_argument(
        "--sac-fpc-model", type=str, default=DEFAULT_SAC_FPC_MODEL_PATH,
    )
    parser.add_argument(
        "--sac-model", type=str, default=DEFAULT_SAC_MODEL_PATH,
    )
    parser.add_argument(
        "--out-dir", type=str,
        default="./test_results_MC50_largeIC_H8_Q20_table_only",
    )
    parser.add_argument(
        "--seed", type=int, default=123,
        help="Base evaluation seed; trial i uses seed+i-1 for every controller.",
    )
    parser.add_argument(
        "--ic-seed", type=int, default=20260810,
        help="Seed used once to generate the shared 50 initial conditions.",
    )
    parser.add_argument("--n-trials", type=int, default=50)
    parser.add_argument("--n-above", type=int, default=25)
    parser.add_argument(
        "--H-pct-min", type=float, default=6.0,
        help="Minimum realized relative-L2 H initial deviation [%%].",
    )
    parser.add_argument(
        "--H-pct-max", type=float, default=7.0,
        help="Maximum realized relative-L2 H initial deviation [%%].",
    )
    parser.add_argument(
        "--Q-pct-min", type=float, default=20.0,
        help="Minimum relative-L2 Q initial deviation [%%].",
    )
    parser.add_argument(
        "--Q-pct-max", type=float, default=21.0,
        help="Maximum relative-L2 Q initial deviation [%%].",
    )
    parser.add_argument("--dH-source", type=float, default=None)

    args, unknown_args = parser.parse_known_args(_clean_runtime_argv())
    if unknown_args:
        print(f"Ignoring unrecognized runtime arguments: {unknown_args}")

    if args.n_trials != 50 or args.n_above != 25:
        print(
            f"[NOTE] Requested design is 50 trials with 25 H-above/Q-below and 25 H-below/Q-above; "
            f"current arguments are n_trials={args.n_trials}, n_above={args.n_above}."
        )

    # Build the one common target equilibrium used by every controller and every trial.
    _, X_ref, dx_ref, Hbar_ref, Qbar_ref = build_collection_matched_reference()

    cases = generate_large_linear_ic_cases(
        X_ref,
        Hbar_ref,
        Qbar_ref,
        n_total=args.n_trials,
        n_above=args.n_above,
        H_pct_min=args.H_pct_min,
        H_pct_max=args.H_pct_max,
        Q_pct_min=args.Q_pct_min,
        Q_pct_max=args.Q_pct_max,
        seed=args.ic_seed,
    )

    print("=" * 110)
    print("Six-learned-controller MC50 large-IC robustness test -- TABLE ONLY")
    print("=" * 110)
    print(
        f"Target equilibrium: Hbar(0)={Hbar_ref[0]:.9f} m, "
        f"Hbar(L)={Hbar_ref[-1]:.9f} m, Qbar={Qbar_ref[0]:.9f} m^3/s, "
        f"dx={dx_ref:.6f} m"
    )
    print(
        f"Initial-condition design: {args.n_trials} total = {args.n_above} H-below/Q-above + "
        f"{args.n_trials-args.n_above} H-above/Q-below."
    )
    print(
        f"Large-IC bands: delta_H,0 in [{args.H_pct_min:g}%, {args.H_pct_max:g}%] "
        f"(center {(args.H_pct_min+args.H_pct_max)/2:g}%), "
        f"delta_Q,0 in [{args.Q_pct_min:g}%, {args.Q_pct_max:g}%] "
        f"(center {(args.Q_pct_min+args.Q_pct_max)/2:g}%)."
    )
    print("H(x,0) is linearly interpolated between H_left and H_right; Q(x,0)=Q_value is spatially constant.")
    print("Trials 1-25: H below equilibrium and Q above equilibrium.")
    print("Trials 26-50: H above equilibrium and Q below equilibrium.")
    print("The exact same 50-case list is used for all six learned controllers.")
    print("No figures are generated.")
    print("=" * 110)

    controller_paths = {
        LABEL_NEW_MODEL: args.new_model,
        LABEL_DFP_DON_SAC: args.tdpsac_fpc_model,
        LABEL_HYBRID: args.dpsac_model,
        LABEL_FP_SAC: args.sac_fpc_model,
        LABEL_SAC: args.do_sac_model,
        LABEL_SAC_CURVE: args.sac_model,
    }

    # Validate all checkpoints before starting the expensive MC experiment.
    for spec in LEARNED_CONTROLLER_SPECS:
        label = spec["label"]
        path = controller_paths[label]
        if not _checkpoint_exists(path):
            raise FileNotFoundError(
                f"No valid checkpoint for {label}: {path!r}. "
                "Fix all model paths before launching MC50."
            )

    raw_rows = []

    # --------------------------------------------------------
    # 1) Six learned controllers -- reversed order, TDPSAC first
    # --------------------------------------------------------
    for spec in LEARNED_CONTROLLER_SPECS:
        label = spec["label"]
        model_kind = spec["model_kind"]
        model_path = controller_paths[label]

        print(f"\n===== {label}: load once, then {args.n_trials} trials =====")
        with _silent_external_output():
            model = load_learned_controller_once(
                model_path,
                model_kind,
                first_case=cases[0],
                seed=args.seed,
                dH_source=args.dH_source,
            )

        for case in cases:
            trial_seed = args.seed + case["trial"] - 1
            print(
                f"[{label}] trial {case['trial']:02d}/{args.n_trials} "
                f"({case['side']}): H0={case['H_left']:.6f}, "
                f"HL={case['H_right']:.6f}, Q={case['Q_value']:.6f}"
            )
            base = {
                "trial": case["trial"],
                "side": case["side"],
                "seed": trial_seed,
                "Controller": label,
                "H_left": case["H_left"],
                "H_right": case["H_right"],
                "Q_value": case["Q_value"],
                "delta_H_L2_pct": case["delta_H_L2_pct"],
                "delta_Q_L2_pct": case["delta_Q_L2_pct"],
                "error_message": "",
            }
            try:
                with _silent_external_output():
                    result = run_loaded_policy_exact(
                        model,
                        model_kind,
                        label,
                        seed=trial_seed,
                        H_left=case["H_left"],
                        H_right=case["H_right"],
                        Q_value=case["Q_value"],
                        dH_source=args.dH_source,
                    )
                base.update(extract_one_mc_row(label, result))
                base.update(build_nonconvergence_diagnostic(result, base, expected_T=SAC_T))

                # TDPSAC-only failure diagnostics.  This does not modify the
                # convergence status, the standard SS metrics, or MC aggregation.
                if (
                    label == LABEL_NEW_MODEL
                    and base.get("Convergence status") != "Converged"
                ):
                    ss_h_diag, ss_q_diag = _tdpsac_failure_ss_diagnostic(
                        result, base
                    )
                    base["TDPSAC failure diagnostic SS_H [m]"] = ss_h_diag
                    base["TDPSAC failure diagnostic SS_Q [m^3/s]"] = ss_q_diag

                    t_arr = np.asarray(result.get("t", []), dtype=float).reshape(-1)
                    t_end = (
                        float(t_arr[-1])
                        if t_arr.size > 0 and np.isfinite(t_arr[-1])
                        else np.nan
                    )
                    print(
                        f"[TDPSAC FAILURE DIAGNOSTIC] trial {case['trial']:02d}: "
                        f"status={base.get('Convergence status')}, "
                        f"SS_H={ss_h_diag:.9g}, SS_Q={ss_q_diag:.9g}, "
                        f"t_end={t_end:.6g} s / expected {SAC_T:.6g} s"
                    )
            except Exception as exc:
                print(f"[FAIL] {label} trial {case['trial']}: {exc!r}")
                for key in MC_METRIC_KEYS:
                    base[key] = np.nan
                base["Convergence status"] = "Diverged"
                base["diagnostic_reason"] = "evaluation_exception"
                base["error_message"] = repr(exc)
                if label == LABEL_NEW_MODEL:
                    base["TDPSAC failure diagnostic SS_H [m]"] = np.nan
                    base["TDPSAC failure diagnostic SS_Q [m^3/s]"] = np.nan
                    print(
                        f"[TDPSAC FAILURE DIAGNOSTIC] trial {case['trial']:02d}: "
                        "SS_H=NaN, SS_Q=NaN because evaluation raised an exception "
                        "before a complete result object was returned."
                    )
            raw_rows.append(base)

        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


    # All trials are complete for the six learned controllers.

    # Aggregate exactly as before; only the displayed columns are reduced.
    summary = aggregate_mc_rows(
        raw_rows,
        controller_order=MC_CONTROLLER_ORDER,
        n_total=args.n_trials,
    )

    # Restore normal printing, display the final table, and save ONLY this table.
    print = _real_print
    table_path = Path("big1988") / "MC50_final_table.txt"
    print_mc_summary_table(summary, save_path=table_path)


if __name__ == "__main__":
    main()
